In [1]:
import os
from kaggle_secrets import UserSecretsClient

# 1. تثبيت rclone
!apt-get update -qq && apt-get install -y -qq rclone

# 2. قراءة الإعدادات من Kaggle Secrets وحفظها
user_secrets = UserSecretsClient()
rclone_config = user_secrets.get_secret("RCLONE_CONFIG")

os.makedirs('/root/.config/rclone', exist_ok=True)
with open('/root/.config/rclone/rclone.conf', 'w') as f:
    f.write(rclone_config)

print("✅ rclone has been set up and connected to the Drive.!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
^C
E: Sub-process /usr/sbin/dpkg-preconfigure --apt || true received signal 2.
E: Failure running script /usr/sbin/dpkg-preconfigure --apt || true
✅ rclone has been set up and connected to the Drive.!


In [ ]:
# ============================================================
# MATERIALS PROJECT API — STABLE ENVIRONMENT SETUP
# ============================================================

import sys
import subprocess

PYTHON = sys.executable

print("Python:", PYTHON)

# Install a mutually compatible MP-API / PyArrow stack
subprocess.check_call([
    PYTHON, "-m", "pip", "install",
    "--upgrade",
    "--force-reinstall",
    "--no-cache-dir",
    "pyarrow>=20,<26",
    "mp-api==0.46.5"
])

print("\nInstallation completed.")
print("IMPORTANT: Restart the runtime NOW.")

In [1]:
!pip install -q --upgrade pip
!pip install -q mace-torch
!pip install -q ase
!pip install torch-dftd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.4 MB/s eta 0:00:0000:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 681.8/681.8 kB 8.7 MB/s  0:00:00


In [2]:
import sys
import importlib.metadata as metadata
import pyarrow

print("Python      :", sys.version)
print("PyArrow     :", pyarrow.__version__)
print("mp-api      :", metadata.version("mp-api"))

from mp_api.client import MPRester

print("MPRester imported successfully.")

Python      : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyArrow     : 25.0.1
mp-api      : 0.46.5
MPRester imported successfully.


In [3]:
# ============================================================
# STEP 0 — Structure Discovery Environment Check
# ============================================================

import sys
import importlib.metadata
import ase
import pymatgen
import spglib
import torch

print("=== Environment Check ===")
print(f"Python      : {sys.version.split()[0]}")
print(f"ASE         : {ase.__version__}")
print(f"pymatgen    : {importlib.metadata.version('pymatgen')}")
print(f"spglib      : {spglib.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
else:
    print("GPU         : Not available")

print("\n=== Core Libraries Ready ===")

=== Environment Check ===
Python      : 3.13.15
ASE         : 3.29.0
pymatgen    : 2026.9.24
spglib      : 2.7.0
PyTorch     : 2.11.0+cu128
CUDA        : True
GPU         : Tesla T4

=== Core Libraries Ready ===


In [4]:
# Core physical system

system = {
    "mobile_ion": "Li+",
    "host_atoms": ["Ti", "Si"],
    "dimension": 3
}

print(system)

{'mobile_ion': 'Li+', 'host_atoms': ['Ti', 'Si'], 'dimension': 3}


In [5]:
import numpy as np
import torch
import gc

from ase import units
from ase.build import bulk
from ase.filters import FrechetCellFilter
from ase.optimize import BFGS
from mace.calculators import mace_mp

# ------------------------------------------------------------
# Step 1: Initialization and Model Loading
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

print("=== Step 1: Loading MACE Model ===")

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64"
)

print("MACE model loaded successfully.")

/usr/local/lib/python3.13/dist-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
=== Step 1: Loading MACE Model ===
Downloading: 100.0% (42.4 MB / 42.4 MB)
Cached MACE model to /root/.cache/mace/20231203mace128L1_epoch199model
Using Materials Project MACE for MACECalculator with /root/.cache/mace/20231203mace128L1_epoch199model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/usr/local/lib/python3.13/dist-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using TorchDFTD3Calculator for D3 dispersion corrections
MACE model loaded successfully.


In [ ]:
import json
import os
import pickle
import subprocess
import sys
from ase.io import read
import pandas as pd

# ============================================================
# PART 1 & 2 — Unified Environment Detection & Drive Path Configuration
# ============================================================
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = os.path.exists("/kaggle")

# Target relative path inside Google Drive
DRIVE_RELATIVE_PATH = (
    "Materials_Project/Top5_search_archive/generation_3_MACE_TOP5_archive"
)

if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    ARCHIVE_DIR = os.path.join("/content/drive/MyDrive", DRIVE_RELATIVE_PATH)

elif IS_KAGGLE:
    # Set up rclone configuration using Kaggle UserSecrets
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    rclone_config = user_secrets.get_secret("RCLONE_CONFIG")

    os.makedirs("/root/.config/rclone", exist_ok=True)
    with open("/root/.config/rclone/rclone.conf", "w") as f:
        f.write(rclone_config)

    # Define a local sync directory for Kaggle runtime
    ARCHIVE_DIR = "/kaggle/working/generation_3_MACE_TOP5_archive"
    os.makedirs(ARCHIVE_DIR, exist_ok=True)

    # Sync files directly from Google Drive remote to local Kaggle path
    remote_path = f"mydrive:{DRIVE_RELATIVE_PATH}"
    subprocess.run(
        ["rclone", "copy", remote_path, ARCHIVE_DIR], check=True
    )

else:
    ARCHIVE_DIR = "./generation_3_MACE_TOP5_archive"

RESULTS_CSV = os.path.join(ARCHIVE_DIR, "MACE_consistent_TOP5_RESULTS.csv")
STRUCTURES_EXTXYZ = os.path.join(
    ARCHIVE_DIR, "MACE_consistent_TOP5_STRUCTURES.extxyz"
)
STRUCTURES_PKL = os.path.join(
    ARCHIVE_DIR, "MACE_TOP5_candidate_to_structure.pkl"
)
ALL_RESULTS_PKL = os.path.join(
    ARCHIVE_DIR, "MACE_TOP5_candidate_to_all_results.pkl"
)

# ============================================================
# PART 3 — Data Archive Deserialization & Verification
# ============================================================
top5_results = pd.read_csv(RESULTS_CSV)

with open(STRUCTURES_PKL, "rb") as f:
    top5_structures = pickle.load(f)

with open(ALL_RESULTS_PKL, "rb") as f:
    top5_all_results = pickle.load(f)

top5_extxyz_structures = read(STRUCTURES_EXTXYZ, index=":")

print("=" * 70)
print("TOP 5 ARCHIVE LOADED SUCCESSFULLY")
print("=" * 70)

print(f"Results rows        : {len(top5_results)}")
print(f"Structure dict      : {len(top5_structures)}")
print(f"All-results dict    : {len(top5_all_results)}")
print(f"EXTXYZ structures  : {len(top5_extxyz_structures)}")

print("\nActual CSV columns:")
print(top5_results.columns.tolist())

# ============================================================
# PART 4 & 5 — Target Candidate Extraction and Preparation
# ============================================================
candidate_row = top5_results.iloc[0].copy()
candidate_id = int(candidate_row["candidate_id"])

candidate = top5_structures[candidate_id].copy()
candidate_results = top5_all_results[candidate_id].copy()

print("\n" + "=" * 70)
print("CANDIDATE 1 READY")
print("=" * 70)

print(f"Candidate ID : {candidate_id}")
print(f"Formula      : {candidate.get_chemical_formula()}")
print(f"Atoms        : {len(candidate)}")
print(f"Volume (Å³)  : {candidate.get_volume():.6f}")
print(f"PBC          : {candidate.get_pbc()}")

print("\nCell:")
print(candidate.cell)

print("\nComplete Candidate 1 results:")
print(candidate_row.to_string())

In [ ]:
# ============================================================
# GENERATION 3 — THERMODYNAMIC-AWARE HEXAGONAL DISCOVERY
# ============================================================
#
# Strategy:
#   1. Use the current MACE Ti-Si hull as a thermodynamic map.
#   2. Generate NEW compositions near promising hull regions.
#   3. Generate multiple structurally different hexagonal
#      structures for each composition.
#   4. Reject obviously bad geometries before MACE.
#   5. Score every surviving structure with the SAME MACE:
#         mace_mp(model="medium",
#                  dispersion=True,
#                  default_dtype="float64")
#   6. Calculate formation energy and energy above hull immediately.
#   7. Rank by thermodynamic quality + structural diversity.
#
# IMPORTANT:
#   - No hard 0.10 eV/atom cutoff.
#   - Known TiSi, TiSi2, Ti5Si3 are NOT generated as targets.
#   - Composition itself is part of the search.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import math
import random
import numpy as np
import pandas as pd

from collections import Counter

from ase import Atoms
from ase.optimize import BFGS
from ase.filters import FrechetCellFilter

from mace.calculators import mace_mp


# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 20260911

random.seed(SEED)
np.random.seed(SEED)


# ============================================================
# 3. GOOGLE DRIVE ARCHIVE
# ============================================================

GEN3_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "generation_3_archive"
)

os.makedirs(
    GEN3_DIR,
    exist_ok=True
)


# ============================================================
# 4. SEARCH PARAMETERS
# ============================================================

# Number of structures to generate BEFORE geometry filtering
N_GENERATED = 180

# Maximum number of MACE candidates actually evaluated
N_MACE_EVALUATIONS = 120

# Number of top candidates saved
N_TOP = 20

# MACE relaxation
FMAX = 0.05
MAX_STEPS = 250


# ============================================================
# 5. CURRENT MACE HULL
# ============================================================
#
# These are the CURRENT thermodynamic reference points obtained
# from the project using the same MACE setup.
#
# Formation energies:
#
# Si       xTi = 0.000000
# TiSi2    xTi = 0.333333
# TiSi     xTi = 0.500000
# Ti5Si3   xTi = 0.625000
# Ti       xTi = 1.000000
#
# We use these ONLY as a thermodynamic map.
# ============================================================

HULL_X = np.array([
    0.000000,
    0.3333333333333333,
    0.5000000000000000,
    0.6250000000000000,
    1.0000000000000000
])

HULL_E = np.array([
    0.000000,
    -0.758522,
    -0.968980,
    -0.890530,
    0.000000
])


# ============================================================
# 6. KNOWN COMPOSITIONS — DO NOT REGENERATE EXACTLY
# ============================================================

KNOWN_FORMULAS = {
    "Ti",
    "Si",
    "TiSi2",
    "TiSi",
    "Ti5Si3"
}


# ============================================================
# 7. HULL INTERPOLATION
# ============================================================

def hull_energy_at_x(x_ti):
    """
    Return the current binary Ti-Si hull energy
    at a requested Ti fraction.

    The hull is piecewise linear between the known
    stable reference points.
    """

    x_ti = float(x_ti)

    if x_ti <= HULL_X[0]:
        return HULL_E[0]

    if x_ti >= HULL_X[-1]:
        return HULL_E[-1]

    for i in range(len(HULL_X) - 1):

        x1 = HULL_X[i]
        x2 = HULL_X[i + 1]

        if x1 <= x_ti <= x2:

            e1 = HULL_E[i]
            e2 = HULL_E[i + 1]

            fraction = (
                (x_ti - x1)
                / (x2 - x1)
            )

            return (
                e1
                + fraction * (e2 - e1)
            )

    raise RuntimeError(
        f"Could not interpolate hull at xTi={x_ti}"
    )


# ============================================================
# 8. TARGET COMPOSITION GENERATION
# ============================================================
#
# We deliberately search NEAR the hull vertices instead of
# randomly distributing compositions over the whole binary line.
#
# But we also avoid reproducing the exact known compounds.
#
# The resulting compositions include nearby ratios such as:
#
#   Ti1Si3
#   Ti3Si5
#   Ti2Si3
#   Ti3Si4
#   Ti4Si5
#   ...
#
# depending on atom-count limits.
# ============================================================

MAX_TOTAL_ATOMS = 12

composition_targets = []


for n_ti in range(1, MAX_TOTAL_ATOMS):

    for n_si in range(1, MAX_TOTAL_ATOMS):

        total_atoms = n_ti + n_si

        if total_atoms > MAX_TOTAL_ATOMS:
            continue

        gcd_value = math.gcd(
            n_ti,
            n_si
        )

        # Reduced chemical formula
        reduced_ti = n_ti // gcd_value
        reduced_si = n_si // gcd_value

        formula = (
            f"Ti{reduced_ti if reduced_ti != 1 else ''}"
            f"Si{reduced_si if reduced_si != 1 else ''}"
        )

        # Never regenerate known compounds
        if formula in KNOWN_FORMULAS:
            continue

        x_ti = n_ti / total_atoms

        # Distance to nearest current hull vertex
        distance_to_hull_vertex = min(
            abs(
                x_ti - x
            )
            for x in HULL_X[1:-1]
        )

        # Thermodynamic map value
        hull_e = hull_energy_at_x(x_ti)

        composition_targets.append({
            "n_Ti": n_ti,
            "n_Si": n_si,
            "total_atoms": total_atoms,
            "formula": formula,
            "x_Ti": x_ti,
            "hull_energy": hull_e,
            "distance_to_hull_vertex":
                distance_to_hull_vertex
        })


composition_df = pd.DataFrame(
    composition_targets
)


# ============================================================
# 9. THERMODYNAMICALLY FOCUSED COMPOSITION WEIGHT
# ============================================================

# Smaller distance to a known hull vertex gets higher priority.
#
# We do NOT use this as a rejection rule.
# It only determines how often a composition is sampled.

composition_df["sampling_weight"] = (
    1.0
    /
    (
        composition_df[
            "distance_to_hull_vertex"
        ]
        + 0.025
    )
)


# Normalize weights
composition_df["sampling_weight"] /= (
    composition_df["sampling_weight"].sum()
)


# ============================================================
# 10. MACE CALCULATOR
# ============================================================

print("=" * 100)
print("INITIALIZING MACE")
print("=" * 100)

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64"
)


# ============================================================
# 11. GEOMETRY PARAMETERS
# ============================================================
#
# Hexagonal cell:
#
#   a = b
#   gamma = 120°
#
# c is chosen relative to a.
#
# We deliberately vary cell dimensions between structures
# so that we explore different structural environments.
# ============================================================

A_RANGE = (
    4.5,
    7.5
)

C_OVER_A_RANGE = (
    1.20,
    2.20
)


# ============================================================
# 12. MINIMUM-DISTANCE CHECK
# ============================================================

def minimum_distance_ok(
    atoms,
    minimum_distance=1.65
):

    try:

        distances = atoms.get_all_distances(
            mic=True
        )

        np.fill_diagonal(
            distances,
            np.inf
        )

        minimum = np.min(
            distances
        )

        return (
            minimum >= minimum_distance,
            minimum
        )

    except Exception:

        return False, 0.0


# ============================================================
# 13. GENERATE RANDOM HEXAGONAL STRUCTURE
# ============================================================

def generate_hexagonal_structure(
    n_ti,
    n_si
):

    symbols = (
        ["Ti"] * n_ti
        +
        ["Si"] * n_si
    )

    random.shuffle(symbols)

    total_atoms = len(symbols)

    # Random hexagonal lattice dimensions
    a = np.random.uniform(
        *A_RANGE
    )

    c_over_a = np.random.uniform(
        *C_OVER_A_RANGE
    )

    c = a * c_over_a

    # Hexagonal cell
    cell = np.array([
        [a, 0.0, 0.0],
        [-0.5 * a,
         np.sqrt(3) * 0.5 * a,
         0.0],
        [0.0, 0.0, c]
    ])

    # Random fractional positions
    scaled_positions = np.random.uniform(
        0.05,
        0.95,
        size=(total_atoms, 3)
    )

    atoms = Atoms(
        symbols=symbols,
        scaled_positions=scaled_positions,
        cell=cell,
        pbc=True
    )

    return atoms


# ============================================================
# 14. STRUCTURAL SANITY FILTER
# ============================================================

def physically_sane_structure(
    atoms
):

    ok, minimum_distance = (
        minimum_distance_ok(
            atoms
        )
    )

    if not ok:
        return False, minimum_distance

    return True, minimum_distance


# ============================================================
# 15. GENERATION
# ============================================================

print("\n" + "=" * 100)
print("GENERATION 3 — COMPOSITION-AWARE HEXAGONAL SEARCH")
print("=" * 100)

generated_structures = {}
generation_records = []

for generation_id in range(
    N_GENERATED
):

    selected_index = np.random.choice(
        composition_df.index,
        p=composition_df[
            "sampling_weight"
        ].values
    )

    target = composition_df.loc[
        selected_index
    ]

    atoms = generate_hexagonal_structure(
        int(target["n_Ti"]),
        int(target["n_Si"])
    )

    sane, minimum_distance = (
        physically_sane_structure(
            atoms
        )
    )

    if not sane:
        continue

    candidate_id = generation_id

    generated_structures[candidate_id] = atoms

    atoms = generated_structures[
    candidate_id].copy()
    generation_records.append({

        "candidate_id":
            candidate_id,

        "formula":
            target["formula"],

        "n_Ti":
            int(target["n_Ti"]),

        "n_Si":
            int(target["n_Si"]),

        "atoms":
            len(atoms),

        "x_Ti":
            target["x_Ti"],

        "hull_energy_target":
            target["hull_energy"],

        "minimum_distance_A":
            minimum_distance
    })


generation_df = pd.DataFrame(
    generation_records
)


print(
    f"\nGenerated structures       : "
    f"{N_GENERATED}"
)

print(
    f"Geometry-sane structures   : "
    f"{len(generated_structures)}"
)


if len(generated_structures) == 0:

    raise RuntimeError(
        "No physically sane structures "
        "were generated."
    )


# ============================================================
# 16. PRIORITIZE STRUCTURES BEFORE MACE
# ============================================================
#
# We first prioritize compositions close to hull vertices.
#
# This is NOT an energy prediction.
# It only decides which structures get MACE evaluated first.
# ============================================================

generation_df = generation_df.sort_values(
    by=[
        "hull_energy_target",
        "minimum_distance_A"
    ],
    ascending=[
        True,
        False
    ]
).reset_index(
    drop=True
)


# ============================================================
# 17. MACE SCREENING
# ============================================================

print("\n" + "=" * 100)
print("EARLY MACE SCREENING")
print("=" * 100)

mace_results = []
mace_structures = {}


n_to_evaluate = min(
    N_MACE_EVALUATIONS,
    len(generated_structures)
)


for screen_index in range(
    n_to_evaluate
):

    row = generation_df.iloc[
        screen_index
    ]

    candidate_id = int(
        row["candidate_id"]
    )

    atoms = generated_structures[
        candidate_id
    ].copy()

    formula = row["formula"]

    x_ti = float(
        row["x_Ti"]
    )

    print(
        f"\n[{screen_index + 1}/"
        f"{n_to_evaluate}] "
        f"Candidate {candidate_id} | "
        f"{formula} | "
        f"xTi={x_ti:.4f}"
    )

    try:

        # ----------------------------------------------------
        # Attach SAME MACE
        # ----------------------------------------------------

        atoms.calc = calc

        # ----------------------------------------------------
        # Relax BOTH positions and cell
        # ----------------------------------------------------

        cell_filter = FrechetCellFilter(
            atoms
        )

        optimizer = BFGS(
            cell_filter,
            logfile=None
        )

        optimizer.run(
            fmax=FMAX,
            steps=MAX_STEPS
        )

        # ----------------------------------------------------
        # Final MACE energy
        # ----------------------------------------------------

        energy_total = (
            atoms.get_potential_energy()
        )

        energy_per_atom = (
            energy_total
            /
            len(atoms)
        )

        # ----------------------------------------------------
        # Formation energy
        #
        # Reference energies will be obtained below from
        # MACE-relaxed elemental references.
        # ----------------------------------------------------

        mace_results.append({

            "candidate_id":
                candidate_id,

            "formula":
                formula,

            "x_Ti":
                x_ti,

            "energy_eV_per_atom":
                energy_per_atom,

            "atoms":
                len(atoms),

            "status":
                "SUCCESS"
        })

        mace_structures[
            candidate_id
        ] = atoms.copy()

        print(
            f"    MACE E/atom = "
            f"{energy_per_atom:.8f} eV"
        )

    except Exception as error:

        print(
            f"    FAILED: {type(error).__name__}: "
            f"{error}"
        )

        mace_results.append({

            "candidate_id":
                candidate_id,

            "formula":
                formula,

            "x_Ti":
                x_ti,

            "energy_eV_per_atom":
                np.nan,

            "atoms":
                len(atoms),

            "status":
                "FAILED"
        })


mace_df = pd.DataFrame(
    mace_results
)


# ============================================================
# 18. REMOVE FAILED MACE STRUCTURES
# ============================================================

mace_df = mace_df[
    mace_df["status"] == "SUCCESS"
].copy()


if len(mace_df) == 0:

    raise RuntimeError(
        "No structure survived MACE screening."
    )


# ============================================================
# 19. RELAXED ELEMENTAL REFERENCES
# ============================================================
#
# These references are calculated using EXACTLY the same
# MACE calculator.
# ============================================================

from ase.build import bulk


print("\n" + "=" * 100)
print("MACE ELEMENTAL REFERENCES")
print("=" * 100)


# ------------------------------------------------------------
# Ti — alpha hcp
# ------------------------------------------------------------

ti_reference = bulk(
    "Ti",
    crystalstructure="hcp",
    a=2.95,
    c=4.68
)

ti_reference.calc = calc

ti_filter = FrechetCellFilter(
    ti_reference
)

ti_optimizer = BFGS(
    ti_filter,
    logfile=None
)

ti_optimizer.run(
    fmax=0.01,
    steps=300
)

ti_energy = (
    ti_reference.get_potential_energy()
    /
    len(ti_reference)
)


# ------------------------------------------------------------
# Si — diamond cubic
# ------------------------------------------------------------

si_reference = bulk(
    "Si",
    crystalstructure="diamond",
    a=5.43
)

si_reference.calc = calc

si_filter = FrechetCellFilter(
    si_reference
)

si_optimizer = BFGS(
    si_filter,
    logfile=None
)

si_optimizer.run(
    fmax=0.01,
    steps=300
)

si_energy = (
    si_reference.get_potential_energy()
    /
    len(si_reference)
)


print(
    f"Ti reference = "
    f"{ti_energy:.8f} eV/atom"
)

print(
    f"Si reference = "
    f"{si_energy:.8f} eV/atom"
)


# ============================================================
# 20. FORMATION ENERGY + HULL DISTANCE
# ============================================================

mace_df["formation_energy_MACE"] = (
    mace_df["energy_eV_per_atom"]
    -
    (
        mace_df["x_Ti"] * ti_energy
        +
        (1.0 - mace_df["x_Ti"]) * si_energy
    )
)


mace_df["hull_energy_MACE"] = (
    mace_df["x_Ti"].apply(
        hull_energy_at_x
    )
)


mace_df["energy_above_hull_MACE"] = (
    mace_df["formation_energy_MACE"]
    -
    mace_df["hull_energy_MACE"]
)


# ============================================================
# 21. STRUCTURAL DIVERSITY METRIC
# ============================================================
#
# We do NOT want 20 copies of essentially the same structure.
#
# A simple structural fingerprint is constructed from:
#
#   - atom count
#   - composition
#   - cell a
#   - cell c
#   - c/a
#
# This is intentionally a lightweight diversity screen.
# ============================================================

diversity_records = []

for candidate_id, atoms in mace_structures.items():

    cell = atoms.cell

    a_length = np.linalg.norm(
        cell[0]
    )

    b_length = np.linalg.norm(
        cell[1]
    )

    c_length = np.linalg.norm(
        cell[2]
    )

    diversity_records.append({

        "candidate_id":
            candidate_id,

        "a_A":
            a_length,

        "b_A":
            b_length,

        "c_A":
            c_length,

        "c_over_a":
            c_length / a_length
            if a_length > 0
            else np.nan,

        "volume_A3":
            atoms.get_volume()
    })


diversity_df = pd.DataFrame(
    diversity_records
)


mace_df = mace_df.merge(
    diversity_df,
    on="candidate_id",
    how="left"
)


# ============================================================
# 22. THERMODYNAMIC RANK
# ============================================================
#
# The PRIMARY criterion is now:
#
#       smaller energy above hull = better
#
# NOT raw energy.
#
# There is intentionally NO hard cutoff.
# ============================================================

mace_df = mace_df.sort_values(
    by=[
        "energy_above_hull_MACE",
        "formula",
        "candidate_id"
    ],
    ascending=[
        True,
        True,
        True
    ]
).reset_index(
    drop=True
)


# ============================================================
# 23. SELECT DIVERSE TOP 20
# ============================================================
#
# First prioritize thermodynamic stability.
# Then avoid filling the final list with many structures
# having exactly the same composition.
# ============================================================

selected_ids = []

composition_counts = Counter()


for _, row in mace_df.iterrows():

    candidate_id = int(
        row["candidate_id"]
    )

    formula = row["formula"]

    # Maximum number of structures from one formula
    # in the final Top 20.
    if composition_counts[formula] >= 4:
        continue

    selected_ids.append(
        candidate_id
    )

    composition_counts[formula] += 1

    if len(selected_ids) >= N_TOP:
        break


top20_df = mace_df[
    mace_df["candidate_id"].isin(
        selected_ids
    )
].copy()


top20_df = top20_df.sort_values(
    by="energy_above_hull_MACE"
).reset_index(
    drop=True
)


# ============================================================
# 24. SAVE TOP 20 STRUCTURES
# ============================================================

from ase.io import write


top20_structures = [
    mace_structures[
        int(candidate_id)
    ]
    for candidate_id
    in top20_df["candidate_id"]
]


TOP20_RESULTS_FILE = os.path.join(
    GEN3_DIR,
    "generation_3_TOP20_RESULTS.csv"
)

TOP20_STRUCTURES_FILE = os.path.join(
    GEN3_DIR,
    "generation_3_TOP20_STRUCTURES.extxyz"
)


top20_df.to_csv(
    TOP20_RESULTS_FILE,
    index=False
)


write(
    TOP20_STRUCTURES_FILE,
    top20_structures,
    format="extxyz"
)


# ============================================================
# 25. SAVE COMPLETE MACE SCREEN
# ============================================================

ALL_MACE_RESULTS_FILE = os.path.join(
    GEN3_DIR,
    "generation_3_ALL_MACE_RESULTS.csv"
)


mace_df.to_csv(
    ALL_MACE_RESULTS_FILE,
    index=False
)


# ============================================================
# 26. FINAL REPORT
# ============================================================

print("\n" + "=" * 100)
print("GENERATION 3 — THERMODYNAMICALLY AWARE RESULTS")
print("=" * 100)

display(
    top20_df[
        [
            "candidate_id",
            "formula",
            "x_Ti",
            "energy_eV_per_atom",
            "formation_energy_MACE",
            "hull_energy_MACE",
            "energy_above_hull_MACE",
            "a_A",
            "c_A",
            "c_over_a",
            "volume_A3"
        ]
    ]
)


print("\n" + "=" * 100)
print("ARCHIVE")
print("=" * 100)

print(
    f"TOP 20 results     : "
    f"{TOP20_RESULTS_FILE}"
)

print(
    f"TOP 20 structures  : "
    f"{TOP20_STRUCTURES_FILE}"
)

print(
    f"ALL MACE results   : "
    f"{ALL_MACE_RESULTS_FILE}"
)

print("\nGeneration 3 completed.")
print("Thermodynamic stability was evaluated during the search.")
print("No hard 0.10 eV/atom cutoff was used.")
print("Known TiSi, TiSi2 and Ti5Si3 compositions were excluded.")

In [ ]:
# ============================================================
# 12A — Retrieve Real Ti-Si Reference Structures
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Materials Project API key
# ------------------------------------------------------------

MP_API_KEY = "cvUeAXbDAqSIOOePiZ10YATWmRYCGhFG"


# ------------------------------------------------------------
# 2. Current best candidate
# ------------------------------------------------------------

candidate_id = 81
candidate_formula = "Ti4Si"

print("=== Current Best Candidate ===")
print(f"Candidate ID : {candidate_id}")
print(f"Formula      : {candidate_formula}")
print("Atom count   : 5")


# ------------------------------------------------------------
# 3. Known Ti-Si reference compositions
# ------------------------------------------------------------

target_formulas = [
    "Ti",
    "Si",
    "Ti2Si",
    "Ti5Si3",
    "TiSi",
    "TiSi2",
]


# ------------------------------------------------------------
# 4. Search Materials Project
# ------------------------------------------------------------

reference_docs = {}

with MPRester(MP_API_KEY) as mpr:

    for formula in target_formulas:

        docs = mpr.materials.summary.search(
            formula=formula,
            fields=[
                "material_id",
                "formula_pretty",
                "structure",
                "energy_per_atom",
                "formation_energy_per_atom",
                "energy_above_hull",
            ],
        )

        if len(docs) == 0:
            print(f"No Materials Project entry found for {formula}")
            continue

        # Choose the entry with the lowest energy above hull
        docs = sorted(
            docs,
            key=lambda x: (
                x.energy_above_hull
                if x.energy_above_hull is not None
                else np.inf
            )
        )

        best = docs[0]

        reference_docs[formula] = best

        print(
            f"{formula:8s} | "
            f"{best.material_id} | "
            f"E_hull = {best.energy_above_hull}"
        )


# ------------------------------------------------------------
# 5. Build a compact summary table
# ------------------------------------------------------------

reference_summary = []

for formula, doc in reference_docs.items():

    reference_summary.append({
        "formula": formula,
        "material_id": str(doc.material_id),
        "energy_eV_per_atom": doc.energy_per_atom,
        "formation_energy_eV_per_atom": doc.formation_energy_per_atom,
        "energy_above_hull_eV_per_atom": doc.energy_above_hull,
        "structure": doc.structure,
    })


reference_df = pd.DataFrame(reference_summary)


# ------------------------------------------------------------
# 6. Display reference structures
# ------------------------------------------------------------

print("\n=== Real Ti-Si Reference Structures ===")

display(
    reference_df[
        [
            "formula",
            "material_id",
            "energy_eV_per_atom",
            "formation_energy_eV_per_atom",
            "energy_above_hull_eV_per_atom",
        ]
    ]
)


# ------------------------------------------------------------
# 7. Confirm the candidate that will be tested
# ------------------------------------------------------------

print("\n=== Candidate Selected for Convex Hull ===")
print(f"Candidate ID : {candidate_id}")
print(f"Formula      : {candidate_formula}")
print("Structure    : generated_structures[81]")
print("Atom count   : 5")

In [ ]:
# ============================================================
# MACE-CONSISTENT FORMATION ENERGY CHECK — CANDIDATE 71
# ============================================================

import numpy as np
import torch

from ase.build import bulk
from ase.optimize import BFGS
from ase.filters import ExpCellFilter
from mace.calculators import mace_mp


# ------------------------------------------------------------
# 1. Select Candidate 81
# ------------------------------------------------------------

candidate_id = 81

candidate_structure = generated_structures[candidate_id].copy()


# ------------------------------------------------------------
# 2. SAME MACE SETUP USED IN THE PROJECT
# ------------------------------------------------------------

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64"
)


# ------------------------------------------------------------
# 3. Calculate Candidate Energy
# ------------------------------------------------------------

candidate_structure.calc = calc

candidate_energy_total = (
    candidate_structure.get_potential_energy()
)

candidate_energy_per_atom = (
    candidate_energy_total / len(candidate_structure)
)


# ------------------------------------------------------------
# 4. Build REAL ELEMENTAL REFERENCE PROTOTYPES
# ------------------------------------------------------------

# Alpha-Ti: HCP
ti_reference = bulk(
    "Ti",
    crystalstructure="hcp",
    a=2.95,
    c=4.68
)

# Silicon: Diamond Cubic
si_reference = bulk(
    "Si",
    crystalstructure="diamond",
    a=5.43
)


# ------------------------------------------------------------
# 5. Relax Ti Reference with the SAME MACE
# ------------------------------------------------------------

ti_reference.calc = calc

ti_cell = ExpCellFilter(ti_reference)

ti_optimizer = BFGS(
    ti_cell,
    logfile=None
)

ti_optimizer.run(
    fmax=0.01,
    steps=300
)

ti_energy_total = (
    ti_reference.get_potential_energy()
)

ti_energy_per_atom = (
    ti_energy_total / len(ti_reference)
)


# ------------------------------------------------------------
# 6. Relax Si Reference with the SAME MACE
# ------------------------------------------------------------

si_reference.calc = calc

si_cell = ExpCellFilter(si_reference)

si_optimizer = BFGS(
    si_cell,
    logfile=None
)

si_optimizer.run(
    fmax=0.01,
    steps=300
)

si_energy_total = (
    si_reference.get_potential_energy()
)

si_energy_per_atom = (
    si_energy_total / len(si_reference)
)


# ------------------------------------------------------------
# 7. MACE-CONSISTENT FORMATION ENERGY
# ------------------------------------------------------------

# Candidate composition: SiTi2
# Ti fraction = 2 / 3
# Si fraction = 1 / 3

x_Ti = 2 / 3
x_Si = 1 / 3

formation_energy_mace = (
    candidate_energy_per_atom
    - (
        x_Ti * ti_energy_per_atom
        + x_Si * si_energy_per_atom
    )
)


# ------------------------------------------------------------
# 8. Report Results
# ------------------------------------------------------------

print("=" * 80)
print("MACE-CONSISTENT FORMATION ENERGY CHECK — CANDIDATE 71")
print("=" * 80)

print(f"Candidate ID : {candidate_id}")
print("Formula      : SiTi2")
print(f"Number of atoms : {len(candidate_structure)}")

print("\n--- Candidate 71 ---")

print(
    f"MACE energy/atom : "
    f"{candidate_energy_per_atom:.8f} eV/atom"
)

print("\n--- Relaxed elemental references ---")

print(
    f"Ti (alpha-hcp) : "
    f"{ti_energy_per_atom:.8f} eV/atom"
)

print(
    f"Si (diamond-cubic) : "
    f"{si_energy_per_atom:.8f} eV/atom"
)

print("\n--- MACE Formation Energy ---")

print(
    f"Formation energy : "
    f"{formation_energy_mace:+.8f} eV/atom"
)

print("=" * 80)

In [ ]:
# ============================================================
# SESSION VARIABLES INSPECTOR
# ============================================================

from IPython import get_ipython
import pandas as pd
import numpy as np

ip = get_ipython()

print("=" * 100)
print("ALL VARIABLES CURRENTLY IN THE SESSION")
print("=" * 100)

user_vars = {}

for name, value in sorted(ip.user_ns.items()):
    # Ignore IPython/internal variables
    if name.startswith("_"):
        continue

    # Ignore imported modules/functions/classes
    if callable(value):
        continue

    user_vars[name] = value

    print(f"\n{'-' * 100}")
    print(f"VARIABLE: {name}")
    print(f"TYPE    : {type(value).__name__}")

    # ----------------------------
    # DataFrame
    # ----------------------------
    if isinstance(value, pd.DataFrame):
        print(f"SHAPE   : {value.shape}")
        print(f"COLUMNS : {list(value.columns)}")
        print("\nCONTENT:")
        print(value.to_string(index=False))

    # ----------------------------
    # Series
    # ----------------------------
    elif isinstance(value, pd.Series):
        print(f"SHAPE   : {value.shape}")
        print("\nCONTENT:")
        print(value.to_string())

    # ----------------------------
    # Dictionary
    # ----------------------------
    elif isinstance(value, dict):
        print(f"LENGTH  : {len(value)}")
        print(f"KEYS    : {list(value.keys())[:30]}")

        if len(value) > 0:
            print("\nFIRST ITEMS:")

            for i, (k, v) in enumerate(value.items()):
                if i >= 10:
                    break

                print(f"\n  KEY: {k}")
                print(f"  VALUE TYPE: {type(v).__name__}")

                if isinstance(v, pd.DataFrame):
                    print(f"  VALUE SHAPE: {v.shape}")
                    print(f"  VALUE COLUMNS: {list(v.columns)}")

                elif hasattr(v, "get_chemical_formula"):
                    print(f"  FORMULA: {v.get_chemical_formula()}")
                    print(f"  ATOMS: {len(v)}")

                else:
                    text = repr(v)
                    if len(text) > 500:
                        text = text[:500] + "..."
                    print(f"  VALUE: {text}")

    # ----------------------------
    # List / Tuple
    # ----------------------------
    elif isinstance(value, (list, tuple)):
        print(f"LENGTH  : {len(value)}")

        if len(value) > 0:
            print("\nFIRST ITEMS:")

            for i, item in enumerate(value[:10]):
                print(f"\n  [{i}] TYPE: {type(item).__name__}")

                if hasattr(item, "get_chemical_formula"):
                    print(f"      FORMULA: {item.get_chemical_formula()}")
                    print(f"      ATOMS  : {len(item)}")
                else:
                    text = repr(item)
                    if len(text) > 500:
                        text = text[:500] + "..."
                    print(f"      VALUE: {text}")

    # ----------------------------
    # ASE Atoms
    # ----------------------------
    elif hasattr(value, "get_chemical_formula") and hasattr(value, "get_positions"):
        print(f"FORMULA : {value.get_chemical_formula()}")
        print(f"ATOMS   : {len(value)}")
        print(f"CELL    : {value.cell.lengths()}")

    # ----------------------------
    # NumPy array
    # ----------------------------
    elif isinstance(value, np.ndarray):
        print(f"SHAPE   : {value.shape}")
        print(f"DTYPE   : {value.dtype}")
        print(f"CONTENT :\n{value}")

    # ----------------------------
    # Other variables
    # ----------------------------
    else:
        text = repr(value)
        if len(text) > 1000:
            text = text[:1000] + "..."
        print(f"VALUE   : {text}")


print("\n" + "=" * 100)
print(f"TOTAL USER VARIABLES: {len(user_vars)}")
print("=" * 100)

In [ ]:
# ============================================================
# MACE-CONSISTENT Ti-Si CONVEX HULL — TOP 5 CANDIDATES
# ============================================================

import re
import numpy as np
import pandas as pd

from ase.optimize import BFGS
from ase.filters import FrechetCellFilter
from mace.calculators import mace_mp
from pymatgen.io.ase import AseAtomsAdaptor


# ------------------------------------------------------------
# 1. Select TOP 5 candidates from Generation 3
# ------------------------------------------------------------

TOP_N = 5

top5 = (
    mace_df
    .sort_values("energy_above_hull_MACE")
    .head(TOP_N)
    .copy()
)

print("=" * 90)
print("TOP 5 GENERATION 3 CANDIDATES")
print("=" * 90)

display(top5)


# ------------------------------------------------------------
# 2. SAME MACE SETUP
# ------------------------------------------------------------

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64"
)


# ------------------------------------------------------------
# 3. Ti fraction
# ------------------------------------------------------------

def ti_fraction(formula):

    matches = re.findall(
        r"(Ti|Si)(\d*)",
        str(formula)
    )

    ti_count = 0
    si_count = 0

    for element, count in matches:

        count = int(count) if count else 1

        if element == "Ti":
            ti_count += count

        elif element == "Si":
            si_count += count

    total = ti_count + si_count

    if total == 0:
        raise ValueError(
            f"Could not parse Ti-Si formula: {formula}"
        )

    return ti_count / total


# ------------------------------------------------------------
# 4. MACE energies of REAL reference structures
# ------------------------------------------------------------

mace_reference_rows = []
mace_reference_structures = {}

for formula, doc in reference_docs.items():

    print(f"Processing MACE reference: {formula}")

    structure = AseAtomsAdaptor.get_atoms(
        doc.structure.copy()
    )

    structure.calc = calc

    cell_filter = FrechetCellFilter(
        structure
    )

    optimizer = BFGS(
        cell_filter,
        logfile=None
    )

    optimizer.run(
        fmax=0.01,
        steps=300
    )

    energy_total = structure.get_potential_energy()

    energy_per_atom = (
        energy_total / len(structure)
    )

    mace_reference_structures[formula] = structure

    mace_reference_rows.append({

        "formula": formula,

        "material_id":
            str(doc.material_id),

        "x_Ti":
            ti_fraction(formula),

        "energy_eV_per_atom":
            energy_per_atom,

        "type":
            "REFERENCE"
    })


mace_reference_df = pd.DataFrame(
    mace_reference_rows
)


# ------------------------------------------------------------
# 5. Elemental reference energies
# ------------------------------------------------------------

ti_energy = float(
    mace_reference_df.loc[
        mace_reference_df["formula"] == "Ti",
        "energy_eV_per_atom"
    ].iloc[0]
)

si_energy = float(
    mace_reference_df.loc[
        mace_reference_df["formula"] == "Si",
        "energy_eV_per_atom"
    ].iloc[0]
)


# ------------------------------------------------------------
# 6. Formation energies of reference phases
# ------------------------------------------------------------

mace_reference_df[
    "formation_energy_eV_per_atom"
] = (
    mace_reference_df["energy_eV_per_atom"]
    - mace_reference_df["x_Ti"] * ti_energy
    - (1.0 - mace_reference_df["x_Ti"]) * si_energy
)


# ------------------------------------------------------------
# 7. Build HULL FROM REFERENCES ONLY
# ------------------------------------------------------------

reference_points = (
    mace_reference_df[
        [
            "x_Ti",
            "formation_energy_eV_per_atom",
            "formula",
            "type"
        ]
    ]
    .sort_values("x_Ti")
    .reset_index(drop=True)
)


def cross(o, a, b):

    return (
        (a["x_Ti"] - o["x_Ti"])
        *
        (
            b["formation_energy_eV_per_atom"]
            - o["formation_energy_eV_per_atom"]
        )

        -

        (
            a["formation_energy_eV_per_atom"]
            - o["formation_energy_eV_per_atom"]
        )
        *
        (b["x_Ti"] - o["x_Ti"])
    )


hull = []

for _, point in reference_points.iterrows():

    while len(hull) >= 2:

        if cross(
            hull[-2],
            hull[-1],
            point
        ) <= 0:

            hull.pop()

        else:

            break

    hull.append(
        point.to_dict()
    )


reference_hull_df = pd.DataFrame(hull)


# ------------------------------------------------------------
# 8. Hull energy at a composition
# ------------------------------------------------------------

def hull_energy_at_x(x, hull_points):

    hull_points = (
        hull_points
        .sort_values("x_Ti")
        .reset_index(drop=True)
    )

    exact = hull_points[
        np.isclose(
            hull_points["x_Ti"],
            x,
            atol=1e-10
        )
    ]

    if len(exact) > 0:

        return exact[
            "formation_energy_eV_per_atom"
        ].min()

    for i in range(
        len(hull_points) - 1
    ):

        left = hull_points.iloc[i]
        right = hull_points.iloc[i + 1]

        if (
            left["x_Ti"]
            <= x
            <= right["x_Ti"]
        ):

            x1 = left["x_Ti"]
            x2 = right["x_Ti"]

            e1 = left[
                "formation_energy_eV_per_atom"
            ]

            e2 = right[
                "formation_energy_eV_per_atom"
            ]

            return (
                e1
                +
                (x - x1)
                / (x2 - x1)
                * (e2 - e1)
            )

    raise ValueError(
        f"Could not determine hull energy at x_Ti={x}"
    )


# ------------------------------------------------------------
# 9. Evaluate TOP 5
# ------------------------------------------------------------

candidate_results = []

for _, row in top5.iterrows():

    candidate_id = int(
        row["candidate_id"]
    )

    candidate_row = mace_df[
    mace_df["candidate_id"] == candidate_id
].iloc[0]

    candidate_formula = (
        candidate_row["formula"]
    )

    candidate_x = ti_fraction(
        candidate_formula
    )

    candidate_energy = float(
        candidate_row["energy_eV_per_atom"]
    )

    candidate_formation_energy = (
        candidate_energy
        - candidate_x * ti_energy
        - (1.0 - candidate_x) * si_energy
    )

    hull_energy = hull_energy_at_x(
        candidate_x,
        reference_hull_df
    )

    energy_above_hull = (
        candidate_formation_energy
        - hull_energy
    )

    candidate_results.append({

        "candidate_id":
            candidate_id,

        "formula":
            candidate_formula,

        "x_Ti":
            candidate_x,

        "energy_eV_per_atom":
            candidate_energy,

        "formation_energy_eV_per_atom":
            candidate_formation_energy,

        "hull_energy_eV_per_atom":
            hull_energy,

        "energy_above_hull_eV_per_atom":
            energy_above_hull
    })


candidate_hull_results = pd.DataFrame(
    candidate_results
)


# ------------------------------------------------------------
# 10. Final ranking
# ------------------------------------------------------------

candidate_hull_results = (
    candidate_hull_results
    .sort_values(
        "energy_above_hull_eV_per_atom"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 11. Display
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("MACE-CONSISTENT TOP 5 HULL RESULTS")
print("=" * 90)

display(
    candidate_hull_results
)


print("\n" + "=" * 90)
print("REFERENCE MACE CONVEX HULL")
print("=" * 90)

display(
    reference_hull_df
)

In [ ]:
# ============================================================
# MACE-CONSISTENT TOP 5 — COMPLETE ARCHIVE
# ============================================================

import os
import json
import pickle
import pandas as pd
import numpy as np
from ase.io import write


# ============================================================
# 1. ARCHIVE DIRECTORY
# ============================================================

TOP5_DIR = os.path.join(
    "Top5_search_archive",
    "generation_3_MACE_TOP5_archive"
)

os.makedirs(TOP5_DIR, exist_ok=True)


# ============================================================
# 2. SELECT MACE-CONSISTENT TOP 5
# ============================================================

TOP5_N = 5

top5_df = (
    mace_df
    .sort_values(
        by="energy_above_hull_MACE",
        ascending=True
    )
    .head(TOP5_N)
    .copy()
    .reset_index(drop=True)
)

top5_ids = [
    int(x)
    for x in top5_df["candidate_id"]
]


# ============================================================
# 3. TOP 5 STRUCTURES
# ============================================================

top5_structures = {
    candidate_id:
        mace_structures[candidate_id].copy()
    for candidate_id in top5_ids
}


# ============================================================
# 4. COMPLETE RESULTS DICTIONARY
# ============================================================

top5_results_dict = {}

for _, row in top5_df.iterrows():

    candidate_id = int(
        row["candidate_id"]
    )

    top5_results_dict[candidate_id] = {
        key: (
            value.item()
            if isinstance(value, np.generic)
            else value
        )
        for key, value in row.to_dict().items()
    }


# ============================================================
# 5. STRUCTURE INFORMATION DICTIONARY
# ============================================================

top5_structure_info = {}

for candidate_id, atoms in top5_structures.items():

    top5_structure_info[candidate_id] = {

        "candidate_id":
            candidate_id,

        "symbols":
            atoms.get_chemical_symbols(),

        "positions_A":
            atoms.get_positions().tolist(),

        "cell_A":
            atoms.cell.array.tolist(),

        "pbc":
            atoms.get_pbc().tolist(),

        "formula":
            atoms.get_chemical_formula(),

        "number_of_atoms":
            len(atoms),

        "volume_A3":
            float(atoms.get_volume())
    }


# ============================================================
# 6. SAVE COMPLETE RESULTS TABLE
# ============================================================

TOP5_RESULTS_FILE = os.path.join(
    TOP5_DIR,
    "MACE_consistent_TOP5_RESULTS.csv"
)

top5_df.to_csv(
    TOP5_RESULTS_FILE,
    index=False
)


# ============================================================
# 7. SAVE ALL 5 STRUCTURES — EXTXYZ
# ============================================================

TOP5_STRUCTURES_FILE = os.path.join(
    TOP5_DIR,
    "MACE_consistent_TOP5_STRUCTURES.extxyz"
)

top5_atoms_list = [
    top5_structures[candidate_id]
    for candidate_id in top5_ids
]

write(
    TOP5_STRUCTURES_FILE,
    top5_atoms_list,
    format="extxyz"
)


# ============================================================
# 8. SAVE candidate_id → structure
# ============================================================

TOP5_STRUCTURE_DICT_FILE = os.path.join(
    TOP5_DIR,
    "MACE_TOP5_candidate_to_structure.pkl"
)

with open(
    TOP5_STRUCTURE_DICT_FILE,
    "wb"
) as f:

    pickle.dump(
        top5_structures,
        f
    )


# ============================================================
# 9. SAVE candidate_id → ALL RESULTS
# ============================================================

TOP5_RESULTS_DICT_FILE = os.path.join(
    TOP5_DIR,
    "MACE_TOP5_candidate_to_all_results.pkl"
)

with open(
    TOP5_RESULTS_DICT_FILE,
    "wb"
) as f:

    pickle.dump(
        top5_results_dict,
        f
    )


# ============================================================
# 10. SAVE STRUCTURE INFORMATION AS JSON
# ============================================================

TOP5_STRUCTURE_INFO_FILE = os.path.join(
    TOP5_DIR,
    "MACE_TOP5_structure_information.json"
)

with open(
    TOP5_STRUCTURE_INFO_FILE,
    "w"
) as f:

    json.dump(
        top5_structure_info,
        f,
        indent=2
    )


# ============================================================
# 11. REFERENCE MACE HULL INFORMATION
# ============================================================

reference_hull = {

    "description":
        "MACE-consistent Ti-Si reference convex hull",

    "calculator":
        "MACE medium",

    "dispersion":
        True,

    "default_dtype":
        "float64",

    "model":
        "20231203mace128L1_epoch199model",

    "reference_elements": {

        "Ti": {
            "crystal_structure":
                "hcp",
            "energy_eV_per_atom":
                float(ti_energy)
        },

        "Si": {
            "crystal_structure":
                "diamond",
            "energy_eV_per_atom":
                float(si_energy)
        }
    },

    "hull_points": [

        {
            "x_Ti": float(x),
            "formation_energy_eV_per_atom":
                float(e)
        }

        for x, e
        in zip(HULL_X, HULL_E)
    ],

    "reference_compounds": [
        "Si",
        "TiSi2",
        "TiSi",
        "Ti5Si3",
        "Ti"
    ]
}


REFERENCE_HULL_FILE = os.path.join(
    TOP5_DIR,
    "MACE_REFERENCE_HULL.json"
)

with open(
    REFERENCE_HULL_FILE,
    "w"
) as f:

    json.dump(
        reference_hull,
        f,
        indent=2
    )


# ============================================================
# 12. SAVE A HUMAN-READABLE SUMMARY
# ============================================================

SUMMARY_FILE = os.path.join(
    TOP5_DIR,
    "MACE_TOP5_ARCHIVE_SUMMARY.txt"
)

with open(
    SUMMARY_FILE,
    "w"
) as f:

    f.write(
        "MACE-CONSISTENT TOP 5 ARCHIVE\n"
    )

    f.write(
        "=" * 70 + "\n\n"
    )

    f.write(
        "MACE model: 20231203mace128L1_epoch199model\n"
    )

    f.write(
        "dtype: float64\n"
    )

    f.write(
        "dispersion: True\n\n"
    )

    f.write(
        "TOP 5 CANDIDATES\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    f.write(
        top5_df.to_string(
            index=False
        )
    )

    f.write(
        "\n\nREFERENCE HULL\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    for point in reference_hull["hull_points"]:

        f.write(
            f"xTi={point['x_Ti']:.6f} | "
            f"Eform={point['formation_energy_eV_per_atom']:.6f} eV/atom\n"
        )

    f.write(
        "\n\nARCHIVED STRUCTURE DATA\n"
    )

    f.write(
        "-" * 70 + "\n"
    )

    for candidate_id in top5_ids:

        info = top5_structure_info[
            candidate_id
        ]

        f.write(
            f"\nCandidate {candidate_id}\n"
        )

        f.write(
            f"Formula: {info['formula']}\n"
        )

        f.write(
            f"Atoms: {info['number_of_atoms']}\n"
        )

        f.write(
            f"Volume: {info['volume_A3']:.6f} A^3\n"
        )

        f.write(
            f"PBC: {info['pbc']}\n"
        )


# ============================================================
# 13. FINAL ARCHIVE REPORT
# ============================================================

print("\n" + "=" * 100)
print("MACE-CONSISTENT TOP 5 ARCHIVE CREATED")
print("=" * 100)

print(
    f"\nArchive directory:\n{TOP5_DIR}\n"
)

print("\nFiles saved:")

print(
    "1. MACE_consistent_TOP5_RESULTS.csv"
)

print(
    "2. MACE_consistent_TOP5_STRUCTURES.extxyz"
)

print(
    "3. MACE_TOP5_candidate_to_structure.pkl"
)

print(
    "4. MACE_TOP5_candidate_to_all_results.pkl"
)

print(
    "5. MACE_TOP5_structure_information.json"
)

print(
    "6. MACE_REFERENCE_HULL.json"
)

print(
    "7. MACE_TOP5_ARCHIVE_SUMMARY.txt"
)

print("\nTop 5 candidate IDs:")

print(top5_ids)

print("\n" + "=" * 100)

In [ ]:
# ============================================================
# TEST 4 — EXTENDED STRUCTURAL STABILITY
# Perturbations: 0.05, 0.10, 0.20 Å
# ============================================================

import numpy as np
import pandas as pd

from ase.optimize import BFGS
from ase.geometry import find_mic


# ============================================================
# PART 1 — Test settings
# ============================================================

PERTURBATION_LEVELS = [0.05, 0.10, 0.20]

N_TESTS_PER_LEVEL = 10

FMAX = 0.05
MAX_STEPS = 250

rng = np.random.default_rng(20260913)


# ============================================================
# PART 2 — Original relaxed structure
# ============================================================

original = candidate.copy()
original.calc = calculator

original_energy = original.get_potential_energy()
original_positions = original.get_positions().copy()
original_cell = original.get_cell().copy()


# ============================================================
# PART 3 — Run perturbation tests
# ============================================================

extended_results = []
extended_structures = {}

for perturbation_std in PERTURBATION_LEVELS:

    for test_number in range(1, N_TESTS_PER_LEVEL + 1):

        test_id = f"{perturbation_std:.2f}_{test_number}"

        test_structure = original.copy()

        # Randomly perturb atomic positions
        displacement = rng.normal(
            loc=0.0,
            scale=perturbation_std,
            size=test_structure.positions.shape
        )

        test_structure.positions += displacement

        # Keep the original cell fixed
        test_structure.set_cell(original_cell)
        test_structure.set_pbc(original.get_pbc())

        test_structure.calc = calculator

        perturbed_energy = test_structure.get_potential_energy()

        # Relax positions only
        optimizer = BFGS(
            test_structure,
            logfile=None
        )

        optimizer.run(
            fmax=FMAX,
            steps=MAX_STEPS
        )

        final_energy = test_structure.get_potential_energy()
        final_positions = test_structure.get_positions()

        # Compare final structure with the original structure
        position_difference = (
            final_positions - original_positions
        )

        mic_difference, _ = find_mic(
            position_difference,
            cell=original_cell,
            pbc=original.get_pbc()
        )

        displacement_lengths = np.linalg.norm(
            mic_difference,
            axis=1
        )

        rmsd = np.sqrt(
            np.mean(displacement_lengths ** 2)
        )

        max_displacement = np.max(displacement_lengths)

        energy_difference = (
            final_energy - original_energy
        )

        extended_results.append({
            "perturbation_std_A": perturbation_std,
            "test_number": test_number,
            "test_id": test_id,
            "perturbed_energy_eV_per_atom":
                perturbed_energy / len(test_structure),
            "final_energy_eV_per_atom":
                final_energy / len(test_structure),
            "energy_difference_eV_per_atom":
                energy_difference / len(test_structure),
            "RMSD_A": rmsd,
            "max_displacement_A": max_displacement
        })

        extended_structures[test_id] = test_structure.copy()


# ============================================================
# PART 4 — Results
# ============================================================

extended_df = pd.DataFrame(extended_results)

print("\n" + "=" * 90)
print("TEST 4 — EXTENDED STRUCTURAL STABILITY")
print("=" * 90)

print(f"Original energy/atom : "
      f"{original_energy / len(original):.6f} eV")

print(f"Tests per level      : {N_TESTS_PER_LEVEL}")
print(f"Perturbation levels  : {PERTURBATION_LEVELS}")

print("\nAll results:")
print(
    extended_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# PART 5 — Summary by perturbation strength
# ============================================================

summary_df = (
    extended_df
    .groupby("perturbation_std_A")
    .agg(
        mean_final_energy_eV_per_atom=(
            "final_energy_eV_per_atom", "mean"
        ),
        mean_energy_difference_eV_per_atom=(
            "energy_difference_eV_per_atom", "mean"
        ),
        mean_RMSD_A=("RMSD_A", "mean"),
        max_RMSD_A=("RMSD_A", "max"),
        mean_max_displacement_A=(
            "max_displacement_A", "mean"
        ),
        max_displacement_A=(
            "max_displacement_A", "max"
        )
    )
    .reset_index()
)

print("\n" + "=" * 90)
print("STRUCTURAL STABILITY SUMMARY BY PERTURBATION")
print("=" * 90)

print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


print("\nTest completed.")

In [ ]:
# ============================================================
# TEST 4 — STRUCTURAL STABILITY
# Periodic Structural Fingerprint
# Candidate 1: Ti4Si
# ============================================================

import numpy as np
import pandas as pd

from ase.optimize import BFGS
from mace.calculators import mace_mp


# ============================================================
# PART 1 — MACE calculator and test settings
# ============================================================

calculator = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64"
)

PERTURBATION_LEVELS = [0.05, 0.10, 0.20]
N_TESTS_PER_LEVEL = 10

FMAX = 0.05
MAX_STEPS = 250

# Maximum distance used for the structural fingerprint
FINGERPRINT_CUTOFF = 6.0  # Å

rng = np.random.default_rng(20260913)


# ============================================================
# PART 2 — Original relaxed structure
# ============================================================

original = candidate.copy()
original.calc = calculator

original_energy = original.get_potential_energy()

original_cell = original.get_cell().copy()
original_pbc = original.get_pbc()

original_symbols = original.get_chemical_symbols()


# ============================================================
# PART 3 — Periodic structural fingerprint
# ============================================================

def get_periodic_fingerprint(atoms, cutoff=6.0):

    symbols = atoms.get_chemical_symbols()
    positions = atoms.get_positions()
    cell = atoms.get_cell()
    pbc = atoms.get_pbc()

    n_atoms = len(atoms)

    all_distances = []
    pair_distances = {
        "Ti-Ti": [],
        "Ti-Si": [],
        "Si-Si": []
    }

    nearest_neighbors = []

    # Fractional coordinates
    scaled_positions = atoms.get_scaled_positions()

    for i in range(n_atoms):

        distances_i = []

        for j in range(n_atoms):

            if i == j:
                continue

            # Fractional-coordinate difference
            delta_frac = scaled_positions[j] - scaled_positions[i]

            # Minimum-image convention for periodic directions
            for axis in range(3):
                if pbc[axis]:
                    delta_frac[axis] -= np.round(delta_frac[axis])

            # Convert to Cartesian
            delta_cart = delta_frac @ cell.array

            distance = np.linalg.norm(delta_cart)

            if distance < cutoff:

                all_distances.append(distance)
                distances_i.append(distance)

                pair = "-".join(
                    sorted([symbols[i], symbols[j]])
                )

                if pair == "Ti-Ti":
                    pair_distances["Ti-Ti"].append(distance)

                elif pair == "Si-Ti":
                    pair_distances["Ti-Si"].append(distance)

                elif pair == "Si-Si":
                    pair_distances["Si-Si"].append(distance)

        if distances_i:
            nearest_neighbors.append(
                min(distances_i)
            )

    fingerprint = {
        "all_distances": np.array(sorted(all_distances)),
        "Ti-Ti": np.array(sorted(pair_distances["Ti-Ti"])),
        "Ti-Si": np.array(sorted(pair_distances["Ti-Si"])),
        "Si-Si": np.array(sorted(pair_distances["Si-Si"])),
        "nearest_neighbors": np.array(sorted(nearest_neighbors))
    }

    return fingerprint


# ============================================================
# PART 4 — Compare two periodic fingerprints
# ============================================================

def compare_fingerprints(original_fp, relaxed_fp):

    comparison = {}

    # --------------------------------------------------------
    # Pair-distance distribution
    # --------------------------------------------------------

    for pair in ["Ti-Ti", "Ti-Si", "Si-Si"]:

        original_distances = original_fp[pair]
        relaxed_distances = relaxed_fp[pair]

        n_common = min(
            len(original_distances),
            len(relaxed_distances)
        )

        if n_common > 0:

            differences = np.abs(
                original_distances[:n_common]
                - relaxed_distances[:n_common]
            )

            comparison[f"{pair}_mean_abs_difference_A"] = (
                np.mean(differences)
            )

            comparison[f"{pair}_max_abs_difference_A"] = (
                np.max(differences)
            )

            comparison[f"{pair}_count_original"] = (
                len(original_distances)
            )

            comparison[f"{pair}_count_relaxed"] = (
                len(relaxed_distances)
            )

        else:

            comparison[f"{pair}_mean_abs_difference_A"] = np.nan
            comparison[f"{pair}_max_abs_difference_A"] = np.nan
            comparison[f"{pair}_count_original"] = len(original_distances)
            comparison[f"{pair}_count_relaxed"] = len(relaxed_distances)

    # --------------------------------------------------------
    # Nearest-neighbor distribution
    # --------------------------------------------------------

    original_nn = original_fp["nearest_neighbors"]
    relaxed_nn = relaxed_fp["nearest_neighbors"]

    n_common = min(
        len(original_nn),
        len(relaxed_nn)
    )

    if n_common > 0:

        nn_difference = np.abs(
            original_nn[:n_common]
            - relaxed_nn[:n_common]
        )

        comparison["nearest_neighbor_mean_difference_A"] = (
            np.mean(nn_difference)
        )

        comparison["nearest_neighbor_max_difference_A"] = (
            np.max(nn_difference)
        )

    else:

        comparison["nearest_neighbor_mean_difference_A"] = np.nan
        comparison["nearest_neighbor_max_difference_A"] = np.nan

    # --------------------------------------------------------
    # Overall pair-distance distribution
    # --------------------------------------------------------

    original_all = original_fp["all_distances"]
    relaxed_all = relaxed_fp["all_distances"]

    n_common = min(
        len(original_all),
        len(relaxed_all)
    )

    if n_common > 0:

        all_difference = np.abs(
            original_all[:n_common]
            - relaxed_all[:n_common]
        )

        comparison["pair_distance_mean_difference_A"] = (
            np.mean(all_difference)
        )

        comparison["pair_distance_max_difference_A"] = (
            np.max(all_difference)
        )

    else:

        comparison["pair_distance_mean_difference_A"] = np.nan
        comparison["pair_distance_max_difference_A"] = np.nan

    return comparison


# ============================================================
# PART 5 — Original structural fingerprint
# ============================================================

original_fingerprint = get_periodic_fingerprint(
    original,
    cutoff=FINGERPRINT_CUTOFF
)


# ============================================================
# PART 6 — Perturb + relax + compare
# ============================================================

structural_results = []
relaxed_structures = {}

for perturbation_std in PERTURBATION_LEVELS:

    for test_number in range(1, N_TESTS_PER_LEVEL + 1):

        test_id = f"{perturbation_std:.2f}_{test_number}"

        test_structure = original.copy()

        # -----------------------------------------------
        # Small random atomic perturbation
        # -----------------------------------------------

        displacement = rng.normal(
            loc=0.0,
            scale=perturbation_std,
            size=test_structure.positions.shape
        )

        test_structure.positions += displacement

        # Keep cell fixed
        test_structure.set_cell(original_cell)
        test_structure.set_pbc(original_pbc)

        test_structure.calc = calculator

        # Energy immediately after perturbation
        perturbed_energy = (
            test_structure.get_potential_energy()
        )

        # -----------------------------------------------
        # Relax atomic positions only
        # -----------------------------------------------

        optimizer = BFGS(
            test_structure,
            logfile=None
        )

        optimizer.run(
            fmax=FMAX,
            steps=MAX_STEPS
        )

        final_energy = (
            test_structure.get_potential_energy()
        )

        # -----------------------------------------------
        # Structural fingerprint after relaxation
        # -----------------------------------------------

        relaxed_fingerprint = get_periodic_fingerprint(
            test_structure,
            cutoff=FINGERPRINT_CUTOFF
        )

        comparison = compare_fingerprints(
            original_fingerprint,
            relaxed_fingerprint
        )

        # -----------------------------------------------
        # Store results
        # -----------------------------------------------

        result = {
            "perturbation_std_A": perturbation_std,
            "test_number": test_number,
            "test_id": test_id,

            "perturbed_energy_eV_per_atom":
                perturbed_energy / len(test_structure),

            "final_energy_eV_per_atom":
                final_energy / len(test_structure),

            "energy_difference_eV_per_atom":
                (
                    final_energy - original_energy
                ) / len(test_structure)
        }

        result.update(comparison)

        structural_results.append(result)

        relaxed_structures[test_id] = (
            test_structure.copy()
        )
        # =============================================================================
# IMMEDIATE CHECKPOINT SAVE
# =============================================================================

# Save the structure immediately
write(
    CHECKPOINT_EXTXYZ,
    system,
    format="extxyz",
    append=os.path.isfile(CHECKPOINT_EXTXYZ)
)

# Save the numerical result immediately
csv_exists = os.path.isfile(CHECKPOINT_CSV)

with open(
    CHECKPOINT_CSV,
    "a",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    if not csv_exists:
        writer.writeheader()

    writer.writerow(result)

print("\nSAVED IMMEDIATELY.")


# ============================================================
# PART 7 — Full results
# ============================================================

structural_df = pd.DataFrame(structural_results)

print("\n" + "=" * 100)
print("TEST 4 — PERIODIC STRUCTURAL STABILITY")
print("=" * 100)

print(
    f"Original energy/atom : "
    f"{original_energy / len(original):.6f} eV"
)

print(
    f"Fingerprint cutoff   : "
    f"{FINGERPRINT_CUTOFF:.2f} Å"
)

print(
    f"Tests per level      : "
    f"{N_TESTS_PER_LEVEL}"
)

print(
    f"Perturbation levels  : "
    f"{PERTURBATION_LEVELS}"
)

print("\nAll results:")

print(
    structural_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# PART 8 — Summary by perturbation strength
# ============================================================

summary_df = (
    structural_df
    .groupby("perturbation_std_A")
    .agg(
        mean_final_energy_eV_per_atom=(
            "final_energy_eV_per_atom",
            "mean"
        ),

        mean_energy_difference_eV_per_atom=(
            "energy_difference_eV_per_atom",
            "mean"
        ),

        mean_nearest_neighbor_difference_A=(
            "nearest_neighbor_mean_difference_A",
            "mean"
        ),

        max_nearest_neighbor_difference_A=(
            "nearest_neighbor_max_difference_A",
            "max"
        ),

        mean_Ti_Ti_difference_A=(
            "Ti-Ti_mean_abs_difference_A",
            "mean"
        ),

        mean_Ti_Si_difference_A=(
            "Ti-Si_mean_abs_difference_A",
            "mean"
        ),

        mean_Si_Si_difference_A=(
            "Si-Si_mean_abs_difference_A",
            "mean"
        ),

        mean_pair_distance_difference_A=(
            "pair_distance_mean_difference_A",
            "mean"
        ),

        max_pair_distance_difference_A=(
            "pair_distance_max_difference_A",
            "max"
        )
    )
    .reset_index()
)


print("\n" + "=" * 100)
print("STRUCTURAL STABILITY SUMMARY")
print("=" * 100)

print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nTest completed.")

In [ ]:
# ============================================================
# MECHANICAL STABILITY — FIRST ELASTIC BENCHMARK
# Candidate 1: Ti4Si
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ase.io import read
from ase.filters import FrechetCellFilter
from mace.calculators import mace_mp


# ============================================================
# 1. LOAD CANDIDATE 1
# ============================================================

ARCHIVE_DIR = "/content/drive/MyDrive/Materials_Project/Top5_search_archive/generation_3_MACE_TOP5_archive"

STRUCTURES_PKL = os.path.join(
    ARCHIVE_DIR,
    "MACE_TOP5_candidate_to_structure.pkl"
)

with open(STRUCTURES_PKL, "rb") as f:
    top5_structures = pickle.load(f)

candidate_id = 81
candidate = top5_structures[candidate_id].copy()

print("=" * 70)
print("CANDIDATE 1 LOADED")
print("=" * 70)
print(f"Candidate ID : {candidate_id}")
print(f"Formula      : {candidate.get_chemical_formula()}")
print(f"Atoms        : {len(candidate)}")
print(f"Volume       : {candidate.get_volume():.6f} Å³")
print(f"PBC          : {candidate.get_pbc()}")


# ============================================================
# 2. MACE CALCULATOR
# ============================================================

calculator = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64"
)


# ============================================================
# 3. STRAIN BENCHMARK
# ============================================================

STRAINS = np.array([
    -0.02,
    -0.01,
     0.00,
     0.01,
     0.02
])

results = []

original_cell = candidate.get_cell().copy()

print("\n" + "=" * 70)
print("RUNNING STRAIN BENCHMARK")
print("=" * 70)

for strain in STRAINS:

    atoms = candidate.copy()
    atoms.calc = calculator

    # Uniaxial strain along x
    strained_cell = original_cell.copy()
    strained_cell[0] *= (1.0 + strain)

    atoms.set_cell(strained_cell, scale_atoms=True)

    energy = atoms.get_potential_energy()
    energy_per_atom = energy / len(atoms)

    stress = atoms.get_stress(voigt=True)

    results.append({
        "strain": strain,
        "strain_percent": strain * 100,
        "energy_eV": energy,
        "energy_eV_per_atom": energy_per_atom,
        "stress_xx_eV_per_A3": stress[0],
        "stress_yy_eV_per_A3": stress[1],
        "stress_zz_eV_per_A3": stress[2],
        "stress_xy_eV_per_A3": stress[3],
        "stress_yz_eV_per_A3": stress[4],
        "stress_xz_eV_per_A3": stress[5],
        "volume_A3": atoms.get_volume()
    })

    print(
        f"strain = {strain:+.3f} | "
        f"E/atom = {energy_per_atom:.8f} eV | "
        f"Sxx = {stress[0]:+.8e} eV/Å³"
    )


# ============================================================
# 4. RESULTS TABLE
# ============================================================

benchmark_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("BENCHMARK RESULTS")
print("=" * 70)

print(
    benchmark_df[
        [
            "strain_percent",
            "energy_eV_per_atom",
            "stress_xx_eV_per_A3",
            "volume_A3"
        ]
    ].to_string(index=False)
)


# ============================================================
# 5. ENERGY–STRAIN CURVE
# ============================================================

plt.figure(figsize=(7, 5))

plt.plot(
    benchmark_df["strain_percent"],
    benchmark_df["energy_eV_per_atom"],
    marker="o"
)

plt.xlabel("Strain (%)")
plt.ylabel("Energy per atom (eV)")
plt.title("Ti4Si — Energy vs Uniaxial Strain")

plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 6. BASIC QUADRATIC FIT
# ============================================================

x = benchmark_df["strain"].to_numpy()
y = benchmark_df["energy_eV_per_atom"].to_numpy()

coefficients = np.polyfit(x, y, 2)
quadratic_fit = np.polyval(coefficients, x)

ss_res = np.sum((y - quadratic_fit) ** 2)
ss_tot = np.sum((y - np.mean(y)) ** 2)

r_squared = 1.0 - ss_res / ss_tot

print("\n" + "=" * 70)
print("QUADRATIC FIT")
print("=" * 70)

print(f"Quadratic coefficient : {coefficients[0]:.8e}")
print(f"Linear coefficient    : {coefficients[1]:.8e}")
print(f"Constant              : {coefficients[2]:.8e}")
print(f"R²                    : {r_squared:.8f}")

print("\nBenchmark complete.")

In [ ]:
# ============================================================
# MECHANICAL STABILITY — FULL ELASTIC CONSTANTS
# Candidate 1: Ti4Si
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd

from mace.calculators import mace_mp


# ============================================================
# 1. LOAD CANDIDATE 1 — ID 81
# ============================================================

ARCHIVE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

STRUCTURES_PKL = os.path.join(
    ARCHIVE_DIR,
    "MACE_TOP5_candidate_to_structure.pkl"
)

with open(STRUCTURES_PKL, "rb") as f:
    top5_structures = pickle.load(f)

candidate_id = 81
candidate = top5_structures[candidate_id].copy()

print("=" * 70)
print("CANDIDATE 1")
print("=" * 70)

print(f"Candidate ID : {candidate_id}")
print(f"Formula      : {candidate.get_chemical_formula()}")
print(f"Atoms        : {len(candidate)}")
print(f"Volume       : {candidate.get_volume():.6f} Å³")
print(f"PBC          : {candidate.get_pbc()}")


# ============================================================
# 2. MACE CALCULATOR
# ============================================================

calculator = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64"
)


# ============================================================
# 3. STRAIN SETTINGS
# ============================================================

# Engineering strain amplitudes
STRAIN_AMPLITUDES = np.array([
    -0.015,
    -0.010,
    -0.005,
     0.005,
     0.010,
     0.015
])

# ASE stress/Voigt order:
# [xx, yy, zz, yz, xz, xy]

MODE_NAMES = [
    "xx",
    "yy",
    "zz",
    "yz",
    "xz",
    "xy"
]

original_cell = candidate.get_cell().copy()


# ============================================================
# 4. FUNCTION: BUILD STRAIN TENSOR
# ============================================================

def make_strain_tensor(mode, amplitude):

    strain = np.zeros((3, 3))

    if mode == 0:
        # epsilon_xx
        strain[0, 0] = amplitude

    elif mode == 1:
        # epsilon_yy
        strain[1, 1] = amplitude

    elif mode == 2:
        # epsilon_zz
        strain[2, 2] = amplitude

    elif mode == 3:
        # engineering shear gamma_yz
        strain[1, 2] = amplitude / 2
        strain[2, 1] = amplitude / 2

    elif mode == 4:
        # engineering shear gamma_xz
        strain[0, 2] = amplitude / 2
        strain[2, 0] = amplitude / 2

    elif mode == 5:
        # engineering shear gamma_xy
        strain[0, 1] = amplitude / 2
        strain[1, 0] = amplitude / 2

    return strain


# ============================================================
# 5. RUN ALL STRAIN TESTS
# ============================================================

results = []

print("\n" + "=" * 70)
print("RUNNING ELASTIC STRAIN TESTS")
print("=" * 70)

for mode, mode_name in enumerate(MODE_NAMES):

    print(f"\n--- Strain mode: {mode_name} ---")

    for amplitude in STRAIN_AMPLITUDES:

        atoms = candidate.copy()

        atoms.calc = calculator

        # Deformation gradient
        F = np.eye(3) + make_strain_tensor(
            mode,
            amplitude
        )

        # Apply deformation to the cell
        # ASE stores cell vectors as rows
        strained_cell = original_cell @ F.T

        atoms.set_cell(
            strained_cell,
            scale_atoms=True
        )

        # Energy
        energy = atoms.get_potential_energy()

        # Stress
        stress = atoms.get_stress(voigt=True)

        results.append({
            "mode": mode_name,
            "mode_index": mode,
            "strain": amplitude,
            "strain_percent": amplitude * 100,

            "energy_eV_per_atom":
                energy / len(atoms),

            "stress_xx":
                stress[0],

            "stress_yy":
                stress[1],

            "stress_zz":
                stress[2],

            "stress_yz":
                stress[3],

            "stress_xz":
                stress[4],

            "stress_xy":
                stress[5],

            "volume_A3":
                atoms.get_volume()
        })

        print(
            f"{mode_name:>2} | "
            f"strain = {amplitude:+.3f} | "
            f"E/atom = {energy/len(atoms):.8f} eV"
        )


# ============================================================
# 6. RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("ALL STRAIN TESTS COMPLETE")
print("=" * 70)

print(results_df.head())


# ============================================================
# 7. EXTRACT ELASTIC STIFFNESS MATRIX
# ============================================================

stress_columns = [
    "stress_xx",
    "stress_yy",
    "stress_zz",
    "stress_yz",
    "stress_xz",
    "stress_xy"
]

C = np.zeros((6, 6))

for mode_index, mode_name in enumerate(MODE_NAMES):

    subset = results_df[
        results_df["mode_index"] == mode_index
    ].copy()

    x = subset["strain"].to_numpy()

    for stress_index, stress_column in enumerate(stress_columns):

        y = subset[stress_column].to_numpy()

        # Linear fit:
        # stress = intercept + C * strain
        coefficients = np.polyfit(x, y, 1)

        slope = coefficients[0]

        C[stress_index, mode_index] = slope


# ============================================================
# 8. SYMMETRIZE THE ELASTIC MATRIX
# ============================================================

C_symmetric = 0.5 * (C + C.T)


# ============================================================
# 9. CONVERT eV/A^3 → GPa
# ============================================================

EV_A3_TO_GPA = 160.21766208

C_GPa = C_symmetric * EV_A3_TO_GPA


# ============================================================
# 10. PRINT ELASTIC CONSTANTS
# ============================================================

print("\n" + "=" * 70)
print("ELASTIC STIFFNESS MATRIX — GPa")
print("=" * 70)

C_df = pd.DataFrame(
    C_GPa,
    index=[
        "xx",
        "yy",
        "zz",
        "yz",
        "xz",
        "xy"
    ],
    columns=[
        "xx",
        "yy",
        "zz",
        "yz",
        "xz",
        "xy"
    ]
)

print(C_df.round(3))


# ============================================================
# 11. MECHANICAL STABILITY
# ============================================================

eigenvalues = np.linalg.eigvalsh(C_symmetric)

eigenvalues_GPa = eigenvalues * EV_A3_TO_GPA

print("\n" + "=" * 70)
print("MECHANICAL STABILITY")
print("=" * 70)

print("Elastic stiffness eigenvalues (GPa):")

for i, value in enumerate(eigenvalues_GPa):

    print(
        f"Mode {i+1}: "
        f"{value:.6f} GPa"
    )


# Positive-definite stiffness matrix = mechanically stable
mechanically_stable = np.all(eigenvalues_GPa > 0)

print("\nMechanical stability:", mechanically_stable)


if mechanically_stable:
    print("RESULT: PASS — stiffness matrix is positive definite.")
else:
    print("RESULT: FAIL — at least one elastic mode is unstable.")


# ============================================================
# 12. SAVE RESULTS
# ============================================================

OUTPUT_CSV = os.path.join(
    ARCHIVE_DIR,
    "candidate_81_elastic_strain_results.csv"
)

OUTPUT_MATRIX = os.path.join(
    ARCHIVE_DIR,
    "candidate_81_elastic_constants_GPa.csv"
)

results_df.to_csv(
    OUTPUT_CSV,
    index=False
)

C_df.to_csv(
    OUTPUT_MATRIX
)

print("\nSaved:")
print(OUTPUT_CSV)
print(OUTPUT_MATRIX)

In [ ]:
# ============================================================
# MECHANICAL STABILITY — ELASTIC CONSTANT ROBUSTNESS CHECK
# Candidate 1: Ti4Si
# ============================================================

import os
import pickle
import numpy as np
import pandas as pd

from mace.calculators import mace_mp


# ============================================================
# 1. LOAD CANDIDATE 1
# ============================================================

ARCHIVE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

STRUCTURES_PKL = os.path.join(
    ARCHIVE_DIR,
    "MACE_TOP5_candidate_to_structure.pkl"
)

with open(STRUCTURES_PKL, "rb") as f:
    top5_structures = pickle.load(f)

candidate_id = 81
candidate = top5_structures[candidate_id].copy()

print("=" * 70)
print("ELASTIC CONSTANT ROBUSTNESS CHECK")
print("=" * 70)

print(f"Candidate ID : {candidate_id}")
print(f"Formula      : {candidate.get_chemical_formula()}")
print(f"Atoms        : {len(candidate)}")
print(f"Volume       : {candidate.get_volume():.6f} Å³")


# ============================================================
# 2. MACE
# ============================================================

calculator = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64"
)


# ============================================================
# 3. STRAIN SETTINGS
# ============================================================

STRAINS = np.array([
    -0.015,
    -0.010,
    -0.005,
     0.000,
     0.005,
     0.010,
     0.015
])

MODE_NAMES = [
    "xx",
    "yy",
    "zz",
    "yz",
    "xz",
    "xy"
]

STRESS_COLUMNS = [
    "stress_xx",
    "stress_yy",
    "stress_zz",
    "stress_yz",
    "stress_xz",
    "stress_xy"
]

original_cell = candidate.get_cell().copy()


# ============================================================
# 4. STRAIN TENSOR
# ============================================================

def make_strain_tensor(mode, amplitude):

    strain = np.zeros((3, 3))

    if mode == 0:
        strain[0, 0] = amplitude

    elif mode == 1:
        strain[1, 1] = amplitude

    elif mode == 2:
        strain[2, 2] = amplitude

    elif mode == 3:
        # Engineering shear gamma_yz
        strain[1, 2] = amplitude / 2
        strain[2, 1] = amplitude / 2

    elif mode == 4:
        # Engineering shear gamma_xz
        strain[0, 2] = amplitude / 2
        strain[2, 0] = amplitude / 2

    elif mode == 5:
        # Engineering shear gamma_xy
        strain[0, 1] = amplitude / 2
        strain[1, 0] = amplitude / 2

    return strain


# ============================================================
# 5. RUN STRAIN CALCULATIONS
# ============================================================

results = []

print("\n" + "=" * 70)
print("RUNNING STRAIN CALCULATIONS")
print("=" * 70)

for mode_index, mode_name in enumerate(MODE_NAMES):

    print(f"\n--- {mode_name} ---")

    for amplitude in STRAINS:

        atoms = candidate.copy()
        atoms.calc = calculator

        F = np.eye(3) + make_strain_tensor(
            mode_index,
            amplitude
        )

        strained_cell = original_cell @ F.T

        atoms.set_cell(
            strained_cell,
            scale_atoms=True
        )

        energy = atoms.get_potential_energy()
        stress = atoms.get_stress(voigt=True)

        results.append({
            "mode": mode_name,
            "mode_index": mode_index,
            "strain": amplitude,
            "strain_percent": amplitude * 100,

            "energy_eV_per_atom":
                energy / len(atoms),

            "stress_xx": stress[0],
            "stress_yy": stress[1],
            "stress_zz": stress[2],
            "stress_yz": stress[3],
            "stress_xz": stress[4],
            "stress_xy": stress[5],

            "volume_A3": atoms.get_volume()
        })

        print(
            f"strain = {amplitude:+.3f} | "
            f"E/atom = {energy/len(atoms):.8f} eV"
        )


results_df = pd.DataFrame(results)


# ============================================================
# 6. CALCULATE C MATRIX
# ============================================================

def calculate_C(data):

    C = np.zeros((6, 6))
    fit_r2 = np.zeros((6, 6))

    for mode_index in range(6):

        subset = data[
            data["mode_index"] == mode_index
        ]

        x = subset["strain"].to_numpy()

        for stress_index, stress_column in enumerate(STRESS_COLUMNS):

            y = subset[stress_column].to_numpy()

            coefficients = np.polyfit(
                x,
                y,
                1
            )

            slope = coefficients[0]

            predicted = np.polyval(
                coefficients,
                x
            )

            ss_res = np.sum(
                (y - predicted) ** 2
            )

            ss_tot = np.sum(
                (y - np.mean(y)) ** 2
            )

            if ss_tot > 0:
                r2 = 1.0 - ss_res / ss_tot
            else:
                r2 = np.nan

            C[stress_index, mode_index] = slope
            fit_r2[stress_index, mode_index] = r2

    return C, fit_r2


# ============================================================
# 7. CALCULATE C FOR DIFFERENT STRAIN WINDOWS
# ============================================================

WINDOWS = {
    "±0.5%": 0.005,
    "±1.0%": 0.010,
    "±1.5%": 0.015
}

C_results = {}
R2_results = {}

for window_name, max_strain in WINDOWS.items():

    subset = results_df[
        np.abs(results_df["strain"]) <= max_strain + 1e-12
    ].copy()

    C_matrix, R2_matrix = calculate_C(subset)

    C_results[window_name] = C_matrix
    R2_results[window_name] = R2_matrix


# ============================================================
# 8. CONVERT TO GPa
# ============================================================

EV_A3_TO_GPA = 160.21766208

C_GPa = {}

for window_name in WINDOWS:

    C_GPa[window_name] = (
        0.5 *
        (
            C_results[window_name]
            +
            C_results[window_name].T
        )
        *
        EV_A3_TO_GPA
    )


# ============================================================
# 9. PRINT MATRICES
# ============================================================

labels = [
    "xx",
    "yy",
    "zz",
    "yz",
    "xz",
    "xy"
]

for window_name in WINDOWS:

    print("\n" + "=" * 70)
    print(f"ELASTIC MATRIX — {window_name}")
    print("=" * 70)

    matrix_df = pd.DataFrame(
        C_GPa[window_name],
        index=labels,
        columns=labels
    )

    print(matrix_df.round(3))


# ============================================================
# 10. COMPARE DIFFERENT STRAIN WINDOWS
# ============================================================

print("\n" + "=" * 70)
print("ROBUSTNESS OF ELASTIC CONSTANTS")
print("=" * 70)

C_reference = C_GPa["±1.0%"]

for window_name in WINDOWS:

    difference = (
        C_GPa[window_name]
        - C_reference
    )

    max_difference = np.max(
        np.abs(difference)
    )

    print(
        f"{window_name:>7} vs ±1.0% | "
        f"maximum difference = "
        f"{max_difference:.4f} GPa"
    )


# ============================================================
# 11. R² CHECK
# ============================================================

print("\n" + "=" * 70)
print("LINEAR STRESS–STRAIN FIT QUALITY")
print("=" * 70)

for window_name in WINDOWS:

    R2 = R2_results[window_name]

    print(f"\n{window_name}")

    for mode_index, mode_name in enumerate(MODE_NAMES):

        # Direct response of the same stress component
        r2 = R2[
            mode_index,
            mode_index
        ]

        print(
            f"{mode_name:>2} : "
            f"R² = {r2:.8f}"
        )


# ============================================================
# 12. MECHANICAL STABILITY FOR EACH WINDOW
# ============================================================

print("\n" + "=" * 70)
print("MECHANICAL STABILITY")
print("=" * 70)

for window_name in WINDOWS:

    matrix_eVA3 = C_GPa[window_name] / EV_A3_TO_GPA

    eigenvalues = np.linalg.eigvalsh(
        matrix_eVA3
    )

    eigenvalues_GPa = (
        eigenvalues *
        EV_A3_TO_GPA
    )

    minimum_eigenvalue = np.min(
        eigenvalues_GPa
    )

    stable = np.all(
        eigenvalues_GPa > 0
    )

    print(f"\n{window_name}")
    print(
        "Eigenvalues (GPa):",
        np.round(eigenvalues_GPa, 6)
    )

    print(
        f"Minimum eigenvalue : "
        f"{minimum_eigenvalue:.6f} GPa"
    )

    print(
        f"Stable              : "
        f"{stable}"
    )


# ============================================================
# 13. SYMMETRY CHECK BEFORE SYMMETRIZATION
# ============================================================

print("\n" + "=" * 70)
print("MATRIX SYMMETRY CHECK")
print("=" * 70)

for window_name in WINDOWS:

    raw_matrix = (
        C_results[window_name]
        * EV_A3_TO_GPA
    )

    asymmetry = np.max(
        np.abs(
            raw_matrix - raw_matrix.T
        )
    )

    print(
        f"{window_name:>7} | "
        f"max |Cij - Cji| = "
        f"{asymmetry:.6f} GPa"
    )


# ============================================================
# 14. SAVE RESULTS
# ============================================================

RESULTS_CSV = os.path.join(
    ARCHIVE_DIR,
    "candidate_81_elastic_robustness_results.csv"
)

MATRICES_CSV = os.path.join(
    ARCHIVE_DIR,
    "candidate_81_elastic_robustness_matrices.csv"
)

results_df.to_csv(
    RESULTS_CSV,
    index=False
)

matrix_rows = []

for window_name in WINDOWS:

    matrix = C_GPa[window_name]

    for i in range(6):
        for j in range(6):

            matrix_rows.append({
                "strain_window": window_name,
                "row": labels[i],
                "column": labels[j],
                "C_GPa": matrix[i, j]
            })

pd.DataFrame(
    matrix_rows
).to_csv(
    MATRICES_CSV,
    index=False
)

print("\nSaved:")
print(RESULTS_CSV)
print(MATRICES_CSV)

print("\nRobustness check complete.")

In [ ]:
# ================================================================================
# CANDIDATE 81 — SAFE RESUME 2×2×2 THERMAL MD
# CPU VERSION
# ================================================================================

import os
import pickle
import random
import numpy as np
import torch

from ase import Atoms
from ase.units import fs
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

from mace.calculators import mace_mp


# ================================================================================
# SETTINGS
# ================================================================================

CANDIDATE_ID = 81

TEMPERATURES = [300, 600, 900, 1200]

TIMESTEP_FS = 0.5

EQUILIBRATION_STEPS = 4000
PRODUCTION_STEPS = 10000
TOTAL_STEPS = EQUILIBRATION_STEPS + PRODUCTION_STEPS

CHECKPOINT_INTERVAL = 500
TRAJECTORY_INTERVAL = 50

FRICTION = 0.01

SUPERCELL = (2, 2, 2)

BASE_SEED = 8100


# ================================================================================
# PATHS
# ================================================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "thermal_stability_candidate_81"
)

MD_DIR = os.path.join(
    BASE_DIR,
    "supercell_2x2x2_thermal_MD_checkpointed"
)

os.makedirs(MD_DIR, exist_ok=True)


# Original relaxed structure
STRUCTURE_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_300K.extxyz"
)


# ================================================================================
# HEADER
# ================================================================================

print("=" * 80)
print("CANDIDATE 81 — SAFE RESUME 2×2×2 THERMAL MD")
print("=" * 80)


# ================================================================================
# LOAD ORIGINAL STRUCTURE
# ================================================================================

atoms_original = read(
    STRUCTURE_FILE,
    index=0
)

print(f"Original atoms : {len(atoms_original)}")
print(f"Formula        : {atoms_original.get_chemical_formula()}")
print(
    f"Original volume: "
    f"{atoms_original.get_volume():.6f} Å³"
)


# ================================================================================
# CREATE 2×2×2 SUPERCELL
# ================================================================================

atoms_template = atoms_original.repeat(
    SUPERCELL
)

print(f"Supercell      : {SUPERCELL}")
print(f"Supercell atoms: {len(atoms_template)}")
print(
    f"Supercell vol. : "
    f"{atoms_template.get_volume():.6f} Å³"
)


# ================================================================================
# LOAD MACE — CPU
# ================================================================================

print("\n" + "=" * 80)
print("LOADING MACE")
print("=" * 80)

DEVICE = "cuda"

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64",
    device=DEVICE
)

print(f"Device : {DEVICE}")
print("MACE   : medium")
print("D3     : True")
print("dtype  : float64")


# ================================================================================
# FILE PATH HELPERS
# ================================================================================

# IMPORTANT:
# These names match the files actually present in your Drive.

def checkpoint_path(T):
    return os.path.join(
        MD_DIR,
        f"candidate_81_{T}K_checkpoint.pkl"
    )


def trajectory_path(T):
    return os.path.join(
        MD_DIR,
        f"candidate_81_{T}K_2x2x2.extxyz"
    )


def complete_path(T):
    return os.path.join(
        MD_DIR,
        f"candidate_81_{T}K_COMPLETE.txt"
    )


# ================================================================================
# SAVE CHECKPOINT
# ================================================================================

def save_checkpoint(T, atoms, step, rng):

    data = {
        "step": int(step),
        "target_temperature": T,

        # Physical state
        "positions": atoms.get_positions().copy(),
        "momenta": atoms.get_momenta().copy(),

        # Cell / PBC
        "cell": atoms.get_cell().copy(),
        "pbc": atoms.get_pbc().copy(),

        # Chemical identity
        "symbols": atoms.get_chemical_symbols(),

        # Global RNG states
        "numpy_global_state": np.random.get_state(),
        "python_random_state": random.getstate(),
        "torch_rng_state": torch.get_rng_state(),

        # CUDA RNG, if ever available
        "torch_cuda_rng_state_all": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else None
        ),

        # Dedicated Langevin RNG
        "langevin_rng_state": rng.bit_generator.state,
    }

    final_path = checkpoint_path(T)
    temp_path = final_path + ".tmp"

    with open(temp_path, "wb") as f:
        pickle.dump(data, f)

    # Replace checkpoint atomically
    os.replace(
        temp_path,
        final_path
    )


# ================================================================================
# LOAD CHECKPOINT
# ================================================================================

def load_checkpoint(T):

    path = checkpoint_path(T)

    if not os.path.exists(path):
        return None

    with open(path, "rb") as f:
        return pickle.load(f)


# ================================================================================
# RESTORE RNG STATES — SAFE VERSION
# ================================================================================

def restore_rng_states(data, rng):

    if "numpy_global_state" in data:

        np.random.set_state(
            data["numpy_global_state"]
        )


    if "python_random_state" in data:

        random.setstate(
            data["python_random_state"]
        )


    if "torch_rng_state" in data:

        torch.set_rng_state(
            data["torch_rng_state"]
        )


    if (
        torch.cuda.is_available()
        and data.get("torch_cuda_rng_state_all") is not None
    ):

        torch.cuda.set_rng_state_all(
            data["torch_cuda_rng_state_all"]
        )


    # ============================================================================
    # DEDICATED LANGEVIN RNG
    # ============================================================================

    langevin_state = data.get(
        "langevin_rng_state",
        None
    )


    if isinstance(langevin_state, dict):

        try:

            rng.bit_generator.state = (
                langevin_state
            )

            return True

        except Exception as e:

            print(
                "WARNING: Langevin RNG state "
                "could not be restored."
            )

            print(
                f"Reason: {e}"
            )

            return False


    else:

        print(
            "WARNING: Langevin RNG state in "
            "checkpoint is not a compatible dict."
        )

        print(
            f"Stored type: "
            f"{type(langevin_state)}"
        )

        print(
            "Physical state will still be resumed."
        )

        return False

# ================================================================================
# APPEND TRAJECTORY FRAME
# ================================================================================

def append_trajectory_frame(
    atoms,
    T,
    global_step,
    phase
):

    frame = atoms.copy()

    frame.info["target_temperature_K"] = int(T)
    frame.info["md_step"] = int(global_step)
    frame.info["phase"] = phase

    write(
        trajectory_path(T),
        frame,
        format="extxyz",
        append=True
    )

# ================================================================================
# RUN ONE TEMPERATURE
# ================================================================================

def run_temperature(T):

    print("\n" + "=" * 80)
    print(f"CHECKING TEMPERATURE = {T} K")
    print("=" * 80)

    # ============================================================================
    # SAFETY CHECK #1 — COMPLETE FILE
    # ============================================================================

    if os.path.exists(complete_path(T)):

        print(
            f"{T} K is already COMPLETE."
        )

        print(
            "NO MD WILL BE RUN."
        )

        return


    # ============================================================================
    # LOAD CHECKPOINT
    # ============================================================================

    data = load_checkpoint(T)


    # ============================================================================
    # CHECKPOINT EXISTS
    # ============================================================================

    if data is not None:

        saved_step = int(
            data["step"]
        )

        print(
            f"Checkpoint found for {T} K."
        )

        print(
            f"Saved progress : "
            f"{saved_step}/{TOTAL_STEPS}"
        )


        # ========================================================================
        # SAFETY CHECK #2 — CHECKPOINT ALREADY COMPLETE
        # ========================================================================

        if saved_step >= TOTAL_STEPS:

            print(
                f"{T} K checkpoint already reached "
                f"{TOTAL_STEPS} steps."
            )

            print(
                "Creating COMPLETE marker."
            )

            with open(
                complete_path(T),
                "w"
            ) as f:

                f.write(
                    f"Candidate 81\n"
                    f"Temperature: {T} K\n"
                    f"Status: COMPLETE\n"
                    f"Steps: {saved_step}/{TOTAL_STEPS}\n"
                )

            print(
                f"{T} K marked COMPLETE."
            )

            return


        # ========================================================================
        # RECONSTRUCT ATOMS FROM CHECKPOINT
        # ========================================================================

        # Use the known 2×2×2 structure as the template.
        # The checkpoint provides the saved physical state.

        atoms = atoms_template.copy()

        atoms.set_positions(
            np.asarray(
                data["positions"]
            )
        )

        atoms.set_momenta(
            np.asarray(
                data["momenta"]
            )
        )

        if "cell" in data:

            atoms.set_cell(
                data["cell"]
            )

        if "pbc" in data:

            atoms.set_pbc(
                data["pbc"]
            )

        atoms.calc = calc


        # ========================================================================
        # RESTORE RANDOM STATE
        # ========================================================================

        seed = BASE_SEED + T

        rng = np.random.default_rng(
            seed
        )

        exact_rng_restored = restore_rng_states(
            data,
            rng
        )

        if exact_rng_restored:

            print(
                "Dedicated Langevin RNG state restored."
            )

        else:

            print(
                "WARNING: checkpoint has no "
                "dedicated Langevin RNG state."
            )


        current_step = saved_step

        print(
            f"RESUMING {T} K FROM STEP "
            f"{current_step}/{TOTAL_STEPS}"
        )


    # ============================================================================
    # NO CHECKPOINT → START NEW
    # ============================================================================

    else:

        print(
            f"No checkpoint found for {T} K."
        )

        print(
            f"STARTING NEW THERMAL MD — {T} K"
        )


        # ------------------------------------------------------------------------
        # CREATE FRESH 2×2×2 SYSTEM
        # ------------------------------------------------------------------------

        atoms = atoms_template.copy()

        atoms.calc = calc

        current_step = 0


        # ------------------------------------------------------------------------
        # DETERMINISTIC SEED
        # ------------------------------------------------------------------------

        seed = BASE_SEED + T

        np.random.seed(
            seed
        )

        random.seed(
            seed
        )

        torch.manual_seed(
            seed
        )

        if torch.cuda.is_available():

            torch.cuda.manual_seed_all(
                seed
            )

        rng = np.random.default_rng(
            seed
        )


        # ------------------------------------------------------------------------
        # INITIAL VELOCITIES
        # ------------------------------------------------------------------------

        MaxwellBoltzmannDistribution(
            atoms,
            temperature_K=T
        )


        # ------------------------------------------------------------------------
        # INITIAL CHECKPOINT
        # ------------------------------------------------------------------------

        save_checkpoint(
            T,
            atoms,
            0,
            rng
        )

        print(
            "Initial checkpoint saved."
        )


    # =============================================================================
    # MAIN MD LOOP
    # =============================================================================

    while current_step < TOTAL_STEPS:

        remaining = (
            TOTAL_STEPS - current_step
        )

        chunk = min(
            CHECKPOINT_INTERVAL,
            remaining
        )

        segment_start = current_step


        # =========================================================================
        # PHASE
        # =========================================================================

        if current_step < EQUILIBRATION_STEPS:

            phase = "equilibration"

        else:

            phase = "production"


        # =========================================================================
        # LANGEVIN
        # =========================================================================

        dyn = Langevin(
            atoms,
            timestep=TIMESTEP_FS * fs,
            temperature_K=T,
            friction=FRICTION,
            fixcm=False,
            rng=rng
        )
        # =========================================================================
        # TRAJECTORY CALLBACK
        # =========================================================================

        def save_frame():

            local_step = dyn.nsteps

            global_step = (
                segment_start + local_step
            )

            if global_step > segment_start:

                current_phase = (
                    "equilibration"
                    if global_step <= EQUILIBRATION_STEPS
                    else "production"
                )

                append_trajectory_frame(
                    atoms,
                    T,
                    global_step,
                    current_phase
                )


        dyn.attach(
            save_frame,
            interval=TRAJECTORY_INTERVAL
        )


        # =========================================================================
        # RUN 500 STEPS
        # =========================================================================

        dyn.run(chunk)

        current_step += chunk


        # =========================================================================
        # SAVE CHECKPOINT
        # =========================================================================

        save_checkpoint(
            T,
            atoms,
            current_step,
            rng
        )


        print(
            f"Progress: "
            f"{current_step}/{TOTAL_STEPS} "
            f"({100 * current_step / TOTAL_STEPS:.1f}%)"
        )


    # =============================================================================
    # MARK COMPLETE
    # =============================================================================

    with open(
        complete_path(T),
        "w"
    ) as f:

        f.write(
            f"Candidate 81\n"
            f"Temperature: {T} K\n"
            f"Status: COMPLETE\n"
            f"Total steps: {TOTAL_STEPS}\n"
        )


    print(
        f"\n{T} K COMPLETE — "
        f"{TOTAL_STEPS}/{TOTAL_STEPS}"
    )


# ================================================================================
# RUN ALL TEMPERATURES
# ================================================================================

for T in TEMPERATURES:

    run_temperature(T)


# ================================================================================
# FINAL STATUS
# ================================================================================

print("\n" + "=" * 80)
print("FINAL THERMAL MD STATUS")
print("=" * 80)

for T in TEMPERATURES:

    if os.path.exists(
        complete_path(T)
    ):

        print(
            f"{T} K : COMPLETE"
        )

    else:

        data = load_checkpoint(T)

        if data is not None:

            print(
                f"{T} K : "
                f"{data['step']}/{TOTAL_STEPS}"
            )

        else:

            print(
                f"{T} K : NOT STARTED"
            )

print("=" * 80)
print("SAFE RESUME FINISHED")
print("=" * 80)

In [ ]:
# ================================================================
# CANDIDATE 81 — FINAL FAIR THERMAL ANALYSIS + ENERGY
#
# NO NEW MD
#
# Strategy:
#   1. Read the ORIGINAL trajectories.
#   2. Use production frames only (MD step >= 4000).
#   3. Select exactly 21 frames uniformly across the AVAILABLE
#      production trajectory of EACH temperature.
#   4. Calculate MACE + D3 potential energy on those SAME selected
#      structures.
#   5. Save energy calculations with checkpointing.
#   6. Produce final thermal + structural + energy summary.
#
# Important:
#   1200 K starts at ~6550 because earlier frames were not saved.
#   We DO NOT invent missing 4000–6500 structures.
# ================================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ase.io import read, write
from ase.geometry import find_mic

# ================================================================
# 1. PATHS
# ================================================================

ROOT_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

MD_DIR = os.path.join(
    ROOT_DIR,
    "thermal_stability_candidate_81",
    "supercell_2x2x2_thermal_MD_checkpointed"
)

FINAL_DIR = os.path.join(
    ROOT_DIR,
    "thermal_analysis_FINAL"
)

os.makedirs(
    FINAL_DIR,
    exist_ok=True
)


# ================================================================
# 2. SETTINGS
# ================================================================

TEMPERATURES = [300, 600, 900, 1200]

EQUIL_STEPS = 4000

TOTAL_STEPS = 14000

TIMESTEP_FS = 0.5

N_ATOMS = 40

# Number of production frames used for the FINAL comparison.
N_SELECTED = 21


# ================================================================
# 3. ORIGINAL TRAJECTORIES
# ================================================================

TRAJECTORIES = {

    300: os.path.join(
        MD_DIR,
        "candidate_81_300K_2x2x2.extxyz"
    ),

    600: os.path.join(
        MD_DIR,
        "candidate_81_600K_2x2x2.extxyz"
    ),

    900: os.path.join(
        MD_DIR,
        "candidate_81_900K_2x2x2.extxyz"
    ),

    1200: os.path.join(
        MD_DIR,
        "candidate_81_1200K_2x2x2.extxyz"
    ),
}


# ================================================================
# 4. GET MD STEP
# ================================================================

def get_md_step(atoms, frame_index, n_frames):

    possible_keys = [
        "md_step",
        "MD_step",
        "step",
        "Step",
        "simulation_step",
    ]

    for key in possible_keys:

        if key in atoms.info:

            try:
                return int(atoms.info[key])
            except:
                pass

    # ------------------------------------------------------------
    # 300/600/900 old trajectories:
    # 29 frames = every 500 steps
    # ------------------------------------------------------------

    if n_frames == 29:

        return frame_index * 500

    # ------------------------------------------------------------
    # If metadata is absent from another trajectory,
    # distribute frames across the known MD interval.
    # ------------------------------------------------------------

    if n_frames > 1:

        return int(round(
            frame_index
            * TOTAL_STEPS
            / (n_frames - 1)
        ))

    return 0


# ================================================================
# 5. TEMPERATURE
# ================================================================

def get_temperature(atoms):

    try:

        value = atoms.get_temperature()

        if np.isfinite(value):

            return float(value)

    except:

        pass

    return np.nan


# ================================================================
# 6. MINIMUM PBC DISTANCE
# ================================================================

def minimum_pbc_distance(atoms):

    positions = atoms.get_positions()

    cell = atoms.get_cell()

    pbc = atoms.get_pbc()

    minimum = np.inf

    for i in range(len(atoms) - 1):

        displacement = (
            positions[i + 1:]
            - positions[i]
        )

        try:

            _, distances = find_mic(
                displacement,
                cell=cell,
                pbc=pbc
            )

        except:

            continue

        distances = np.asarray(
            distances
        )

        if distances.size == 0:

            continue

        local_minimum = np.min(
            distances
        )

        minimum = min(
            minimum,
            local_minimum
        )

    if np.isfinite(minimum):

        return float(minimum)

    return np.nan


# ================================================================
# 7. LOAD ORIGINAL TRAJECTORY
# ================================================================

def load_trajectory(T):

    path = TRAJECTORIES[T]

    print()
    print("=" * 80)
    print(f"LOADING {T} K")
    print("=" * 80)

    print(path)

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"Trajectory not found:\n{path}"
        )

    frames = read(
        path,
        index=":"
    )

    print(
        f"Total frames : {len(frames)}"
    )

    return frames


# ================================================================
# 8. BUILD DATAFRAME FOR ALL FRAMES
# ================================================================

def build_frame_dataframe(frames):

    n_frames = len(frames)

    records = []

    for i, atoms in enumerate(frames):

        step = get_md_step(
            atoms,
            i,
            n_frames
        )

        records.append({

            "original_frame": i,

            "md_step": step,

            "time_ps":
                step
                * TIMESTEP_FS
                / 1000.0,

            "temperature_K":
                get_temperature(atoms),

            "min_pbc_distance_A":
                minimum_pbc_distance(atoms),

        })

    df = pd.DataFrame(records)

    df = df.sort_values(
        "md_step"
    ).reset_index(
        drop=True
    )

    return df


# ================================================================
# 9. FAIR PRODUCTION SAMPLING
# ================================================================

def select_uniform_production_frames(
    frames,
    df,
    T
):

    production_mask = (
        df["md_step"] >= EQUIL_STEPS
    )

    production_df = df[
        production_mask
    ].copy()

    if len(production_df) == 0:

        raise RuntimeError(
            f"No production frames for {T} K."
        )

    # ------------------------------------------------------------
    # Select exactly N_SELECTED frames uniformly over the
    # AVAILABLE production trajectory.
    #
    # This is NOT claiming that the temperatures have identical
    # absolute time coverage.
    # It only makes the statistical sampling density comparable.
    # ------------------------------------------------------------

    if len(production_df) <= N_SELECTED:

        selected_df = production_df.copy()

    else:

        selected_positions = np.linspace(
            0,
            len(production_df) - 1,
            N_SELECTED
        )

        selected_positions = np.rint(
            selected_positions
        ).astype(int)

        selected_df = production_df.iloc[
            selected_positions
        ].copy()

    selected_df = selected_df.drop_duplicates(
        subset=["md_step"]
    ).reset_index(
        drop=True
    )

    selected_frames = [

        frames[int(original_index)].copy()

        for original_index in
        selected_df["original_frame"]

    ]

    # Explicit metadata for downstream use.
    for atoms, step in zip(
        selected_frames,
        selected_df["md_step"]
    ):

        atoms.info["md_step"] = int(step)

    print()
    print(
        f"{T} K AVAILABLE PRODUCTION:"
    )

    print(
        f"  Frames : {len(production_df)}"
    )

    print(
        f"  Range  : "
        f"{production_df['md_step'].iloc[0]} → "
        f"{production_df['md_step'].iloc[-1]} steps"
    )

    print()
    print(
        f"{T} K SELECTED:"
    )

    print(
        f"  Frames : {len(selected_df)}"
    )

    print(
        f"  Range  : "
        f"{selected_df['md_step'].iloc[0]} → "
        f"{selected_df['md_step'].iloc[-1]} steps"
    )

    print(
        "  Steps  :"
    )

    print(
        selected_df["md_step"].to_list()
    )

    return (
        selected_df,
        selected_frames
    )


# ================================================================
# 10. SAVE SELECTED STRUCTURES
# ================================================================

def save_selected_trajectory(
    T,
    selected_frames
):

    path = os.path.join(
        FINAL_DIR,
        f"candidate_81_{T}K_FINAL_selected.extxyz"
    )

    write(
        path,
        selected_frames,
        format="extxyz"
    )

    print()
    print(
        f"Selected structures saved:"
    )

    print(path)

    return path


# ================================================================
# 11. LOAD MACE + D3
# ================================================================

print()
print("=" * 80)
print("LOADING MACE FOR SINGLE-POINT ENERGY CALCULATIONS")
print("=" * 80)

from mace.calculators import mace_mp

device = "cuda"

try:

    import torch

    if not torch.cuda.is_available():

        device = "cpu"

except:

    device = "cpu"


print(
    f"Device : {device}"
)

calculator = mace_mp(

    model="medium",

    dispersion=True,

    default_dtype="float64",

    device=device
)

print(
    "MACE   : medium"
)

print(
    "D3     : True"
)

print(
    "dtype  : float64"
)


# ================================================================
# 12. ENERGY CHECKPOINT
# ================================================================

ENERGY_CHECKPOINT = os.path.join(
    FINAL_DIR,
    "candidate_81_energy_checkpoint.csv"
)


# ================================================================
# LOAD PREVIOUS ENERGY CHECKPOINT
# ================================================================

if os.path.exists(
    ENERGY_CHECKPOINT
):

    energy_df = pd.read_csv(
        ENERGY_CHECKPOINT
    )

    print()
    print(
        "✓ Existing energy checkpoint found."
    )

    print(
        f"Already calculated : "
        f"{len(energy_df)} structures"
    )

else:

    energy_df = pd.DataFrame(
        columns=[
            "temperature_K",
            "selected_frame",
            "md_step",
            "potential_energy_eV",
        ]
    )


# ================================================================
# 13. CALCULATE ENERGY
# ================================================================

def calculate_energy(
    T,
    selected_df,
    selected_frames
):

    global energy_df

    print()
    print("=" * 80)
    print(
        f"ENERGY CALCULATION — {T} K"
    )
    print("=" * 80)

    for local_index, (
        atoms,
        (_, row)
    ) in enumerate(
        zip(
            selected_frames,
            selected_df.iterrows()
        )
    ):

        step = int(
            row["md_step"]
        )

        # --------------------------------------------------------
        # Check if this exact calculation already exists.
        # --------------------------------------------------------

        already_done = energy_df[
            (
                energy_df[
                    "temperature_K"
                ].astype(int)
                == int(T)
            )
            &
            (
                energy_df[
                    "md_step"
                ].astype(int)
                == step
            )
        ]

        if len(already_done) > 0:

            print(
                f"[{local_index + 1}/"
                f"{len(selected_frames)}] "
                f"{T} K — step {step} "
                f"already calculated"
            )

            continue

        # --------------------------------------------------------
        # Attach SAME MACE + D3 calculator.
        # --------------------------------------------------------

        atoms.calc = calculator

        try:

            energy = atoms.get_potential_energy()

        except Exception as e:

            print()
            print(
                f"❌ Energy calculation failed "
                f"at {T} K, step {step}"
            )

            print(e)

            raise

        if not np.isfinite(energy):

            raise RuntimeError(
                f"Non-finite energy at "
                f"{T} K, step {step}"
            )

        new_row = pd.DataFrame({

            "temperature_K": [T],

            "selected_frame": [
                local_index
            ],

            "md_step": [step],

            "potential_energy_eV": [
                float(energy)
            ],

        })

        energy_df = pd.concat(
            [
                energy_df,
                new_row
            ],
            ignore_index=True
        )

        # --------------------------------------------------------
        # SAVE AFTER EVERY FRAME.
        # --------------------------------------------------------

        energy_df.to_csv(
            ENERGY_CHECKPOINT,
            index=False
        )

        print(
            f"[{local_index + 1}/"
            f"{len(selected_frames)}] "
            f"{T} K — step {step} — "
            f"E = {energy:.8f} eV"
        )

    print()
    print(
        f"✓ {T} K energy calculations complete."
    )


# ================================================================
# 14. PROCESS ALL TEMPERATURES
# ================================================================

all_selected_data = {}

for T in TEMPERATURES:

    frames = load_trajectory(T)

    df = build_frame_dataframe(
        frames
    )

    selected_df, selected_frames = (
        select_uniform_production_frames(
            frames,
            df,
            T
        )
    )

    selected_path = save_selected_trajectory(
        T,
        selected_frames
    )

    # Save selected metadata.
    selected_csv = os.path.join(
        FINAL_DIR,
        f"candidate_81_{T}K_FINAL_selected.csv"
    )

    selected_df.to_csv(
        selected_csv,
        index=False
    )

    all_selected_data[T] = (
        selected_df,
        selected_frames
    )

    calculate_energy(
        T,
        selected_df,
        selected_frames
    )


# ================================================================
# 15. FINAL ENERGY DATA
# ================================================================

energy_df = energy_df.sort_values(
    [
        "temperature_K",
        "md_step"
    ]
).reset_index(
    drop=True
)

energy_df.to_csv(
    os.path.join(
        FINAL_DIR,
        "candidate_81_FINAL_energy_data.csv"
    ),
    index=False
)


# ================================================================
# 16. FINAL ANALYSIS
# ================================================================

final_results = []


for T in TEMPERATURES:

    selected_df, selected_frames = (
        all_selected_data[T]
    )

    # ------------------------------------------------------------
    # Temperature
    # ------------------------------------------------------------

    temperatures = selected_df[
        "temperature_K"
    ].dropna().to_numpy()

    mean_temp = np.mean(
        temperatures
    )

    std_temp = (
        np.std(
            temperatures,
            ddof=1
        )
        if len(temperatures) > 1
        else 0.0
    )

    temp_error = (
        (mean_temp - T)
        / T
        * 100.0
    )

    # ------------------------------------------------------------
    # Temperature trend
    # ------------------------------------------------------------

    if len(temperatures) >= 2:

        slope, _ = np.polyfit(
            selected_df["time_ps"],
            selected_df["temperature_K"],
            1
        )

        temp_trend = slope

    else:

        temp_trend = np.nan

    # ------------------------------------------------------------
    # Expected temperature fluctuation
    # ------------------------------------------------------------

    dof = (
        3 * N_ATOMS
        - 3
    )

    expected_std = (
        T
        * np.sqrt(
            2.0 / dof
        )
    )

    std_ratio = (
        std_temp
        / expected_std
    )

    # ------------------------------------------------------------
    # PBC distance
    # ------------------------------------------------------------

    distances = selected_df[
        "min_pbc_distance_A"
    ].dropna().to_numpy()

    minimum_distance = np.min(
        distances
    )

    mean_distance = np.mean(
        distances
    )

    maximum_distance = np.max(
        distances
    )

    # ------------------------------------------------------------
    # Energy
    # ------------------------------------------------------------

    E = energy_df[
        energy_df["temperature_K"].astype(int)
        == int(T)
    ][
        "potential_energy_eV"
    ].dropna().to_numpy()

    if len(E) > 0:

        mean_energy = np.mean(E)

        std_energy = (
            np.std(
                E,
                ddof=1
            )
            if len(E) > 1
            else 0.0
        )

        energy_range = (
            np.max(E)
            - np.min(E)
        )

        energy_available = True

    else:

        mean_energy = np.nan

        std_energy = np.nan

        energy_range = np.nan

        energy_available = False

    # ------------------------------------------------------------
    # Thermal control
    # ------------------------------------------------------------

    if abs(temp_error) <= 5:

        thermal_control = "PASS"

    else:

        thermal_control = "CHECK"

    # ------------------------------------------------------------
    # Save result
    # ------------------------------------------------------------

    final_results.append({

        "Temperature_K": T,

        "Selected_frames":
            len(selected_df),

        "Available_production_frames":
            len(
                selected_df
            ),

        "Production_start_step":
            int(
                selected_df["md_step"].iloc[0]
            ),

        "Production_end_step":
            int(
                selected_df["md_step"].iloc[-1]
            ),

        "Mean_temperature_K":
            mean_temp,

        "Temperature_error_%":
            temp_error,

        "Temperature_std_K":
            std_temp,

        "Expected_std_K":
            expected_std,

        "Std_ratio":
            std_ratio,

        "Temperature_trend_K_per_ps":
            temp_trend,

        "Minimum_PBC_distance_A":
            minimum_distance,

        "Mean_minimum_distance_A":
            mean_distance,

        "Maximum_minimum_distance_A":
            maximum_distance,

        "Mean_potential_energy_eV":
            mean_energy,

        "Potential_energy_std_eV":
            std_energy,

        "Potential_energy_range_eV":
            energy_range,

        "Energy_available":
            energy_available,

        "Thermal_control":
            thermal_control,

    })


# ================================================================
# 17. FINAL SUMMARY
# ================================================================

summary = pd.DataFrame(
    final_results
)

final_summary_path = os.path.join(
    FINAL_DIR,
    "candidate_81_FINAL_THERMAL_STABILITY_SUMMARY.csv"
)

summary.to_csv(
    final_summary_path,
    index=False
)


# ================================================================
# 18. PRINT FINAL TABLE
# ================================================================

print()
print("=" * 120)
print(
    "FINAL THERMAL STABILITY SUMMARY"
)
print(
    "CANDIDATE 81 | SiTi4 | 2×2×2 | 40 ATOMS"
)
print(
    "21 UNIFORMLY SAMPLED PRODUCTION FRAMES PER TEMPERATURE"
)
print("=" * 120)

display(summary)


# ================================================================
# 19. PLOT — TEMPERATURE
# ================================================================

for T in TEMPERATURES:

    selected_df, _ = (
        all_selected_data[T]
    )

    plt.figure(
        figsize=(9, 5)
    )

    plt.plot(
        selected_df["time_ps"],
        selected_df["temperature_K"],
        marker="o",
        linewidth=1.2,
        markersize=4
    )

    plt.axhline(
        T,
        linestyle="--",
        linewidth=1.5
    )

    plt.xlabel(
        "MD time (ps)"
    )

    plt.ylabel(
        "Temperature (K)"
    )

    plt.title(
        f"Candidate 81 — {T} K "
        "Production Sampling"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    path = os.path.join(
        FINAL_DIR,
        f"candidate_81_{T}K_FINAL_temperature.png"
    )

    plt.savefig(
        path,
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

    plt.close()


# ================================================================
# 20. PLOT — POTENTIAL ENERGY
# ================================================================

for T in TEMPERATURES:

    data = energy_df[
        energy_df["temperature_K"].astype(int)
        == int(T)
    ].sort_values(
        "md_step"
    )

    plt.figure(
        figsize=(9, 5)
    )

    plt.plot(
        data["md_step"],
        data["potential_energy_eV"],
        marker="o",
        linewidth=1.2,
        markersize=4
    )

    plt.xlabel(
        "MD step"
    )

    plt.ylabel(
        "Potential energy (eV)"
    )

    plt.title(
        f"Candidate 81 — {T} K "
        "Potential Energy"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    path = os.path.join(
        FINAL_DIR,
        f"candidate_81_{T}K_FINAL_energy.png"
    )

    plt.savefig(
        path,
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

    plt.close()


# ================================================================
# 21. FINAL FILE LIST
# ================================================================

print()
print("=" * 100)
print("FINAL OUTPUTS")
print("=" * 100)

print()
print("FINAL SUMMARY:")
print(final_summary_path)

print()
print("ENERGY CHECKPOINT:")
print(ENERGY_CHECKPOINT)

print()
print("FINAL ENERGY DATA:")
print(
    os.path.join(
        FINAL_DIR,
        "candidate_81_FINAL_energy_data.csv"
    )
)

print()
print("SELECTED TRAJECTORIES:")
for T in TEMPERATURES:

    print(
        os.path.join(
            FINAL_DIR,
            f"candidate_81_{T}K_FINAL_selected.extxyz"
        )
    )

print()
print("=" * 100)
print("DONE — NO NEW MD WAS RUN")
print("=" * 100)

In [ ]:
# =============================================================================
# CANDIDATE 81 — CLEAN 1200 K THERMAL MD
# RESUMABLE EVERY 500 STEPS
# =============================================================================

import os
import pickle
import random
import numpy as np
import torch

from ase import Atoms, units
from ase.io import write, read
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from mace.calculators import mace_mp


# =============================================================================
# 1. MOUNT GOOGLE DRIVE
# =============================================================================

from google.colab import drive

drive.mount("/content/drive")


# =============================================================================
# 2. PATHS
# =============================================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

PKL_FILE = os.path.join(
    BASE_DIR,
    "MACE_TOP5_candidate_to_structure.pkl"
)

# IMPORTANT:
# This is a NEW dedicated folder for this resumable 1200 K run.
# We will NOT use the old thermal_MD_checkpointed folder.

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "thermal_stability_candidate_81",
    "supercell_2x2x2_thermal_MD_1200K_RESUMABLE"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# 3. FILE NAMES
# =============================================================================

TRAJ_FILE = os.path.join(
    OUTPUT_DIR,
    "candidate_81_1200K_2x2x2_CLEAN.extxyz"
)

STEP_LOG = os.path.join(
    OUTPUT_DIR,
    "candidate_81_1200K_CLEAN_step_log.txt"
)

CHECKPOINT_FILE = os.path.join(
    OUTPUT_DIR,
    "candidate_81_1200K_checkpoint.pkl"
)

INFO_FILE = os.path.join(
    OUTPUT_DIR,
    "candidate_81_1200K_CLEAN_info.txt"
)

COMPLETE_FILE = os.path.join(
    OUTPUT_DIR,
    "candidate_81_1200K_CLEAN_COMPLETE.txt"
)


# =============================================================================
# 4. MD PARAMETERS
# =============================================================================

candidate_id = 81

TEMPERATURE = 1200.0
TIMESTEP_FS = 0.5

EQUILIBRATION_STEPS = 4000
PRODUCTION_STEPS = 10000

TOTAL_STEPS = (
    EQUILIBRATION_STEPS +
    PRODUCTION_STEPS
)

SAVE_INTERVAL = 500

EXPECTED_FRAMES = (
    PRODUCTION_STEPS // SAVE_INTERVAL
) + 1

EXPECTED_PRODUCTION_STEPS = list(
    range(
        EQUILIBRATION_STEPS,
        TOTAL_STEPS + 1,
        SAVE_INTERVAL
    )
)


# =============================================================================
# 5. PRINT PARAMETERS
# =============================================================================

print("=" * 80)
print("CANDIDATE 81 — CLEAN 1200 K RESUMABLE MD")
print("=" * 80)

print("Output directory :", OUTPUT_DIR)
print()
print("Temperature      :", TEMPERATURE, "K")
print("Timestep         :", TIMESTEP_FS, "fs")
print("Equilibration    :", EQUILIBRATION_STEPS)
print("Production       :", PRODUCTION_STEPS)
print("Total steps      :", TOTAL_STEPS)
print("Checkpoint       : every", SAVE_INTERVAL, "steps")
print("Trajectory frames:", EXPECTED_FRAMES)

print()
print("Expected production frames:")
print(EXPECTED_PRODUCTION_STEPS)


# =============================================================================
# 6. LOAD MACE
# =============================================================================

print()
print("=" * 80)
print("LOADING MACE")
print("=" * 80)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device :", device)

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64",
    device=device
)

print("MACE   : medium")
print("D3     : True")
print("dtype  : float64")


# =============================================================================
# 7. HELPER — SAVE CHECKPOINT SAFELY
# =============================================================================

def save_checkpoint(
    path,
    step,
    atoms,
    langevin_rng
):

    checkpoint_data = {

        "step": int(step),

        "positions":
            atoms.get_positions().copy(),

        "momenta":
            atoms.get_momenta().copy(),

        "cell":
            atoms.get_cell().array.copy(),

        "pbc":
            atoms.get_pbc().copy(),

        "symbols":
            atoms.get_chemical_symbols(),

        # RNG states
        "numpy_rng_state":
            np.random.get_state(),

        "python_rng_state":
            random.getstate(),

        "torch_rng_state":
            torch.get_rng_state(),

        "langevin_rng_state":
            langevin_rng.bit_generator.state,
    }

    if torch.cuda.is_available():
        checkpoint_data[
            "torch_cuda_rng_state_all"
        ] = torch.cuda.get_rng_state_all()

    tmp_path = path + ".tmp"

    with open(tmp_path, "wb") as f:

        pickle.dump(
            checkpoint_data,
            f,
            protocol=pickle.HIGHEST_PROTOCOL
        )

        f.flush()
        os.fsync(f.fileno())

    # Atomic replacement
    os.replace(
        tmp_path,
        path
    )

    print(
        f"CHECKPOINT SAVED → step {step}"
    )


# =============================================================================
# 8. HELPER — LOAD CHECKPOINT
# =============================================================================

def load_checkpoint(path):

    with open(path, "rb") as f:
        checkpoint = pickle.load(f)

    required = [
        "step",
        "positions",
        "momenta",
        "cell",
        "pbc",
        "symbols",
        "langevin_rng_state"
    ]

    missing = [
        key
        for key in required
        if key not in checkpoint
    ]

    if missing:

        raise RuntimeError(
            "Checkpoint is missing required data:\n"
            + str(missing)
        )

    return checkpoint


# =============================================================================
# 9. DETERMINE WHETHER WE ARE STARTING OR RESUMING
# =============================================================================

print()
print("=" * 80)
print("CHECKING FOR EXISTING CHECKPOINT")
print("=" * 80)

resume = os.path.exists(CHECKPOINT_FILE)

if resume:

    checkpoint = load_checkpoint(
        CHECKPOINT_FILE
    )

    saved_step = int(
        checkpoint["step"]
    )

    print("CHECKPOINT FOUND")
    print("Saved step :", saved_step)
    print("Resuming from checkpoint.")

else:

    checkpoint = None
    saved_step = 0

    print("NO CHECKPOINT FOUND")
    print("Starting MD from the beginning.")


# =============================================================================
# 10. RECONSTRUCT OR INITIALIZE ATOMS
# =============================================================================

if resume:

    print()
    print("=" * 80)
    print("RESTORING STRUCTURE FROM CHECKPOINT")
    print("=" * 80)

    atoms = Atoms(
        symbols=checkpoint["symbols"],
        positions=checkpoint["positions"],
        cell=checkpoint["cell"],
        pbc=checkpoint["pbc"]
    )

    atoms.set_momenta(
        checkpoint["momenta"]
    )

    print(
        "Restored formula :",
        atoms.get_chemical_formula()
    )

    print(
        "Restored atoms   :",
        len(atoms)
    )

    print(
        "Restored volume  :",
        atoms.get_volume(),
        "Å^3"
    )

else:

    # =========================================================================
    # LOAD RELAXED CANDIDATE 81
    # =========================================================================

    print()
    print("=" * 80)
    print("LOADING RELAXED CANDIDATE 81")
    print("=" * 80)

    if not os.path.exists(PKL_FILE):

        raise FileNotFoundError(
            f"Could not find:\n{PKL_FILE}"
        )

    with open(PKL_FILE, "rb") as f:

        candidate_structures = pickle.load(f)

    if not isinstance(
        candidate_structures,
        dict
    ):

        raise TypeError(
            "The PKL file is not a dictionary "
            "of candidate structures."
        )

    if candidate_id in candidate_structures:

        atoms = (
            candidate_structures[
                candidate_id
            ].copy()
        )

    elif str(candidate_id) in candidate_structures:

        atoms = (
            candidate_structures[
                str(candidate_id)
            ].copy()
        )

    else:

        raise KeyError(
            f"Candidate {candidate_id} "
            "was not found in the PKL file."
        )

    print(
        "Original formula :",
        atoms.get_chemical_formula()
    )

    print(
        "Original atoms   :",
        len(atoms)
    )

    print(
        "Original volume  :",
        atoms.get_volume(),
        "Å^3"
    )

    # =========================================================================
    # 2x2x2 SUPERCELL
    # =========================================================================

    print()
    print("=" * 80)
    print("BUILDING 2x2x2 SUPERCELL")
    print("=" * 80)

    atoms = atoms.repeat(
        (2, 2, 2)
    )

    print(
        "Supercell formula :",
        atoms.get_chemical_formula()
    )

    print(
        "Supercell atoms   :",
        len(atoms)
    )

    print(
        "Supercell volume  :",
        atoms.get_volume(),
        "Å^3"
    )

    print(
        "PBC               :",
        atoms.get_pbc()
    )

    # =========================================================================
    # INITIAL VELOCITIES
    # =========================================================================

    print()
    print("=" * 80)
    print("INITIALIZING VELOCITIES")
    print("=" * 80)

    MaxwellBoltzmannDistribution(
        atoms,
        temperature_K=TEMPERATURE
    )

    print(
        "Velocities initialized at",
        TEMPERATURE,
        "K"
    )


# =============================================================================
# 11. SAFETY CHECKS
# =============================================================================

if len(atoms) != 40:

    raise RuntimeError(
        f"Expected 40 atoms, found {len(atoms)}."
    )

if atoms.get_chemical_formula() != "Si8Ti32":

    raise RuntimeError(
        "Unexpected formula: "
        + atoms.get_chemical_formula()
    )


# =============================================================================
# 12. ATTACH MACE
# =============================================================================

atoms.calc = calc


# =============================================================================
# 13. RESTORE RNG OR CREATE NEW RNG
# =============================================================================

if resume:

    print()
    print("=" * 80)
    print("RESTORING RNG STATES")
    print("=" * 80)

    if "numpy_rng_state" in checkpoint:
        np.random.set_state(
            checkpoint["numpy_rng_state"]
        )

    if "python_rng_state" in checkpoint:
        random.setstate(
            checkpoint["python_rng_state"]
        )

    if "torch_rng_state" in checkpoint:
        torch.set_rng_state(
            checkpoint["torch_rng_state"]
        )

    if (
        torch.cuda.is_available()
        and
        "torch_cuda_rng_state_all"
        in checkpoint
    ):

        torch.cuda.set_rng_state_all(
            checkpoint[
                "torch_cuda_rng_state_all"
            ]
        )

    langevin_rng = np.random.default_rng()

    langevin_rng.bit_generator.state = (
        checkpoint[
            "langevin_rng_state"
        ]
    )

    print("All available RNG states restored.")

else:

    print()
    print("=" * 80)
    print("CREATING NEW LANGEVIN RNG")
    print("=" * 80)

    langevin_rng = np.random.default_rng()


# =============================================================================
# 14. CREATE LANGEVIN DYNAMICS
# =============================================================================

dyn = Langevin(
    atoms,
    timestep=TIMESTEP_FS * units.fs,
    temperature_K=TEMPERATURE,
    friction=0.01 / units.fs,
    fixcm=False,
    rng=langevin_rng
)


# =============================================================================
# 15. WRITE INFO FILE
# =============================================================================

if not os.path.exists(INFO_FILE):

    with open(INFO_FILE, "w") as f:

        f.write(
            "Candidate 81 — Clean 1200 K "
            "Resumable Thermal MD\n"
        )

        f.write("=" * 70 + "\n")

        f.write(
            f"Candidate ID: {candidate_id}\n"
        )

        f.write(
            f"Formula: {atoms.get_chemical_formula()}\n"
        )

        f.write(
            f"Atoms: {len(atoms)}\n"
        )

        f.write(
            f"Temperature: {TEMPERATURE} K\n"
        )

        f.write(
            f"Timestep: {TIMESTEP_FS} fs\n"
        )

        f.write(
            f"Equilibration: {EQUILIBRATION_STEPS}\n"
        )

        f.write(
            f"Production: {PRODUCTION_STEPS}\n"
        )

        f.write(
            f"Total steps: {TOTAL_STEPS}\n"
        )

        f.write(
            f"Checkpoint interval: "
            f"{SAVE_INTERVAL}\n"
        )

        f.write(
            "MACE: medium\n"
        )

        f.write(
            "D3: True\n"
        )

        f.write(
            "dtype: float64\n"
        )

        f.write(
            "fixcm: False\n"
        )

        f.write(
            f"Device: {device}\n"
        )


# =============================================================================
# 16. DETERMINE EXISTING PRODUCTION FRAMES
# =============================================================================

saved_production_steps = []

if os.path.exists(STEP_LOG):

    with open(STEP_LOG, "r") as f:

        for line in f:

            line = line.strip()

            if line:

                try:
                    step = int(line)

                    if (
                        step >= EQUILIBRATION_STEPS
                        and
                        step <= TOTAL_STEPS
                        and
                        (
                            step
                            - EQUILIBRATION_STEPS
                        ) % SAVE_INTERVAL == 0
                    ):

                        saved_production_steps.append(
                            step
                        )

                except ValueError:
                    pass


saved_production_steps = sorted(
    set(saved_production_steps)
)


# =============================================================================
# 17. IF STARTING FRESH, SAVE INITIAL CHECKPOINT AT STEP 0
# =============================================================================

if not resume:

    print()
    print("=" * 80)
    print("STARTING FRESH MD")
    print("=" * 80)

    save_checkpoint(
        CHECKPOINT_FILE,
        0,
        atoms,
        langevin_rng
    )

    current_step = 0

else:

    current_step = saved_step


# =============================================================================
# 18. RUN IN 500-STEP CHUNKS
# =============================================================================

print()
print("=" * 80)
print("STARTING / RESUMING MD")
print("=" * 80)

print(
    "Current step :",
    current_step
)

while current_step < TOTAL_STEPS:

    next_step = min(
        current_step + SAVE_INTERVAL,
        TOTAL_STEPS
    )

    chunk = (
        next_step
        - current_step
    )

    print()
    print(
        f"Running steps "
        f"{current_step} → {next_step}"
    )

    # -------------------------------------------------------------------------
    # RUN EXACTLY 500 STEPS
    # -------------------------------------------------------------------------

    dyn.run(chunk)

    current_step = next_step

    # -------------------------------------------------------------------------
    # CHECKPOINT EVERY 500 STEPS
    # -------------------------------------------------------------------------

    save_checkpoint(
        CHECKPOINT_FILE,
        current_step,
        atoms,
        langevin_rng
    )

    # -------------------------------------------------------------------------
    # PRODUCTION FRAME
    #
    # We save ONLY:
    #
    # 4000, 4500, 5000, ..., 14000
    # -------------------------------------------------------------------------

    if current_step >= EQUILIBRATION_STEPS:

        if (
            current_step
            - EQUILIBRATION_STEPS
        ) % SAVE_INTERVAL == 0:

            if (
                current_step
                not in saved_production_steps
            ):

                # Save energy and temperature
                atoms.info[
                    "md_step"
                ] = current_step

                atoms.info[
                    "temperature_K"
                ] = float(
                    atoms.get_temperature()
                )

                atoms.info[
                    "potential_energy_eV"
                ] = float(
                    atoms.get_potential_energy()
                )

                write(
                    TRAJ_FILE,
                    atoms,
                    format="extxyz",
                    append=os.path.exists(
                        TRAJ_FILE
                    )
                )

                saved_production_steps.append(
                    current_step
                )

                saved_production_steps = sorted(
                    set(saved_production_steps)
                )

                print(
                    "Production frame saved:",
                    current_step
                )

    # -------------------------------------------------------------------------
    # UPDATE STEP LOG
    # -------------------------------------------------------------------------

    with open(STEP_LOG, "w") as f:

        for step in saved_production_steps:

            f.write(
                f"{step}\n"
            )

    print(
        f"Progress: "
        f"{current_step}/{TOTAL_STEPS}"
    )


# =============================================================================
# 19. FINAL VERIFICATION
# =============================================================================

print()
print("=" * 80)
print("FINAL VERIFICATION")
print("=" * 80)

expected_steps = EXPECTED_PRODUCTION_STEPS

saved_production_steps = sorted(
    set(saved_production_steps)
)

print()
print("Expected production steps:")
print(expected_steps)

print()
print("Actual production steps:")
print(saved_production_steps)

if saved_production_steps != expected_steps:

    raise RuntimeError(
        "\nProduction step sequence is incorrect.\n"
        f"Expected: {expected_steps}\n"
        f"Actual:   {saved_production_steps}"
    )


# =============================================================================
# 20. VERIFY TRAJECTORY
# =============================================================================

saved_frames = read(
    TRAJ_FILE,
    index=":"
)

print()
print("=" * 80)
print("TRAJECTORY VERIFICATION")
print("=" * 80)

print(
    "Trajectory file :",
    TRAJ_FILE
)

print(
    "Number of frames:",
    len(saved_frames)
)

print(
    "Expected frames :",
    EXPECTED_FRAMES
)

print(
    "Atoms/frame     :",
    len(saved_frames[0])
)

if len(saved_frames) != EXPECTED_FRAMES:

    raise RuntimeError(
        f"Expected {EXPECTED_FRAMES} frames, "
        f"found {len(saved_frames)}."
    )

if len(saved_frames[0]) != 40:

    raise RuntimeError(
        "Trajectory does not contain 40 atoms/frame."
    )


# =============================================================================
# 21. COMPLETE MARKER
# =============================================================================

with open(COMPLETE_FILE, "w") as f:

    f.write(
        "CANDIDATE 81 — CLEAN 1200 K MD COMPLETE\n"
    )

    f.write("=" * 70 + "\n")

    f.write(
        f"Final step: {current_step}\n"
    )

    f.write(
        f"Frames: {len(saved_frames)}\n"
    )

    f.write(
        f"Saved steps: {saved_production_steps}\n"
    )

    f.write(
        f"Trajectory: {TRAJ_FILE}\n"
    )

    f.write(
        f"Checkpoint: {CHECKPOINT_FILE}\n"
    )


print()
print("=" * 80)
print("SUCCESS — CLEAN 1200 K MD COMPLETE")
print("=" * 80)

print()
print("Final step:", current_step)

print(
    "Final checkpoint:",
    CHECKPOINT_FILE
)

print(
    "Trajectory:",
    TRAJ_FILE
)

print(
    "Frames:",
    len(saved_frames)
)

In [ ]:
# ================================================================
# CANDIDATE 81 — FIXED FINAL ANALYSIS DATA LOADER
#
# NO MD
# NO new simulation
# NO modification of trajectories
#
# Fixes:
#   1. NEVER use *_COMPLETE.txt as numerical data
#   2. Prefer real *_step_log.txt files
#   3. Search existing CSV analysis/results files
#   4. Use .extxyz for structural information
#   5. Merge whatever valid energy/temperature data already exists
# ================================================================

import os
import glob
import re
import numpy as np
import pandas as pd

# ================================================================
# 1. BASE PATH
# ================================================================

BASE = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "thermal_stability_candidate_81"
)

OUTDIR = os.path.join(
    BASE,
    "FINAL_THERMAL_ANALYSIS_V3"
)

os.makedirs(
    OUTDIR,
    exist_ok=True
)

TEMPERATURES = [300, 600, 900, 1200]

print("=" * 80)
print("CANDIDATE 81 — FIXED ANALYSIS DATA LOADER")
print("=" * 80)
print("NO MD will be run.")
print("NO trajectory will be modified.")
print("Output:", OUTDIR)


# ================================================================
# 2. FIND ALL FILES
# ================================================================

all_txt = sorted(
    glob.glob(
        os.path.join(BASE, "**", "*.txt"),
        recursive=True
    )
)

all_csv = sorted(
    glob.glob(
        os.path.join(BASE, "**", "*.csv"),
        recursive=True
    )
)

all_extxyz = sorted(
    glob.glob(
        os.path.join(BASE, "**", "*.extxyz"),
        recursive=True
    )
)


# ================================================================
# 3. NEVER USE COMPLETE.TXT AS NUMERICAL LOG
# ================================================================

real_txt_logs = [
    f for f in all_txt
    if "complete" not in os.path.basename(f).lower()
]


print("\n" + "=" * 80)
print("REAL TXT LOG CANDIDATES")
print("=" * 80)

for f in real_txt_logs:
    print(f)


# ================================================================
# 4. FIND REAL STEP LOGS
# ================================================================

step_logs = [
    f for f in real_txt_logs
    if "step_log" in os.path.basename(f).lower()
]


print("\n" + "=" * 80)
print("STEP LOGS")
print("=" * 80)

for f in step_logs:
    print(f)


def find_step_log(T):

    candidates = [
        f for f in step_logs
        if re.search(
            rf"{T}\s*K|{T}K|{T}_K|{T}-K",
            os.path.basename(f),
            re.IGNORECASE
        )
    ]

    # Special handling for 1200 K because its folder name
    # contains 1200K even if filename does not.
    if not candidates:

        candidates = [
            f for f in step_logs
            if re.search(
                rf"{T}K",
                f,
                re.IGNORECASE
            )
        ]

    if not candidates:
        return None

    return candidates[-1]


selected_step_logs = {
    T: find_step_log(T)
    for T in TEMPERATURES
}


print("\n" + "=" * 80)
print("SELECTED REAL STEP LOGS")
print("=" * 80)

for T in TEMPERATURES:

    print(f"\n{T} K:")
    print(selected_step_logs[T])


# ================================================================
# 5. FIND ALL EXISTING ANALYSIS CSVs
# ================================================================

print("\n" + "=" * 80)
print("EXISTING CSV FILES")
print("=" * 80)

for f in all_csv:
    print(f)


# ================================================================
# 6. READ CSV SAFELY
# ================================================================

def safe_read_csv(path):

    try:

        df = pd.read_csv(path)

        if len(df) == 0:
            return None

        return df

    except Exception as e:

        print(
            f"Could not read CSV: {os.path.basename(path)}"
        )

        return None


# ================================================================
# 7. NORMALIZE COLUMN NAMES
# ================================================================

def norm(x):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(x).lower()
    )


def find_col(df, patterns):

    if df is None:
        return None

    normalized = {
        norm(c): c
        for c in df.columns
    }

    # exact
    for p in patterns:

        p = norm(p)

        if p in normalized:
            return normalized[p]

    # substring
    for n, original in normalized.items():

        for p in patterns:

            p = norm(p)

            if p in n or n in p:
                return original

    return None


STEP_PATTERNS = [
    "md_step",
    "step",
    "steps",
    "timestep",
    "iteration"
]

ENERGY_PATTERNS = [
    "potential_energy_eV",
    "potential_energy",
    "energy_eV",
    "energy",
    "epot",
    "potential"
]

TEMP_PATTERNS = [
    "temperature_K",
    "temperature",
    "temp_K",
    "temp",
    "temperaturek"
]


# ================================================================
# 8. SEARCH EXISTING CSVs FOR NUMERICAL MD DATA
# ================================================================

csv_candidates = []

for path in all_csv:

    name = os.path.basename(path).lower()

    # Ignore files generated by this current run
    if "final_thermal_analysis_v3" in path.lower():
        continue

    df = safe_read_csv(path)

    if df is None:
        continue

    step_col = find_col(
        df,
        STEP_PATTERNS
    )

    energy_col = find_col(
        df,
        ENERGY_PATTERNS
    )

    temp_col = find_col(
        df,
        TEMP_PATTERNS
    )

    score = 0

    if step_col is not None:
        score += 2

    if energy_col is not None:
        score += 4

    if temp_col is not None:
        score += 4

    if score > 0:

        csv_candidates.append({

            "path": path,
            "df": df,
            "step_col": step_col,
            "energy_col": energy_col,
            "temp_col": temp_col,
            "score": score

        })


print("\n" + "=" * 80)
print("NUMERICAL CSV CANDIDATES")
print("=" * 80)

for item in csv_candidates:

    print("\n", item["path"])
    print(
        "  step  =",
        item["step_col"]
    )
    print(
        "  energy =",
        item["energy_col"]
    )
    print(
        "  temp   =",
        item["temp_col"]
    )
    print(
        "  score  =",
        item["score"]
    )


# ================================================================
# 9. EXTRACT NUMERICAL DATA FROM BEST CSVs
# ================================================================

def extract_numeric_data(item):

    df = item["df"]

    out = pd.DataFrame()

    if item["step_col"] is not None:

        out["md_step"] = pd.to_numeric(
            df[item["step_col"]],
            errors="coerce"
        )

    if item["energy_col"] is not None:

        out["potential_energy_eV"] = pd.to_numeric(
            df[item["energy_col"]],
            errors="coerce"
        )

    if item["temp_col"] is not None:

        out["temperature_K"] = pd.to_numeric(
            df[item["temp_col"]],
            errors="coerce"
        )

    return out


numeric_csv_data = []

for item in csv_candidates:

    data = extract_numeric_data(item)

    if len(data) > 0:

        numeric_csv_data.append({

            "path": item["path"],
            "data": data,
            "score": item["score"]

        })


# ================================================================
# 10. CHOOSE BEST NUMERICAL SOURCE
# ================================================================

def choose_best_csv():

    if not numeric_csv_data:
        return None

    # Prefer files explicitly related to thermal MD
    ranked = []

    for item in numeric_csv_data:

        name = os.path.basename(
            item["path"]
        ).lower()

        bonus = 0

        if "thermal_md_results" in name:
            bonus += 20

        if "energy_vs_time" in name:
            bonus += 15

        if "temperature_vs_time" in name:
            bonus += 15

        if "thermal" in name:
            bonus += 5

        if "summary" in name:
            bonus += 2

        ranked.append(
            (
                item["score"] + bonus,
                len(item["data"]),
                item
            )
        )

    ranked.sort(
        key=lambda x: (x[0], x[1])
    )

    return ranked[-1][2]


best_csv = choose_best_csv()

print("\n" + "=" * 80)
print("BEST EXISTING NUMERICAL CSV")
print("=" * 80)

if best_csv is not None:

    print(best_csv["path"])
    print("\nRows:", len(best_csv["data"]))
    print(
        "\nColumns:",
        list(best_csv["data"].columns)
    )

else:

    print("NO suitable numerical CSV found.")


# ================================================================
# 11. PARSE THE 1200 K STEP LOG DIRECTLY
# ================================================================

def parse_step_log(path):

    if path is None:
        return None

    print("\nParsing step log:")
    print(path)

    with open(
        path,
        "r",
        errors="ignore"
    ) as f:

        lines = f.readlines()

    rows = []

    for line in lines:

        # Extract all numerical values from the line
        nums = re.findall(
            r"[-+]?(?:\d*\.\d+|\d+\.?)(?:[Ee][-+]?\d+)?",
            line
        )

        if len(nums) < 2:
            continue

        # We only accept lines that clearly look like
        # MD-step records.
        lower = line.lower()

        looks_like_step = (
            "step" in lower
            or "energy" in lower
            or "temp" in lower
            or "temperature" in lower
        )

        if not looks_like_step:
            continue

        values = []

        for x in nums:

            try:
                values.append(float(x))
            except:
                pass

        if len(values) >= 2:

            rows.append(values)

    if not rows:

        print(
            "No numerical MD records detected."
        )

        return None

    # Put into a flexible dataframe
    max_len = max(
        len(r)
        for r in rows
    )

    padded = [
        r + [np.nan] * (
            max_len - len(r)
        )
        for r in rows
    ]

    df = pd.DataFrame(
        padded
    )

    print(
        "Detected numerical lines:",
        len(df)
    )

    return df


log_parsed_1200 = parse_step_log(
    selected_step_logs.get(1200)
)


# ================================================================
# 12. LOAD EXTXYZ STRUCTURAL DATA
# ================================================================

from ase.io import read


def choose_extxyz(T):

    candidates = [
        f for f in all_extxyz
        if re.search(
            rf"{T}K",
            os.path.basename(f),
            re.IGNORECASE
        )
    ]

    # 1200 K: final resumable trajectory
    if T == 1200:

        preferred = [
            f for f in candidates
            if "resumable" in f.lower()
        ]

        if preferred:
            return preferred[-1]

    # Other temperatures:
    preferred = [
        f for f in candidates
        if "checkpointed" in f.lower()
        and "2x2x2" in f.lower()
    ]

    if preferred:
        return preferred[-1]

    return candidates[-1] if candidates else None


trajectories = {}

print("\n" + "=" * 80)
print("LOADING STRUCTURES")
print("=" * 80)

for T in TEMPERATURES:

    path = choose_extxyz(T)

    print(f"\n{T} K:")
    print(path)

    frames = read(
        path,
        index=":"
    )

    trajectories[T] = frames

    print(
        "Frames:",
        len(frames)
    )

    print(
        "Atoms:",
        len(frames[0])
    )


# ================================================================
# 13. STRUCTURAL METRICS
# ================================================================

def structural_metrics(atoms):

    symbols = np.array(
        atoms.get_chemical_symbols()
    )

    lengths = atoms.cell.lengths()
    angles = atoms.cell.angles()

    def min_dist(A, B):

        i1 = np.where(
            symbols == A
        )[0]

        i2 = np.where(
            symbols == B
        )[0]

        if len(i1) == 0 or len(i2) == 0:
            return np.nan

        minimum = np.inf

        for i in i1:

            for j in i2:

                if A == B and i >= j:
                    continue

                d = atoms.get_distance(
                    i,
                    j,
                    mic=True
                )

                if d > 1e-8:
                    minimum = min(
                        minimum,
                        d
                    )

        return minimum

    return {

        "volume_A3":
            atoms.get_volume(),

        "a_A":
            lengths[0],

        "b_A":
            lengths[1],

        "c_A":
            lengths[2],

        "alpha_deg":
            angles[0],

        "beta_deg":
            angles[1],

        "gamma_deg":
            angles[2],

        "min_Ti_Si_A":
            min_dist("Ti", "Si"),

        "min_Ti_Ti_A":
            min_dist("Ti", "Ti"),

        "min_Si_Si_A":
            min_dist("Si", "Si"),

    }


structural_tables = {}

for T, frames in trajectories.items():

    rows = []

    for i, atoms in enumerate(frames):

        row = structural_metrics(
            atoms
        )

        row["frame_index"] = i

        row["embedded_step"] = atoms.info.get(
            "md_step",
            np.nan
        )

        row["embedded_energy_eV"] = atoms.info.get(
            "potential_energy_eV",
            np.nan
        )

        row["embedded_temperature_K"] = atoms.info.get(
            "temperature_K",
            np.nan
        )

        rows.append(row)

    structural_tables[T] = pd.DataFrame(
        rows
    )


# ================================================================
# 14. SAVE STRUCTURAL TABLES
# ================================================================

for T, df in structural_tables.items():

    path = os.path.join(
        OUTDIR,
        f"candidate_81_{T}K_STRUCTURAL.csv"
    )

    df.to_csv(
        path,
        index=False
    )


# ================================================================
# 15. STRUCTURAL SUMMARY
# ================================================================

summary_rows = []

for T, df in structural_tables.items():

    V = df["volume_A3"].dropna()

    summary_rows.append({

        "T_K": T,

        "frames":
            len(df),

        "volume_initial_A3":
            V.iloc[0],

        "volume_final_A3":
            V.iloc[-1],

        "volume_change_percent":
            100
            * (
                V.iloc[-1] - V.iloc[0]
            )
            / V.iloc[0],

        "mean_min_Ti_Si_A":
            df["min_Ti_Si_A"].mean(),

        "min_Ti_Si_A":
            df["min_Ti_Si_A"].min(),

        "mean_min_Ti_Ti_A":
            df["min_Ti_Ti_A"].mean(),

        "min_Ti_Ti_A":
            df["min_Ti_Ti_A"].min(),

        "mean_min_Si_Si_A":
            df["min_Si_Si_A"].mean(),

        "min_Si_Si_A":
            df["min_Si_Si_A"].min(),

    })


structural_summary = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 80)
print("STRUCTURAL SUMMARY")
print("=" * 80)

print(
    structural_summary.to_string(
        index=False
    )
)

structural_summary.to_csv(
    os.path.join(
        OUTDIR,
        "candidate_81_STRUCTURAL_SUMMARY.csv"
    ),
    index=False
)


# ================================================================
# 16. IMPORTANT DIAGNOSTIC
# ================================================================

print("\n" + "=" * 80)
print("DATA SOURCE DIAGNOSTIC")
print("=" * 80)

if best_csv is not None:

    data = best_csv["data"]

    print(
        "\nBest numerical source:"
    )

    print(
        best_csv["path"]
    )

    print(
        "\nAvailable numerical columns:"
    )

    print(
        list(data.columns)
    )

    print(
        "\nFirst rows:"
    )

    print(
        data.head(10).to_string(
            index=False
        )
    )

else:

    print(
        "\nNo existing numerical CSV was recognized."
    )


print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print(
    "\nThis cell DID NOT run MD."
)

print(
    "It only loaded existing trajectories/logs/results."
)

print(
    "\nResults:",
    OUTDIR
)

In [ ]:
# ================================================================
# CANDIDATE 81 — CLEAN THERMAL STABILITY ANALYSIS V4
# ================================================================
#
# IMPORTANT:
#   This analysis uses ONLY the existing EXTXYZ trajectories.
#   Legacy COMBINED CSV files are NOT used as authoritative data.
#
# System:
#   Candidate 81
#   Si8Ti32
#   2x2x2 supercell
#   40 atoms
#   PBC
#
# Temperatures:
#   300 / 600 / 900 / 1200 K
#
# Analysis:
#   1. trajectory reconstruction
#   2. 21-frame selection
#   3. energy
#   4. temperature
#   5. PBC-corrected displacement
#   6. RMSD
#   7. pair distances
#   8. neighbor topology
#   9. volume / cell evolution
#  10. energy stability
#  11. temperature stability
#  12. final evidence summary
#
# ================================================================


import os
import gc
import warnings

import numpy as np
import pandas as pd

from ase.io import read
from ase.geometry import find_mic

warnings.filterwarnings("ignore")


# ================================================================
# 1. SETTINGS
# ================================================================

CANDIDATE_ID = 81

TEMPERATURES = [300, 600, 900, 1200]

N_SELECTED_FRAMES = 21

# Pair cutoffs used ONLY for topology tracking.
# They are deliberately explicit so the definition is transparent.
PAIR_CUTOFFS = {
    "Ti-Ti": 3.50,
    "Ti-Si": 3.20,
    "Si-Si": 3.00,
}


# ================================================================
# 2. PATHS
# ================================================================

ARCHIVE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "thermal_stability_candidate_81"
)

TRAJECTORY_DIR = os.path.join(
    ARCHIVE_DIR,
    "supercell_2x2x2_thermal_MD_checkpointed"
)

TRAJECTORY_1200_DIR = os.path.join(
    ARCHIVE_DIR,
    "supercell_2x2x2_thermal_MD_1200K_RESUMABLE"
)

OUT = os.path.join(
    ARCHIVE_DIR,
    "FINAL_THERMAL_ANALYSIS_V4"
)

os.makedirs(OUT, exist_ok=True)


TRAJECTORIES = {

    300: os.path.join(
        TRAJECTORY_DIR,
        "candidate_81_300K_2x2x2.extxyz"
    ),

    600: os.path.join(
        TRAJECTORY_DIR,
        "candidate_81_600K_2x2x2.extxyz"
    ),

    900: os.path.join(
        TRAJECTORY_DIR,
        "candidate_81_900K_2x2x2.extxyz"
    ),

    1200: os.path.join(
        TRAJECTORY_1200_DIR,
        "candidate_81_1200K_2x2x2_CLEAN.extxyz"
    ),
}


# ================================================================
# 3. BASIC CHECKS
# ================================================================

print("\n" + "=" * 80)
print("CANDIDATE 81 — CLEAN THERMAL STABILITY ANALYSIS V4")
print("=" * 80)

print("\nOutput directory:")
print(OUT)

print("\nTrajectory files:")

for T, path in TRAJECTORIES.items():

    print(f"{T} K : {path}")

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"\nMissing trajectory for {T} K:\n{path}"
        )


# ================================================================
# 4. LOAD TRAJECTORIES
# ================================================================

print("\n" + "=" * 80)
print("LOADING EXISTING TRAJECTORIES")
print("=" * 80)


raw_trajectories = {}

for T in TEMPERATURES:

    frames = read(
        TRAJECTORIES[T],
        index=":"
    )

    if len(frames) == 0:

        raise RuntimeError(
            f"{T} K trajectory contains zero frames."
        )

    # ------------------------------------------------------------
    # Basic structural checks
    # ------------------------------------------------------------

    for i, atoms in enumerate(frames):

        if len(atoms) != 40:

            raise RuntimeError(
                f"{T} K frame {i} has "
                f"{len(atoms)} atoms instead of 40."
            )

        if not np.all(atoms.pbc):

            raise RuntimeError(
                f"{T} K frame {i} does not have "
                f"full periodic boundary conditions."
            )

    raw_trajectories[T] = frames

    print(
        f"{T:4d} K : "
        f"{len(frames)} frames | "
        f"{len(frames[0])} atoms | "
        f"{frames[0].get_chemical_formula()}"
    )


# ================================================================
# 5. SELECT EXACTLY 21 FRAMES
# ================================================================

print("\n" + "=" * 80)
print("SELECTING EXACTLY 21 FRAMES")
print("=" * 80)


trajectories = {}

frame_selection = []


for T in TEMPERATURES:

    frames = raw_trajectories[T]

    n_total = len(frames)

    if n_total < N_SELECTED_FRAMES:

        raise RuntimeError(
            f"{T} K has only {n_total} frames. "
            f"Cannot select {N_SELECTED_FRAMES}."
        )

    # ------------------------------------------------------------
    # Uniformly distribute 21 frames over the COMPLETE trajectory.
    #
    # This is important:
    # we do NOT simply take the first 21 frames when 29 exist.
    # ------------------------------------------------------------

    indices = np.linspace(
        0,
        n_total - 1,
        N_SELECTED_FRAMES
    )

    indices = np.rint(indices).astype(int)

    # Remove accidental duplicates, then verify.
    indices = np.unique(indices)

    if len(indices) != N_SELECTED_FRAMES:

        raise RuntimeError(
            f"{T} K produced {len(indices)} unique selected "
            f"frames instead of {N_SELECTED_FRAMES}."
        )

    trajectories[T] = [
        frames[i]
        for i in indices
    ]

    for selected_number, original_index in enumerate(indices):

        frame_selection.append({

            "T_K": T,

            "selected_frame":
                selected_number,

            "original_trajectory_frame":
                int(original_index),

            "total_trajectory_frames":
                n_total,
        })

    print(
        f"{T:4d} K : "
        f"{n_total} available -> "
        f"{len(indices)} selected"
    )


frame_selection_df = pd.DataFrame(
    frame_selection
)

frame_selection_df.to_csv(
    os.path.join(
        OUT,
        "candidate_81_V4_frame_selection.csv"
    ),
    index=False
)


# ================================================================
# 6. HELPER — EXTRACT TEMPERATURE
# ================================================================

def extract_temperature(atoms):

    # ------------------------------------------------------------
    # First try common ASE/EXTXYZ info keys.
    # ------------------------------------------------------------

    possible_keys = [

        "temperature",

        "temperature_K",

        "temp",

        "Temp",

        "Temperature",

        "kinetic_temperature",

        "kinetic_temperature_K",
    ]

    for key in possible_keys:

        if key in atoms.info:

            try:

                value = float(
                    atoms.info[key]
                )

                if np.isfinite(value):

                    return value

            except Exception:

                pass

    # ------------------------------------------------------------
    # If momenta exist, calculate temperature from kinetic energy.
    #
    # ASE:
    #   KE = sum(p^2 / 2m)
    #   T  = 2 KE / (3 N kB)
    #
    # Center-of-mass removal is NOT assumed here because the
    # original MD used fixcm=False.
    # ------------------------------------------------------------

    try:

        momenta = atoms.get_momenta()

        if momenta is not None:

            masses = atoms.get_masses()

            if (
                momenta.shape[0] == len(atoms)
                and len(masses) == len(atoms)
            ):

                # ASE kinetic energy is in eV when using ASE units.
                from ase import units

                kinetic_energy_eV = np.sum(
                    momenta ** 2 /
                    (2.0 * masses[:, None])
                )

                temperature = (
                    2.0 * kinetic_energy_eV /
                    (
                        3.0 *
                        len(atoms) *
                        units.kB
                    )
                )

                if np.isfinite(temperature):

                    return float(temperature)

    except Exception:

        pass

    return np.nan


# ================================================================
# 7. HELPER — EXTRACT STORED ENERGY
# ================================================================

def extract_stored_energy(atoms):

    possible_keys = [

        "energy",

        "potential_energy",

        "Energy",

        "PotentialEnergy",

        "free_energy",
    ]

    for key in possible_keys:

        if key in atoms.info:

            try:

                value = float(
                    atoms.info[key]
                )

                if np.isfinite(value):

                    return value

            except Exception:

                pass

    return np.nan


# ================================================================
# 8. HELPER — PBC DISPLACEMENT
# ================================================================

def pbc_displacement(
    reference,
    current
):

    displacement = (
        current.get_positions()
        -
        reference.get_positions()
    )

    # ------------------------------------------------------------
    # MIC gives the physically shortest displacement under PBC.
    # ------------------------------------------------------------

    mic_displacement, _ = find_mic(
        displacement,
        cell=reference.get_cell(),
        pbc=reference.pbc
    )

    return mic_displacement


# ================================================================
# 9. HELPER — RMSD
# ================================================================

def calculate_rmsd(
    reference,
    current
):

    displacement = pbc_displacement(
        reference,
        current
    )

    squared = np.sum(
        displacement ** 2,
        axis=1
    )

    return float(
        np.sqrt(
            np.mean(squared)
        )
    )


# ================================================================
# 10. HELPER — PAIR DISTANCES
# ================================================================

def pair_distances(
    atoms
):

    symbols = np.array(
        atoms.get_chemical_symbols()
    )

    positions = atoms.get_positions()

    cell = atoms.get_cell()

    pbc = atoms.pbc

    results = {}

    pair_definitions = {

        "Ti-Ti": ("Ti", "Ti"),

        "Ti-Si": ("Ti", "Si"),

        "Si-Si": ("Si", "Si"),
    }

    for pair_name, (
        element_a,
        element_b
    ) in pair_definitions.items():

        distances = []

        indices_a = np.where(
            symbols == element_a
        )[0]

        indices_b = np.where(
            symbols == element_b
        )[0]

        for i in indices_a:

            for j in indices_b:

                # Avoid duplicate same-element pairs.
                if (
                    element_a == element_b
                    and j <= i
                ):

                    continue

                vector = (
                    positions[j]
                    -
                    positions[i]
                )

                mic_vector, distance = find_mic(
                    vector,
                    cell=cell,
                    pbc=pbc
                )

                distances.append(
                    float(distance)
                )

        if len(distances) == 0:

            results[pair_name] = {

                "min": np.nan,

                "mean": np.nan,

                "median": np.nan,

                "max": np.nan,

                "neighbor_count": 0,
            }

        else:

            distances = np.array(
                distances
            )

            cutoff = PAIR_CUTOFFS[
                pair_name
            ]

            results[pair_name] = {

                "min":
                    float(np.min(distances)),

                "mean":
                    float(np.mean(distances)),

                "median":
                    float(np.median(distances)),

                "max":
                    float(np.max(distances)),

                "neighbor_count":
                    int(
                        np.sum(
                            distances <= cutoff
                        )
                    ),
            }

    return results


# ================================================================
# 11. RECONSTRUCT NUMERICAL DATA DIRECTLY FROM TRAJECTORIES
# ================================================================

print("\n" + "=" * 80)
print("REBUILDING NUMERICAL DATA FROM EXTXYZ")
print("=" * 80)


numerical = {}

for T in TEMPERATURES:

    rows = []

    frames = trajectories[T]

    for frame_index, atoms in enumerate(frames):

        energy = extract_stored_energy(
            atoms
        )

        temperature = extract_temperature(
            atoms
        )

        rows.append({

            "T_K":
                T,

            "frame_index":
                frame_index,

            "energy_eV":
                energy,

            "temperature_K":
                temperature,

            "volume_A3":
                float(
                    atoms.get_volume()
                ),
        })

    df = pd.DataFrame(
        rows
    )

    numerical[T] = df

    print(
        f"\n{T} K"
    )

    print(
        f"frames = {len(df)}"
    )

    print(
        f"valid stored energy = "
        f"{df['energy_eV'].notna().sum()}"
    )

    print(
        f"valid temperature = "
        f"{df['temperature_K'].notna().sum()}"
    )


# ================================================================
# 12. ENERGY RECOVERY USING MACE IF NECESSARY
# ================================================================

# We only load MACE if the trajectory did NOT store energies.
# This prevents unnecessary model evaluation when energy is
# already available in the EXTXYZ.

need_mace = any(
    numerical[T]["energy_eV"].isna().any()
    for T in TEMPERATURES
)


if need_mace:

    print("\n" + "=" * 80)
    print("SOME ENERGIES ARE NOT STORED — LOADING MACE")
    print("=" * 80)

    import torch

    from mace.calculators import mace_mp

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        f"Device: {device}"
    )

    calc = mace_mp(
        model="medium",
        dispersion=True,
        default_dtype="float64",
        device=device
    )

    for T in TEMPERATURES:

        df = numerical[T]

        for i, atoms in enumerate(
            trajectories[T]
        ):

            if np.isfinite(
                df.loc[
                    i,
                    "energy_eV"
                ]
            ):

                continue

            atoms.calc = calc

            energy = atoms.get_potential_energy()

            df.loc[
                i,
                "energy_eV"
            ] = float(
                energy
            )

        numerical[T] = df

    del calc

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ================================================================
# 13. HARD NUMERICAL VALIDATION
# ================================================================

print("\n" + "=" * 80)
print("NUMERICAL VALIDATION")
print("=" * 80)


for T in TEMPERATURES:

    df = numerical[T]

    if len(df) != N_SELECTED_FRAMES:

        raise RuntimeError(
            f"{T} K has {len(df)} numerical rows "
            f"instead of {N_SELECTED_FRAMES}."
        )

    if df["energy_eV"].isna().any():

        raise RuntimeError(
            f"{T} K still contains missing energy values."
        )

    print(
        f"{T} K : "
        f"{len(df)} frames | "
        f"energy OK"
    )


# ================================================================
# 14. SAVE CLEAN NUMERICAL DATA
# ================================================================

for T in TEMPERATURES:

    numerical[T].to_csv(

        os.path.join(
            OUT,
            f"candidate_81_{T}K_V4_numerical.csv"
        ),

        index=False
    )


# ================================================================
# 15. STRUCTURAL ANALYSIS
# ================================================================

print("\n" + "=" * 80)
print("STRUCTURAL ANALYSIS")
print("=" * 80)


structural_rows = []

for T in TEMPERATURES:

    frames = trajectories[T]

    reference = frames[0]

    for frame_index, atoms in enumerate(
        frames
    ):

        displacement = pbc_displacement(
            reference,
            atoms
        )

        displacement_magnitudes = np.linalg.norm(
            displacement,
            axis=1
        )

        rmsd = calculate_rmsd(
            reference,
            atoms
        )

        pair_data = pair_distances(
            atoms
        )

        row = {

            "T_K":
                T,

            "frame_index":
                frame_index,

            "rmsd_A":
                rmsd,

            "max_displacement_A":
                float(
                    np.max(
                        displacement_magnitudes
                    )
                ),

            "mean_displacement_A":
                float(
                    np.mean(
                        displacement_magnitudes
                    )
                ),

            "volume_A3":
                float(
                    atoms.get_volume()
                ),
        }

        for pair_name in pair_data:

            row[
                f"{pair_name}_min_A"
            ] = pair_data[
                pair_name
            ]["min"]

            row[
                f"{pair_name}_mean_A"
            ] = pair_data[
                pair_name
            ]["mean"]

            row[
                f"{pair_name}_neighbors"
            ] = pair_data[
                pair_name
            ]["neighbor_count"]

        structural_rows.append(
            row
        )


structural_df = pd.DataFrame(
    structural_rows
)

structural_df.to_csv(

    os.path.join(
        OUT,
        "candidate_81_V4_structural_trajectory.csv"
    ),

    index=False
)


# ================================================================
# 16. ENERGY STABILITY ANALYSIS
# ================================================================

print("\n" + "=" * 80)
print("ENERGY STABILITY ANALYSIS")
print("=" * 80)


energy_summary = []


for T in TEMPERATURES:

    df = numerical[T]

    e = df[
        "energy_eV"
    ].to_numpy(
        dtype=float
    )

    steps = np.arange(
        len(e),
        dtype=float
    )

    n = len(e)

    if n != N_SELECTED_FRAMES:

        raise RuntimeError(
            f"{T} K energy analysis received "
            f"{n} frames instead of "
            f"{N_SELECTED_FRAMES}."
        )

    window = max(
        3,
        n // 5
    )

    first = e[:window]

    last = e[-window:]

    if n >= 2:

        slope, intercept = np.polyfit(
            steps,
            e,
            1
        )

    else:

        slope = np.nan

    diffs = np.diff(e)

    energy_summary.append({

        "T_K":
            T,

        "frames":
            n,

        "mean_energy_eV":
            np.mean(e),

        "std_energy_eV":
            np.std(e),

        "min_energy_eV":
            np.min(e),

        "max_energy_eV":
            np.max(e),

        "first_window_mean_eV":
            np.mean(first),

        "last_window_mean_eV":
            np.mean(last),

        "last_minus_first_eV":
            np.mean(last)
            -
            np.mean(first),

        "linear_drift_eV_per_frame":
            slope,

        "largest_frame_jump_eV":
            (
                np.max(
                    np.abs(diffs)
                )
                if len(diffs)
                else 0.0
            ),

        "mean_abs_frame_change_eV":
            (
                np.mean(
                    np.abs(diffs)
                )
                if len(diffs)
                else 0.0
            ),
    })


energy_summary_df = pd.DataFrame(
    energy_summary
)

print(
    energy_summary_df.to_string(
        index=False
    )
)

energy_summary_df.to_csv(

    os.path.join(
        OUT,
        "candidate_81_V4_energy_stability.csv"
    ),

    index=False
)


# ================================================================
# 17. TEMPERATURE STABILITY
# ================================================================

print("\n" + "=" * 80)
print("TEMPERATURE ANALYSIS")
print("=" * 80)


temperature_summary = []


for T in TEMPERATURES:

    temp = numerical[T][
        "temperature_K"
    ].to_numpy(
        dtype=float
    )

    valid = temp[
        np.isfinite(temp)
    ]

    if len(valid) == 0:

        temperature_summary.append({

            "T_K":
                T,

            "valid_temperature_frames":
                0,

            "mean_temperature_K":
                np.nan,

            "std_temperature_K":
                np.nan,

            "min_temperature_K":
                np.nan,

            "max_temperature_K":
                np.nan,

            "mean_minus_target_K":
                np.nan,
        })

        continue

    temperature_summary.append({

        "T_K":
            T,

        "valid_temperature_frames":
            len(valid),

        "mean_temperature_K":
            np.mean(valid),

        "std_temperature_K":
            np.std(valid),

        "min_temperature_K":
            np.min(valid),

        "max_temperature_K":
            np.max(valid),

        "mean_minus_target_K":
            np.mean(valid) - T,
    })


temperature_summary_df = pd.DataFrame(
    temperature_summary
)

print(
    temperature_summary_df.to_string(
        index=False
    )
)

temperature_summary_df.to_csv(

    os.path.join(
        OUT,
        "candidate_81_V4_temperature_stability.csv"
    ),

    index=False
)


# ================================================================
# 18. TOPOLOGY SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("NEIGHBOR TOPOLOGY ANALYSIS")
print("=" * 80)


topology_rows = []


for T in TEMPERATURES:

    df = structural_df[
        structural_df["T_K"] == T
    ]

    for pair_name in PAIR_CUTOFFS:

        column = (
            f"{pair_name}_neighbors"
        )

        values = df[
            column
        ].to_numpy(
            dtype=float
        )

        topology_rows.append({

            "T_K":
                T,

            "pair":
                pair_name,

            "cutoff_A":
                PAIR_CUTOFFS[
                    pair_name
                ],

            "initial_neighbors":
                values[0],

            "final_neighbors":
                values[-1],

            "minimum_neighbors":
                np.min(values),

            "maximum_neighbors":
                np.max(values),

            "mean_neighbors":
                np.mean(values),

            "frames_different_from_initial":
                np.sum(
                    values != values[0]
                ),

            "frames_returning_to_initial":
                np.sum(
                    values == values[0]
                ),
        })


topology_summary_df = pd.DataFrame(
    topology_rows
)

print(
    topology_summary_df.to_string(
        index=False
    )
)

topology_summary_df.to_csv(

    os.path.join(
        OUT,
        "candidate_81_V4_neighbor_topology.csv"
    ),

    index=False
)


# ================================================================
# 19. FINAL STRUCTURAL SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("FINAL STRUCTURAL SUMMARY")
print("=" * 80)


final_rows = []


for T in TEMPERATURES:

    df = structural_df[
        structural_df["T_K"] == T
    ]

    energy = numerical[T][
        "energy_eV"
    ].to_numpy(
        dtype=float
    )

    temperature = numerical[T][
        "temperature_K"
    ].to_numpy(
        dtype=float
    )

    rmsd = df[
        "rmsd_A"
    ].to_numpy(
        dtype=float
    )

    volume = df[
        "volume_A3"
    ].to_numpy(
        dtype=float
    )

    final_rows.append({

        "T_K":
            T,

        "frames":
            len(df),

        "final_rmsd_A":
            rmsd[-1],

        "max_rmsd_A":
            np.max(rmsd),

        "mean_rmsd_A":
            np.mean(rmsd),

        "volume_initial_A3":
            volume[0],

        "volume_final_A3":
            volume[-1],

        "volume_change_percent":
            100.0 *
            (
                volume[-1] -
                volume[0]
            )
            /
            volume[0],

        "energy_first_mean_eV":
            np.mean(
                energy[:max(3, len(energy)//5)]
            ),

        "energy_last_mean_eV":
            np.mean(
                energy[-max(3, len(energy)//5):]
            ),

        "energy_total_change_eV":
            energy[-1] -
            energy[0],

        "temperature_mean_K":
            (
                np.nanmean(temperature)
                if np.isfinite(
                    temperature
                ).any()
                else np.nan
            ),
    })


final_summary_df = pd.DataFrame(
    final_rows
)

print(
    final_summary_df.to_string(
        index=False
    )
)

final_summary_df.to_csv(

    os.path.join(
        OUT,
        "candidate_81_V4_FINAL_SUMMARY.csv"
    ),

    index=False
)


# ================================================================
# 20. ANALYSIS COMPLETION REPORT
# ================================================================

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

print("\nCandidate 81")
print("System      : Si8Ti32")
print("Atoms       : 40")
print("Temperatures: 300 / 600 / 900 / 1200 K")
print("Frames/T    : 21")
print("PBC         : Corrected using MIC")
print()
print("IMPORTANT:")
print("No legacy COMBINED CSV was used as the analysis source.")
print("All numerical analysis was reconstructed from EXTXYZ.")
print()
print("Output directory:")
print(OUT)

print("\nGenerated files:")

for filename in sorted(
    os.listdir(OUT)
):

    print(
        " -",
        filename
    )

print("\n" + "=" * 80)
print("READY FOR SCIENTIFIC INTERPRETATION")
print("=" * 80)

In [ ]:
%env LD_LIBRARY_PATH=/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib:/usr/lib64-nvidia

In [ ]:
# =============================================================================
# CANDIDATE 81 — FINAL FULLY-FLEXIBLE VARIABLE-CELL NPT THERMAL MD
# RESTART-SAFE VERSION
# =============================================================================

import os
import glob
import json
import pickle
import hashlib

import numpy as np
import pandas as pd
import torch

from ase import units
from ase.io import read, write
from ase.geometry import cell_to_cellpar
from ase.md.npt import NPT
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

from mace.calculators import mace_mp


# =============================================================================
# 1. DRIVE
# =============================================================================

from google.colab import drive

drive.mount("/content/drive")


# =============================================================================
# 2. PATHS
# =============================================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

PKL_FILE = os.path.join(
    BASE_DIR,
    "MACE_TOP5_candidate_to_structure.pkl"
)

THERMAL_ROOT = os.path.join(
    BASE_DIR,
    "thermal_stability_candidate_81"
)

BASE_OUTPUT = os.path.join(
    THERMAL_ROOT,
    "FINAL_FULLY_FLEXIBLE_NPT"
)


# -----------------------------------------------------------------------------
# IMPORTANT:
#
# Determine a NEW output directory.
# Never overwrite an old run.
#
# Old FINAL_FULLY_FLEXIBLE_NPT* directories remain available for restart
# discovery and are READ ONLY from this script's point of view.
# -----------------------------------------------------------------------------

OUTPUT_DIR = BASE_OUTPUT

if os.path.exists(OUTPUT_DIR):

    version = 2

    while os.path.exists(
        f"{BASE_OUTPUT}_v{version}"
    ):
        version += 1

    OUTPUT_DIR = (
        f"{BASE_OUTPUT}_v{version}"
    )

os.makedirs(
    OUTPUT_DIR,
    exist_ok=False
)


# =============================================================================
# 3. GLOBAL PARAMETERS
# =============================================================================

CANDIDATE_ID = 81

TEMPERATURES_K = [
    300,
    600,
    900,
    1200
]

SUPERCELL = (2, 2, 2)

N_ATOMS_EXPECTED = 40

TIMESTEP_FS = 0.5

EQUILIBRATION_STEPS = 4000

PRODUCTION_STEPS = 20000

FRAME_INTERVAL = 1000

CHECKPOINT_INTERVAL = 2000

EXPECTED_PRODUCTION_FRAMES = (
    PRODUCTION_STEPS // FRAME_INTERVAL + 1
)

assert EXPECTED_PRODUCTION_FRAMES == 21


# =============================================================================
# 4. FULLY FLEXIBLE NPT PARAMETERS
# =============================================================================

# Target external pressure:
# 0 GPa = 0 eV/A^3

EXTERNAL_PRESSURE_GPA = 0.0

EXTERNAL_PRESSURE_AU = (
    EXTERNAL_PRESSURE_GPA
    * units.GPa
)


# -----------------------------------------------------------------------------
# Thermostat time scale
# -----------------------------------------------------------------------------

THERMOSTAT_TIME_FS = 25.0


# -----------------------------------------------------------------------------
# Barostat time scale
# -----------------------------------------------------------------------------

BAROSTAT_TIME_FS = 100.0


# -----------------------------------------------------------------------------
# Approximate bulk modulus used ONLY to construct pfactor.
#
# This is a dynamical parameter.
# It is NOT a measured material property.
# -----------------------------------------------------------------------------

BULK_MODULUS_GPA = 100.0

BULK_MODULUS_AU = (
    BULK_MODULUS_GPA
    * units.GPa
)

PFAC = (
    (BAROSTAT_TIME_FS * units.fs) ** 2
    * BULK_MODULUS_AU
)


# =============================================================================
# 5. PROTOCOL FINGERPRINT
# =============================================================================

PROTOCOL_DEFINITION = {

    "protocol_name":
        "candidate81_final_fully_flexible_npt_restart_safe_v1",

    "candidate_id":
        CANDIDATE_ID,

    "formula":
        "Ti4Si",

    "supercell":
        list(SUPERCELL),

    "n_atoms":
        N_ATOMS_EXPECTED,

    "temperatures_K":
        list(TEMPERATURES_K),

    "timestep_fs":
        TIMESTEP_FS,

    "equilibration_steps":
        EQUILIBRATION_STEPS,

    "production_steps":
        PRODUCTION_STEPS,

    "frame_interval":
        FRAME_INTERVAL,

    "checkpoint_interval":
        CHECKPOINT_INTERVAL,

    "external_pressure_GPa":
        EXTERNAL_PRESSURE_GPA,

    "thermostat_time_fs":
        THERMOSTAT_TIME_FS,

    "barostat_time_fs":
        BAROSTAT_TIME_FS,

    "bulk_modulus_GPa_for_pfactor":
        BULK_MODULUS_GPA,

    "npt_mask":
        None,

    "mace_model":
        "medium",

    "dispersion":
        True,

    "default_dtype":
        "float64",
}


PROTOCOL_JSON = json.dumps(
    PROTOCOL_DEFINITION,
    sort_keys=True,
    separators=(",", ":")
)

PROTOCOL_HASH = hashlib.sha256(
    PROTOCOL_JSON.encode("utf-8")
).hexdigest()


# =============================================================================
# 6. HEADER
# =============================================================================

print("=" * 80)
print(
    "CANDIDATE 81 — FINAL FULLY-FLEXIBLE "
    "VARIABLE-CELL NPT THERMAL MD"
)
print("RESTART-SAFE VERSION")
print("=" * 80)

print("Source PKL :", PKL_FILE)
print("Candidate  :", CANDIDATE_ID)
print("Temperatures:", TEMPERATURES_K)
print("Supercell  :", SUPERCELL)
print("Expected atoms:", N_ATOMS_EXPECTED)
print("Equilibration:", EQUILIBRATION_STEPS)
print("Production   :", PRODUCTION_STEPS)
print("Timestep     :", TIMESTEP_FS, "fs")
print("Frame interval:", FRAME_INTERVAL)
print(
    "Expected production frames:",
    EXPECTED_PRODUCTION_FRAMES
)
print(
    "Checkpoint interval:",
    CHECKPOINT_INTERVAL
)
print("Cell:", "FULLY FLEXIBLE")
print(
    "Pressure:",
    EXTERNAL_PRESSURE_GPA,
    "GPa"
)
print("Protocol hash:", PROTOCOL_HASH)
print(
    "New output directory:",
    OUTPUT_DIR
)

print("=" * 80)


# =============================================================================
# 7. LOAD MACE
# =============================================================================

print()
print("=" * 80)
print("LOADING MACE")
print("=" * 80)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64",
    device=DEVICE
)

print("MACE   : medium")
print("D3     : True")
print("dtype  : float64")


# =============================================================================
# 8. LOAD CANDIDATE 81 FROM PKL
# =============================================================================

print()
print("=" * 80)
print("LOADING CANDIDATE 81 FROM PKL")
print("=" * 80)

if not os.path.exists(PKL_FILE):

    raise FileNotFoundError(
        f"\nPKL file not found:\n{PKL_FILE}\n"
    )


with open(
    PKL_FILE,
    "rb"
) as f:

    candidate_structures = pickle.load(f)


print(
    "PKL type:",
    type(candidate_structures)
)


if not isinstance(
    candidate_structures,
    dict
):

    raise TypeError(
        "Expected the PKL to contain a dictionary "
        "of candidate structures."
    )


if CANDIDATE_ID in candidate_structures:

    atoms = (
        candidate_structures[
            CANDIDATE_ID
        ]
        .copy()
    )

elif str(CANDIDATE_ID) in candidate_structures:

    atoms = (
        candidate_structures[
            str(CANDIDATE_ID)
        ]
        .copy()
    )

else:

    raise KeyError(
        f"Candidate {CANDIDATE_ID} "
        f"was not found in PKL.\n"
        f"Available keys: "
        f"{list(candidate_structures.keys())[:30]}"
    )


# =============================================================================
# 9. VALIDATE ORIGINAL CANDIDATE
# =============================================================================

print()
print("=" * 80)
print("REFERENCE CANDIDATE")
print("=" * 80)

print(
    "Candidate ID:",
    CANDIDATE_ID
)

print(
    "Formula:",
    atoms.get_chemical_formula()
)

print(
    "Atoms:",
    len(atoms)
)

print(
    "Volume:",
    atoms.get_volume(),
    "A^3"
)

print(
    "PBC:",
    atoms.get_pbc()
)


if len(atoms) != 5:

    raise ValueError(
        f"Candidate 81 is expected to contain "
        f"5 atoms (Ti4Si), but contains {len(atoms)}."
    )


formula = atoms.get_chemical_formula()

if formula not in (
    "SiTi4",
    "Ti4Si"
):

    raise ValueError(
        f"Candidate 81 does not look like Ti4Si. "
        f"Detected formula: {formula}"
    )


# Ensure full periodicity

atoms.set_pbc(
    [True, True, True]
)


# =============================================================================
# 10. BUILD 2x2x2 SUPERCELL
# =============================================================================

print()
print("=" * 80)
print("BUILDING 2x2x2 SUPERCELL")
print("=" * 80)

atoms = atoms.repeat(
    SUPERCELL
)

print(
    "Formula:",
    atoms.get_chemical_formula()
)

print(
    "Atoms:",
    len(atoms)
)

print(
    "Volume:",
    atoms.get_volume(),
    "A^3"
)

print(
    "PBC:",
    atoms.get_pbc()
)


if len(atoms) != N_ATOMS_EXPECTED:

    raise ValueError(
        f"Supercell contains {len(atoms)} atoms, "
        f"expected {N_ATOMS_EXPECTED}."
    )


# =============================================================================
# 11. CONVERT CELL TO TRIANGULAR STANDARD FORM
# =============================================================================

print()
print("=" * 80)
print("STANDARDIZING INITIAL CELL")
print("=" * 80)

old_cell = atoms.cell.copy()

new_cell, rotation = (
    atoms.cell.standard_form(
        form="upper"
    )
)


# Convert Cartesian positions consistently
# with the cell rotation.

new_positions = (
    atoms.positions
    @ rotation.T
)

atoms.set_positions(
    new_positions
)

atoms.set_cell(
    new_cell,
    scale_atoms=False
)

atoms.wrap()


print(
    "Initial cell before standardization:"
)

print(old_cell)

print()

print(
    "Initial cell used by NPT:"
)

print(atoms.cell)


# Verify triangular form

cell_array = np.array(
    atoms.cell
)

lower_triangle = np.tril(
    cell_array,
    k=-1
)

if not np.allclose(
    lower_triangle,
    0.0,
    atol=1e-10
):

    raise RuntimeError(
        "Cell is not triangular after standardization."
    )


# =============================================================================
# 12. ATTACH CALCULATOR
# =============================================================================

atoms.calc = calc


# =============================================================================
# 13. HELPER FUNCTIONS
# =============================================================================

def cell_metadata(system):

    cp = cell_to_cellpar(
        system.cell
    )

    return {

        "a_A":
            float(cp[0]),

        "b_A":
            float(cp[1]),

        "c_A":
            float(cp[2]),

        "alpha_deg":
            float(cp[3]),

        "beta_deg":
            float(cp[4]),

        "gamma_deg":
            float(cp[5]),

        "volume_A3":
            float(system.get_volume())
    }


def full_metadata_row(
    system,
    temperature_target,
    production_step
):

    potential = float(
        system.get_potential_energy()
    )

    kinetic = float(
        system.get_kinetic_energy()
    )

    total = (
        potential
        +
        kinetic
    )

    temp_now = float(
        system.get_temperature()
    )

    meta = cell_metadata(
        system
    )

    return {

        "candidate_id":
            CANDIDATE_ID,

        "target_temperature_K":
            float(temperature_target),

        "production_step":
            int(production_step),

        "potential_energy_eV":
            potential,

        "kinetic_energy_eV":
            kinetic,

        "total_energy_eV":
            total,

        "temperature_K":
            temp_now,

        "volume_A3":
            meta["volume_A3"],

        "a_A":
            meta["a_A"],

        "b_A":
            meta["b_A"],

        "c_A":
            meta["c_A"],

        "alpha_deg":
            meta["alpha_deg"],

        "beta_deg":
            meta["beta_deg"],

        "gamma_deg":
            meta["gamma_deg"],

        "cell_xx":
            float(system.cell[0, 0]),

        "cell_xy":
            float(system.cell[0, 1]),

        "cell_xz":
            float(system.cell[0, 2]),

        "cell_yx":
            float(system.cell[1, 0]),

        "cell_yy":
            float(system.cell[1, 1]),

        "cell_yz":
            float(system.cell[1, 2]),

        "cell_zx":
            float(system.cell[2, 0]),

        "cell_zy":
            float(system.cell[2, 1]),

        "cell_zz":
            float(system.cell[2, 2])
    }


def put_frame_metadata_in_info(
    system,
    temperature_target,
    production_step,
    checkpoint=False,
    precomputed_row=None
):

    if precomputed_row is not None:

        row = precomputed_row

    else:

        row = full_metadata_row(
            system,
            temperature_target,
            production_step
        )


    system.info["candidate_id"] = CANDIDATE_ID

    system.info["formula"] = "Ti4Si"

    system.info["supercell_formula"] = "Ti32Si8"

    system.info["target_temperature_K"] = float(
        temperature_target
    )

    system.info["production_step"] = int(
        production_step
    )

    system.info["potential_energy_eV"] = row[
        "potential_energy_eV"
    ]

    system.info["kinetic_energy_eV"] = row[
        "kinetic_energy_eV"
    ]

    system.info["total_energy_eV"] = row[
        "total_energy_eV"
    ]

    system.info["temperature_K"] = row[
        "temperature_K"
    ]

    system.info["volume_A3"] = row[
        "volume_A3"
    ]

    system.info["a_A"] = row["a_A"]
    system.info["b_A"] = row["b_A"]
    system.info["c_A"] = row["c_A"]

    system.info["alpha_deg"] = row[
        "alpha_deg"
    ]

    system.info["beta_deg"] = row[
        "beta_deg"
    ]

    system.info["gamma_deg"] = row[
        "gamma_deg"
    ]

    system.info["cell_xx"] = row["cell_xx"]
    system.info["cell_xy"] = row["cell_xy"]
    system.info["cell_xz"] = row["cell_xz"]

    system.info["cell_yx"] = row["cell_yx"]
    system.info["cell_yy"] = row["cell_yy"]
    system.info["cell_yz"] = row["cell_yz"]

    system.info["cell_zx"] = row["cell_zx"]
    system.info["cell_zy"] = row["cell_zy"]
    system.info["cell_zz"] = row["cell_zz"]

    system.info["checkpoint"] = bool(
        checkpoint
    )

    system.info["protocol_hash"] = (
        PROTOCOL_HASH
    )

    return row


def save_frame(
    system,
    temperature_target,
    production_step,
    output_file,
    checkpoint=False
):

    row = put_frame_metadata_in_info(
        system,
        temperature_target,
        production_step,
        checkpoint=checkpoint
    )

    write(
        output_file,
        system,
        format="extxyz",
        append=True
    )

    return row


def atomic_write_json(
    final_file,
    payload
):

    temp_file = (
        final_file
        + ".tmp"
    )

    with open(
        temp_file,
        "w"
    ) as f:

        json.dump(
            payload,
            f,
            indent=2,
            sort_keys=True
        )

        f.flush()

        os.fsync(
            f.fileno()
        )

    os.replace(
        temp_file,
        final_file
    )


def valid_composition(
    system
):

    symbols = (
        system.get_chemical_symbols()
    )

    return (
        len(symbols) == N_ATOMS_EXPECTED
        and symbols.count("Ti") == 32
        and symbols.count("Si") == 8
    )


def expected_frame_steps_through(
    checkpoint_step
):

    return list(
        range(
            0,
            checkpoint_step + 1,
            FRAME_INTERVAL
        )
    )


def find_source_trajectory(
    temp_dir
):

    candidates = sorted(
        glob.glob(
            os.path.join(
                temp_dir,
                "candidate_81_*K_FULLY_FLEXIBLE.extxyz"
            )
        )
    )

    if not candidates:
        return None

    candidates.sort(
        key=lambda p: os.path.getmtime(p),
        reverse=True
    )

    return candidates[0]


def validate_source_trajectory(
    trajectory_file,
    target_temperature,
    checkpoint_step
):
    """
    The source trajectory must contain every production frame:

        0, 1000, 2000, ..., checkpoint_step

    before a checkpoint can be used.
    """

    if (
        trajectory_file is None
        or not os.path.isfile(trajectory_file)
    ):

        return (
            False,
            None,
            "source trajectory is missing"
        )


    try:

        frames = read(
            trajectory_file,
            index=":"
        )

    except Exception as exc:

        return (
            False,
            None,
            f"cannot read source trajectory: {exc}"
        )


    required_steps = (
        expected_frame_steps_through(
            checkpoint_step
        )
    )

    frame_by_step = {}


    for frame in frames:

        try:

            step = int(
                frame.info[
                    "production_step"
                ]
            )

            frame_temperature = float(
                frame.info[
                    "target_temperature_K"
                ]
            )

        except Exception:

            continue


        if not np.isclose(
            frame_temperature,
            float(target_temperature),
            rtol=0.0,
            atol=1e-8
        ):

            continue


        if step in required_steps:

            frame_by_step[
                step
            ] = frame


    if (
        set(frame_by_step.keys())
        !=
        set(required_steps)
    ):

        missing = sorted(
            set(required_steps)
            -
            set(frame_by_step.keys())
        )

        return (
            False,
            None,
            (
                "source trajectory is missing "
                f"production frames {missing}"
            )
        )


    ordered_frames = [

        frame_by_step[
            step
        ]

        for step in required_steps
    ]


    for step, frame in zip(
        required_steps,
        ordered_frames
    ):

        if not valid_composition(
            frame
        ):

            return (
                False,
                None,
                f"bad composition in source frame {step}"
            )


    return (
        True,
        ordered_frames,
        "OK"
    )


def copy_historical_frames_to_new_run(
    source_frames,
    trajectory_file,
    target_temperature
):

    metadata_rows = []


    for source_frame in source_frames:

        frame = source_frame.copy()

        step = int(
            frame.info[
                "production_step"
            ]
        )


        required_info = [

            "potential_energy_eV",
            "kinetic_energy_eV",
            "total_energy_eV",
            "temperature_K",
            "volume_A3",
            "a_A",
            "b_A",
            "c_A",
            "alpha_deg",
            "beta_deg",
            "gamma_deg",
        ]


        if not all(
            key in frame.info
            for key in required_info
        ):

            raise RuntimeError(
                f"Source frame {step} does not contain "
                "the required saved metadata."
            )


        write(
            trajectory_file,
            frame,
            format="extxyz",
            append=True
        )


        cp = np.asarray(
            frame.cell
        )


        metadata_rows.append({

            "candidate_id":
                CANDIDATE_ID,

            "target_temperature_K":
                float(target_temperature),

            "production_step":
                step,

            "potential_energy_eV":
                float(
                    frame.info[
                        "potential_energy_eV"
                    ]
                ),

            "kinetic_energy_eV":
                float(
                    frame.info[
                        "kinetic_energy_eV"
                    ]
                ),

            "total_energy_eV":
                float(
                    frame.info[
                        "total_energy_eV"
                    ]
                ),

            "temperature_K":
                float(
                    frame.info[
                        "temperature_K"
                    ]
                ),

            "volume_A3":
                float(
                    frame.info[
                        "volume_A3"
                    ]
                ),

            "a_A":
                float(
                    frame.info["a_A"]
                ),

            "b_A":
                float(
                    frame.info["b_A"]
                ),

            "c_A":
                float(
                    frame.info["c_A"]
                ),

            "alpha_deg":
                float(
                    frame.info["alpha_deg"]
                ),

            "beta_deg":
                float(
                    frame.info["beta_deg"]
                ),

            "gamma_deg":
                float(
                    frame.info["gamma_deg"]
                ),

            "cell_xx":
                float(cp[0, 0]),

            "cell_xy":
                float(cp[0, 1]),

            "cell_xz":
                float(cp[0, 2]),

            "cell_yx":
                float(cp[1, 0]),

            "cell_yy":
                float(cp[1, 1]),

            "cell_yz":
                float(cp[1, 2]),

            "cell_zx":
                float(cp[2, 0]),

            "cell_zy":
                float(cp[2, 1]),

            "cell_zz":
                float(cp[2, 2])
        })


    return metadata_rows


# =============================================================================
# 14. RESTART CHECKPOINT WRITER
# =============================================================================

def _validate_npt_init_data(
    init_data
):

    if not isinstance(
        init_data,
        dict
    ):

        return (
            False,
            "npt_init is not a dictionary"
        )


    required = {

        "dt",
        "temperature",
        "desiredEkin",
        "externalstress",
        "mask",
        "ttime",
        "tfact",
        "pfactor_given",
        "pfact",
        "frac_traceless",
    }


    missing = sorted(
        required
        -
        set(init_data)
    )


    if missing:

        return (
            False,
            f"npt_init missing keys: {missing}"
        )


    return (
        True,
        "OK"
    )


def _validate_npt_state_data(
    state_data
):

    if not isinstance(
        state_data,
        dict
    ):

        return (
            False,
            "npt_dynamics is not a dictionary"
        )


    required = {

        "eta",
        "eta_past",
        "zeta",
        "zeta_past",
        "zeta_integrated",
        "h",
        "h_past",
        "timeelapsed",
    }


    missing = sorted(
        required
        -
        set(state_data)
    )


    if missing:

        return (
            False,
            f"npt_dynamics missing keys: {missing}"
        )


    for key, value in state_data.items():

        try:

            arr = np.asarray(
                value
            )

            if arr.dtype.kind in "biufc":

                if not np.all(
                    np.isfinite(arr)
                ):

                    return (
                        False,
                        (
                            "npt_dynamics contains "
                            f"non-finite data: {key}"
                        )
                    )

        except Exception:

            return (
                False,
                (
                    "cannot validate "
                    f"npt_dynamics field: {key}"
                )
            )


    return (
        True,
        "OK"
    )


def _restore_npt_from_checkpoint(
    checkpoint_file
):
    """
    Restore:

        Atoms
        positions
        momenta
        cell
        PBC
        NPT initialization state
        NPT dynamical state

    from our custom pickle checkpoint.

    BundleTrajectory is intentionally NOT used.
    """

    with open(
        checkpoint_file,
        "rb"
    ) as f:

        payload = pickle.load(f)


    if not isinstance(
        payload,
        dict
    ):

        raise RuntimeError(
            "Checkpoint payload is not a dictionary."
        )


    required = {

        "format_version",
        "candidate_id",
        "target_temperature_K",
        "production_step",
        "protocol_hash",
        "protocol",
        "atoms",
        "npt_init",
        "npt_dynamics",
    }


    missing = sorted(
        required
        -
        set(payload)
    )


    if missing:

        raise RuntimeError(
            f"Checkpoint payload is missing "
            f"required fields: {missing}"
        )


    if int(
        payload["candidate_id"]
    ) != CANDIDATE_ID:

        raise RuntimeError(
            "Checkpoint candidate ID does not match."
        )


    if not np.isfinite(
        float(
            payload[
                "target_temperature_K"
            ]
        )
    ):

        raise RuntimeError(
            "Invalid checkpoint target temperature."
        )


    if payload[
        "protocol_hash"
    ] != PROTOCOL_HASH:

        raise RuntimeError(
            "Checkpoint protocol fingerprint does not match."
        )


    if int(
        payload[
            "format_version"
        ]
    ) != 1:

        raise RuntimeError(
            "Unsupported checkpoint format version."
        )


    if payload[
        "protocol"
    ] != PROTOCOL_DEFINITION:

        raise RuntimeError(
            "Checkpoint protocol definition does not match."
        )


    production_step = int(
        payload[
            "production_step"
        ]
    )


    if production_step <= 0:

        raise RuntimeError(
            "Checkpoint production step must be positive."
        )


    restored_atoms = (
        payload["atoms"].copy()
    )

    init_data = payload[
        "npt_init"
    ]

    state_data = payload[
        "npt_dynamics"
    ]


    ok, reason = (
        _validate_npt_init_data(
            init_data
        )
    )

    if not ok:

        raise RuntimeError(
            reason
        )


    ok, reason = (
        _validate_npt_state_data(
            state_data
        )
    )

    if not ok:

        raise RuntimeError(
            reason
        )


    if not valid_composition(
        restored_atoms
    ):

        raise RuntimeError(
            "Checkpoint has wrong composition."
        )


    if not np.all(
        restored_atoms.get_pbc()
    ):

        raise RuntimeError(
            "Checkpoint is not fully periodic."
        )


    momenta = (
        restored_atoms.get_momenta()
    )


    if (
        momenta is None
        or np.shape(momenta)
        != (N_ATOMS_EXPECTED, 3)
        or not np.all(
            np.isfinite(momenta)
        )
    ):

        raise RuntimeError(
            "Checkpoint momenta are missing or invalid."
        )


    positions = (
        restored_atoms.get_positions()
    )


    if not np.all(
        np.isfinite(positions)
    ):

        raise RuntimeError(
            "Checkpoint positions are invalid."
        )


    cell = np.asarray(
        restored_atoms.cell
    )


    if (
        cell.shape != (3, 3)
        or not np.all(
            np.isfinite(cell)
        )
        or restored_atoms.get_volume() <= 0.0
    ):

        raise RuntimeError(
            "Checkpoint cell is invalid."
        )


    # -------------------------------------------------------------------------
    # Recreate NPT object.
    # -------------------------------------------------------------------------

    dyn = NPT(

        restored_atoms,

        timestep=init_data[
            "dt"
        ],

        temperature_K=(
            init_data[
                "temperature"
            ]
            /
            units.kB
        ),

        externalstress=init_data[
            "externalstress"
        ],

        ttime=init_data[
            "ttime"
        ],

        pfactor=init_data[
            "pfact"
        ],

        mask=init_data[
            "mask"
        ],
    )


    # -------------------------------------------------------------------------
    # Restore internal NPT state.
    # -------------------------------------------------------------------------

    for key, value in (
        state_data.items()
    ):

        if isinstance(
            value,
            np.ndarray
        ):

            value = value.copy()

        setattr(
            dyn,
            key,
            value
        )


    # Explicitly restore NPT initialization values.

    dyn.dt = init_data[
        "dt"
    ]

    dyn.temperature = init_data[
        "temperature"
    ]

    dyn.desiredEkin = init_data[
        "desiredEkin"
    ]

    dyn.externalstress = init_data[
        "externalstress"
    ]

    dyn.mask = init_data[
        "mask"
    ]

    dyn.ttime = init_data[
        "ttime"
    ]

    dyn.tfact = init_data[
        "tfact"
    ]

    dyn.pfactor_given = init_data[
        "pfactor_given"
    ]

    dyn.pfact = init_data[
        "pfact"
    ]

    dyn.frac_traceless = init_data[
        "frac_traceless"
    ]


    return (
        dyn,
        restored_atoms
    )


def save_restart_checkpoint(
    dyn,
    system,
    target_temperature,
    production_step,
    checkpoint_dir,
    source_trajectory_file
):
    """
    Save a TRUE restart checkpoint.

    Checkpoint contains:

        positions
        momenta
        cell
        PBC
        NPT initialization state
        NPT dynamical state

    The checkpoint is written to a temporary file first.

    It is loaded back and reconstructed before being published.

    The manifest is written LAST.
    """

    if production_step <= 0:

        raise ValueError(
            "Restart checkpoint production_step must be > 0."
        )


    if (
        production_step
        %
        CHECKPOINT_INTERVAL
        != 0
    ):

        raise ValueError(
            f"{production_step} is not a valid "
            "checkpoint step."
        )


    os.makedirs(
        checkpoint_dir,
        exist_ok=True
    )


    checkpoint_name = (
        f"candidate_81_{target_temperature}K_"
        f"restart_step_{production_step:06d}.pkl"
    )


    manifest_name = (
        f"candidate_81_{target_temperature}K_"
        f"restart_step_{production_step:06d}.json"
    )


    final_checkpoint = os.path.join(
        checkpoint_dir,
        checkpoint_name
    )


    final_manifest = os.path.join(
        checkpoint_dir,
        manifest_name
    )


    temporary_checkpoint = (
        final_checkpoint
        +
        ".tmp"
    )


    if os.path.exists(
        temporary_checkpoint
    ):

        os.remove(
            temporary_checkpoint
        )


    if os.path.exists(
        final_checkpoint
    ):

        raise FileExistsError(
            f"Checkpoint already exists: "
            f"{final_checkpoint}"
        )


    # -------------------------------------------------------------------------
    # IMPORTANT:
    #
    # Calculate metadata while the live system still has the calculator.
    # -------------------------------------------------------------------------

    checkpoint_row = full_metadata_row(
        system,
        target_temperature,
        production_step
    )


    # -------------------------------------------------------------------------
    # Detached copy.
    #
    # This copy intentionally does NOT carry the MACE calculator.
    # -------------------------------------------------------------------------

    checkpoint_atoms = (
        system.copy()
    )


    put_frame_metadata_in_info(

        checkpoint_atoms,

        target_temperature,

        production_step,

        checkpoint=True,

        precomputed_row=checkpoint_row
    )


    # -------------------------------------------------------------------------
    # Capture NPT state.
    # -------------------------------------------------------------------------

    init_data = (
        dyn.get_init_data()
    )

    state_data = (
        dyn.get_data()
    )


    ok, reason = (
        _validate_npt_init_data(
            init_data
        )
    )


    if not ok:

        raise RuntimeError(
            "Cannot save NPT initialization state: "
            f"{reason}"
        )


    ok, reason = (
        _validate_npt_state_data(
            state_data
        )
    )


    if not ok:

        raise RuntimeError(
            "Cannot save NPT dynamical state: "
            f"{reason}"
        )


    # -------------------------------------------------------------------------
    # Build payload.
    # -------------------------------------------------------------------------

    payload = {

        "format_version":
            1,

        "candidate_id":
            CANDIDATE_ID,

        "formula":
            "Ti4Si",

        "supercell_formula":
            "Ti32Si8",

        "target_temperature_K":
            float(target_temperature),

        "production_step":
            int(production_step),

        "protocol_hash":
            PROTOCOL_HASH,

        "protocol":
            PROTOCOL_DEFINITION,

        "atoms":
            checkpoint_atoms,

        "npt_init":
            init_data,

        "npt_dynamics":
            state_data,
    }


    # -------------------------------------------------------------------------
    # Write temporary checkpoint.
    # -------------------------------------------------------------------------

    try:

        with open(
            temporary_checkpoint,
            "wb"
        ) as f:

            pickle.dump(
                payload,
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )

            f.flush()

            os.fsync(
                f.fileno()
            )


        # ---------------------------------------------------------------------
        # Decisive verification:
        #
        # Load checkpoint and reconstruct a fresh NPT object.
        # ---------------------------------------------------------------------

        test_dyn, test_atoms = (
            _restore_npt_from_checkpoint(
                temporary_checkpoint
            )
        )


        if not valid_composition(
            test_atoms
        ):

            raise RuntimeError(
                "Checkpoint verification failed: "
                "wrong composition."
            )


        if test_atoms.get_momenta() is None:

            raise RuntimeError(
                "Checkpoint verification failed: "
                "momenta missing."
            )


        if not np.all(
            np.isfinite(
                test_atoms.get_positions()
            )
        ):

            raise RuntimeError(
                "Checkpoint verification failed: "
                "invalid positions."
            )


        if test_atoms.get_volume() <= 0.0:

            raise RuntimeError(
                "Checkpoint verification failed: "
                "invalid cell volume."
            )


        del test_dyn


    except Exception:

        if os.path.isfile(
            temporary_checkpoint
        ):

            os.remove(
                temporary_checkpoint
            )

        raise


    # -------------------------------------------------------------------------
    # Publish checkpoint atomically.
    # -------------------------------------------------------------------------

    os.replace(
        temporary_checkpoint,
        final_checkpoint
    )


    # -------------------------------------------------------------------------
    # Manifest is written LAST.
    # -------------------------------------------------------------------------

    manifest = {

        "complete":
            True,

        "checkpoint_format":
            "pickle_npt_state_v1",

        "candidate_id":
            CANDIDATE_ID,

        "formula":
            "Ti4Si",

        "supercell_formula":
            "Ti32Si8",

        "target_temperature_K":
            float(target_temperature),

        "production_step":
            int(production_step),

        "checkpoint_interval":
            CHECKPOINT_INTERVAL,

        "protocol_hash":
            PROTOCOL_HASH,

        "protocol":
            PROTOCOL_DEFINITION,

        "checkpoint_file":
            checkpoint_name,

        "source_trajectory_file":
            os.path.basename(
                source_trajectory_file
            ),
    }


    atomic_write_json(
        final_manifest,
        manifest
    )


    return (
        final_checkpoint,
        final_manifest
    )


# =============================================================================
# 15. RESTART CHECKPOINT VALIDATION
# =============================================================================

def validate_restart_candidate(
    manifest_file,
    target_temperature
):

    # -------------------------------------------------------------------------
    # Read manifest.
    # -------------------------------------------------------------------------

    try:

        with open(
            manifest_file,
            "r"
        ) as f:

            manifest = json.load(f)

    except Exception as exc:

        return (
            False,
            None,
            f"manifest unreadable: {exc}"
        )


    try:

        if manifest.get(
            "complete",
            False
        ) is not True:

            return (
                False,
                None,
                "manifest is not marked complete"
            )


        if manifest.get(
            "checkpoint_format"
        ) != "pickle_npt_state_v1":

            return (
                False,
                None,
                "unsupported checkpoint format"
            )


        if int(
            manifest.get(
                "candidate_id",
                -1
            )
        ) != CANDIDATE_ID:

            return (
                False,
                None,
                "wrong candidate"
            )


        if not np.isclose(

            float(
                manifest.get(
                    "target_temperature_K",
                    np.nan
                )
            ),

            float(target_temperature),

            rtol=0.0,

            atol=1e-8
        ):

            return (
                False,
                None,
                "wrong target temperature"
            )


        step = int(
            manifest[
                "production_step"
            ]
        )


        if (
            step <= 0
            or step > PRODUCTION_STEPS
            or step % CHECKPOINT_INTERVAL != 0
        ):

            return (
                False,
                None,
                "not a real production checkpoint step"
            )


        if manifest.get(
            "protocol_hash"
        ) != PROTOCOL_HASH:

            return (
                False,
                None,
                "protocol fingerprint does not match"
            )


        if manifest.get(
            "protocol"
        ) != PROTOCOL_DEFINITION:

            return (
                False,
                None,
                "protocol definition does not match"
            )


        checkpoint_name = manifest[
            "checkpoint_file"
        ]


        checkpoint_dir = os.path.dirname(
            manifest_file
        )


        checkpoint_file = os.path.join(
            checkpoint_dir,
            checkpoint_name
        )


        if not os.path.isfile(
            checkpoint_file
        ):

            return (
                False,
                None,
                "checkpoint file is missing"
            )


        restored_dyn, restored_atoms = (
            _restore_npt_from_checkpoint(
                checkpoint_file
            )
        )


        if not valid_composition(
            restored_atoms
        ):

            return (
                False,
                None,
                "restored system has wrong composition"
            )


        if not np.all(
            restored_atoms.get_pbc()
        ):

            return (
                False,
                None,
                "restored system is not fully periodic"
            )


        momenta = (
            restored_atoms.get_momenta()
        )


        if (
            momenta is None
            or np.shape(momenta)
            != (N_ATOMS_EXPECTED, 3)
            or not np.all(
                np.isfinite(momenta)
            )
        ):

            return (
                False,
                None,
                "restored momenta are missing or invalid"
            )


        positions = (
            restored_atoms.get_positions()
        )


        if not np.all(
            np.isfinite(positions)
        ):

            return (
                False,
                None,
                "restored positions are invalid"
            )


        cell = np.asarray(
            restored_atoms.cell
        )


        if (
            cell.shape != (3, 3)
            or not np.all(
                np.isfinite(cell)
            )
            or restored_atoms.get_volume() <= 0.0
        ):

            return (
                False,
                None,
                "restored cell is invalid"
            )


        # ---------------------------------------------------------------------
        # The source trajectory must live in the SAME run-temperature
        # directory as this checkpoint.
        # ---------------------------------------------------------------------

        temp_dir = os.path.dirname(
            checkpoint_dir
        )


        source_trajectory = os.path.join(

            temp_dir,

            manifest[
                "source_trajectory_file"
            ]
        )


        ok, source_frames, reason = (
            validate_source_trajectory(

                source_trajectory,

                target_temperature,

                step
            )
        )


        if not ok:

            return (
                False,
                None,
                reason
            )


        candidate = {

            "manifest_file":
                manifest_file,

            "checkpoint_file":
                checkpoint_file,

            "source_trajectory":
                source_trajectory,

            "source_frames":
                source_frames,

            "production_step":
                step,

            "restored_dyn":
                restored_dyn,

            "restored_atoms":
                restored_atoms,

            "mtime":
                os.path.getmtime(
                    manifest_file
                ),
        }


        return (
            True,
            candidate,
            "OK"
        )


    except Exception as exc:

        return (
            False,
            None,
            f"checkpoint validation exception: {exc}"
        )


# =============================================================================
# 16. SEARCH ALL RUNS FOR BEST CHECKPOINT
# =============================================================================

def find_best_restart_checkpoint(
    target_temperature
):
    """
    Search every:

        FINAL_FULLY_FLEXIBLE_NPT*

    directory under THERMAL_ROOT.

    Selection:

        1. Highest valid production step.
        2. If tied, newest manifest.
    """

    run_dirs = sorted(

        path

        for path in glob.glob(
            os.path.join(
                THERMAL_ROOT,
                "FINAL_FULLY_FLEXIBLE_NPT*"
            )
        )

        if os.path.isdir(path)
    )


    valid_candidates = []


    print()

    print(
        f"{target_temperature} K: searching "
        f"{len(run_dirs)} FINAL_FULLY_FLEXIBLE_NPT* "
        "run directories for restart checkpoints..."
    )


    for run_dir in run_dirs:

        temp_dir = os.path.join(
            run_dir,
            f"{target_temperature}K"
        )


        checkpoint_dir = os.path.join(
            temp_dir,
            "checkpoints"
        )


        if not os.path.isdir(
            checkpoint_dir
        ):

            continue


        manifest_pattern = os.path.join(

            checkpoint_dir,

            (
                f"candidate_81_{target_temperature}K_"
                "restart_step_*.json"
            )
        )


        manifest_files = sorted(
            glob.glob(
                manifest_pattern
            )
        )


        if not manifest_files:

            print(
                "  NO MANIFESTS:",
                checkpoint_dir
            )

            continue


        for manifest_file in manifest_files:

            ok, candidate, reason = (
                validate_restart_candidate(

                    manifest_file,

                    target_temperature
                )
            )


            if ok:

                valid_candidates.append(
                    candidate
                )


                print(

                    "  VALID:",

                    os.path.basename(
                        run_dir
                    ),

                    "| step",

                    candidate[
                        "production_step"
                    ]
                )


            else:

                print(

                    "  IGNORE:",

                    manifest_file,

                    "|",

                    reason
                )


    if not valid_candidates:

        print(

            f"{target_temperature} K: "
            "no valid restart checkpoint found."
        )

        return None


    valid_candidates.sort(

        key=lambda item: (

            item[
                "production_step"
            ],

            item[
                "mtime"
            ]
        ),

        reverse=True
    )


    best = valid_candidates[0]


    print(

        f"{target_temperature} K: selected restart "
        "checkpoint at production step "
        f"{best['production_step']}"
    )


    print(
        "  Checkpoint:",
        best["checkpoint_file"]
    )


    return best


# =============================================================================
# 17. CREATE NPT DYNAMICS
# =============================================================================

def create_fresh_npt(
    system,
    target_temperature,
    log_file
):

    return NPT(

        system,

        timestep=(
            TIMESTEP_FS
            * units.fs
        ),

        temperature_K=target_temperature,

        externalstress=EXTERNAL_PRESSURE_AU,

        ttime=(
            THERMOSTAT_TIME_FS
            * units.fs
        ),

        pfactor=PFAC,

        mask=None,

        logfile=log_file,

        loginterval=100
    )


def attach_restart_log(
    dyn,
    system,
    log_file,
    target_temperature,
    start_production_step
):
    """
    Attach a fresh log for the new run.

    This does not modify the NPT equations of motion.
    """

    header_needed = not os.path.exists(
        log_file
    )


    if header_needed:

        with open(
            log_file,
            "w"
        ) as f:

            f.write(
                "# restarted NPT log\n"
            )

            f.write(
                "# production_step target_T_K "
                "actual_T_K potential_eV "
                "kinetic_eV total_eV volume_A3\n"
            )


    restart_counter = {

        "production_step":
            int(start_production_step)
    }


    def write_restart_log():

        potential = float(
            system.get_potential_energy()
        )

        kinetic = float(
            system.get_kinetic_energy()
        )


        with open(
            log_file,
            "a"
        ) as f:

            f.write(

                f"{restart_counter['production_step']} "

                f"{float(target_temperature):.8f} "

                f"{float(system.get_temperature()):.8f} "

                f"{potential:.16e} "

                f"{kinetic:.16e} "

                f"{(potential + kinetic):.16e} "

                f"{float(system.get_volume()):.16e}\n"
            )


    def advance_restart_counter():

        restart_counter[
            "production_step"
        ] += 100

        write_restart_log()


    dyn.attach(
        advance_restart_counter,
        interval=100
    )


# =============================================================================
# 18. RUN ONE TEMPERATURE
# =============================================================================

def run_temperature(
    target_temperature
):

    print()
    print("=" * 80)

    print(
        f"STARTING {target_temperature} K — "
        "FULLY FLEXIBLE VARIABLE-CELL NPT"
    )

    print("=" * 80)


    # -------------------------------------------------------------------------
    # Files for THIS new run only.
    # -------------------------------------------------------------------------

    temp_dir = os.path.join(

        OUTPUT_DIR,

        f"{target_temperature}K"
    )


    os.makedirs(
        temp_dir,
        exist_ok=False
    )


    trajectory_file = os.path.join(

        temp_dir,

        (
            f"candidate_81_"
            f"{target_temperature}K_"
            "FULLY_FLEXIBLE.extxyz"
        )
    )


    checkpoint_dir = os.path.join(

        temp_dir,

        "checkpoints"
    )


    os.makedirs(
        checkpoint_dir,
        exist_ok=False
    )


    log_file = os.path.join(

        temp_dir,

        (
            f"candidate_81_"
            f"{target_temperature}K_"
            "step_log.txt"
        )
    )


    summary_file = os.path.join(

        temp_dir,

        (
            f"candidate_81_"
            f"{target_temperature}K_"
            "frame_metadata.csv"
        )
    )


    metadata_rows = []


    # -------------------------------------------------------------------------
    # Search ALL old/current FINAL_FULLY_FLEXIBLE_NPT* directories.
    # -------------------------------------------------------------------------

    restart = find_best_restart_checkpoint(
        target_temperature
    )


    # =========================================================================
    # CASE A — VALID RESTART
    # =========================================================================

    if restart is not None:

        start_production_step = int(
            restart[
                "production_step"
            ]
        )


        dyn = restart[
            "restored_dyn"
        ]


        system = restart[
            "restored_atoms"
        ]


        # Reattach calculator to the restored Atoms.

        system.calc = calc


        # ---------------------------------------------------------------------
        # Copy historical frames into THIS new run.
        #
        # Old trajectory remains READ ONLY.
        # ---------------------------------------------------------------------

        metadata_rows = (
            copy_historical_frames_to_new_run(

                restart[
                    "source_frames"
                ],

                trajectory_file,

                target_temperature
            )
        )


        attach_restart_log(

            dyn,

            system,

            log_file,

            target_temperature,

            start_production_step
        )


        print(

            f"{target_temperature} K: "
            "RESTARTING from production step "
            f"{start_production_step}"
        )


        print(

            f"{target_temperature} K: copied "
            f"{len(metadata_rows)} historical production "
            "frames into the new run trajectory."
        )


        # ---------------------------------------------------------------------
        # Publish selected restart state into CURRENT run.
        #
        # This makes the new run independently restartable.
        # ---------------------------------------------------------------------

        if start_production_step > 0:

            save_restart_checkpoint(

                dyn,

                system,

                target_temperature,

                start_production_step,

                checkpoint_dir,

                trajectory_file
            )


    # =========================================================================
    # CASE B — NO VALID CHECKPOINT
    # =========================================================================

    else:

        start_production_step = 0


        system = atoms.copy()

        system.calc = calc


        MaxwellBoltzmannDistribution(

            system,

            temperature_K=target_temperature
        )


        dyn = create_fresh_npt(

            system,

            target_temperature,

            log_file
        )


        print(

            f"{target_temperature} K: "
            "no valid checkpoint -> "
            "starting this temperature from zero."
        )


        print(

            f"{target_temperature} K: "
            f"equilibration = "
            f"{EQUILIBRATION_STEPS} steps"
        )


        # ---------------------------------------------------------------------
        # Equilibration
        # ---------------------------------------------------------------------

        dyn.run(
            EQUILIBRATION_STEPS
        )


        # ---------------------------------------------------------------------
        # Production frame 0
        # ---------------------------------------------------------------------

        row = save_frame(

            system,

            target_temperature,

            0,

            trajectory_file,

            checkpoint=False
        )


        metadata_rows.append(
            row
        )


    # =========================================================================
    # ALREADY COMPLETE?
    # =========================================================================

    if (
        start_production_step
        ==
        PRODUCTION_STEPS
    ):

        print(

            f"{target_temperature} K: "
            "selected checkpoint is already the final "
            f"production step {PRODUCTION_STEPS}."
        )


    # =========================================================================
    # CONTINUE PRODUCTION
    # =========================================================================

    else:

        next_frame_step = (

            start_production_step
            +
            FRAME_INTERVAL
        )


        for production_step in range(

            next_frame_step,

            PRODUCTION_STEPS + 1,

            FRAME_INTERVAL
        ):

            # -----------------------------------------------------------------
            # Advance exactly 1000 MD steps.
            # -----------------------------------------------------------------

            dyn.run(
                FRAME_INTERVAL
            )


            is_checkpoint = (

                production_step
                %
                CHECKPOINT_INTERVAL
                ==
                0
            )


            # -----------------------------------------------------------------
            # Save production frame.
            # -----------------------------------------------------------------

            row = save_frame(

                system,

                target_temperature,

                production_step,

                trajectory_file,

                checkpoint=is_checkpoint
            )


            metadata_rows.append(
                row
            )


            # -----------------------------------------------------------------
            # Save TRUE restart checkpoint every 2000 steps.
            # -----------------------------------------------------------------

            if is_checkpoint:

                save_restart_checkpoint(

                    dyn,

                    system,

                    target_temperature,

                    production_step,

                    checkpoint_dir,

                    trajectory_file
                )


                print(

                    f"{target_temperature} K | "

                    f"production step "
                    f"{production_step:5d} / "
                    f"{PRODUCTION_STEPS} | "

                    "RESTART CHECKPOINT SAVED"
                )


            else:

                print(

                    f"{target_temperature} K | "

                    f"production step "
                    f"{production_step:5d} / "
                    f"{PRODUCTION_STEPS}"
                )


    # =========================================================================
    # SAVE METADATA CSV
    # =========================================================================

    metadata_df = pd.DataFrame(
        metadata_rows
    )


    metadata_df = (

        metadata_df

        .sort_values(
            "production_step"
        )

        .drop_duplicates(
            subset=[
                "production_step"
            ],
            keep="last"
        )

        .reset_index(
            drop=True
        )
    )


    metadata_df.to_csv(

        summary_file,

        index=False
    )


    # =========================================================================
    # FINAL VALIDATION
    # =========================================================================

    expected_steps = list(

        range(

            0,

            PRODUCTION_STEPS + 1,

            FRAME_INTERVAL
        )
    )


    actual_steps = (

        metadata_df[
            "production_step"
        ]

        .astype(int)

        .tolist()
    )


    if actual_steps != expected_steps:

        raise RuntimeError(

            f"{target_temperature} K: "
            "production frame sequence is wrong.\n"

            f"Expected: {expected_steps}\n"

            f"Actual:   {actual_steps}"
        )


    if len(
        metadata_df
    ) != EXPECTED_PRODUCTION_FRAMES:

        raise RuntimeError(

            f"{target_temperature} K: "
            f"expected {EXPECTED_PRODUCTION_FRAMES} "
            "production frames but recorded "
            f"{len(metadata_df)}."
        )


    # =========================================================================
    # FINAL SUMMARY
    # =========================================================================

    final_meta = cell_metadata(
        system
    )


    print()

    print(
        f"{target_temperature} K COMPLETE"
    )


    print(
        "Production frames:",
        len(metadata_df)
    )


    print(
        "Final temperature:",
        system.get_temperature(),
        "K"
    )


    print(
        "Final volume:",
        final_meta["volume_A3"],
        "A^3"
    )


    print(
        "Final cell parameters:",
        final_meta
    )


    print(
        "Trajectory:",
        trajectory_file
    )


    print(
        "Checkpoint directory:",
        checkpoint_dir
    )


    return metadata_df


# =============================================================================
# 19. RUN ALL FOUR TEMPERATURES
# =============================================================================

all_temperature_data = {}


for T in TEMPERATURES_K:

    df = run_temperature(
        T
    )

    all_temperature_data[
        T
    ] = df


# =============================================================================
# 20. COMBINE ALL NUMERICAL DATA
# =============================================================================

combined_rows = []


for T, df in (
    all_temperature_data.items()
):

    combined_rows.append(
        df
    )


combined_df = pd.concat(

    combined_rows,

    ignore_index=True
)


combined_file = os.path.join(

    OUTPUT_DIR,

    (
        "candidate_81_FINAL_FULLY_FLEXIBLE_NPT_"
        "ALL_TEMPERATURES.csv"
    )
)


combined_df.to_csv(

    combined_file,

    index=False
)


# =============================================================================
# 21. FINAL PROTOCOL VALIDATION
# =============================================================================

print()
print("=" * 80)
print("FINAL PROTOCOL VALIDATION")
print("=" * 80)

print(
    "Candidate:",
    CANDIDATE_ID
)

print(
    "Formula:",
    "Ti4Si"
)

print(
    "MD system:",
    "Ti32Si8"
)

print(
    "Atoms:",
    N_ATOMS_EXPECTED
)

print(
    "Temperatures:",
    TEMPERATURES_K
)

print(
    "Production steps/T:",
    PRODUCTION_STEPS
)

print(
    "Timestep:",
    TIMESTEP_FS,
    "fs"
)

print(
    "Production frames/T:",
    EXPECTED_PRODUCTION_FRAMES
)

print(
    "Checkpoint interval:",
    CHECKPOINT_INTERVAL
)

print(
    "Cell lengths:",
    "VARIABLE"
)

print(
    "Cell angles:",
    "VARIABLE"
)

print(
    "NPT mask:",
    "None = FULLY FLEXIBLE"
)

print(
    "External pressure:",
    EXTERNAL_PRESSURE_GPA,
    "GPa"
)

print(
    "Protocol hash:",
    PROTOCOL_HASH
)

print(
    "Restart checkpoint format:",
    "pickle_npt_state_v1"
)

print(
    "Combined CSV:",
    combined_file
)

print(
    "Run directory:",
    OUTPUT_DIR
)

print()

print(
    "ALL TEMPERATURES COMPLETED."
)

print("=" * 80)
print("THERMAL MD RUN COMPLETE")
print("=" * 80)

In [ ]:
from pathlib import Path
import pandas as pd
from ase.io import read

# ============================================================
# MINI-EXPERIMENT 1
# Inventory the actual thermal-MD data
# NO MODIFICATION — READ ONLY
# ============================================================

BASE = Path(
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "thermal_stability_candidate_81/"
    "FINAL_FULLY_FLEXIBLE_NPT_v11"
)

CSV_PATH = BASE / "candidate_81_FINAL_FULLY_FLEXIBLE_NPT_ALL_TEMPERATURES.csv"

print("=" * 70)
print("MINI-EXPERIMENT 1 — DATA INVENTORY")
print("=" * 70)

# ------------------------------------------------------------
# 1. CSV
# ------------------------------------------------------------

print("\n[1] CSV")
print("-" * 70)
print("Path:", CSV_PATH)

df = pd.read_csv(CSV_PATH)

print("\nShape:", df.shape)
print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

print("\nFirst 3 rows:")
print(df.head(3).to_string(index=False))

print("\nLast 3 rows:")
print(df.tail(3).to_string(index=False))

# ------------------------------------------------------------
# 2. Trajectories
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("[2] TRAJECTORIES")
print("=" * 70)

for T in [300, 600, 900, 1200]:

    traj = BASE / f"{T}K" / f"candidate_81_{T}K_FULLY_FLEXIBLE.extxyz"

    print(f"\n--- {T} K ---")
    print("Path:", traj)

    if not traj.exists():
        print("❌ FILE NOT FOUND")
        continue

    # Read all frames because expected production frames = 21
    frames = read(traj, index=":")

    print("Frames:", len(frames))
    print("Atoms/frame:", len(frames[0]))

    # First and last frame
    first = frames[0]
    last = frames[-1]

    print("\nFirst frame:")
    print("  Cell:")
    print(first.cell)
    print("  PBC:", first.pbc)

    print("\nLast frame:")
    print("  Cell:")
    print(last.cell)
    print("  PBC:", last.pbc)

    # What ASE stored
    print("\nFirst-frame arrays:")
    print(" ", list(first.arrays.keys()))

    print("\nFirst-frame info:")
    print(" ", list(first.info.keys()))

    print("\nLast-frame info:")
    print(" ", list(last.info.keys()))

print("\n" + "=" * 70)
print("END OF MINI-EXPERIMENT 1")
print("=" * 70)

In [ ]:
# ============================================================
# Thermal MD stability analysis
# Energy / Temperature / Volume / Cell lengths / Cell angles
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Locate the existing combined CSV
# ------------------------------------------------------------

run_dir = "/content/drive/MyDrive/Materials_Project/Top5_search_archive/generation_3_MACE_TOP5_archive/thermal_stability_candidate_81/FINAL_FULLY_FLEXIBLE_NPT_v11"

csv_candidates = []

for root, dirs, files in os.walk(run_dir):
    for f in files:
        if f.endswith(".csv"):
            csv_candidates.append(os.path.join(root, f))

print("CSV files found:")
for p in csv_candidates:
    print(" ", p)

if not csv_candidates:
    raise FileNotFoundError("No CSV file found.")

# Prefer a combined CSV if there is one
combined = [p for p in csv_candidates if "combined" in os.path.basename(p).lower()]

csv_path = combined[0] if combined else csv_candidates[0]

print("\nUsing:")
print(csv_path)

df = pd.read_csv(csv_path)

print("\nData shape:", df.shape)
print("Temperatures:", sorted(df["target_temperature_K"].unique()))


# ------------------------------------------------------------
# 2. Define quantities to analyze
# ------------------------------------------------------------

quantities = {
    "potential_energy_eV": "Potential Energy (eV)",
    "kinetic_energy_eV": "Kinetic Energy (eV)",
    "total_energy_eV": "Total Energy (eV)",
    "temperature_K": "Temperature (K)",
    "volume_A3": "Volume (Å³)",
    "a_A": "a (Å)",
    "b_A": "b (Å)",
    "c_A": "c (Å)",
    "alpha_deg": "α (deg)",
    "beta_deg": "β (deg)",
    "gamma_deg": "γ (deg)",
}


# ------------------------------------------------------------
# 3. Numerical stability summary
# ------------------------------------------------------------

summary_rows = []

for T in sorted(df["target_temperature_K"].unique()):

    d = df[df["target_temperature_K"] == T].copy()
    d = d.sort_values("production_step")

    # Time in ps
    time_ps = d["production_step"].to_numpy() * 0.5 / 1000.0

    for col, label in quantities.items():

        y = d[col].to_numpy(dtype=float)

        # Linear trend: y = slope*time + intercept
        if len(y) >= 2:
            slope = np.polyfit(time_ps, y, 1)[0]
        else:
            slope = np.nan

        initial = y[0]
        final = y[-1]

        summary_rows.append({
            "Temperature_K": T,
            "Quantity": label,
            "Initial": initial,
            "Final": final,
            "Final_minus_initial": final - initial,
            "Mean": np.mean(y),
            "Std": np.std(y, ddof=1),
            "Min": np.min(y),
            "Max": np.max(y),
            "Range": np.max(y) - np.min(y),
            "Linear_slope_per_ps": slope,
        })

summary = pd.DataFrame(summary_rows)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

print("\n============================================================")
print("NUMERICAL STABILITY SUMMARY")
print("============================================================\n")

print(summary.to_string(index=False))


# ------------------------------------------------------------
# 4. Plot every quantity vs time
# ------------------------------------------------------------

for col, label in quantities.items():

    plt.figure(figsize=(10, 6))

    for T in sorted(df["target_temperature_K"].unique()):

        d = df[df["target_temperature_K"] == T].copy()
        d = d.sort_values("production_step")

        time_ps = d["production_step"].to_numpy() * 0.5 / 1000.0
        y = d[col].to_numpy(dtype=float)

        plt.plot(time_ps, y, marker="o", label=f"{T} K")

    plt.xlabel("Production time (ps)")
    plt.ylabel(label)
    plt.title(f"{label} vs time")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ------------------------------------------------------------
# 5. Print compact per-temperature interpretation data
# ------------------------------------------------------------

print("\n============================================================")
print("PER-TEMPERATURE SUMMARY")
print("============================================================")

for T in sorted(df["target_temperature_K"].unique()):

    d = df[df["target_temperature_K"] == T].copy()
    d = d.sort_values("production_step")

    print(f"\n--- {T} K ---")

    for col, label in quantities.items():

        y = d[col].to_numpy(dtype=float)

        print(
            f"{label:28s} | "
            f"mean={np.mean(y):10.4f} | "
            f"std={np.std(y, ddof=1):10.4f} | "
            f"range={np.ptp(y):10.4f} | "
            f"Δ(final-start)={y[-1]-y[0]:10.4f}"
        )


print("\n============================================================")
print("DONE")
print("============================================================")
print("Next step: interpret drift / jumps / fluctuations before analyzing atomic positions.")

In [ ]:
# ============================================================
# STRUCTURAL STABILITY — NEIGHBOR NETWORK ANALYSIS
# Candidate 81 | Ti32Si8 | Fully Flexible NPT
#
# Question:
# Do Ti/Si neighbor relationships remain structurally intact,
# or do neighbor identities/distances change persistently?
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ase.io import read


# ============================================================
# 1. Locate trajectories
# ============================================================

run_dir = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "thermal_stability_candidate_81/"
    "FINAL_FULLY_FLEXIBLE_NPT_v11"
)

traj_paths = {}

for T in [300, 600, 900, 1200]:

    folder = os.path.join(run_dir, f"{T}K")

    candidates = []
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.endswith(".extxyz") or f.endswith(".xyz"):
                candidates.append(os.path.join(root, f))

    if not candidates:
        raise FileNotFoundError(
            f"No trajectory found for {T} K in {folder}"
        )

    # Prefer the largest trajectory file if several exist
    traj = max(candidates, key=os.path.getsize)
    traj_paths[T] = traj

    print(f"{T} K -> {traj}")


# ============================================================
# 2. Read trajectories
# ============================================================

trajectories = {}

for T, path in traj_paths.items():

    frames = read(path, index=":")

    if len(frames) == 0:
        raise RuntimeError(f"No frames read from {path}")

    trajectories[T] = frames

    print(
        f"{T} K: {len(frames)} frames, "
        f"{len(frames[0])} atoms"
    )


# ============================================================
# 3. Identify Ti / Si atoms
# ============================================================

# Candidate 81 = Ti32Si8
# We determine indices directly from atomic symbols.

symbols = trajectories[300][0].get_chemical_symbols()

ti_indices = np.array(
    [i for i, s in enumerate(symbols) if s == "Ti"],
    dtype=int
)

si_indices = np.array(
    [i for i, s in enumerate(symbols) if s == "Si"],
    dtype=int
)

print("\nAtom counts:")
print("Ti:", len(ti_indices))
print("Si:", len(si_indices))

if len(ti_indices) != 32 or len(si_indices) != 8:
    raise ValueError(
        "Unexpected composition. Expected Ti32Si8."
    )


# ============================================================
# 4. Cutoffs
#
# These are analysis cutoffs, NOT claims about a universal
# chemical bond cutoff.
#
# We use generous local-neighbor windows and inspect the
# distributions afterward.
# ============================================================

cutoffs = {
    "Ti-Ti": 3.2,
    "Ti-Si": 3.3,
    "Si-Si": 3.5,
}


# ============================================================
# 5. Helper: pair distances with PBC
# ============================================================

def pair_distances(atoms, indices_a, indices_b, same_species=False):

    # ASE handles the variable cell + PBC correctly.
    distances = []

    for i in indices_a:

        if same_species:
            js = indices_b[indices_b > i]
        else:
            js = indices_b

        if len(js) == 0:
            continue

        # get_distances(..., mic=True) applies minimum image
        d = atoms.get_distances(
            i,
            js,
            mic=True
        )

        distances.extend(d.tolist())

    return np.asarray(distances)


# ============================================================
# 6. Collect distance distributions over all frames
# ============================================================

distance_records = []

for T, frames in trajectories.items():

    for frame_id, atoms in enumerate(frames):

        for pair_type, cutoff in cutoffs.items():

            if pair_type == "Ti-Ti":
                d = pair_distances(
                    atoms,
                    ti_indices,
                    ti_indices,
                    same_species=True
                )

            elif pair_type == "Ti-Si":
                d = pair_distances(
                    atoms,
                    ti_indices,
                    si_indices,
                    same_species=False
                )

            elif pair_type == "Si-Si":
                d = pair_distances(
                    atoms,
                    si_indices,
                    si_indices,
                    same_species=True
                )

            for x in d:

                distance_records.append({
                    "Temperature_K": T,
                    "Frame": frame_id,
                    "Pair": pair_type,
                    "Distance_A": x,
                    "Within_cutoff": x <= cutoff
                })


distance_df = pd.DataFrame(distance_records)

print("\nDistance records:", distance_df.shape)


# ============================================================
# 7. Distance statistics
# ============================================================

print("\n============================================================")
print("DISTANCE DISTRIBUTIONS")
print("============================================================")

for T in [300, 600, 900, 1200]:

    print(f"\n--- {T} K ---")

    dT = distance_df[
        distance_df["Temperature_K"] == T
    ]

    for pair_type in cutoffs:

        d = dT[
            dT["Pair"] == pair_type
        ]["Distance_A"].to_numpy()

        print(
            f"{pair_type:6s} | "
            f"mean={np.mean(d):7.3f} Å | "
            f"std={np.std(d):7.3f} Å | "
            f"min={np.min(d):7.3f} Å | "
            f"max={np.max(d):7.3f} Å"
        )


# ============================================================
# 8. Coordination number per frame
# ============================================================

coord_records = []

for T, frames in trajectories.items():

    for frame_id, atoms in enumerate(frames):

        for pair_type, cutoff in cutoffs.items():

            if pair_type == "Ti-Ti":
                A = ti_indices
                B = ti_indices
                same = True

            elif pair_type == "Ti-Si":
                A = ti_indices
                B = si_indices
                same = False

            else:
                A = si_indices
                B = si_indices
                same = True

            coordination = []

            for i in A:

                if same:
                    js = B[B != i]
                else:
                    js = B

                if len(js) == 0:
                    coordination.append(0)
                    continue

                distances = atoms.get_distances(
                    i,
                    js,
                    mic=True
                )

                coordination.append(
                    np.sum(distances <= cutoff)
                )

            coordination = np.asarray(coordination)

            coord_records.append({
                "Temperature_K": T,
                "Frame": frame_id,
                "Pair": pair_type,
                "Mean_coordination": np.mean(coordination),
                "Std_coordination": np.std(coordination),
                "Min_coordination": np.min(coordination),
                "Max_coordination": np.max(coordination),
            })


coord_df = pd.DataFrame(coord_records)


# ============================================================
# 9. Coordination summary
# ============================================================

print("\n============================================================")
print("COORDINATION SUMMARY")
print("============================================================")

for T in [300, 600, 900, 1200]:

    print(f"\n--- {T} K ---")

    dT = coord_df[
        coord_df["Temperature_K"] == T
    ]

    for pair_type in cutoffs:

        d = dT[
            dT["Pair"] == pair_type
        ]

        print(
            f"{pair_type:6s} | "
            f"mean coordination = "
            f"{d['Mean_coordination'].mean():6.2f} | "
            f"frame-to-frame std = "
            f"{d['Mean_coordination'].std():6.2f} | "
            f"min = {d['Min_coordination'].min():.0f} | "
            f"max = {d['Max_coordination'].max():.0f}"
        )


# ============================================================
# 10. Track Ti-Si neighbor identity
#
# This is the strongest part of the test:
# for every Ti atom, identify its Ti-Si neighbors at the
# first frame and compare them with later frames.
# ============================================================

def neighbor_set(atoms, atom_index, candidate_indices, cutoff):

    distances = atoms.get_distances(
        atom_index,
        candidate_indices,
        mic=True
    )

    return set(
        candidate_indices[distances <= cutoff].tolist()
    )


neighbor_change_records = []

for T, frames in trajectories.items():

    first = frames[0]

    # Reference Ti-Si neighbors at first frame
    reference = {}

    for ti in ti_indices:
        reference[ti] = neighbor_set(
            first,
            ti,
            si_indices,
            cutoffs["Ti-Si"]
        )

    for frame_id, atoms in enumerate(frames):

        overlaps = []
        changed_counts = []

        for ti in ti_indices:

            current = neighbor_set(
                atoms,
                ti,
                si_indices,
                cutoffs["Ti-Si"]
            )

            ref = reference[ti]

            union = ref | current
            intersection = ref & current

            if len(union) == 0:
                overlap = 1.0
            else:
                overlap = len(intersection) / len(union)

            overlaps.append(overlap)
            changed_counts.append(
                len(ref.symmetric_difference(current))
            )

        neighbor_change_records.append({
            "Temperature_K": T,
            "Frame": frame_id,
            "Mean_TiSi_neighbor_overlap": np.mean(overlaps),
            "Min_TiSi_neighbor_overlap": np.min(overlaps),
            "Mean_changed_TiSi_neighbors": np.mean(changed_counts),
            "Max_changed_TiSi_neighbors": np.max(changed_counts),
        })


neighbor_change_df = pd.DataFrame(
    neighbor_change_records
)


# ============================================================
# 11. Print neighbor identity results
# ============================================================

print("\n============================================================")
print("Ti-Si NEIGHBOR IDENTITY STABILITY")
print("============================================================")

for T in [300, 600, 900, 1200]:

    d = neighbor_change_df[
        neighbor_change_df["Temperature_K"] == T
    ]

    print(f"\n--- {T} K ---")

    print(
        f"Mean overlap over trajectory : "
        f"{d['Mean_TiSi_neighbor_overlap'].mean():.3f}"
    )

    print(
        f"Final mean overlap           : "
        f"{d['Mean_TiSi_neighbor_overlap'].iloc[-1]:.3f}"
    )

    print(
        f"Final minimum overlap        : "
        f"{d['Min_TiSi_neighbor_overlap'].iloc[-1]:.3f}"
    )

    print(
        f"Final mean changed neighbors : "
        f"{d['Mean_changed_TiSi_neighbors'].iloc[-1]:.2f}"
    )

    print(
        f"Maximum changed neighbors    : "
        f"{d['Max_changed_TiSi_neighbors'].max():.0f}"
    )


# ============================================================
# 12. Plot coordination vs time
# ============================================================

for pair_type in cutoffs:

    plt.figure(figsize=(10, 6))

    for T in [300, 600, 900, 1200]:

        d = coord_df[
            (coord_df["Temperature_K"] == T) &
            (coord_df["Pair"] == pair_type)
        ].sort_values("Frame")

        time_ps = d["Frame"].to_numpy() * 0.5

        plt.plot(
            time_ps,
            d["Mean_coordination"],
            marker="o",
            label=f"{T} K"
        )

    plt.xlabel("Production time (ps)")
    plt.ylabel(f"Mean {pair_type} coordination")
    plt.title(f"{pair_type} coordination vs time")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ============================================================
# 13. Plot Ti-Si neighbor overlap vs time
# ============================================================

plt.figure(figsize=(10, 6))

for T in [300, 600, 900, 1200]:

    d = neighbor_change_df[
        neighbor_change_df["Temperature_K"] == T
    ].sort_values("Frame")

    time_ps = d["Frame"].to_numpy() * 0.5

    plt.plot(
        time_ps,
        d["Mean_TiSi_neighbor_overlap"],
        marker="o",
        label=f"{T} K"
    )

plt.xlabel("Production time (ps)")
plt.ylabel("Mean Ti-Si neighbor overlap")
plt.title("Persistence of Ti-Si neighbor identities")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# ============================================================
# 14. Histograms of pair distances
# ============================================================

for T in [300, 600, 900, 1200]:

    plt.figure(figsize=(10, 6))

    dT = distance_df[
        distance_df["Temperature_K"] == T
    ]

    for pair_type in cutoffs:

        d = dT[
            dT["Pair"] == pair_type
        ]["Distance_A"]

        plt.hist(
            d,
            bins=60,
            alpha=0.45,
            label=pair_type
        )

    plt.xlabel("Pair distance (Å)")
    plt.ylabel("Count")
    plt.title(f"Pair-distance distribution — {T} K")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


print("\n============================================================")
print("STRUCTURAL NEIGHBOR ANALYSIS COMPLETE")
print("============================================================")
print("Next: interpret neighbor overlap + coordination + distance distributions.")

In [ ]:
# ============================================================
# MINI-EXPERIMENT 3
# RDF -> First Minimum -> Data-driven Coordination
#
# Candidate 81 | Ti32Si8
# Fully flexible NPT
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ase.io import read


# ============================================================
# 1. Locate trajectories
# ============================================================

run_dir = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "thermal_stability_candidate_81/"
    "FINAL_FULLY_FLEXIBLE_NPT_v11"
)

traj_paths = {}

for T in [300, 600, 900, 1200]:

    folder = os.path.join(run_dir, f"{T}K")

    candidates = []

    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.endswith(".extxyz"):
                candidates.append(os.path.join(root, f))

    if not candidates:
        raise FileNotFoundError(
            f"No EXTXYZ trajectory found for {T} K"
        )

    traj_paths[T] = max(
        candidates,
        key=os.path.getsize
    )

    print(f"{T} K -> {traj_paths[T]}")


# ============================================================
# 2. Read trajectories
# ============================================================

trajectories = {}

for T, path in traj_paths.items():

    frames = read(path, index=":")

    trajectories[T] = frames

    print(
        f"{T} K: {len(frames)} frames, "
        f"{len(frames[0])} atoms"
    )


# ============================================================
# 3. Identify Ti / Si atoms
# ============================================================

symbols = trajectories[300][0].get_chemical_symbols()

ti_indices = np.array(
    [i for i, s in enumerate(symbols) if s == "Ti"],
    dtype=int
)

si_indices = np.array(
    [i for i, s in enumerate(symbols) if s == "Si"],
    dtype=int
)

print("\nTi atoms:", len(ti_indices))
print("Si atoms:", len(si_indices))


# ============================================================
# 4. Pair-distance extraction
# ============================================================

def get_pair_distances(
    atoms,
    indices_a,
    indices_b,
    same_species=False
):

    distances = []

    for i in indices_a:

        if same_species:
            js = indices_b[indices_b > i]
        else:
            js = indices_b

        if len(js) == 0:
            continue

        d = atoms.get_distances(
            i,
            js,
            mic=True
        )

        distances.extend(d.tolist())

    return np.asarray(distances)


# ============================================================
# 5. Build RDF-like histograms
#
# IMPORTANT:
# We calculate the distance distribution first.
# Then normalize it by shell volume and atom counts.
# This gives an RDF-style g(r).
# ============================================================

pair_definitions = {
    "Ti-Ti": (ti_indices, ti_indices, True),
    "Ti-Si": (ti_indices, si_indices, False),
    "Si-Si": (si_indices, si_indices, True),
}


rdf_results = {}

r_max = 8.5
dr = 0.02

r_edges = np.arange(
    0.0,
    r_max + dr,
    dr
)

r_centers = 0.5 * (
    r_edges[:-1] + r_edges[1:]
)


for T, frames in trajectories.items():

    rdf_results[T] = {}

    for pair_name, (
        indices_a,
        indices_b,
        same_species
    ) in pair_definitions.items():

        all_distances = []

        for atoms in frames:

            d = get_pair_distances(
                atoms,
                indices_a,
                indices_b,
                same_species=same_species
            )

            all_distances.extend(d.tolist())

        all_distances = np.asarray(all_distances)

        counts, _ = np.histogram(
            all_distances,
            bins=r_edges
        )

        # Basic RDF-style normalization.
        #
        # We use the average cell volume over the trajectory.
        avg_volume = np.mean([
            atoms.get_volume()
            for atoms in frames
        ])

        shell_volume = (
            4.0
            * np.pi
            * r_centers**2
            * dr
        )

        if same_species:

            N_A = len(indices_a)

            # Number of unique A-A pairs
            number_of_pairs = (
                N_A * (N_A - 1) / 2
            )

            density_factor = (
                number_of_pairs / avg_volume
            )

        else:

            N_A = len(indices_a)
            N_B = len(indices_b)

            number_of_pairs = N_A * N_B

            density_factor = (
                number_of_pairs / avg_volume
            )

        # Avoid division by zero at r=0
        normalization = (
            len(frames)
            * density_factor
            * shell_volume
        )

        g_r = np.zeros_like(r_centers)

        valid = normalization > 0

        g_r[valid] = (
            counts[valid]
            / normalization[valid]
        )

        rdf_results[T][pair_name] = {
            "r": r_centers,
            "g_r": g_r,
            "raw_distances": all_distances,
        }


# ============================================================
# 6. Find first peak + first minimum after it
# ============================================================

def find_first_peak_and_minimum(
    r,
    g,
    min_r=1.5,
    max_r=7.0
):

    mask = (
        (r >= min_r) &
        (r <= max_r)
    )

    rr = r[mask]
    gg = g[mask]

    # First meaningful peak:
    # maximum in the first half of the local-distance region.
    peak_index = np.argmax(gg)

    peak_r = rr[peak_index]
    peak_g = gg[peak_index]

    # Search for the first local minimum after the peak.
    min_index = None

    for i in range(
        peak_index + 2,
        len(gg) - 2
    ):

        if (
            gg[i] <= gg[i - 1]
            and
            gg[i] <= gg[i + 1]
        ):
            min_index = i
            break

    if min_index is None:

        # Fallback: minimum after peak
        if peak_index + 1 < len(gg):
            min_index = (
                peak_index + 1
                +
                np.argmin(
                    gg[peak_index + 1:]
                )
            )
        else:
            return peak_r, peak_g, np.nan, np.nan

    minimum_r = rr[min_index]
    minimum_g = gg[min_index]

    return (
        peak_r,
        peak_g,
        minimum_r,
        minimum_g
    )


cutoff_results = []

for T in [300, 600, 900, 1200]:

    for pair_name in pair_definitions:

        result = rdf_results[T][pair_name]

        peak_r, peak_g, minimum_r, minimum_g = (
            find_first_peak_and_minimum(
                result["r"],
                result["g_r"]
            )
        )

        cutoff_results.append({
            "Temperature_K": T,
            "Pair": pair_name,
            "First_peak_r_A": peak_r,
            "First_peak_g": peak_g,
            "First_minimum_r_A": minimum_r,
            "First_minimum_g": minimum_g,
        })


cutoff_df = pd.DataFrame(cutoff_results)

print("\n============================================================")
print("RDF PEAKS AND DATA-DRIVEN CUTOFFS")
print("============================================================\n")

print(
    cutoff_df.to_string(index=False)
)


# ============================================================
# 7. Plot RDFs with detected cutoff
# ============================================================

for pair_name in pair_definitions:

    plt.figure(figsize=(10, 6))

    for T in [300, 600, 900, 1200]:

        result = rdf_results[T][pair_name]

        r = result["r"]
        g = result["g_r"]

        row = cutoff_df[
            (cutoff_df["Temperature_K"] == T) &
            (cutoff_df["Pair"] == pair_name)
        ].iloc[0]

        cutoff = row["First_minimum_r_A"]

        plt.plot(
            r,
            g,
            label=f"{T} K"
        )

        if np.isfinite(cutoff):
            plt.axvline(
                cutoff,
                linestyle="--",
                alpha=0.5
            )

    plt.xlabel("r (Å)")
    plt.ylabel("g(r)")
    plt.title(
        f"RDF — {pair_name}"
    )
    plt.xlim(1.5, 7.0)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# ============================================================
# 8. Recalculate coordination using RDF-derived cutoffs
# ============================================================

coord_records = []


for T, frames in trajectories.items():

    for pair_name, (
        indices_a,
        indices_b,
        same_species
    ) in pair_definitions.items():

        cutoff_row = cutoff_df[
            (cutoff_df["Temperature_K"] == T) &
            (cutoff_df["Pair"] == pair_name)
        ].iloc[0]

        cutoff = cutoff_row[
            "First_minimum_r_A"
        ]

        if not np.isfinite(cutoff):
            continue

        for frame_id, atoms in enumerate(frames):

            coordination = []

            for i in indices_a:

                if same_species:
                    js = indices_b[
                        indices_b != i
                    ]
                else:
                    js = indices_b

                if len(js) == 0:
                    coordination.append(0)
                    continue

                distances = atoms.get_distances(
                    i,
                    js,
                    mic=True
                )

                coordination.append(
                    np.sum(
                        distances <= cutoff
                    )
                )

            coordination = np.asarray(
                coordination
            )

            coord_records.append({
                "Temperature_K": T,
                "Frame": frame_id,
                "Pair": pair_name,
                "Cutoff_A": cutoff,
                "Mean_coordination": (
                    np.mean(coordination)
                ),
                "Std_coordination": (
                    np.std(coordination)
                ),
                "Min_coordination": (
                    np.min(coordination)
                ),
                "Max_coordination": (
                    np.max(coordination)
                ),
            })


rdf_coord_df = pd.DataFrame(
    coord_records
)


# ============================================================
# 9. Coordination summary
# ============================================================

print("\n============================================================")
print("COORDINATION USING RDF-DERIVED CUTOFFS")
print("============================================================")

for T in [300, 600, 900, 1200]:

    print(f"\n--- {T} K ---")

    dT = rdf_coord_df[
        rdf_coord_df["Temperature_K"] == T
    ]

    for pair_name in pair_definitions:

        d = dT[
            dT["Pair"] == pair_name
        ]

        print(
            f"{pair_name:6s} | "
            f"cutoff={d['Cutoff_A'].iloc[0]:.3f} Å | "
            f"mean CN={d['Mean_coordination'].mean():.3f} | "
            f"frame std={d['Mean_coordination'].std():.3f} | "
            f"min={d['Min_coordination'].min():.0f} | "
            f"max={d['Max_coordination'].max():.0f}"
        )


# ============================================================
# 10. Compare cutoffs across temperatures
# ============================================================

print("\n============================================================")
print("CUTOFF TEMPERATURE DEPENDENCE")
print("============================================================")

for pair_name in pair_definitions:

    d = cutoff_df[
        cutoff_df["Pair"] == pair_name
    ].sort_values("Temperature_K")

    print(f"\n{pair_name}")

    for _, row in d.iterrows():

        print(
            f"{int(row['Temperature_K'])} K : "
            f"{row['First_minimum_r_A']:.3f} Å"
        )


print("\n============================================================")
print("RDF ANALYSIS COMPLETE")
print("============================================================")

In [ ]:
# ============================================================
# DIAGNOSTIC INSPECTION — MACE TOP 5 EXTXYZ
# ============================================================

from ase.io import read
import os
import numpy as np

# ------------------------------------------------------------
# 1. FILE
# ------------------------------------------------------------

STRUCTURES_FILE = os.path.join(
    "/content/drive/MyDrive/Materials_Project",
    "Top5_search_archive",
    "generation_3_MACE_TOP5_archive",
    "MACE_consistent_TOP5_STRUCTURES.extxyz"
)

print("=" * 90)
print("MACE TOP 5 EXTXYZ — DIAGNOSTIC INSPECTION")
print("=" * 90)

print(f"\nFile:")
print(STRUCTURES_FILE)

if not os.path.exists(STRUCTURES_FILE):
    raise FileNotFoundError(
        f"\nFile not found:\n{STRUCTURES_FILE}"
    )

# ------------------------------------------------------------
# 2. READ ALL STRUCTURES
# ------------------------------------------------------------

structures = read(
    STRUCTURES_FILE,
    index=":"
)

print(f"\nNumber of structures found: {len(structures)}")

# ------------------------------------------------------------
# 3. INSPECT EVERY STRUCTURE
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("STRUCTURE INVENTORY")
print("=" * 90)

candidate_ids = []

for i, atoms in enumerate(structures):

    candidate_id = atoms.info.get(
        "candidate_id",
        None
    )

    candidate_ids.append(candidate_id)

    print(f"\nStructure #{i}")

    print(f"  candidate_id : {candidate_id}")
    print(f"  formula      : {atoms.get_chemical_formula()}")
    print(f"  atoms        : {len(atoms)}")
    print(f"  volume (A^3) : {atoms.get_volume():.6f}")
    print(f"  PBC          : {atoms.get_pbc().tolist()}")

    print("  cell (A):")
    print(atoms.cell.array)

    print("  info keys:")
    print(list(atoms.info.keys()))

    print("  arrays keys:")
    print(list(atoms.arrays.keys()))

# ------------------------------------------------------------
# 4. FIND CANDIDATE 81
# ------------------------------------------------------------

matches = [
    (i, atoms)
    for i, atoms in enumerate(structures)
    if atoms.info.get("candidate_id", None) == 81
]

print("\n" + "=" * 90)
print("CANDIDATE 81 CHECK")
print("=" * 90)

if not matches:

    print("\nCandidate 81 was NOT found using atoms.info['candidate_id'].")

    print("\nStored candidate IDs:")
    print(candidate_ids)

else:

    i, atoms = matches[0]

    print(f"\nCandidate 81 found at structure index: {i}")

    print(f"Formula      : {atoms.get_chemical_formula()}")
    print(f"Number atoms : {len(atoms)}")
    print(f"Volume (A^3) : {atoms.get_volume():.6f}")
    print(f"PBC          : {atoms.get_pbc().tolist()}")

    print("\nCell (A):")
    print(atoms.cell.array)

    print("\nChemical symbols:")
    print(atoms.get_chemical_symbols())

    print("\nPositions (A):")
    print(atoms.get_positions())

    print("\nASE atoms.info:")
    for key, value in atoms.info.items():
        print(f"  {key}: {value}")

    print("\nASE atoms.arrays:")
    for key, value in atoms.arrays.items():
        print(
            f"  {key}: shape={value.shape}, dtype={value.dtype}"
        )

# ------------------------------------------------------------
# 5. FINAL DIAGNOSTIC SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FINAL DIAGNOSTIC SUMMARY")
print("=" * 90)

print(f"\nStructures in file : {len(structures)}")
print(f"Candidate IDs      : {candidate_ids}")

if matches:
    atoms = matches[0][1]

    print("\nCandidate 81:")
    print("  FOUND")
    print(f"  Atoms   : {len(atoms)}")
    print(f"  Formula : {atoms.get_chemical_formula()}")
    print(f"  Volume  : {atoms.get_volume():.6f} A^3")
    print(
        f"  Cell variable? "
        f"{not np.allclose(atoms.cell.array, np.diag(np.diag(atoms.cell.array)))}"
    )

    print(
        "\nThis file can now be used to determine "
        "exactly what structure was archived for Candidate 81."
    )

else:
    print("\nCandidate 81: NOT FOUND")

print("\n" + "=" * 90)
print("END OF DIAGNOSTIC")
print("=" * 90)

In [ ]:
# =============================================================================
# CANDIDATE 81 — 40-ATOM FULLY RELAXED HOST
# =============================================================================
# Purpose:
# Build the 2x2x2 (40-atom) host structure from Candidate 81
# and perform a full variable-cell MACE relaxation.
#
# This structure will be used as the host for Li insertion/diffusion studies.
# =============================================================================

import os
import pickle
import numpy as np

from google.colab import drive
from ase.io import write
from ase.optimize import BFGS
from ase.filters import FrechetCellFilter
from mace.calculators import mace_mp
import torch


# =============================================================================
# 1. MOUNT GOOGLE DRIVE
# =============================================================================

drive.mount("/content/drive")


# =============================================================================
# 2. PATHS
# =============================================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

PKL_FILE = os.path.join(
    BASE_DIR,
    "MACE_TOP5_candidate_to_structure.pkl"
)

OUTPUT_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_40atom_relaxed_host.extxyz"
)


# =============================================================================
# 3. PARAMETERS
# =============================================================================

CANDIDATE_ID = 81

SUPERCELL = (2, 2, 2)

EXPECTED_ATOMS = 40

FMAX = 0.01

MAX_STEPS = 1000


# =============================================================================
# 4. HEADER
# =============================================================================

print("=" * 90)
print("CANDIDATE 81 — 40-ATOM FULLY RELAXED HOST")
print("=" * 90)

print(f"Candidate       : {CANDIDATE_ID}")
print(f"Supercell       : {SUPERCELL}")
print(f"Expected atoms  : {EXPECTED_ATOMS}")
print(f"Force threshold : {FMAX} eV/A")
print(f"Maximum steps   : {MAX_STEPS}")
print(f"Output          : {OUTPUT_FILE}")

print("=" * 90)


# =============================================================================
# 5. LOAD MACE
# =============================================================================

print("\nLoading MACE...")

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64",
    device=DEVICE
)

print(f"Device : {DEVICE}")
print("MACE   : medium")
print("D3     : True")
print("dtype  : float64")


# =============================================================================
# 6. LOAD CANDIDATE 81
# =============================================================================

print("\nLoading Candidate 81...")

with open(
    PKL_FILE,
    "rb"
) as f:
    candidate_structures = pickle.load(f)


if CANDIDATE_ID not in candidate_structures:
    raise KeyError(
        f"Candidate {CANDIDATE_ID} was not found in the PKL."
    )


host_5 = candidate_structures[
    CANDIDATE_ID
].copy()


# =============================================================================
# 7. CHECK ORIGINAL 5-ATOM HOST
# =============================================================================

print("\n" + "=" * 90)
print("ORIGINAL HOST")
print("=" * 90)

print("Formula :", host_5.get_chemical_formula())
print("Atoms   :", len(host_5))
print("Volume  :", host_5.get_volume(), "A^3")
print("PBC     :", host_5.get_pbc())


if len(host_5) != 5:
    raise ValueError(
        f"Expected Candidate 81 to contain 5 atoms, "
        f"but found {len(host_5)}."
    )


# =============================================================================
# 8. BUILD 2x2x2 SUPERCELL
# =============================================================================

print("\n" + "=" * 90)
print("BUILDING 2x2x2 SUPERCELL")
print("=" * 90)

host_40 = host_5.repeat(
    SUPERCELL
)

print("Formula :", host_40.get_chemical_formula())
print("Atoms   :", len(host_40))
print("Volume  :", host_40.get_volume(), "A^3")
print("PBC     :", host_40.get_pbc())


if len(host_40) != EXPECTED_ATOMS:
    raise ValueError(
        f"Expected {EXPECTED_ATOMS} atoms, "
        f"but generated {len(host_40)}."
    )


# =============================================================================
# 9. ATTACH MACE
# =============================================================================

host_40.calc = calc


# =============================================================================
# 10. ENERGY BEFORE RELAXATION
# =============================================================================

energy_before = (
    host_40.get_potential_energy()
)

forces_before = (
    host_40.get_forces()
)

max_force_before = np.max(
    np.linalg.norm(
        forces_before,
        axis=1
    )
)


print("\n" + "=" * 90)
print("BEFORE RELAXATION")
print("=" * 90)

print(
    f"Energy        : "
    f"{energy_before:.8f} eV"
)

print(
    f"Energy/atom   : "
    f"{energy_before / len(host_40):.8f} eV/atom"
)

print(
    f"Maximum force : "
    f"{max_force_before:.8f} eV/A"
)

print(
    f"Volume        : "
    f"{host_40.get_volume():.8f} A^3"
)


# =============================================================================
# 11. FULL VARIABLE-CELL RELAXATION
# =============================================================================

print("\n" + "=" * 90)
print("STARTING 40-ATOM FULL VARIABLE-CELL RELAXATION")
print("=" * 90)

cell_filter = FrechetCellFilter(
    host_40
)

optimizer = BFGS(
    cell_filter,
    logfile="-"
)

optimizer.run(
    fmax=FMAX,
    steps=MAX_STEPS
)


# =============================================================================
# 12. FINAL PROPERTIES
# =============================================================================

energy_after = (
    host_40.get_potential_energy()
)

forces_after = (
    host_40.get_forces()
)

max_force_after = np.max(
    np.linalg.norm(
        forces_after,
        axis=1
    )
)

volume_after = (
    host_40.get_volume()
)


# =============================================================================
# 13. FINAL REPORT
# =============================================================================

print("\n" + "=" * 90)
print("RELAXATION COMPLETE")
print("=" * 90)

print(
    f"Energy before     : "
    f"{energy_before:.8f} eV"
)

print(
    f"Energy after      : "
    f"{energy_after:.8f} eV"
)

print(
    f"Energy/atom       : "
    f"{energy_after / len(host_40):.8f} eV/atom"
)

print(
    f"Maximum force     : "
    f"{max_force_after:.8f} eV/A"
)

print(
    f"Final volume      : "
    f"{volume_after:.8f} A^3"
)

print(
    f"Optimizer steps   : "
    f"{optimizer.nsteps}"
)

print(
    f"Converged         : "
    f"{max_force_after <= FMAX}"
)


# =============================================================================
# 14. SAVE RELAXED 40-ATOM HOST
# =============================================================================

host_40.info["candidate_id"] = CANDIDATE_ID
host_40.info["structure_role"] = (
    "40-atom fully relaxed host for Li insertion/diffusion"
)
host_40.info["relaxation"] = (
    "MACE full variable-cell relaxation"
)
host_40.info["relaxation_fmax_eV_per_A"] = FMAX

write(
    OUTPUT_FILE,
    host_40,
    format="extxyz"
)


# =============================================================================
# 15. FINAL VALIDATION
# =============================================================================

print("\n" + "=" * 90)
print("SAVED HOST VALIDATION")
print("=" * 90)

print("File :", OUTPUT_FILE)
print("Atoms:", len(host_40))
print("Formula:", host_40.get_chemical_formula())
print("Volume:", host_40.get_volume(), "A^3")
print("PBC:", host_40.get_pbc())

print("\n" + "=" * 90)
print("DONE")
print("=" * 90)

In [ ]:
# =============================================================================
# CANDIDATE 81 — DATA-DRIVEN Li INSERTION SCREENING
# 40-ATOM HOST + 10 Li
# =============================================================================
#
# Purpose:
#   1. Load the fully relaxed 40-atom Candidate 81 host.
#   2. Find interstitial sites from the host geometry itself.
#   3. DO NOT assume a hand-picked Li-host distance cutoff.
#   4. Select 10 geometrically distinct interstitial sites.
#   5. Insert 10 Li atoms.
#   6. Relax atomic positions with FIXED CELL.
#
# This is an insertion screening experiment.
# Diffusion / NEB will be done later.
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import os
import numpy as np
import torch

from google.colab import drive

from ase.io import read, write
from ase import Atoms
from ase.optimize import BFGS

from scipy.spatial import Voronoi


# =============================================================================
# 2. MOUNT DRIVE
# =============================================================================

drive.mount("/content/drive")


# =============================================================================
# 3. PATHS
# =============================================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

HOST_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_40atom_relaxed_host.extxyz"
)

OUTPUT_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_40atom_10Li_inserted_relaxed.extxyz"
)

SITE_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_10Li_interstitial_sites.npy"
)


# =============================================================================
# 4. PARAMETERS
# =============================================================================

CANDIDATE_ID = 81

N_LI = 10

FMAX = 0.01
MAX_STEPS = 1000

# Number of periodic copies used only to construct the Voronoi geometry.
# This is NOT a physical distance cutoff.
REPLICATIONS = [-1, 0, 1]

# Numerical tolerance for identifying duplicate wrapped sites.
DUPLICATE_TOL = 1e-5


# =============================================================================
# 5. HEADER
# =============================================================================

print("=" * 100)
print("CANDIDATE 81 — DATA-DRIVEN Li INSERTION SCREENING")
print("=" * 100)

print(f"Host file       : {HOST_FILE}")
print(f"Li atoms        : {N_LI}")
print(f"Method          : Periodic Voronoi interstitial-site search")
print(f"Li-host cutoff  : NONE")
print(f"Grid            : NONE")
print(f"Relaxation      : Atomic positions only")
print(f"Cell            : Fixed")
print(f"Force threshold : {FMAX} eV/A")
print(f"Maximum steps   : {MAX_STEPS}")

print("=" * 100)


# =============================================================================
# 6. LOAD HOST
# =============================================================================

print("\nLoading fully relaxed 40-atom host...")

if not os.path.isfile(HOST_FILE):
    raise FileNotFoundError(
        f"\nHost file not found:\n{HOST_FILE}"
    )

host = read(
    HOST_FILE,
    index=0
)

host = host.copy()


# =============================================================================
# 7. HOST VALIDATION
# =============================================================================

print("\n" + "=" * 100)
print("HOST VALIDATION")
print("=" * 100)

print("Formula :", host.get_chemical_formula())
print("Atoms   :", len(host))
print("Volume  :", host.get_volume(), "A^3")
print("PBC     :", host.get_pbc())

if len(host) != 40:
    raise ValueError(
        f"Expected 40 host atoms, found {len(host)}."
    )

if not np.all(host.get_pbc()):
    raise ValueError(
        "Host is not fully periodic."
    )

cell = np.asarray(
    host.cell.array,
    dtype=float
)

if cell.shape != (3, 3):
    raise ValueError(
        "Invalid cell shape."
    )

if host.get_volume() <= 0:
    raise ValueError(
        "Host cell volume is not positive."
    )


# =============================================================================
# 8. CONVERT HOST POSITIONS TO FRACTIONAL COORDINATES
# =============================================================================

positions = host.get_positions()

fractional = host.get_scaled_positions(
    wrap=False
)


# =============================================================================
# 9. BUILD PERIODIC IMAGES
# =============================================================================
#
# We replicate the HOST positions around the central cell.
#
# This is required because a normal Voronoi calculation does not understand
# periodic boundary conditions by itself.
#
# We are NOT creating a physical 3x3x3 supercell for the material.
# We are only giving Voronoi the neighboring periodic images needed to
# correctly determine the geometry near the cell boundaries.
# =============================================================================

print("\n" + "=" * 100)
print("BUILDING PERIODIC VORONOI GEOMETRY")
print("=" * 100)

image_positions = []

for ix in REPLICATIONS:
    for iy in REPLICATIONS:
        for iz in REPLICATIONS:

            shift = np.array(
                [ix, iy, iz],
                dtype=float
            )

            frac_image = fractional + shift

            cart_image = np.dot(
                frac_image,
                cell
            )

            image_positions.append(
                cart_image
            )

image_positions = np.concatenate(
    image_positions,
    axis=0
)

print(
    "Original host atoms :",
    len(host)
)

print(
    "Periodic image atoms:",
    len(image_positions)
)


# =============================================================================
# 10. VORONOI ANALYSIS
# =============================================================================

print("\n" + "=" * 100)
print("SEARCHING FOR GEOMETRIC INTERSTITIAL SITES")
print("=" * 100)

vor = Voronoi(
    image_positions
)

vertices = np.asarray(
    vor.vertices,
    dtype=float
)

print(
    "Raw Voronoi vertices:",
    len(vertices)
)


# =============================================================================
# 11. KEEP ONLY VORONOI VERTICES INSIDE CENTRAL CELL
# =============================================================================

candidate_sites = []

for vertex in vertices:

    # Convert Cartesian -> fractional coordinates
    frac = np.linalg.solve(
        cell.T,
        vertex
    )

    # Wrap into the central periodic cell
    frac_wrapped = frac % 1.0

    # Convert back to Cartesian
    cart_wrapped = np.dot(
        frac_wrapped,
        cell
    )

    candidate_sites.append(
        (
            frac_wrapped,
            cart_wrapped
        )
    )


# =============================================================================
# 12. REMOVE DUPLICATE PERIODIC REPRESENTATIONS
# =============================================================================

unique_sites = []

for frac, cart in candidate_sites:

    duplicate = False

    for old_frac, old_cart in unique_sites:

        # Minimum-image comparison in fractional coordinates
        delta = frac - old_frac
        delta -= np.round(delta)

        if np.linalg.norm(delta) < DUPLICATE_TOL:
            duplicate = True
            break

    if not duplicate:
        unique_sites.append(
            (
                frac,
                cart
            )
        )


print(
    "Unique Voronoi sites:",
    len(unique_sites)
)


# =============================================================================
# 13. CALCULATE NATURAL HOST CLEARANCE
# =============================================================================
#
# IMPORTANT:
#
# We do NOT say:
#
#     "Li must be at least 1.80 A away."
#
# Instead, for every geometric site we simply calculate:
#
#     distance from this site to its nearest host atom.
#
# This is an observed geometric property of THIS material.
#
# Larger clearance = more open local environment.
#
# =============================================================================

host_positions = host.get_positions()

site_records = []

for site_index, (frac, cart) in enumerate(
    unique_sites
):

    # Minimum-image distances from this site to every host atom
    displacement = (
        host_positions
        - cart
    )

    displacement -= np.round(
        np.dot(
            displacement,
            np.linalg.inv(cell)
        )
    ) @ cell

    distances = np.linalg.norm(
        displacement,
        axis=1
    )

    nearest_distance = np.min(
        distances
    )

    nearest_atom_index = int(
        np.argmin(distances)
    )

    nearest_atom_symbol = (
        host[nearest_atom_index]
        .symbol
    )

    site_records.append(
        {
            "site_index":
                site_index,

            "fractional":
                frac.copy(),

            "cartesian":
                cart.copy(),

            "nearest_host_distance":
                float(nearest_distance),

            "nearest_host_atom":
                nearest_atom_index,

            "nearest_host_element":
                nearest_atom_symbol
        }
    )


# =============================================================================
# 14. SORT BY NATURAL GEOMETRIC CLEARANCE
# =============================================================================

site_records.sort(
    key=lambda x:
        x["nearest_host_distance"],
    reverse=True
)


print("\nTop geometric sites by natural host clearance:")
print("-" * 100)

for rank, record in enumerate(
    site_records[:20],
    start=1
):

    print(
        f"{rank:3d} | "
        f"site={record['site_index']:4d} | "
        f"nearest host = "
        f"{record['nearest_host_distance']:.4f} A | "
        f"nearest element = "
        f"{record['nearest_host_element']}"
    )


# =============================================================================
# 15. SELECT 10 SPATIALLY DISTINCT SITES
# =============================================================================
#
# We do not impose a minimum separation between Li sites.
#
# Instead we use a greedy max-min selection:
#
#   - first choose the most open site
#   - then choose the site that maximizes its distance from the
#     already selected sites
#
# This gives spatial coverage without inventing a physical cutoff.
# =============================================================================

if len(site_records) < N_LI:
    raise RuntimeError(
        f"Only {len(site_records)} unique Voronoi sites were found; "
        f"cannot select {N_LI} sites."
    )


selected_sites = []

# First site = largest natural host clearance
selected_sites.append(
    site_records[0]
)


while len(selected_sites) < N_LI:

    best_record = None
    best_score = -np.inf

    for record in site_records:

        if record in selected_sites:
            continue

        candidate_cart = record[
            "cartesian"
        ]

        min_distance_to_selected = np.inf

        for selected in selected_sites:

            selected_cart = selected[
                "cartesian"
            ]

            delta = (
                candidate_cart
                - selected_cart
            )

            delta_frac = np.dot(
                delta,
                np.linalg.inv(cell)
            )

            delta_frac -= np.round(
                delta_frac
            )

            delta_cart = np.dot(
                delta_frac,
                cell
            )

            d = np.linalg.norm(
                delta_cart
            )

            min_distance_to_selected = min(
                min_distance_to_selected,
                d
            )

        # Main criterion:
        # maximize separation from already selected sites.
        #
        # Small tie-breaking contribution:
        # prefer naturally more open sites.
        score = (
            min_distance_to_selected
            +
            1e-6
            *
            record[
                "nearest_host_distance"
            ]
        )

        if score > best_score:

            best_score = score
            best_record = record

    selected_sites.append(
        best_record
    )


# =============================================================================
# 16. PRINT FINAL SELECTED SITES
# =============================================================================

print("\n" + "=" * 100)
print("SELECTED 10 Li INSERTION SITES")
print("=" * 100)

for i, record in enumerate(
    selected_sites,
    start=1
):

    print(
        f"\nLi site {i}"
    )

    print(
        f"  site index             : "
        f"{record['site_index']}"
    )

    print(
        f"  fractional coordinates : "
        f"{record['fractional']}"
    )

    print(
        f"  Cartesian coordinates  : "
        f"{record['cartesian']}"
    )

    print(
        f"  nearest host distance  : "
        f"{record['nearest_host_distance']:.6f} A"
    )

    print(
        f"  nearest host element   : "
        f"{record['nearest_host_element']}"
    )


# =============================================================================
# 17. SAVE SITE INFORMATION
# =============================================================================

site_array = np.array(
    [
        record["cartesian"]
        for record in selected_sites
    ],
    dtype=float
)

np.save(
    SITE_FILE,
    site_array
)

print(
    f"\nSelected site coordinates saved to:\n{SITE_FILE}"
)


# =============================================================================
# 18. INSERT 10 Li
# =============================================================================

print("\n" + "=" * 100)
print("INSERTING 10 Li")
print("=" * 100)

li_positions = np.array(
    [
        record["cartesian"]
        for record in selected_sites
    ],
    dtype=float
)

li_atoms = Atoms(
    symbols=["Li"] * N_LI,
    positions=li_positions,
    cell=cell,
    pbc=host.get_pbc()
)


# Combine host + Li
system = host.copy()

system.extend(
    li_atoms
)


# =============================================================================
# 19. VALIDATE COMPOSITION
# =============================================================================

print("\nSystem before relaxation:")

print(
    "Formula :",
    system.get_chemical_formula()
)

print(
    "Atoms   :",
    len(system)
)

print(
    "Host    :",
    len(host)
)

print(
    "Li      :",
    N_LI
)

if len(system) != 50:
    raise ValueError(
        "Expected 50 total atoms."
    )


# =============================================================================
# 20. LOAD MACE
# =============================================================================

print("\n" + "=" * 100)
print("LOADING MACE")
print("=" * 100)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

from mace.calculators import mace_mp

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64",
    device=DEVICE
)

print(
    "Device :",
    DEVICE
)

print(
    "MACE   : medium"
)

print(
    "D3     : True"
)

print(
    "dtype  : float64"
)


# =============================================================================
# 21. ENERGY BEFORE RELAXATION
# =============================================================================

system.calc = calc

energy_before = (
    system.get_potential_energy()
)

forces_before = (
    system.get_forces()
)

max_force_before = np.max(
    np.linalg.norm(
        forces_before,
        axis=1
    )
)


print("\n" + "=" * 100)
print("BEFORE Li INSERTION RELAXATION")
print("=" * 100)

print(
    f"Total energy       : "
    f"{energy_before:.8f} eV"
)

print(
    f"Energy/atom        : "
    f"{energy_before / len(system):.8f} eV/atom"
)

print(
    f"Maximum force      : "
    f"{max_force_before:.8f} eV/A"
)

print(
    f"Cell volume        : "
    f"{system.get_volume():.8f} A^3"
)


# =============================================================================
# 22. RELAX ATOMIC POSITIONS
# =============================================================================
#
# No ExpCellFilter.
#
# Therefore:
#   - atomic positions can move
#   - Li can move
#   - host atoms can move
#   - the cell is fixed
#
# This is intentional for this first insertion experiment.
# =============================================================================

print("\n" + "=" * 100)
print("RELAXING Li + HOST POSITIONS")
print("CELL FIXED")
print("=" * 100)

optimizer = BFGS(
    system,
    logfile=None
)

optimizer.run(
    fmax=FMAX,
    steps=MAX_STEPS
)


# =============================================================================
# 23. FINAL PROPERTIES
# =============================================================================

energy_after = (
    system.get_potential_energy()
)

forces_after = (
    system.get_forces()
)

max_force_after = np.max(
    np.linalg.norm(
        forces_after,
        axis=1
    )
)


# =============================================================================
# 24. Li FINAL POSITIONS
# =============================================================================

final_li_positions = (
    system.get_positions()[-N_LI:]
)


# =============================================================================
# 25. FINAL REPORT
# =============================================================================

print("\n" + "=" * 100)
print("Li INSERTION RELAXATION COMPLETE")
print("=" * 100)

print(
    f"Initial energy       : "
    f"{energy_before:.8f} eV"
)

print(
    f"Final energy         : "
    f"{energy_after:.8f} eV"
)

print(
    f"Energy change        : "
    f"{energy_after - energy_before:.8f} eV"
)

print(
    f"Energy/atom          : "
    f"{energy_after / len(system):.8f} eV/atom"
)

print(
    f"Final maximum force  : "
    f"{max_force_after:.8f} eV/A"
)

print(
    f"Optimizer steps      : "
    f"{optimizer.nsteps}"
)

print(
    f"Converged            : "
    f"{max_force_after <= FMAX}"
)

print(
    f"Final cell volume    : "
    f"{system.get_volume():.8f} A^3"
)


# =============================================================================
# 26. CHECK WHETHER Li MOVED
# =============================================================================

li_displacements = []

for initial, final in zip(
    li_positions,
    final_li_positions
):

    delta = final - initial

    delta_frac = np.dot(
        delta,
        np.linalg.inv(cell)
    )

    delta_frac -= np.round(
        delta_frac
    )

    delta_cart = np.dot(
        delta_frac,
        cell
    )

    li_displacements.append(
        np.linalg.norm(
            delta_cart
        )
    )

li_displacements = np.array(
    li_displacements
)


print("\n" + "=" * 100)
print("Li MOVEMENT DURING INSERTION RELAXATION")
print("=" * 100)

for i, displacement in enumerate(
    li_displacements,
    start=1
):

    print(
        f"Li {i:2d} : "
        f"{displacement:.6f} A"
    )

print(
    "\nMean Li displacement : "
    f"{np.mean(li_displacements):.6f} A"
)

print(
    "Maximum Li displacement : "
    f"{np.max(li_displacements):.6f} A"
)


# =============================================================================
# 27. SAVE METADATA
# =============================================================================

system.info["candidate_id"] = (
    CANDIDATE_ID
)

system.info["host_atoms"] = 40

system.info["li_atoms"] = N_LI

system.info["insertion_method"] = (
    "Periodic Voronoi geometry; "
    "no manually imposed Li-host cutoff"
)

system.info["relaxation_method"] = (
    "BFGS atomic-position relaxation; "
    "cell fixed"
)

system.info["relaxation_fmax_eV_per_A"] = (
    FMAX
)

system.info["relaxation_converged"] = (
    bool(max_force_after <= FMAX)
)

system.info["initial_energy_eV"] = (
    float(energy_before)
)

system.info["final_energy_eV"] = (
    float(energy_after)
)


# =============================================================================
# 28. SAVE FINAL STRUCTURE
# =============================================================================

write(
    OUTPUT_FILE,
    system,
    format="extxyz"
)


# =============================================================================
# 29. FINAL VALIDATION
# =============================================================================

print("\n" + "=" * 100)
print("FINAL SAVED STRUCTURE")
print("=" * 100)

print(
    "File    :",
    OUTPUT_FILE
)

print(
    "Formula :",
    system.get_chemical_formula()
)

print(
    "Atoms   :",
    len(system)
)

print(
    "Host    :",
    40
)

print(
    "Li      :",
    N_LI
)

print(
    "Volume  :",
    system.get_volume(),
    "A^3"
)

print(
    "PBC     :",
    system.get_pbc()
)

print("\n" + "=" * 100)
print("Li INSERTION SCREENING COMPLETE")
print("=" * 100)

In [ ]:
# =============================================================================
# CANDIDATE 81 — POST-RELAXATION Li INSERTION ANALYSIS
# =============================================================================
#
# Purpose:
# Analyze the final relaxed Li10Si8Ti32 structure and determine:
#
# 1. Final Li positions
# 2. Nearest Ti and nearest Si for every Li
# 3. Li-Ti and Li-Si distances
# 4. Whether Li atoms occupy distinct regions or cluster
# 5. Li-Li distances
# 6. Local host distortion
# 7. Initial Li sites -> final Li positions
#
# IMPORTANT:
# - No arbitrary Li-host cutoff is assumed.
# - No arbitrary "good" distance is assumed.
# - Distances are obtained directly from the relaxed structure.
# - Periodic boundary conditions are included through MIC.
# =============================================================================

import os
import numpy as np

from google.colab import drive
from ase.io import read


# =============================================================================
# 1. MOUNT GOOGLE DRIVE
# =============================================================================

drive.mount("/content/drive")


# =============================================================================
# 2. PATHS
# =============================================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

HOST_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_40atom_relaxed_host.extxyz"
)

INITIAL_LI_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_10Li_interstitial_sites.npy"
)

FINAL_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_40atom_10Li_inserted_relaxed.extxyz"
)

REPORT_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_10Li_post_relaxation_analysis.txt"
)


# =============================================================================
# 3. HEADER
# =============================================================================

print("=" * 100)
print("CANDIDATE 81 — POST-RELAXATION Li INSERTION ANALYSIS")
print("=" * 100)

print("Original relaxed host :")
print(HOST_FILE)

print("\nInitial Li sites      :")
print(INITIAL_LI_FILE)

print("\nFinal relaxed system  :")
print(FINAL_FILE)

print("=" * 100)


# =============================================================================
# 4. LOAD STRUCTURES
# =============================================================================

print("\nLoading structures...")

host_initial = read(HOST_FILE)
system_final = read(FINAL_FILE)

initial_li_positions = np.asarray(
    np.load(INITIAL_LI_FILE),
    dtype=float
)


# =============================================================================
# 5. VALIDATION
# =============================================================================

print("\n" + "=" * 100)
print("VALIDATION")
print("=" * 100)

n_host = len(host_initial)
n_final = len(system_final)
n_initial_li = len(initial_li_positions)

print("Initial host atoms :", n_host)
print("Final system atoms :", n_final)
print("Initial Li sites   :", n_initial_li)


if n_host != 40:
    raise ValueError(
        f"Expected 40 host atoms, found {n_host}."
    )

if n_initial_li != 10:
    raise ValueError(
        f"Expected 10 initial Li sites, found {n_initial_li}."
    )

if n_final != 50:
    raise ValueError(
        f"Expected 50 atoms in final system, found {n_final}."
    )


# Identify atoms from chemical symbols.
symbols = np.array(
    system_final.get_chemical_symbols()
)

li_indices = np.where(
    symbols == "Li"
)[0]

host_indices = np.where(
    symbols != "Li"
)[0]

ti_indices = np.where(
    symbols == "Ti"
)[0]

si_indices = np.where(
    symbols == "Si"
)[0]


print("\nLi atoms identified   :", len(li_indices))
print("Host atoms identified :", len(host_indices))
print("Ti atoms identified   :", len(ti_indices))
print("Si atoms identified   :", len(si_indices))


if len(li_indices) != 10:
    raise ValueError(
        f"Expected 10 Li atoms, found {len(li_indices)}."
    )

if len(host_indices) != 40:
    raise ValueError(
        f"Expected 40 host atoms, found {len(host_indices)}."
    )


# =============================================================================
# 6. CELL
# =============================================================================

print("\n" + "=" * 100)
print("CELL")
print("=" * 100)

initial_volume = host_initial.get_volume()
final_volume = system_final.get_volume()

cell_difference = (
    host_initial.cell.array
    - system_final.cell.array
)

cell_difference_max = np.max(
    np.abs(cell_difference)
)

print(
    f"Initial host volume : "
    f"{initial_volume:.8f} A^3"
)

print(
    f"Final system volume : "
    f"{final_volume:.8f} A^3"
)

print(
    f"Cell difference max : "
    f"{cell_difference_max:.12e} A"
)


# =============================================================================
# 7. INITIAL -> FINAL Li MOVEMENT
# =============================================================================

print("\n" + "=" * 100)
print("Li INITIAL -> FINAL MOVEMENT")
print("=" * 100)

final_li_positions = (
    system_final.positions[li_indices]
)

li_displacements = []

movement_lines = []

for i in range(10):

    initial_pos = initial_li_positions[i]

    final_pos = final_li_positions[i]

    # IMPORTANT:
    # Calculate displacement using ASE's periodic MIC handling.
    #
    # We create a temporary 2-atom system containing:
    # initial Li position + final Li position.
    #
    # This avoids confusing Cartesian coordinates with atom indices.

    from ase import Atoms

    temp = Atoms(
        symbols=["Li", "Li"],
        positions=[
            initial_pos,
            final_pos
        ],
        cell=system_final.cell,
        pbc=system_final.pbc
    )

    displacement_vector = temp.get_distance(
        0,
        1,
        mic=True,
        vector=True
    )

    displacement = np.linalg.norm(
        displacement_vector
    )

    li_displacements.append(
        displacement
    )

    line = (
        f"Li {i+1:2d} | "
        f"initial = {np.array2string(initial_pos, precision=8)} | "
        f"final = {np.array2string(final_pos, precision=8)} | "
        f"displacement = {displacement:.6f} A"
    )

    print(line)

    movement_lines.append(line)


li_displacements = np.asarray(
    li_displacements
)

mean_displacement = np.mean(
    li_displacements
)

median_displacement = np.median(
    li_displacements
)

min_displacement = np.min(
    li_displacements
)

max_displacement = np.max(
    li_displacements
)

print("\n" + "-" * 100)

print(
    f"Mean Li displacement   : "
    f"{mean_displacement:.6f} A"
)

print(
    f"Median Li displacement : "
    f"{median_displacement:.6f} A"
)

print(
    f"Minimum Li displacement: "
    f"{min_displacement:.6f} A"
)

print(
    f"Maximum Li displacement: "
    f"{max_displacement:.6f} A"
)


# =============================================================================
# 8. NEAREST Ti AND Si FOR EACH Li
# =============================================================================

print("\n" + "=" * 100)
print("NEAREST HOST ATOMS")
print("=" * 100)

nearest_host_lines = []

nearest_ti_data = []
nearest_si_data = []

for i, li_idx in enumerate(li_indices):

    # -------------------------------------------------------------------------
    # Li -> Ti
    # -------------------------------------------------------------------------

    if len(ti_indices) > 0:

        ti_distances = system_final.get_distances(
            li_idx,
            ti_indices,
            mic=True
        )

        ti_local = np.argmin(
            ti_distances
        )

        nearest_ti_idx = int(
            ti_indices[ti_local]
        )

        nearest_ti_distance = float(
            ti_distances[ti_local]
        )

    else:

        nearest_ti_idx = None
        nearest_ti_distance = np.nan


    # -------------------------------------------------------------------------
    # Li -> Si
    # -------------------------------------------------------------------------

    if len(si_indices) > 0:

        si_distances = system_final.get_distances(
            li_idx,
            si_indices,
            mic=True
        )

        si_local = np.argmin(
            si_distances
        )

        nearest_si_idx = int(
            si_indices[si_local]
        )

        nearest_si_distance = float(
            si_distances[si_local]
        )

    else:

        nearest_si_idx = None
        nearest_si_distance = np.nan


    nearest_ti_data.append(
        (
            i + 1,
            nearest_ti_idx,
            nearest_ti_distance
        )
    )

    nearest_si_data.append(
        (
            i + 1,
            nearest_si_idx,
            nearest_si_distance
        )
    )


    print(
        f"\nLi {i+1:2d}"
    )

    print(
        f"  nearest Ti : "
        f"atom {nearest_ti_idx} | "
        f"{nearest_ti_distance:.6f} A"
    )

    print(
        f"  nearest Si : "
        f"atom {nearest_si_idx} | "
        f"{nearest_si_distance:.6f} A"
    )


# =============================================================================
# 9. COMPLETE Li-HOST DISTANCE INFORMATION
# =============================================================================

print("\n" + "=" * 100)
print("Li-HOST DISTANCE DISTRIBUTIONS")
print("=" * 100)

li_ti_distance_matrix = np.zeros(
    (len(li_indices), len(ti_indices))
)

li_si_distance_matrix = np.zeros(
    (len(li_indices), len(si_indices))
)

for i, li_idx in enumerate(li_indices):

    li_ti_distance_matrix[i, :] = (
        system_final.get_distances(
            li_idx,
            ti_indices,
            mic=True
        )
    )

    li_si_distance_matrix[i, :] = (
        system_final.get_distances(
            li_idx,
            si_indices,
            mic=True
        )
    )


all_li_ti_distances = (
    li_ti_distance_matrix.flatten()
)

all_li_si_distances = (
    li_si_distance_matrix.flatten()
)


print(
    "\nLi-Ti distances:"
)

print(
    f"  minimum : "
    f"{np.min(all_li_ti_distances):.6f} A"
)

print(
    f"  mean    : "
    f"{np.mean(all_li_ti_distances):.6f} A"
)

print(
    f"  maximum : "
    f"{np.max(all_li_ti_distances):.6f} A"
)


print(
    "\nLi-Si distances:"
)

print(
    f"  minimum : "
    f"{np.min(all_li_si_distances):.6f} A"
)

print(
    f"  mean    : "
    f"{np.mean(all_li_si_distances):.6f} A"
)

print(
    f"  maximum : "
    f"{np.max(all_li_si_distances):.6f} A"
)


# =============================================================================
# 10. Li-Li DISTANCES
# =============================================================================

print("\n" + "=" * 100)
print("Li-Li DISTANCES")
print("=" * 100)

li_li_distance_matrix = np.zeros(
    (len(li_indices), len(li_indices))
)

for i, li_i in enumerate(li_indices):

    for j, li_j in enumerate(li_indices):

        if i == j:
            continue

        li_li_distance_matrix[i, j] = (
            system_final.get_distance(
                li_i,
                li_j,
                mic=True
            )
        )


# Extract unique pair distances only.
li_li_pairs = []

for i in range(len(li_indices)):

    for j in range(i + 1, len(li_indices)):

        d = system_final.get_distance(
            li_indices[i],
            li_indices[j],
            mic=True
        )

        li_li_pairs.append(
            (
                i + 1,
                j + 1,
                float(d)
            )
        )


li_li_pairs.sort(
    key=lambda x: x[2]
)

print(
    f"Number of unique Li-Li pairs : "
    f"{len(li_li_pairs)}"
)

print("\nLi-Li pairs sorted by distance:")

for pair in li_li_pairs:

    i, j, d = pair

    print(
        f"Li {i:2d} - Li {j:2d} : "
        f"{d:.6f} A"
    )


# =============================================================================
# 11. HOST DISTORTION
# =============================================================================
#
# We compare every host atom in:
#
#     original relaxed 40-atom host
#
# against the same host atom in:
#
#     final Li10 + host structure
#
# The atom ordering is preserved because the insertion code appended Li atoms
# to the original host.
#
# We use periodic MIC distances for the displacement.
# =============================================================================

print("\n" + "=" * 100)
print("HOST DISTORTION AFTER Li INSERTION")
print("=" * 100)

host_displacements = []

for i in range(40):

    host_temp = system_final.copy()

    # The original host atom is atom i in both structures.
    #
    # Construct a two-atom temporary system containing the initial
    # and final position of that same host atom.

    from ase import Atoms

    temp = Atoms(
        symbols=[
            host_initial[i].symbol,
            system_final[i].symbol
        ],
        positions=[
            host_initial[i].position,
            system_final[i].position
        ],
        cell=system_final.cell,
        pbc=system_final.pbc
    )

    displacement = temp.get_distance(
        0,
        1,
        mic=True
    )

    host_displacements.append(
        float(displacement)
    )


host_displacements = np.asarray(
    host_displacements
)

print(
    f"Mean host displacement   : "
    f"{np.mean(host_displacements):.6f} A"
)

print(
    f"Median host displacement : "
    f"{np.median(host_displacements):.6f} A"
)

print(
    f"Minimum host displacement: "
    f"{np.min(host_displacements):.6f} A"
)

print(
    f"Maximum host displacement: "
    f"{np.max(host_displacements):.6f} A"
)


# Print the largest host movements.

largest_host_indices = np.argsort(
    host_displacements
)[::-1]


print("\nLargest host-atom movements:")

for rank in range(
    min(10, len(largest_host_indices))
):

    idx = int(
        largest_host_indices[rank]
    )

    print(
        f"{rank+1:2d} | "
        f"host atom {idx:2d} | "
        f"{host_initial[idx].symbol} | "
        f"{host_displacements[idx]:.6f} A"
    )


# =============================================================================
# 12. FINAL Li POSITIONS
# =============================================================================

print("\n" + "=" * 100)
print("FINAL Li POSITIONS")
print("=" * 100)

final_position_lines = []

for i, li_idx in enumerate(li_indices):

    pos = system_final.positions[li_idx]

    line = (
        f"Li {i+1:2d} | "
        f"atom index = {li_idx:2d} | "
        f"position = "
        f"{np.array2string(pos, precision=8)}"
    )

    print(line)

    final_position_lines.append(line)


# =============================================================================
# 13. Li FRACTIONAL POSITIONS
# =============================================================================

print("\n" + "=" * 100)
print("FINAL Li FRACTIONAL POSITIONS")
print("=" * 100)

final_fractional_lines = []

scaled_positions = (
    system_final.get_scaled_positions(
        wrap=True
    )
)

for i, li_idx in enumerate(li_indices):

    frac = scaled_positions[li_idx]

    line = (
        f"Li {i+1:2d} | "
        f"fractional = "
        f"{np.array2string(frac, precision=8)}"
    )

    print(line)

    final_fractional_lines.append(line)


# =============================================================================
# 14. BASIC CLUSTERING INFORMATION
# =============================================================================
#
# We do NOT impose a clustering cutoff.
#
# Instead we report the actual Li-Li distance distribution and identify
# the closest Li pairs. Whether those distances indicate meaningful
# clustering will be interpreted from the structure rather than from
# an arbitrary threshold here.
# =============================================================================

print("\n" + "=" * 100)
print("Li-Li SPATIAL ORGANIZATION")
print("=" * 100)

closest_pairs = li_li_pairs[
    :min(10, len(li_li_pairs))
]

print(
    "\nClosest Li-Li pairs:"
)

for i, j, d in closest_pairs:

    print(
        f"Li {i:2d} - Li {j:2d} : "
        f"{d:.6f} A"
    )


# =============================================================================
# 15. SUMMARY STATISTICS
# =============================================================================

print("\n" + "=" * 100)
print("ANALYSIS SUMMARY")
print("=" * 100)

print(
    f"Li count                    : {len(li_indices)}"
)

print(
    f"Host count                  : {len(host_indices)}"
)

print(
    f"Mean Li displacement        : "
    f"{mean_displacement:.6f} A"
)

print(
    f"Maximum Li displacement     : "
    f"{max_displacement:.6f} A"
)

print(
    f"Mean host displacement      : "
    f"{np.mean(host_displacements):.6f} A"
)

print(
    f"Maximum host displacement   : "
    f"{np.max(host_displacements):.6f} A"
)

print(
    f"Minimum Li-Li distance      : "
    f"{np.min([x[2] for x in li_li_pairs]):.6f} A"
)

print(
    f"Mean Li-Li pair distance    : "
    f"{np.mean([x[2] for x in li_li_pairs]):.6f} A"
)

print(
    f"Initial host volume         : "
    f"{initial_volume:.6f} A^3"
)

print(
    f"Final cell volume           : "
    f"{final_volume:.6f} A^3"
)


# =============================================================================
# 16. SAVE HUMAN-READABLE REPORT
# =============================================================================

print("\n" + "=" * 100)
print("SAVING ANALYSIS REPORT")
print("=" * 100)

with open(
    REPORT_FILE,
    "w"
) as f:

    f.write(
        "CANDIDATE 81 — POST-RELAXATION Li INSERTION ANALYSIS\n"
    )

    f.write(
        "=" * 100 + "\n\n"
    )

    f.write(
        f"Initial host file:\n{HOST_FILE}\n\n"
    )

    f.write(
        f"Initial Li sites:\n{INITIAL_LI_FILE}\n\n"
    )

    f.write(
        f"Final structure:\n{FINAL_FILE}\n\n"
    )


    # -------------------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------------------

    f.write(
        "VALIDATION\n"
    )

    f.write(
        "-" * 100 + "\n"
    )

    f.write(
        f"Initial host atoms : {n_host}\n"
    )

    f.write(
        f"Final system atoms : {n_final}\n"
    )

    f.write(
        f"Initial Li sites   : {n_initial_li}\n"
    )

    f.write(
        f"Final Li atoms     : {len(li_indices)}\n"
    )

    f.write(
        f"Final host atoms   : {len(host_indices)}\n\n"
    )


    # -------------------------------------------------------------------------
    # Cell
    # -------------------------------------------------------------------------

    f.write(
        "CELL\n"
    )

    f.write(
        "-" * 100 + "\n"
    )

    f.write(
        f"Initial host volume : {initial_volume:.10f} A^3\n"
    )

    f.write(
        f"Final system volume : {final_volume:.10f} A^3\n"
    )

    f.write(
        f"Cell difference max : {cell_difference_max:.12e} A\n\n"
    )


    # -------------------------------------------------------------------------
    # Li movement
    # -------------------------------------------------------------------------

    f.write(
        "Li INITIAL -> FINAL MOVEMENT\n"
    )

    f.write(
        "-" * 100 + "\n"
    )

    for line in movement_lines:

        f.write(
            line + "\n"
        )

    f.write("\n")

    f.write(
        f"Mean Li displacement   : "
        f"{mean_displacement:.8f} A\n"
    )

    f.write(
        f"Median Li displacement : "
        f"{median_displacement:.8f} A\n"
    )

    f.write(
        f"Minimum Li displacement: "
        f"{min_displacement:.8f} A\n"
    )

    f.write(
        f"Maximum Li displacement: "
        f"{max_displacement:.8f} A\n\n"
    )


    # -------------------------------------------------------------------------
    # Nearest host
    # -------------------------------------------------------------------------

    f.write(
        "NEAREST Ti AND Si FOR EACH Li\n"
    )

    f.write(
        "-" * 100 + "\n"
    )

    for i in range(10):

        li_number = i + 1

        ti_number = nearest_ti_data[i][1]
        ti_distance = nearest_ti_data[i][2]

        si_number = nearest_si_data[i][1]
        si_distance = nearest_si_data[i][2]

        f.write(
            f"Li {li_number:2d} | "
            f"nearest Ti atom {ti_number} : "
            f"{ti_distance:.8f} A | "
            f"nearest Si atom {si_number} : "
            f"{si_distance:.8f} A\n"
        )

    f.write("\n")


    # -------------------------------------------------------------------------
    # Li-Li
    # -------------------------------------------------------------------------

    f.write(
        "Li-Li DISTANCES\n"
    )

    f.write(
        "-" * 100 + "\n"
    )

    for i, j, d in li_li_pairs:

        f.write(
            f"Li {i:2d} - Li {j:2d} : "
            f"{d:.8f} A\n"
        )

    f.write("\n")


    # -------------------------------------------------------------------------
    # Host distortion
    # -------------------------------------------------------------------------

    f.write(
        "HOST DISTORTION\n"
    )

    f.write(
        "-" * 100 + "\n"
    )

    f.write(
        f"Mean host displacement   : "
        f"{np.mean(host_displacements):.8f} A\n"
    )

    f.write(
        f"Median host displacement : "
        f"{np.median(host_displacements):.8f} A\n"
    )

    f.write(
        f"Minimum host displacement: "
        f"{np.min(host_displacements):.8f} A\n"
    )

    f.write(
        f"Maximum host displacement: "
        f"{np.max(host_displacements):.8f} A\n\n"
    )

    f.write(
        "Largest host movements:\n"
    )

    for rank in range(
        min(10, len(largest_host_indices))
    ):

        idx = int(
            largest_host_indices[rank]
        )

        f.write(
            f"{rank+1:2d} | "
            f"host atom {idx:2d} | "
            f"{host_initial[idx].symbol} | "
            f"{host_displacements[idx]:.8f} A\n"
        )

    f.write("\n")


    # -------------------------------------------------------------------------
    # Final Li positions
    # -------------------------------------------------------------------------

    f.write(
        "FINAL Li POSITIONS\n"
    )

    f.write(
        "-" * 100 + "\n"
    )

    for line in final_position_lines:

        f.write(
            line + "\n"
        )

    f.write("\n")


    f.write(
        "FINAL Li FRACTIONAL POSITIONS\n"
    )

    f.write(
        "-" * 100 + "\n"
    )

    for line in final_fractional_lines:

        f.write(
            line + "\n"
        )

    f.write("\n")


    # -------------------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------------------

    f.write(
        "SUMMARY\n"
    )

    f.write(
        "-" * 100 + "\n"
    )

    f.write(
        f"Li count                  : {len(li_indices)}\n"
    )

    f.write(
        f"Host count                : {len(host_indices)}\n"
    )

    f.write(
        f"Mean Li displacement      : "
        f"{mean_displacement:.8f} A\n"
    )

    f.write(
        f"Maximum Li displacement   : "
        f"{max_displacement:.8f} A\n"
    )

    f.write(
        f"Mean host displacement    : "
        f"{np.mean(host_displacements):.8f} A\n"
    )

    f.write(
        f"Maximum host displacement : "
        f"{np.max(host_displacements):.8f} A\n"
    )

    f.write(
        f"Minimum Li-Li distance    : "
        f"{np.min([x[2] for x in li_li_pairs]):.8f} A\n"
    )


# =============================================================================
# 17. FINAL
# =============================================================================

print("\n" + "=" * 100)
print("ANALYSIS COMPLETE")
print("=" * 100)

print(
    "\nReport saved to:"
)

print(
    REPORT_FILE
)

print("\n" + "=" * 100)

In [ ]:
# =============================================================================
# CANDIDATE 81 — DATA-DRIVEN Li NETWORK / BASIN ANALYSIS
# =============================================================================
#
# Purpose:
#
# Analyze the final relaxed Li10Si8Ti32 structure and determine whether
# the relaxed Li atoms form:
#
#   1. distinct local basins,
#   2. connected groups,
#   3. or a distributed Li network.
#
# IMPORTANT:
# No hand-picked Li-Li cutoff is used.
# No assumed "good" Li-Li distance is used.
# No assumed diffusion bottleneck is used.
#
# The actual Li-Li distance distribution determines the graph structure.
#
# We also compare:
#
#   - initial Li insertion sites
#   - final relaxed Li positions
#   - Li-Li distances
#   - nearest Ti / Si atoms
#   - host atomic displacements
#
# This is still NOT a diffusion calculation.
# It is a structural diagnostic before selecting diffusion pathways.
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import os
import numpy as np
import pandas as pd

from google.colab import drive
from ase.io import read
from scipy.sparse.csgraph import minimum_spanning_tree, connected_components


# =============================================================================
# 2. MOUNT GOOGLE DRIVE
# =============================================================================

drive.mount("/content/drive")


# =============================================================================
# 3. PATHS
# =============================================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

HOST_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_40atom_relaxed_host.extxyz"
)

INITIAL_LI_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_10Li_interstitial_sites.npy"
)

FINAL_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_40atom_10Li_inserted_relaxed.extxyz"
)

OUTPUT_LI_CSV = os.path.join(
    BASE_DIR,
    "candidate_81_Li_network_analysis.csv"
)

OUTPUT_LILI_CSV = os.path.join(
    BASE_DIR,
    "candidate_81_Li_Li_distance_matrix.csv"
)

OUTPUT_GRAPH_CSV = os.path.join(
    BASE_DIR,
    "candidate_81_Li_network_edges.csv"
)

OUTPUT_HOST_CSV = os.path.join(
    BASE_DIR,
    "candidate_81_host_distortion_analysis.csv"
)


# =============================================================================
# 4. HEADER
# =============================================================================

print("=" * 100)
print("CANDIDATE 81 — DATA-DRIVEN Li NETWORK / BASIN ANALYSIS")
print("=" * 100)

print("\nHost:")
print(HOST_FILE)

print("\nInitial Li sites:")
print(INITIAL_LI_FILE)

print("\nFinal relaxed Li-host structure:")
print(FINAL_FILE)

print("=" * 100)


# =============================================================================
# 5. LOAD STRUCTURES
# =============================================================================

print("\nLoading structures...")

host_initial = read(
    HOST_FILE,
    index=0
)

system_final = read(
    FINAL_FILE,
    index=0
)

initial_li_positions = np.asarray(
    np.load(INITIAL_LI_FILE),
    dtype=float
)


# =============================================================================
# 6. VALIDATION
# =============================================================================

print("\n" + "=" * 100)
print("VALIDATION")
print("=" * 100)

if len(host_initial) != 40:
    raise ValueError(
        f"Expected 40 host atoms, found {len(host_initial)}"
    )

if len(system_final) != 50:
    raise ValueError(
        f"Expected 50 atoms in final system, found {len(system_final)}"
    )

if len(initial_li_positions) != 10:
    raise ValueError(
        f"Expected 10 initial Li sites, "
        f"found {len(initial_li_positions)}"
    )


# Identify atoms from chemical symbols
symbols_final = np.array(
    system_final.get_chemical_symbols()
)

li_indices = np.where(
    symbols_final == "Li"
)[0]

host_indices = np.where(
    symbols_final != "Li"
)[0]

ti_indices = np.where(
    symbols_final == "Ti"
)[0]

si_indices = np.where(
    symbols_final == "Si"
)[0]


if len(li_indices) != 10:
    raise ValueError(
        f"Expected 10 Li atoms, found {len(li_indices)}"
    )

if len(host_indices) != 40:
    raise ValueError(
        f"Expected 40 host atoms, found {len(host_indices)}"
    )

if len(ti_indices) != 32:
    raise ValueError(
        f"Expected 32 Ti atoms, found {len(ti_indices)}"
    )

if len(si_indices) != 8:
    raise ValueError(
        f"Expected 8 Si atoms, found {len(si_indices)}"
    )

print(
    f"Initial host atoms : {len(host_initial)}"
)

print(
    f"Final system atoms : {len(system_final)}"
)

print(
    f"Li atoms           : {len(li_indices)}"
)

print(
    f"Ti atoms           : {len(ti_indices)}"
)

print(
    f"Si atoms           : {len(si_indices)}"
)


# =============================================================================
# 7. BASIC CELL CHECK
# =============================================================================

print("\n" + "=" * 100)
print("CELL")
print("=" * 100)

initial_volume = (
    host_initial.get_volume()
)

final_volume = (
    system_final.get_volume()
)

cell_difference = np.max(
    np.abs(
        host_initial.cell.array
        - system_final.cell.array
    )
)

print(
    f"Initial host volume : "
    f"{initial_volume:.8f} A^3"
)

print(
    f"Final system volume : "
    f"{final_volume:.8f} A^3"
)

print(
    f"Maximum cell difference : "
    f"{cell_difference:.12e} A"
)


# =============================================================================
# 8. FINAL Li POSITIONS
# =============================================================================

final_li_positions = (
    system_final.positions[
        li_indices
    ].copy()
)


print("\n" + "=" * 100)
print("FINAL Li POSITIONS")
print("=" * 100)

for i in range(len(li_indices)):

    print(
        f"Li {i+1:2d} | "
        f"atom index = {li_indices[i]:2d} | "
        f"position = "
        f"{final_li_positions[i]}"
    )


# =============================================================================
# 9. INITIAL -> FINAL Li MOVEMENT
# =============================================================================
#
# IMPORTANT:
# ASE's get_distances is used with the final cell and MIC.
# This avoids problems caused by atoms crossing periodic boundaries.
# =============================================================================

print("\n" + "=" * 100)
print("INITIAL -> FINAL Li MOVEMENT")
print("=" * 100)

li_displacements = []

movement_records = []

for i in range(10):

    final_index = int(
        li_indices[i]
    )

    distance = system_final.get_distance(
        final_index,
        final_index,
        mic=False
    )

    # Direct vector between the initial insertion site
    # and the corresponding final Li position.
    #
    # We use ASE's MIC through a temporary two-point distance
    # calculation implemented from fractional coordinates below.

    initial_cart = initial_li_positions[i]

    final_cart = final_li_positions[i]

    # Convert both positions to fractional coordinates.
    initial_frac = np.linalg.solve(
        system_final.cell.array.T,
        initial_cart
    )

    final_frac = np.linalg.solve(
        system_final.cell.array.T,
        final_cart
    )

    delta_frac = (
        final_frac - initial_frac
    )

    # Apply periodic minimum image.
    delta_frac -= np.round(
        delta_frac
    )

    delta_cart = (
        delta_frac
        @ system_final.cell.array
    )

    displacement = float(
        np.linalg.norm(delta_cart)
    )

    li_displacements.append(
        displacement
    )

    movement_records.append({

        "Li": i + 1,

        "initial_x_A":
            initial_cart[0],

        "initial_y_A":
            initial_cart[1],

        "initial_z_A":
            initial_cart[2],

        "final_x_A":
            final_cart[0],

        "final_y_A":
            final_cart[1],

        "final_z_A":
            final_cart[2],

        "displacement_A":
            displacement
    })

    print(
        f"Li {i+1:2d} | "
        f"displacement = "
        f"{displacement:.6f} A"
    )


li_displacements = np.asarray(
    li_displacements
)

print(
    f"\nMean displacement   : "
    f"{np.mean(li_displacements):.6f} A"
)

print(
    f"Median displacement : "
    f"{np.median(li_displacements):.6f} A"
)

print(
    f"Minimum displacement : "
    f"{np.min(li_displacements):.6f} A"
)

print(
    f"Maximum displacement : "
    f"{np.max(li_displacements):.6f} A"
)


# =============================================================================
# 10. NEAREST Ti AND Si FOR EACH Li
# =============================================================================

print("\n" + "=" * 100)
print("FINAL Li LOCAL ENVIRONMENT")
print("=" * 100)

li_environment_records = []

for i in range(10):

    li_index = int(
        li_indices[i]
    )

    # Distances from this Li to every Ti
    ti_distances = np.array([
        system_final.get_distance(
            li_index,
            int(ti_index),
            mic=True
        )
        for ti_index in ti_indices
    ])

    # Distances from this Li to every Si
    si_distances = np.array([
        system_final.get_distance(
            li_index,
            int(si_index),
            mic=True
        )
        for si_index in si_indices
    ])

    ti_order = np.argsort(
        ti_distances
    )

    si_order = np.argsort(
        si_distances
    )

    nearest_ti_local = (
        ti_order[0]
    )

    nearest_si_local = (
        si_order[0]
    )

    nearest_ti_atom = int(
        ti_indices[nearest_ti_local]
    )

    nearest_si_atom = int(
        si_indices[nearest_si_local]
    )

    nearest_ti_distance = float(
        ti_distances[nearest_ti_local]
    )

    nearest_si_distance = float(
        si_distances[nearest_si_local]
    )

    print(
        f"\nLi {i+1}"
    )

    print(
        f"  nearest Ti : "
        f"atom {nearest_ti_atom}, "
        f"{nearest_ti_distance:.6f} A"
    )

    print(
        f"  nearest Si : "
        f"atom {nearest_si_atom}, "
        f"{nearest_si_distance:.6f} A"
    )

    li_environment_records.append({

        "Li":
            i + 1,

        "nearest_Ti_atom":
            nearest_ti_atom,

        "nearest_Ti_distance_A":
            nearest_ti_distance,

        "nearest_Si_atom":
            nearest_si_atom,

        "nearest_Si_distance_A":
            nearest_si_distance
    })


# =============================================================================
# 11. FULL Li-Li DISTANCE MATRIX
# =============================================================================
#
# No Li-Li cutoff is chosen here.
# We simply calculate ALL pairwise periodic distances.
# =============================================================================

print("\n" + "=" * 100)
print("ALL PAIRWISE Li-Li DISTANCES")
print("=" * 100)

n_li = len(li_indices)

li_distance_matrix = np.zeros(
    (n_li, n_li)
)

li_pair_records = []

for i in range(n_li):

    for j in range(n_li):

        if i == j:
            continue

        distance = system_final.get_distance(
            int(li_indices[i]),
            int(li_indices[j]),
            mic=True
        )

        li_distance_matrix[i, j] = (
            distance
        )

        if j > i:

            li_pair_records.append({

                "Li_i":
                    i + 1,

                "Li_j":
                    j + 1,

                "distance_A":
                    float(distance)
            })


li_li_df = pd.DataFrame(
    li_pair_records
)

li_li_df = li_li_df.sort_values(
    "distance_A"
).reset_index(
    drop=True
)


print(
    li_li_df.to_string(
        index=False
    )
)


# =============================================================================
# 12. DATA-DRIVEN Li GRAPH
# =============================================================================
#
# We construct the Minimum Spanning Tree (MST).
#
# Why?
#
# The MST connects all Li atoms using the shortest possible total
# connection length without us inventing a distance cutoff.
#
# Then we inspect the actual MST edge-length distribution.
#
# If there is a strong gap between short and long MST edges,
# that gap can reveal natural separation between Li groups.
#
# This is a diagnostic, NOT a claim that the gap itself is a
# physical diffusion barrier.
# =============================================================================

print("\n" + "=" * 100)
print("DATA-DRIVEN Li GRAPH")
print("=" * 100)

mst = minimum_spanning_tree(
    li_distance_matrix
)

mst_dense = mst.toarray()

mst_edges = []

for i in range(n_li):

    for j in range(n_li):

        if mst_dense[i, j] > 0:

            mst_edges.append({

                "Li_i":
                    i + 1,

                "Li_j":
                    j + 1,

                "distance_A":
                    float(
                        mst_dense[i, j]
                    )
            })


mst_edges_df = pd.DataFrame(
    mst_edges
)

mst_edges_df = mst_edges_df.sort_values(
    "distance_A"
).reset_index(
    drop=True
)

print(
    "\nMST edges:"
)

print(
    mst_edges_df.to_string(
        index=False
    )
)


# =============================================================================
# 13. FIND LARGEST NATURAL GAP IN MST EDGE LENGTHS
# =============================================================================

mst_lengths = (
    mst_edges_df[
        "distance_A"
    ].to_numpy()
)

if len(mst_lengths) > 1:

    sorted_lengths = np.sort(
        mst_lengths
    )

    gaps = np.diff(
        sorted_lengths
    )

    largest_gap_index = int(
        np.argmax(gaps)
    )

    largest_gap = float(
        gaps[largest_gap_index]
    )

    lower_edge = float(
        sorted_lengths[
            largest_gap_index
        ]
    )

    upper_edge = float(
        sorted_lengths[
            largest_gap_index + 1
        ]
    )

    print(
        "\nNatural MST distance distribution:"
    )

    print(
        f"Small side of largest gap : "
        f"{lower_edge:.6f} A"
    )

    print(
        f"Large side of largest gap : "
        f"{upper_edge:.6f} A"
    )

    print(
        f"Largest gap               : "
        f"{largest_gap:.6f} A"
    )

else:

    largest_gap_index = None

    print(
        "\nNot enough MST edges to calculate "
        "a gap."
    )


# =============================================================================
# 14. NATURAL Li GROUPS FROM MST
# =============================================================================
#
# We cut ONLY the single largest MST edge.
#
# This is not a manually chosen distance.
#
# It is a structural diagnostic:
# "What happens if the strongest separation in the actual Li
#  distance distribution is used to split the graph?"
#
# We explicitly report the result rather than calling it
# a physical diffusion network.
# =============================================================================

if len(mst_edges_df) > 1:

    largest_edge_row = (
        mst_edges_df[
            "distance_A"
        ].idxmax()
    )

    graph_matrix = (
        li_distance_matrix.copy()
    )

    # Remove the largest MST edge.
    #
    # Find the corresponding pair.
    edge_i = int(
        mst_edges_df.loc[
            largest_edge_row,
            "Li_i"
        ]
    ) - 1

    edge_j = int(
        mst_edges_df.loc[
            largest_edge_row,
            "Li_j"
        ]
    ) - 1

    graph_matrix[
        edge_i,
        edge_j
    ] = 0.0

    graph_matrix[
        edge_j,
        edge_i
    ] = 0.0

    # Convert every remaining MST edge into connectivity.
    adjacency = np.zeros(
        (n_li, n_li),
        dtype=int
    )

    for _, row in mst_edges_df.iterrows():

        i = int(
            row["Li_i"]
        ) - 1

        j = int(
            row["Li_j"]
        ) - 1

        if (
            (i == edge_i and j == edge_j)
            or
            (i == edge_j and j == edge_i)
        ):
            continue

        adjacency[i, j] = 1
        adjacency[j, i] = 1

    n_groups, labels = connected_components(
        adjacency,
        directed=False
    )

    print("\n" + "=" * 100)
    print("NATURAL Li GROUPS AFTER REMOVING LARGEST MST GAP")
    print("=" * 100)

    print(
        f"Number of groups : {n_groups}"
    )

    group_records = []

    for group_id in range(
        n_groups
    ):

        members = (
            np.where(
                labels == group_id
            )[0]
            + 1
        )

        print(
            f"Group {group_id + 1}: "
            f"Li {members.tolist()}"
        )

        group_records.append({

            "group":
                group_id + 1,

            "Li_members":
                ",".join(
                    map(
                        str,
                        members.tolist()
                    )
                ),

            "size":
                len(members)
        })

else:

    n_groups = 1

    labels = np.zeros(
        n_li,
        dtype=int
    )

    group_records = [{
        "group": 1,
        "Li_members": ",".join(
            map(
                str,
                range(1, n_li + 1)
            )
        ),
        "size": n_li
    }]


# =============================================================================
# 15. Li-Li LOCAL NEIGHBOR ORDER
# =============================================================================
#
# Instead of defining a cutoff, report each Li's nearest Li neighbors.
# This lets us see whether Li atoms have repeated close neighbors.
# =============================================================================

print("\n" + "=" * 100)
print("Li LOCAL NEIGHBORS")
print("=" * 100)

for i in range(n_li):

    order = np.argsort(
        li_distance_matrix[i]
    )

    order = order[
        order != i
    ]

    print(
        f"\nLi {i+1}:"
    )

    for rank, j in enumerate(
        order[:3],
        start=1
    ):

        print(
            f"  neighbor {rank}: "
            f"Li {j+1}, "
            f"{li_distance_matrix[i, j]:.6f} A"
        )


# =============================================================================
# 16. HOST ATOM DISTORTION
# =============================================================================
#
# The final system contains:
#
#   first 40 non-Li atoms = host
#   10 Li atoms
#
# We identify host atoms by element, not by assuming that atom ordering
# must remain identical.
#
# For each host atom we compare the initial host structure to the
# corresponding final host atom by element + minimum periodic displacement.
#
# This is a structural diagnostic, not a predefined "large distortion"
# threshold.
# =============================================================================

print("\n" + "=" * 100)
print("HOST ATOM DISTORTION")
print("=" * 100)

initial_host_symbols = (
    np.array(
        host_initial.get_chemical_symbols()
    )
)

final_host_symbols = (
    symbols_final[host_indices]
)

if not np.array_equal(
    initial_host_symbols,
    final_host_symbols
):

    raise ValueError(
        "Host atom ordering/elements changed between "
        "initial and final structures. "
        "A direct atom-by-atom host comparison is unsafe."
    )


host_displacements = []

host_records = []

for i in range(40):

    initial_pos = (
        host_initial.positions[i]
    )

    final_index = int(
        host_indices[i]
    )

    final_pos = (
        system_final.positions[
            final_index
        ]
    )

    initial_frac = np.linalg.solve(
        system_final.cell.array.T,
        initial_pos
    )

    final_frac = np.linalg.solve(
        system_final.cell.array.T,
        final_pos
    )

    delta_frac = (
        final_frac
        - initial_frac
    )

    delta_frac -= np.round(
        delta_frac
    )

    delta_cart = (
        delta_frac
        @ system_final.cell.array
    )

    displacement = float(
        np.linalg.norm(delta_cart)
    )

    host_displacements.append(
        displacement
    )

    host_records.append({

        "host_atom":
            i,

        "element":
            initial_host_symbols[i],

        "displacement_A":
            displacement
    })


host_displacements = np.asarray(
    host_displacements
)

host_df = pd.DataFrame(
    host_records
)

host_df = host_df.sort_values(
    "displacement_A",
    ascending=False
).reset_index(
    drop=True
)

print(
    host_df.to_string(
        index=False
    )
)

print(
    f"\nMean host displacement : "
    f"{np.mean(host_displacements):.6f} A"
)

print(
    f"Median host displacement : "
    f"{np.median(host_displacements):.6f} A"
)

print(
    f"Maximum host displacement : "
    f"{np.max(host_displacements):.6f} A"
)


# =============================================================================
# 17. COMBINE Li INFORMATION
# =============================================================================

movement_df = pd.DataFrame(
    movement_records
)

environment_df = pd.DataFrame(
    li_environment_records
)

network_labels = pd.DataFrame({

    "Li":
        np.arange(
            1,
            n_li + 1
        ),

    "natural_group":
        labels + 1
})

li_analysis_df = (
    movement_df
    .merge(
        environment_df,
        on="Li"
    )
    .merge(
        network_labels,
        on="Li"
    )
)


# =============================================================================
# 18. SAVE RESULTS
# =============================================================================

li_analysis_df.to_csv(
    OUTPUT_LI_CSV,
    index=False
)

li_li_df.to_csv(
    OUTPUT_LILI_CSV,
    index=False
)

mst_edges_df.to_csv(
    OUTPUT_GRAPH_CSV,
    index=False
)

host_df.to_csv(
    OUTPUT_HOST_CSV,
    index=False
)


# =============================================================================
# 19. FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("FINAL DATA-DRIVEN SUMMARY")
print("=" * 100)

print(
    f"Final structure : "
    f"{system_final.get_chemical_formula()}"
)

print(
    f"Li atoms        : "
    f"{n_li}"
)

print(
    f"Mean Li movement : "
    f"{np.mean(li_displacements):.6f} A"
)

print(
    f"Maximum Li movement : "
    f"{np.max(li_displacements):.6f} A"
)

print(
    f"Shortest Li-Li distance : "
    f"{li_li_df['distance_A'].min():.6f} A"
)

print(
    f"Longest Li-Li distance : "
    f"{li_li_df['distance_A'].max():.6f} A"
)

print(
    f"Mean host displacement : "
    f"{np.mean(host_displacements):.6f} A"
)

print(
    f"Maximum host displacement : "
    f"{np.max(host_displacements):.6f} A"
)

print(
    f"Natural Li groups from MST diagnostic : "
    f"{n_groups}"
)

if len(mst_edges_df) > 1:

    print(
        f"Largest MST separation : "
        f"{largest_gap:.6f} A"
    )

print("\nSaved files:")

print(
    OUTPUT_LI_CSV
)

print(
    OUTPUT_LILI_CSV
)

print(
    OUTPUT_GRAPH_CSV
)

print(
    OUTPUT_HOST_CSV
)

print("\n" + "=" * 100)
print("Li NETWORK / BASIN ANALYSIS COMPLETE")
print("=" * 100)

In [ ]:
# =============================================================================
# CANDIDATE 81 — COMPLETE Li INTERSTITIAL ENERGY LANDSCAPE SCREENING
# RESTART-SAFE + IMMEDIATE CHECKPOINT VERSION
# =============================================================================
#
# PURPOSE
# -------
# Determine which periodic Voronoi interstitial sites of the fully relaxed
# 40-atom Candidate 81 host correspond to genuine Li local energy minima.
#
# IMPORTANT SCIENTIFIC RULES
# --------------------------
# 1. NO hand-picked Li-host distance cutoff.
# 2. NO artificial "good distance".
# 3. NO diffusion / NEB yet.
# 4. Voronoi geometry generates the candidate sites.
# 5. MACE energy + relaxation determines the local minimum.
# 6. Different initial sites may converge to the same final basin.
#
# RESTART / CHECKPOINT RULES
# --------------------------
# 7. Every COMPLETED site is saved immediately.
# 8. On restart, already-saved sites are detected by Voronoi site index.
# 9. Previously completed sites are NEVER recalculated.
# 10. CSV and EXTXYZ are both used for recovery.
# 11. There is NO MAX_SITES_TO_TEST limit.
# 12. Runtime interruption does not erase completed sites.
#
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import os
import csv
import numpy as np
import torch

from google.colab import drive

from ase import Atoms
from ase.io import read, write
from ase.optimize import BFGS

from scipy.spatial import Voronoi

from mace.calculators import mace_mp


# =============================================================================
# 2. MOUNT GOOGLE DRIVE
# =============================================================================

drive.mount("/content/drive")


# =============================================================================
# 3. PATHS
# =============================================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

HOST_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_40atom_relaxed_host.extxyz"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "candidate_81_Li_single_site_energy_screening"
)

RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "candidate_81_all_tested_Li_sites_results.extxyz"
)

SUMMARY_FILE = os.path.join(
    OUTPUT_DIR,
    "candidate_81_Li_site_energy_summary.csv"
)

SITE_FILE = os.path.join(
    OUTPUT_DIR,
    "candidate_81_all_periodic_voronoi_sites.npy"
)


# =============================================================================
# 4. PARAMETERS
# =============================================================================

CANDIDATE_ID = 81

FMAX = 0.01

MAX_STEPS = 1000

REPLICATIONS = [-1, 0, 1]

DUPLICATE_TOL = 1e-5


# =============================================================================
# 5. CREATE OUTPUT DIRECTORY
# =============================================================================

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# =============================================================================
# 6. HEADER
# =============================================================================

print("=" * 100)
print("CANDIDATE 81 — COMPLETE Li INTERSTITIAL ENERGY LANDSCAPE SCREENING")
print("RESTART-SAFE + IMMEDIATE CHECKPOINT VERSION")
print("=" * 100)

print("Host file       :", HOST_FILE)
print("Output directory:", OUTPUT_DIR)
print("Method          :", "Periodic Voronoi + single-Li MACE relaxation")
print("Li-host cutoff  :", "NONE")
print("Artificial distance:", "NONE")
print("Cell            :", "FIXED")
print("Force threshold :", FMAX, "eV/A")
print("Maximum steps   :", MAX_STEPS)
print("Total sites     :", "ALL 866")
print("Checkpoint      :", "IMMEDIATE")
print("Restart mode    :", "AUTOMATIC")
print("=" * 100)


# =============================================================================
# 7. LOAD HOST
# =============================================================================

print("\nLoading fully relaxed Candidate 81 host...")

if not os.path.isfile(HOST_FILE):
    raise FileNotFoundError(
        f"\nHost file not found:\n{HOST_FILE}"
    )

host = read(
    HOST_FILE,
    index=0
).copy()


# =============================================================================
# 8. VALIDATE HOST
# =============================================================================

print("\n" + "=" * 100)
print("HOST VALIDATION")
print("=" * 100)

print("Formula :", host.get_chemical_formula())
print("Atoms   :", len(host))
print("Volume  :", host.get_volume(), "A^3")
print("PBC     :", host.get_pbc())

if len(host) != 40:
    raise ValueError(
        f"Expected 40 host atoms, found {len(host)}."
    )

if not np.all(host.get_pbc()):
    raise ValueError(
        "Host must be fully periodic."
    )

cell = np.asarray(
    host.cell.array,
    dtype=float
)

if cell.shape != (3, 3):
    raise ValueError(
        "Invalid cell matrix."
    )

if host.get_volume() <= 0:
    raise ValueError(
        "Host volume must be positive."
    )


# =============================================================================
# 9. BUILD PERIODIC HOST IMAGES
# =============================================================================

print("\n" + "=" * 100)
print("BUILDING PERIODIC VORONOI GEOMETRY")
print("=" * 100)

fractional = host.get_scaled_positions(
    wrap=False
)

image_positions = []

for ix in REPLICATIONS:

    for iy in REPLICATIONS:

        for iz in REPLICATIONS:

            shift = np.array(
                [ix, iy, iz],
                dtype=float
            )

            frac_image = (
                fractional + shift
            )

            cart_image = np.dot(
                frac_image,
                cell
            )

            image_positions.append(
                cart_image
            )


image_positions = np.concatenate(
    image_positions,
    axis=0
)

print(
    "Original host atoms :",
    len(host)
)

print(
    "Periodic image atoms:",
    len(image_positions)
)


# =============================================================================
# 10. GENERATE VORONOI SITES
# =============================================================================

print("\n" + "=" * 100)
print("GENERATING VORONOI SITES")
print("=" * 100)

vor = Voronoi(
    image_positions
)

vertices = np.asarray(
    vor.vertices,
    dtype=float
)

print(
    "Raw Voronoi vertices:",
    len(vertices)
)


# =============================================================================
# 11. WRAP VORONOI VERTICES INTO CENTRAL CELL
# =============================================================================

candidate_sites = []

for vertex in vertices:

    frac = np.linalg.solve(
        cell.T,
        vertex
    )

    frac_wrapped = (
        frac % 1.0
    )

    cart_wrapped = np.dot(
        frac_wrapped,
        cell
    )

    candidate_sites.append(
        (
            frac_wrapped,
            cart_wrapped
        )
    )


# =============================================================================
# 12. REMOVE PERIODIC DUPLICATES
# =============================================================================

unique_sites = []

for frac, cart in candidate_sites:

    duplicate = False

    for old_frac, old_cart in unique_sites:

        delta = frac - old_frac

        delta -= np.round(
            delta
        )

        if np.linalg.norm(delta) < DUPLICATE_TOL:

            duplicate = True
            break

    if not duplicate:

        unique_sites.append(
            (
                frac,
                cart
            )
        )


print(
    "Unique Voronoi sites:",
    len(unique_sites)
)

if len(unique_sites) != 866:

    raise ValueError(
        f"Expected exactly 866 unique Voronoi sites, "
        f"but generated {len(unique_sites)}."
    )


# =============================================================================
# 13. CALCULATE NATURAL GEOMETRIC CLEARANCE
# =============================================================================

print("\n" + "=" * 100)
print("CALCULATING NATURAL SITE CLEARANCES")
print("=" * 100)

host_positions = host.get_positions()

site_records = []

for site_index, (frac, cart) in enumerate(
    unique_sites
):

    delta = (
        host_positions
        - cart[np.newaxis, :]
    )

    frac_delta = np.linalg.solve(
        cell.T,
        delta.T
    ).T

    frac_delta -= np.round(
        frac_delta
    )

    cart_delta = np.dot(
        frac_delta,
        cell
    )

    site_host_distances = np.linalg.norm(
        cart_delta,
        axis=1
    )

    nearest_index = int(
        np.argmin(
            site_host_distances
        )
    )

    nearest_distance = float(
        site_host_distances[
            nearest_index
        ]
    )

    nearest_element = (
        host[
            nearest_index
        ].symbol
    )

    site_records.append(
        {
            "site_index": site_index,

            "fractional": frac.copy(),

            "cartesian": cart.copy(),

            "nearest_host_distance":
                nearest_distance,

            "nearest_host_index":
                nearest_index,

            "nearest_host_element":
                nearest_element
        }
    )


# =============================================================================
# 14. SORT BY NATURAL CLEARANCE
# =============================================================================

site_records.sort(
    key=lambda x:
        x["nearest_host_distance"],
    reverse=True
)


print(
    "Total unique sites:",
    len(site_records)
)


# =============================================================================
# 15. SAVE / LOAD ALL VORONOI SITES
# =============================================================================
#
# IMPORTANT:
# The site file contains the coordinates in the SAME ordering used below.
# It is overwritten with the deterministic regenerated geometry.
#
# This file is NOT the checkpoint.
# The checkpoint is the per-site EXTXYZ + CSV.
#
# =============================================================================

np.save(
    SITE_FILE,
    np.array(
        [
            record["cartesian"]
            for record in site_records
        ],
        dtype=float
    )
)

print(
    "\nAll Voronoi sites saved to:"
)

print(
    SITE_FILE
)


# =============================================================================
# 16. LOAD MACE
# =============================================================================

print("\n" + "=" * 100)
print("LOADING MACE")
print("=" * 100)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64",
    device=DEVICE
)

print("Device :", DEVICE)
print("MACE   :", "medium")
print("D3     :", True)
print("dtype  :", "float64")


# =============================================================================
# 17. CHECKPOINT RECOVERY
# =============================================================================
#
# We recover completed sites from BOTH:
#
#   A) existing EXTXYZ
#   B) existing CSV
#
# The key is:
#
#       voronoi_site_index
#
# NOT the test number.
#
# Therefore, if sites 219-291 are already present, they are skipped.
#
# =============================================================================

print("\n" + "=" * 100)
print("CHECKPOINT RECOVERY")
print("=" * 100)


completed_site_indices = set()


# -----------------------------------------------------------------------------
# 17A. RECOVER FROM EXTXYZ
# -----------------------------------------------------------------------------

if os.path.isfile(RESULTS_FILE):

    print(
        "\nExisting EXTXYZ found:"
    )

    print(
        RESULTS_FILE
    )

    try:

        existing_structures = read(
            RESULTS_FILE,
            index=":"
        )

        print(
            "Existing saved structures:",
            len(existing_structures)
        )

        for structure in existing_structures:

            if "voronoi_site_index" in structure.info:

                try:

                    site_index = int(
                        structure.info[
                            "voronoi_site_index"
                        ]
                    )

                    completed_site_indices.add(
                        site_index
                    )

                except Exception:
                    pass

    except Exception as exc:

        print(
            "\nWARNING: Could not fully read existing EXTXYZ."
        )

        print(
            "Reason:",
            repr(exc)
        )


# -----------------------------------------------------------------------------
# 17B. RECOVER FROM CSV
# -----------------------------------------------------------------------------

if os.path.isfile(SUMMARY_FILE):

    print(
        "\nExisting CSV found:"
    )

    print(
        SUMMARY_FILE
    )

    try:

        with open(
            SUMMARY_FILE,
            "r",
            newline=""
        ) as f:

            reader = csv.DictReader(
                f
            )

            for row in reader:

                value = row.get(
                    "site_index"
                )

                if value is None:
                    continue

                try:

                    site_index = int(
                        value
                    )

                    completed_site_indices.add(
                        site_index
                    )

                except Exception:
                    pass

    except Exception as exc:

        print(
            "\nWARNING: Could not fully read existing CSV."
        )

        print(
            "Reason:",
            repr(exc)
        )


print(
    "\nRecovered completed sites:",
    len(completed_site_indices)
)


if completed_site_indices:

    sorted_completed = sorted(
        completed_site_indices
    )

    print(
        "First saved site indices:",
        sorted_completed[:20]
    )

    print(
        "Last saved site indices :",
        sorted_completed[-20:]
    )


# =============================================================================
# 18. PREPARE CSV CHECKPOINT
# =============================================================================

FIELDNAMES = [

    "test_number",

    "site_index",

    "initial_x",
    "initial_y",
    "initial_z",

    "final_x",
    "final_y",
    "final_z",

    "natural_clearance_A",

    "initial_energy_eV",
    "final_energy_eV",
    "energy_change_eV",

    "initial_max_force_eV_A",
    "final_max_force_eV_A",

    "Li_displacement_A",

    "final_nearest_host_distance_A",

    "final_nearest_host_element",

    "final_nearest_host_index",

    "optimizer_steps",

    "converged"
]


csv_exists = os.path.isfile(
    SUMMARY_FILE
)

csv_is_empty = (
    not csv_exists
    or os.path.getsize(
        SUMMARY_FILE
    ) == 0
)


# =============================================================================
# 19. SCREENING SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("SITE ENERGY SCREENING")
print("=" * 100)

print(
    "Total sites generated :",
    len(site_records)
)

print(
    "Already completed      :",
    len(completed_site_indices)
)

print(
    "Remaining              :",
    len(site_records)
    - len(completed_site_indices)
)

print(
    "Checkpoint mode        :",
    "SAVE AFTER EVERY SITE"
)

print("=" * 100)


# =============================================================================
# 20. MAIN SCREENING LOOP
# =============================================================================

try:

    for test_number, record in enumerate(
        site_records,
        start=1
    ):

        site_index = int(
            record["site_index"]
        )


        # =====================================================================
        # SKIP ALREADY COMPLETED SITE
        # =====================================================================

        if site_index in completed_site_indices:

            print(
                f"\nSKIP {test_number:3d} / "
                f"{len(site_records)} "
                f"| Voronoi site index = "
                f"{site_index}"
                f" | ALREADY SAVED"
            )

            continue


        # =====================================================================
        # SITE INFORMATION
        # =====================================================================

        initial_li_position = (
            record["cartesian"].copy()
        )

        print("\n")
        print("=" * 100)

        print(
            f"SITE TEST {test_number} / "
            f"{len(site_records)}"
        )

        print("=" * 100)

        print(
            "Voronoi site index :",
            site_index
        )

        print(
            "Initial Li position:",
            initial_li_position
        )

        print(
            "Natural clearance  :",
            f"{record['nearest_host_distance']:.6f} A"
        )


        # =====================================================================
        # CREATE ONE-Li SYSTEM
        # =====================================================================

        system = host.copy()

        li = Atoms(
            symbols=["Li"],
            positions=[
                initial_li_position
            ]
        )

        system += li

        system.set_pbc(
            host.get_pbc()
        )

        system.set_cell(
            host.cell
        )

        system.calc = calc

        li_index = (
            len(system) - 1
        )


        # =====================================================================
        # INITIAL ENERGY
        # =====================================================================

        initial_energy = (
            system.get_potential_energy()
        )

        initial_forces = (
            system.get_forces()
        )

        initial_max_force = float(
            np.max(
                np.linalg.norm(
                    initial_forces,
                    axis=1
                )
            )
        )


        # =====================================================================
        # RELAXATION
        # =====================================================================

        print(
            "\nStarting BFGS relaxation..."
        )

        optimizer = BFGS(
            system,
            logfile=None
        )

        optimizer.run(
            fmax=FMAX,
            steps=MAX_STEPS
        )


        # =====================================================================
        # FINAL ENERGY
        # =====================================================================

        final_energy = (
            system.get_potential_energy()
        )

        final_forces = (
            system.get_forces()
        )

        final_max_force = float(
            np.max(
                np.linalg.norm(
                    final_forces,
                    axis=1
                )
            )
        )

        final_li_position = (
            system.positions[
                li_index
            ].copy()
        )


        # =====================================================================
        # Li DISPLACEMENT WITH PERIODIC BOUNDARY CONDITIONS
        # =====================================================================

        delta = (
            final_li_position
            - initial_li_position
        )

        frac_delta = np.linalg.solve(
            cell.T,
            delta
        )

        frac_delta -= np.round(
            frac_delta
        )

        cart_delta = np.dot(
            frac_delta,
            cell
        )

        li_displacement = float(
            np.linalg.norm(
                cart_delta
            )
        )


        # =====================================================================
        # FINAL Li-HOST DISTANCES
        # =====================================================================

        final_host_positions = (
            system.positions[
                :len(host)
            ]
        )

        delta_host = (
            final_host_positions
            - final_li_position[
                np.newaxis, :
            ]
        )

        frac_host = np.linalg.solve(
            cell.T,
            delta_host.T
        ).T

        frac_host -= np.round(
            frac_host
        )

        cart_host = np.dot(
            frac_host,
            cell
        )

        final_distances = np.linalg.norm(
            cart_host,
            axis=1
        )

        nearest_final_host_index = int(
            np.argmin(
                final_distances
            )
        )

        nearest_final_host_distance = float(
            final_distances[
                nearest_final_host_index
            ]
        )

        nearest_final_host_element = (
            host[
                nearest_final_host_index
            ].symbol
        )


        # =====================================================================
        # ENERGY CHANGE
        # =====================================================================

        energy_change = (
            final_energy
            - initial_energy
        )


        # =====================================================================
        # CONVERGENCE
        # =====================================================================

        converged = (
            final_max_force <= FMAX
        )


        # =====================================================================
        # RESULT DICTIONARY
        # =====================================================================

        result = {

            "test_number":
                test_number,

            "site_index":
                site_index,

            "initial_x":
                float(
                    initial_li_position[0]
                ),

            "initial_y":
                float(
                    initial_li_position[1]
                ),

            "initial_z":
                float(
                    initial_li_position[2]
                ),

            "final_x":
                float(
                    final_li_position[0]
                ),

            "final_y":
                float(
                    final_li_position[1]
                ),

            "final_z":
                float(
                    final_li_position[2]
                ),

            "natural_clearance_A":
                float(
                    record[
                        "nearest_host_distance"
                    ]
                ),

            "initial_energy_eV":
                float(
                    initial_energy
                ),

            "final_energy_eV":
                float(
                    final_energy
                ),

            "energy_change_eV":
                float(
                    energy_change
                ),

            "initial_max_force_eV_A":
                float(
                    initial_max_force
                ),

            "final_max_force_eV_A":
                float(
                    final_max_force
                ),

            "Li_displacement_A":
                float(
                    li_displacement
                ),

            "final_nearest_host_distance_A":
                float(
                    nearest_final_host_distance
                ),

            "final_nearest_host_element":
                nearest_final_host_element,

            "final_nearest_host_index":
                int(
                    nearest_final_host_index
                ),

            "optimizer_steps":
                int(
                    optimizer.nsteps
                ),

            "converged":
                bool(
                    converged
                )
        }


        # =====================================================================
        # PUT METADATA INTO STRUCTURE
        # =====================================================================

        system.info[
            "candidate_id"
        ] = CANDIDATE_ID

        system.info[
            "voronoi_site_index"
        ] = site_index

        system.info[
            "test_number"
        ] = test_number

        system.info[
            "initial_li_x"
        ] = float(
            initial_li_position[0]
        )

        system.info[
            "initial_li_y"
        ] = float(
            initial_li_position[1]
        )

        system.info[
            "initial_li_z"
        ] = float(
            initial_li_position[2]
        )

        system.info[
            "final_li_x"
        ] = float(
            final_li_position[0]
        )

        system.info[
            "final_li_y"
        ] = float(
            final_li_position[1]
        )

        system.info[
            "final_li_z"
        ] = float(
            final_li_position[2]
        )

        system.info[
            "final_li_displacement_A"
        ] = float(
            li_displacement
        )

        system.info[
            "initial_energy_eV"
        ] = float(
            initial_energy
        )

        system.info[
            "final_energy_eV"
        ] = float(
            final_energy
        )

        system.info[
            "energy_change_eV"
        ] = float(
            energy_change
        )

        system.info[
            "natural_clearance_A"
        ] = float(
            record[
                "nearest_host_distance"
            ]
        )

        system.info[
            "final_nearest_host_distance_A"
        ] = float(
            nearest_final_host_distance
        )

        system.info[
            "final_nearest_host_element"
        ] = nearest_final_host_element

        system.info[
            "final_nearest_host_index"
        ] = int(
            nearest_final_host_index
        )

        system.info[
            "final_max_force_eV_A"
        ] = float(
            final_max_force
        )

        system.info[
            "optimizer_steps"
        ] = int(
            optimizer.nsteps
        )

        system.info[
            "converged"
        ] = bool(
            converged
        )


        # =====================================================================
        # PRINT RESULT BEFORE SAVING
        # =====================================================================

        print("\nRESULT")

        print(
            "Initial energy :",
            f"{initial_energy:.8f} eV"
        )

        print(
            "Final energy   :",
            f"{final_energy:.8f} eV"
        )

        print(
            "Energy change  :",
            f"{energy_change:.8f} eV"
        )

        print(
            "Li displacement:",
            f"{li_displacement:.8f} A"
        )

        print(
            "Final nearest host:",
            f"{nearest_final_host_distance:.8f} A",
            nearest_final_host_element
        )

        print(
            "Final max force:",
            f"{final_max_force:.8f} eV/A"
        )

        print(
            "Optimizer steps:",
            optimizer.nsteps
        )

        print(
            "Converged:",
            converged
        )


        # =====================================================================
        # IMMEDIATE EXTXYZ CHECKPOINT
        # =====================================================================
        #
        # SAVE THIS SITE BEFORE DOING ANYTHING ELSE.
        #
        # If runtime dies after this line, this site is recoverable.
        #
        # =====================================================================

        write(
            RESULTS_FILE,
            system,
            format="extxyz",
            append=True
        )


        # =====================================================================
        # IMMEDIATE CSV CHECKPOINT
        # =====================================================================

        with open(
            SUMMARY_FILE,
            "a",
            newline=""
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=FIELDNAMES
            )

            if csv_is_empty:

                writer.writeheader()

                csv_is_empty = False

            writer.writerow(
                result
            )


        # =====================================================================
        # FORCE FILE FLUSH TO DISK
        # =====================================================================

        # Re-open/close above guarantees the CSV buffer is flushed.
        # EXTXYZ write() also closes its file handle after writing.


        # =====================================================================
        # MARK SITE AS COMPLETED
        # =====================================================================

        completed_site_indices.add(
            site_index
        )


        # =====================================================================
        # CONFIRM CHECKPOINT
        # =====================================================================

        print("\n")
        print("*" * 100)

        print(
            "SAVED IMMEDIATELY."
        )

        print(
            "Saved Voronoi site index:",
            site_index
        )

        print(
            "Completed checkpoints:",
            len(completed_site_indices),
            "/",
            len(site_records)
        )

        print(
            "Remaining:",
            len(site_records)
            - len(completed_site_indices)
        )

        print("*" * 100)


# =============================================================================
# 21. INTERRUPT HANDLING
# =============================================================================

except KeyboardInterrupt:

    print("\n")
    print("=" * 100)
    print("RUNTIME INTERRUPTED BY USER")
    print("=" * 100)

    print(
        "Completed sites were preserved."
    )

    print(
        "Completed checkpoints:",
        len(completed_site_indices),
        "/",
        len(site_records)
    )

    print(
        "Remaining:",
        len(site_records)
        - len(completed_site_indices)
    )

    print(
        "\nWhen restarted, the script will automatically skip"
        " all saved Voronoi site indices."
    )

    print("=" * 100)


# =============================================================================
# 22. FINAL RECOVERY / SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("SCREENING STATUS")
print("=" * 100)

print(
    "Total Voronoi sites:",
    len(site_records)
)

print(
    "Completed:",
    len(completed_site_indices)
)

print(
    "Remaining:",
    len(site_records)
    - len(completed_site_indices)
)

print(
    "Results EXTXYZ:",
    RESULTS_FILE
)

print(
    "Results CSV:",
    SUMMARY_FILE
)

print(
    "All Voronoi sites:",
    SITE_FILE
)


# =============================================================================
# 23. LIST REMAINING SITES
# =============================================================================

remaining_sites = [

    record["site_index"]

    for record in site_records

    if record["site_index"]
    not in completed_site_indices
]


if remaining_sites:

    print(
        "\nFirst remaining Voronoi site indices:"
    )

    print(
        remaining_sites[:20]
    )

else:

    print(
        "\nALL 866 SITES ARE COMPLETE."
    )


print("=" * 100)
print("DONE")
print("=" * 100)

In [ ]:
# =============================================================================
# CANDIDATE 81 — CSV SCREENING ANALYSIS
# Read-only analysis of the completed 866-site MACE screening
# =============================================================================

import os
import numpy as np
import pandas as pd

# -----------------------------------------------------------------------------
# 1. PATH
# -----------------------------------------------------------------------------

CSV_FILE = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_Li_single_site_energy_screening/"
    "candidate_81_Li_site_energy_summary.csv"
)

# -----------------------------------------------------------------------------
# 2. LOAD
# -----------------------------------------------------------------------------

print("=" * 100)
print("CANDIDATE 81 — CSV SCREENING ANALYSIS")
print("=" * 100)

if not os.path.isfile(CSV_FILE):
    raise FileNotFoundError(f"\nCSV not found:\n{CSV_FILE}")

df = pd.read_csv(CSV_FILE)

print(f"\nCSV loaded successfully.")
print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

print("\nColumns:")
for col in df.columns:
    print(" -", col)

# -----------------------------------------------------------------------------
# 3. BASIC VALIDATION
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("BASIC VALIDATION")
print("=" * 100)

print(f"Rows in CSV : {len(df)}")
print(f"Duplicate rows : {df.duplicated().sum()}")

if len(df) != 866:
    print(
        f"WARNING: Expected 866 rows, but found {len(df)}."
    )
else:
    print("PASS: All 866 site results are present.")

# -----------------------------------------------------------------------------
# 4. FIND ENERGY COLUMN
# -----------------------------------------------------------------------------

energy_candidates = [
    "Final energy",
    "Final energy (eV)",
    "final_energy",
    "final_energy_eV",
    "Energy",
    "energy",
]

energy_col = None

for col in energy_candidates:
    if col in df.columns:
        energy_col = col
        break

if energy_col is None:
    raise ValueError(
        "\nCould not automatically identify the final-energy column.\n"
        "Available columns:\n"
        + "\n".join(map(str, df.columns))
    )

print(f"\nEnergy column detected: {energy_col}")

df[energy_col] = pd.to_numeric(
    df[energy_col],
    errors="coerce"
)

if df[energy_col].isna().any():
    raise ValueError(
        "Some energy values could not be converted to numbers."
    )

# -----------------------------------------------------------------------------
# 5. LOWEST-ENERGY SITES
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("LOWEST-ENERGY SITES")
print("=" * 100)

sorted_df = df.sort_values(
    by=energy_col,
    ascending=True
).reset_index(drop=True)

N_LOWEST = 20

print(f"\nLowest {N_LOWEST} sites:\n")

print(
    sorted_df.head(N_LOWEST).to_string(
        index=False
    )
)

# -----------------------------------------------------------------------------
# 6. ENERGY RANGE
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("ENERGY DISTRIBUTION")
print("=" * 100)

emin = sorted_df[energy_col].min()
emax = sorted_df[energy_col].max()

print(f"Minimum final energy : {emin:.10f} eV")
print(f"Maximum final energy : {emax:.10f} eV")
print(f"Energy range         : {emax - emin:.10f} eV")

# -----------------------------------------------------------------------------
# 7. NEARLY IDENTICAL ENERGIES
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NEARLY IDENTICAL FINAL ENERGIES")
print("=" * 100)

# Energies are rounded to 6 decimal places only for grouping.
# This does NOT modify the original dataframe.

energy_groups = (
    sorted_df
    .assign(
        energy_group=sorted_df[energy_col].round(6)
    )
    .groupby("energy_group")
    .size()
    .sort_values(ascending=False)
)

repeated_groups = energy_groups[
    energy_groups > 1
]

print(
    f"\nNumber of energy groups containing multiple sites: "
    f"{len(repeated_groups)}"
)

if len(repeated_groups) > 0:
    print("\nMost repeated energy groups:")
    print(repeated_groups.head(20).to_string())

# -----------------------------------------------------------------------------
# 8. SHOW SITES BELONGING TO THE LOWEST ENERGY GROUPS
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SITES WITH THE LOWEST ENERGY VALUES")
print("=" * 100)

lowest_energy_groups = (
    sorted_df[energy_col]
    .round(6)
    .drop_duplicates()
    .head(10)
)

for energy_value in lowest_energy_groups:

    group = sorted_df[
        sorted_df[energy_col].round(6)
        == energy_value
    ]

    print("\n" + "-" * 100)
    print(
        f"Energy group: {energy_value:.6f} eV"
    )
    print(
        f"Number of sites: {len(group)}"
    )

    print(
        group.to_string(index=False)
    )

# -----------------------------------------------------------------------------
# 9. SAVE ANALYSIS TABLE
# -----------------------------------------------------------------------------

OUTPUT_CSV = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_Li_single_site_energy_screening/"
    "candidate_81_energy_sorted_analysis.csv"
)

sorted_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\n" + "=" * 100)
print("ANALYSIS COMPLETE")
print("=" * 100)

print(f"\nSorted analysis saved to:")
print(OUTPUT_CSV)

print("\nNO MACE CALCULATIONS WERE PERFORMED.")
print("NO STRUCTURES WERE MODIFIED.")
print("This was a read-only CSV analysis.")

In [ ]:
# ============================================================
# CANDIDATE 81 — BASIN ANALYSIS OF ALL 866 Li SCREENING RESULTS
# READ-ONLY ANALYSIS — NO MACE / NO RELAXATION
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PATH
# ------------------------------------------------------------

CSV_FILE = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_Li_single_site_energy_screening/"
    "candidate_81_Li_site_energy_summary.csv"
)

OUTPUT_DIR = os.path.dirname(CSV_FILE)

BASIN_FILE = os.path.join(
    OUTPUT_DIR,
    "candidate_81_Li_basin_analysis.csv"
)

MINIMA_FILE = os.path.join(
    OUTPUT_DIR,
    "candidate_81_Li_distinct_minima.csv"
)

# ------------------------------------------------------------
# 2. ANALYSIS PARAMETERS
# ------------------------------------------------------------

# Energy tolerance for grouping nearly identical minima
ENERGY_TOL = 1e-4       # eV

# Position tolerance for identifying the same basin
POSITION_TOL = 0.10     # Angstrom

# ------------------------------------------------------------
# 3. LOAD CSV
# ------------------------------------------------------------

df = pd.read_csv(CSV_FILE)

print("=" * 80)
print("CANDIDATE 81 — Li BASIN ANALYSIS")
print("=" * 80)

print("\nCSV loaded successfully.")
print("Rows:", len(df))

print("\nColumns:")
for c in df.columns:
    print(" -", c)

# ------------------------------------------------------------
# 4. BASIC VALIDATION
# ------------------------------------------------------------

required_columns = [
    "final_energy_eV",
    "converged",
    "final_x",
    "final_y",
    "final_z",
    "site_index",
    "test_number",
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        "Missing required columns: " + str(missing)
    )

# ------------------------------------------------------------
# 5. NORMALIZE CONVERGENCE COLUMN
# ------------------------------------------------------------

df["converged"] = (
    df["converged"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
)

if df["converged"].isna().any():
    raise ValueError(
        "Could not interpret some values in 'converged'."
    )

# ------------------------------------------------------------
# 6. CONVERT NUMERICAL COLUMNS
# ------------------------------------------------------------

numeric_columns = [
    "final_energy_eV",
    "final_x",
    "final_y",
    "final_z",
    "site_index",
    "test_number",
]

for column in numeric_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

if df[numeric_columns].isna().any().any():

    bad_columns = (
        df[numeric_columns]
        .columns[
            df[numeric_columns].isna().any()
        ]
        .tolist()
    )

    raise ValueError(
        "Non-numeric or missing values found in: "
        + str(bad_columns)
    )

# ------------------------------------------------------------
# 7. SCREEN VALID RESULTS
# ------------------------------------------------------------

valid = df[
    df["converged"] == True
].copy()

invalid = df[
    df["converged"] == False
].copy()

print("\n" + "=" * 80)
print("CONVERGENCE CHECK")
print("=" * 80)

print("Total rows    :", len(df))
print("Converged     :", len(valid))
print("Not converged :", len(invalid))

if len(valid) == 0:
    raise ValueError(
        "No converged results available."
    )

# ------------------------------------------------------------
# 8. SORT BY FINAL ENERGY
# ------------------------------------------------------------

valid = valid.sort_values(
    "final_energy_eV",
    ascending=True
).reset_index(drop=True)

print("\nLowest final energies:")

print(
    valid[
        [
            "site_index",
            "test_number",
            "final_energy_eV",
            "final_x",
            "final_y",
            "final_z"
        ]
    ]
    .head(20)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 9. CANDIDATE 81 CELL
# ------------------------------------------------------------

CELL = np.array([
    [7.039, 0.000, 0.000],
    [0.000, 6.978, 0.000],
    [0.000, 0.000, 12.411]
])

# ------------------------------------------------------------
# 10. PBC DISTANCE FUNCTION
# ------------------------------------------------------------

def pbc_distance(r1, r2, cell):

    r1 = np.asarray(r1, dtype=float)
    r2 = np.asarray(r2, dtype=float)

    # Cartesian -> fractional
    f1 = r1 @ np.linalg.inv(cell)
    f2 = r2 @ np.linalg.inv(cell)

    df = f1 - f2

    # Minimum-image convention
    df -= np.round(df)

    # Fractional -> Cartesian
    dr = df @ cell

    return np.linalg.norm(dr)

# ------------------------------------------------------------
# 11. BASIN CLUSTERING
# ------------------------------------------------------------

basins = []

for idx, row in valid.iterrows():

    position = np.array([
        row["final_x"],
        row["final_y"],
        row["final_z"]
    ])

    energy = float(
        row["final_energy_eV"]
    )

    assigned = False

    for basin in basins:

        representative = (
            basin["representative_position"]
        )

        representative_energy = (
            basin["representative_energy"]
        )

        distance = pbc_distance(
            position,
            representative,
            CELL
        )

        energy_difference = abs(
            energy - representative_energy
        )

        if (
            distance <= POSITION_TOL
            and
            energy_difference <= ENERGY_TOL
        ):

            basin["members"].append(idx)

            assigned = True
            break

    if not assigned:

        basins.append({

            "representative_position":
                position,

            "representative_energy":
                energy,

            "members": [idx]
        })

# ------------------------------------------------------------
# 12. BUILD BASIN TABLES
# ------------------------------------------------------------

basin_rows = []
minimum_rows = []

for basin_id, basin in enumerate(
    basins,
    start=1
):

    members = valid.loc[
        basin["members"]
    ].copy()

    members = members.sort_values(
        "final_energy_eV"
    )

    best = members.iloc[0]

    basin_rows.append({

        "basin_id":
            basin_id,

        "population":
            len(members),

        "minimum_energy_eV":
            float(
                best["final_energy_eV"]
            ),

        "representative_x_A":
            float(
                best["final_x"]
            ),

        "representative_y_A":
            float(
                best["final_y"]
            ),

        "representative_z_A":
            float(
                best["final_z"]
            ),

        "best_original_site_index":
            int(
                best["site_index"]
            ),

        "best_screening_test_number":
            int(
                best["test_number"]
            )
    })

    minimum_rows.append({

        "basin_id":
            basin_id,

        "population":
            len(members),

        **best.to_dict()
    })

# ------------------------------------------------------------
# 13. DATAFRAMES
# ------------------------------------------------------------

basin_df = pd.DataFrame(
    basin_rows
)

minima_df = pd.DataFrame(
    minimum_rows
)

# Sort by minimum energy
basin_df = basin_df.sort_values(
    "minimum_energy_eV",
    ascending=True
).reset_index(drop=True)

minima_df = minima_df.sort_values(
    "final_energy_eV",
    ascending=True
).reset_index(drop=True)

# Re-number basins after energy sorting
basin_df["basin_id"] = np.arange(
    1,
    len(basin_df) + 1
)

minima_df["basin_id"] = np.arange(
    1,
    len(minima_df) + 1
)

# ------------------------------------------------------------
# 14. SAVE
# ------------------------------------------------------------

basin_df.to_csv(
    BASIN_FILE,
    index=False
)

minima_df.to_csv(
    MINIMA_FILE,
    index=False
)

# ------------------------------------------------------------
# 15. REPORT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("BASIN ANALYSIS RESULT")
print("=" * 80)

print("Converged results :", len(valid))
print("Distinct basins   :", len(basin_df))

print("\n" + "-" * 80)
print("DISTINCT MINIMA")
print("-" * 80)

display(
    basin_df.head(30)
)

print("\n" + "-" * 80)
print("BASIN POPULATIONS")
print("-" * 80)

display(
    basin_df[
        [
            "basin_id",
            "population",
            "minimum_energy_eV"
        ]
    ].head(30)
)

print("\n" + "=" * 80)
print("FILES SAVED")
print("=" * 80)

print("Basin analysis:")
print(BASIN_FILE)

print("\nDistinct minima:")
print(MINIMA_FILE)

print("\n" + "=" * 80)
print("DONE — NO NEW MACE CALCULATION WAS PERFORMED")
print("=" * 80)

In [ ]:
# ============================================================
# CANDIDATE 81 — FAILED-SITES DIAGNOSTIC ANALYSIS
# READ-ONLY — NO MACE / NO RELAXATION
# ============================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. PATH
# ------------------------------------------------------------

CSV_FILE = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_Li_single_site_energy_screening/"
    "candidate_81_Li_site_energy_summary.csv"
)

OUTPUT_DIR = os.path.dirname(CSV_FILE)

FAILED_FILE = os.path.join(
    OUTPUT_DIR,
    "candidate_81_failed_sites_diagnostic.csv"
)

# ------------------------------------------------------------
# 2. LOAD CSV
# ------------------------------------------------------------

df = pd.read_csv(CSV_FILE)

print("=" * 80)
print("CANDIDATE 81 — FAILED-SITES DIAGNOSTIC ANALYSIS")
print("=" * 80)

print("\nCSV loaded successfully.")
print("Total rows:", len(df))

# ------------------------------------------------------------
# 3. REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "site_index",
    "test_number",
    "natural_clearance_A",
    "initial_energy_eV",
    "final_energy_eV",
    "energy_change_eV",
    "initial_max_force_eV_A",
    "final_max_force_eV_A",
    "Li_displacement_A",
    "final_nearest_host_distance_A",
    "final_nearest_host_element",
    "final_nearest_host_index",
    "optimizer_steps",
    "converged"
]

missing = [
    c for c in required_columns
    if c not in df.columns
]

if missing:
    raise ValueError(
        "Missing required columns: " + str(missing)
    )

# ------------------------------------------------------------
# 4. NORMALIZE CONVERGENCE
# ------------------------------------------------------------

df["converged"] = (
    df["converged"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
)

if df["converged"].isna().any():
    raise ValueError(
        "Could not interpret some values in 'converged'."
    )

# ------------------------------------------------------------
# 5. EXTRACT FAILED SITES
# ------------------------------------------------------------

failed = df[
    df["converged"] == False
].copy()

print("\n" + "=" * 80)
print("FAILURE INVENTORY")
print("=" * 80)

print("Total sites      :", len(df))
print("Converged        :", (df["converged"] == True).sum())
print("Not converged    :", len(failed))

if len(failed) == 0:
    print("\nNo failed sites found.")
    raise SystemExit

# ------------------------------------------------------------
# 6. NUMERIC CONVERSION
# ------------------------------------------------------------

numeric_columns = [
    "natural_clearance_A",
    "initial_energy_eV",
    "final_energy_eV",
    "energy_change_eV",
    "initial_max_force_eV_A",
    "final_max_force_eV_A",
    "Li_displacement_A",
    "final_nearest_host_distance_A",
    "optimizer_steps"
]

for column in numeric_columns:

    failed[column] = pd.to_numeric(
        failed[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# 7. BASIC STATISTICS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FAILED-SITE STATISTICS")
print("=" * 80)

print("\nNatural clearance:")
print(
    failed["natural_clearance_A"]
    .describe()
    .to_string()
)

print("\nInitial maximum force:")
print(
    failed["initial_max_force_eV_A"]
    .describe()
    .to_string()
)

print("\nFinal maximum force:")
print(
    failed["final_max_force_eV_A"]
    .describe()
    .to_string()
)

print("\nInitial energy:")
print(
    failed["initial_energy_eV"]
    .describe()
    .to_string()
)

print("\nFinal energy:")
print(
    failed["final_energy_eV"]
    .describe()
    .to_string()
)

print("\nLi displacement:")
print(
    failed["Li_displacement_A"]
    .describe()
    .to_string()
)

print("\nOptimizer steps:")
print(
    failed["optimizer_steps"]
    .describe()
    .to_string()
)

# ------------------------------------------------------------
# 8. MAX-STEP CHECK
# ------------------------------------------------------------

MAX_STEPS = 1000

max_step_failed = failed[
    failed["optimizer_steps"] >= MAX_STEPS
]

print("\n" + "=" * 80)
print("MAX-STEP CHECK")
print("=" * 80)

print(
    "Failed sites reaching max steps:",
    len(max_step_failed)
)

print(
    "Failed sites stopping before max steps:",
    len(failed) - len(max_step_failed)
)

# ------------------------------------------------------------
# 9. SORT BY INITIAL FORCE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("HIGHEST INITIAL FORCES")
print("=" * 80)

display(
    failed[
        [
            "site_index",
            "test_number",
            "natural_clearance_A",
            "initial_energy_eV",
            "initial_max_force_eV_A",
            "optimizer_steps"
        ]
    ]
    .sort_values(
        "initial_max_force_eV_A",
        ascending=False
    )
    .head(20)
)

# ------------------------------------------------------------
# 10. SORT BY NATURAL CLEARANCE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("SMALLEST NATURAL CLEARANCE")
print("=" * 80)

display(
    failed[
        [
            "site_index",
            "test_number",
            "natural_clearance_A",
            "initial_energy_eV",
            "initial_max_force_eV_A",
            "optimizer_steps"
        ]
    ]
    .sort_values(
        "natural_clearance_A",
        ascending=True
    )
    .head(20)
)

# ------------------------------------------------------------
# 11. LARGEST FINAL FORCES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LARGEST FINAL FORCES")
print("=" * 80)

display(
    failed[
        [
            "site_index",
            "test_number",
            "final_energy_eV",
            "final_max_force_eV_A",
            "Li_displacement_A",
            "optimizer_steps"
        ]
    ]
    .sort_values(
        "final_max_force_eV_A",
        ascending=False
    )
    .head(20)
)

# ------------------------------------------------------------
# 12. ENERGY EXTREMES
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ENERGY EXTREMES AMONG FAILED SITES")
print("=" * 80)

print("\nLowest final energies among failed:")
display(
    failed[
        [
            "site_index",
            "test_number",
            "initial_energy_eV",
            "final_energy_eV",
            "energy_change_eV",
            "optimizer_steps"
        ]
    ]
    .sort_values(
        "final_energy_eV",
        ascending=True
    )
    .head(20)
)

print("\nHighest final energies among failed:")
display(
    failed[
        [
            "site_index",
            "test_number",
            "initial_energy_eV",
            "final_energy_eV",
            "energy_change_eV",
            "optimizer_steps"
        ]
    ]
    .sort_values(
        "final_energy_eV",
        ascending=False
    )
    .head(20)
)

# ------------------------------------------------------------
# 13. CLEARANCE BINS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FAILURES BY NATURAL-CLEARANCE RANGE")
print("=" * 80)

clearance_bins = pd.cut(
    failed["natural_clearance_A"],
    bins=[
        -np.inf,
        1.5,
        2.0,
        2.5,
        3.0,
        3.5,
        np.inf
    ]
)

clearance_summary = (
    failed
    .groupby(
        clearance_bins,
        observed=False
    )
    .agg(
        sites=("site_index", "count"),
        mean_initial_force=(
            "initial_max_force_eV_A",
            "mean"
        ),
        mean_initial_energy=(
            "initial_energy_eV",
            "mean"
        ),
        mean_steps=(
            "optimizer_steps",
            "mean"
        )
    )
    .reset_index()
)

display(clearance_summary)

# ------------------------------------------------------------
# 14. INITIAL-FORCE BINS
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FAILURES BY INITIAL-FORCE RANGE")
print("=" * 80)

force_bins = pd.cut(
    failed["initial_max_force_eV_A"],
    bins=[
        -np.inf,
        1,
        10,
        100,
        1000,
        1e4,
        np.inf
    ]
)

force_summary = (
    failed
    .groupby(
        force_bins,
        observed=False
    )
    .agg(
        sites=("site_index", "count"),
        mean_clearance=(
            "natural_clearance_A",
            "mean"
        ),
        mean_initial_energy=(
            "initial_energy_eV",
            "mean"
        ),
        mean_steps=(
            "optimizer_steps",
            "mean"
        )
    )
    .reset_index()
)

display(force_summary)

# ------------------------------------------------------------
# 15. SAVE COMPLETE FAILED-SITE TABLE
# ------------------------------------------------------------

failed = failed.sort_values(
    [
        "optimizer_steps",
        "natural_clearance_A"
    ],
    ascending=[
        False,
        True
    ]
)

failed.to_csv(
    FAILED_FILE,
    index=False
)

# ------------------------------------------------------------
# 16. FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("DIAGNOSTIC FILE SAVED")
print("=" * 80)

print(FAILED_FILE)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print(
    "No MACE calculation was performed."
)

print(
    "No relaxation was performed."
)

print(
    "No original screening result was modified."
)

In [ ]:
# =============================================================================
# CANDIDATE 81 — SUCCESS vs FAILURE COMPARISON
# Read-only comparison of the 787 converged vs 79 failed Li sites
# =============================================================================

import os
import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# 1. PATH
# -----------------------------------------------------------------------------

CSV_FILE = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_Li_single_site_energy_screening/"
    "candidate_81_Li_site_energy_summary.csv"
)

OUTPUT_DIR = os.path.dirname(CSV_FILE)


# -----------------------------------------------------------------------------
# 2. LOAD DATA
# -----------------------------------------------------------------------------

df = pd.read_csv(CSV_FILE)

print("=" * 80)
print("CANDIDATE 81 — SUCCESS vs FAILURE COMPARISON")
print("=" * 80)

print(f"\nTotal rows: {len(df)}")


# -----------------------------------------------------------------------------
# 3. NORMALIZE CONVERGED COLUMN
# -----------------------------------------------------------------------------

if df["converged"].dtype == bool:
    converged = df["converged"]
else:
    converged = (
        df["converged"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )

if converged.isna().any():
    raise ValueError("Could not interpret some values in 'converged'.")

df["converged_clean"] = converged


# -----------------------------------------------------------------------------
# 4. SPLIT SUCCESS / FAILURE
# -----------------------------------------------------------------------------

success = df[df["converged_clean"] == True].copy()
failed = df[df["converged_clean"] == False].copy()

n_success = len(success)
n_failed = len(failed)
n_total = len(df)

print("\n" + "-" * 80)
print("1) BASIC COUNTS")
print("-" * 80)

print(f"Successful / converged : {n_success}")
print(f"Failed / not converged : {n_failed}")
print(f"Total                  : {n_total}")

print(f"Success rate           : {100*n_success/n_total:.2f}%")
print(f"Failure rate           : {100*n_failed/n_total:.2f}%")


# -----------------------------------------------------------------------------
# 5. COLUMNS WE CARE ABOUT
# -----------------------------------------------------------------------------

COMPARE_COLUMNS = [
    "natural_clearance_A",
    "initial_max_force_eV_A",
    "initial_energy_eV",
    "optimizer_steps",
]

for col in COMPARE_COLUMNS:
    success[col] = pd.to_numeric(success[col], errors="coerce")
    failed[col] = pd.to_numeric(failed[col], errors="coerce")


# -----------------------------------------------------------------------------
# 6. DESCRIPTIVE STATISTICS
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("2) SUCCESS vs FAILURE — DESCRIPTIVE STATISTICS")
print("-" * 80)

for col in COMPARE_COLUMNS:

    print(f"\n### {col}")

    comparison = pd.DataFrame({
        "SUCCESS_787": success[col].describe(),
        "FAILURE_79": failed[col].describe(),
    })

    print(comparison)


# -----------------------------------------------------------------------------
# 7. IMPORTANT QUANTILES
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("3) KEY QUANTILES")
print("-" * 80)

quantiles = [0.00, 0.10, 0.25, 0.50, 0.75, 0.90, 1.00]

for col in [
    "natural_clearance_A",
    "initial_max_force_eV_A",
    "initial_energy_eV",
]:

    print(f"\n### {col}")

    q = pd.DataFrame({
        "SUCCESS_787": success[col].quantile(quantiles),
        "FAILURE_79": failed[col].quantile(quantiles),
    })

    q.index = [
        "min",
        "10%",
        "25%",
        "median",
        "75%",
        "90%",
        "max",
    ]

    print(q)


# -----------------------------------------------------------------------------
# 8. CLEARANCE BINS
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("4) FAILURE RATE BY NATURAL CLEARANCE")
print("-" * 80)

clearance_bins = [
    -np.inf,
    0.30,
    0.50,
    0.80,
    1.00,
    1.50,
    np.inf,
]

clearance_labels = [
    "<= 0.30 Å",
    "0.30–0.50 Å",
    "0.50–0.80 Å",
    "0.80–1.00 Å",
    "1.00–1.50 Å",
    "> 1.50 Å",
]

df["clearance_bin"] = pd.cut(
    pd.to_numeric(df["natural_clearance_A"], errors="coerce"),
    bins=clearance_bins,
    labels=clearance_labels,
    include_lowest=True,
)

clearance_table = (
    df.groupby("clearance_bin", observed=False)
    .agg(
        total_sites=("converged_clean", "size"),
        successful=("converged_clean", "sum"),
        failed=("converged_clean", lambda x: (~x).sum()),
    )
)

clearance_table["success_rate_%"] = (
    100 * clearance_table["successful"]
    / clearance_table["total_sites"]
)

clearance_table["failure_rate_%"] = (
    100 * clearance_table["failed"]
    / clearance_table["total_sites"]
)

print(clearance_table)


# -----------------------------------------------------------------------------
# 9. INITIAL FORCE BINS
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("5) FAILURE RATE BY INITIAL MAX FORCE")
print("-" * 80)

force_bins = [
    -np.inf,
    10,
    100,
    1000,
    10000,
    np.inf,
]

force_labels = [
    "<= 10 eV/Å",
    "10–100 eV/Å",
    "100–1,000 eV/Å",
    "1,000–10,000 eV/Å",
    "> 10,000 eV/Å",
]

df["force_bin"] = pd.cut(
    pd.to_numeric(df["initial_max_force_eV_A"], errors="coerce"),
    bins=force_bins,
    labels=force_labels,
    include_lowest=True,
)

force_table = (
    df.groupby("force_bin", observed=False)
    .agg(
        total_sites=("converged_clean", "size"),
        successful=("converged_clean", "sum"),
        failed=("converged_clean", lambda x: (~x).sum()),
    )
)

force_table["success_rate_%"] = (
    100 * force_table["successful"]
    / force_table["total_sites"]
)

force_table["failure_rate_%"] = (
    100 * force_table["failed"]
    / force_table["total_sites"]
)

print(force_table)


# -----------------------------------------------------------------------------
# 10. SUCCESS SITES WITH VERY LOW CLEARANCE
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("6) SUCCESSFUL SITES WITH VERY LOW CLEARANCE")
print("-" * 80)

low_clearance_success = (
    success[
        success["natural_clearance_A"] <= 0.50
    ]
    .sort_values("natural_clearance_A")
)

print(
    low_clearance_success[
        [
            "site_index",
            "natural_clearance_A",
            "initial_max_force_eV_A",
            "initial_energy_eV",
            "optimizer_steps",
        ]
    ]
    .head(20)
)


# -----------------------------------------------------------------------------
# 11. FAILED SITES WITH RELATIVELY MODERATE CLEARANCE
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("7) FAILED SITES WITH RELATIVELY LARGER CLEARANCE")
print("-" * 80)

moderate_clearance_failed = (
    failed[
        failed["natural_clearance_A"] > 0.50
    ]
    .sort_values("natural_clearance_A", ascending=False)
)

print(
    moderate_clearance_failed[
        [
            "site_index",
            "natural_clearance_A",
            "initial_max_force_eV_A",
            "initial_energy_eV",
            "optimizer_steps",
        ]
    ]
    .head(20)
)


# -----------------------------------------------------------------------------
# 12. CORRELATION BETWEEN CLEARANCE AND INITIAL FORCE
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("8) CLEARANCE vs INITIAL FORCE")
print("-" * 80)

corr_df = df[
    [
        "natural_clearance_A",
        "initial_max_force_eV_A",
    ]
].apply(pd.to_numeric, errors="coerce")

print(
    "Pearson correlation:",
    corr_df["natural_clearance_A"].corr(
        corr_df["initial_max_force_eV_A"]
    )
)

print(
    "\nInterpretation reminder:"
    "\nThis correlation describes association in the dataset."
    "\nIt does NOT by itself prove causation."
)


# -----------------------------------------------------------------------------
# 13. SAVE MAIN COMPARISON TABLE
# -----------------------------------------------------------------------------

comparison_summary = pd.DataFrame({
    "metric": COMPARE_COLUMNS,
    "success_mean": [
        success[c].mean() for c in COMPARE_COLUMNS
    ],
    "success_median": [
        success[c].median() for c in COMPARE_COLUMNS
    ],
    "success_min": [
        success[c].min() for c in COMPARE_COLUMNS
    ],
    "success_max": [
        success[c].max() for c in COMPARE_COLUMNS
    ],
    "failure_mean": [
        failed[c].mean() for c in COMPARE_COLUMNS
    ],
    "failure_median": [
        failed[c].median() for c in COMPARE_COLUMNS
    ],
    "failure_min": [
        failed[c].min() for c in COMPARE_COLUMNS
    ],
    "failure_max": [
        failed[c].max() for c in COMPARE_COLUMNS
    ],
})

comparison_file = os.path.join(
    OUTPUT_DIR,
    "candidate_81_success_vs_failure_comparison.csv"
)

comparison_summary.to_csv(
    comparison_file,
    index=False
)


# -----------------------------------------------------------------------------
# 14. SAVE FAILURE-RATE TABLES
# -----------------------------------------------------------------------------

clearance_file = os.path.join(
    OUTPUT_DIR,
    "candidate_81_failure_rate_by_clearance.csv"
)

force_file = os.path.join(
    OUTPUT_DIR,
    "candidate_81_failure_rate_by_initial_force.csv"
)

clearance_table.to_csv(clearance_file)
force_table.to_csv(force_file)


# -----------------------------------------------------------------------------
# 15. FINAL
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

print("\nSaved:")
print(comparison_file)
print(clearance_file)
print(force_file)

print("\nNO MACE calculation was performed.")
print("NO relaxation was performed.")
print("Original screening CSV was not modified.")

In [ ]:
# =============================================================================
# CANDIDATE 81 — REPRESENTATIVE RELAXATION TRAJECTORY ANALYSIS
# Read-only inspection of successful vs failed Li relaxations
# =============================================================================

import os
import numpy as np
import pandas as pd
from ase.io import read


# =============================================================================
# 1. PATHS
# =============================================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_Li_single_site_energy_screening"
)

CSV_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_Li_site_energy_summary.csv"
)

EXTXYZ_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_all_tested_Li_sites_results.extxyz"
)

OUTPUT_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_representative_relaxation_trajectory_analysis.csv"
)


# =============================================================================
# 2. LOAD SCREENING SUMMARY
# =============================================================================

print("=" * 80)
print("CANDIDATE 81 — REPRESENTATIVE RELAXATION TRAJECTORY ANALYSIS")
print("=" * 80)

df = pd.read_csv(CSV_FILE)

# Normalize converged
if df["converged"].dtype == bool:
    df["converged_clean"] = df["converged"]
else:
    df["converged_clean"] = (
        df["converged"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
    )

if df["converged_clean"].isna().any():
    raise ValueError("Could not interpret values in 'converged'.")


# =============================================================================
# 3. NUMERIC COLUMNS
# =============================================================================

NUMERIC_COLUMNS = [
    "site_index",
    "natural_clearance_A",
    "initial_energy_eV",
    "final_energy_eV",
    "initial_max_force_eV_A",
    "final_max_force_eV_A",
    "Li_displacement_A",
    "optimizer_steps",
]

for col in NUMERIC_COLUMNS:
    df[col] = pd.to_numeric(df[col], errors="coerce")


# =============================================================================
# 4. CHOOSE REPRESENTATIVE SITES
# =============================================================================
#
# We deliberately do NOT choose sites randomly.
#
# We want four scientifically useful cases:
#
#   A) Successful / extreme starting condition
#   B) Successful / ordinary starting condition
#   C) Failed / moderate starting condition
#   D) Failed / extreme starting condition
#
# This lets us inspect HOW the relaxation behaves.
# =============================================================================

success = df[df["converged_clean"]].copy()
failed = df[~df["converged_clean"]].copy()


# -----------------------------------------------------------------------------
# A) Successful / extreme
# -----------------------------------------------------------------------------

successful_extreme = (
    success
    .sort_values(
        ["initial_max_force_eV_A", "natural_clearance_A"],
        ascending=[False, True]
    )
    .iloc[0]
)


# -----------------------------------------------------------------------------
# B) Successful / ordinary
#
# Pick a site near the median initial force among successful sites.
# -----------------------------------------------------------------------------

success_force_median = success["initial_max_force_eV_A"].median()

successful_ordinary = (
    success
    .assign(
        force_distance=(
            success["initial_max_force_eV_A"]
            - success_force_median
        ).abs()
    )
    .sort_values("force_distance")
    .iloc[0]
)


# -----------------------------------------------------------------------------
# C) Failed / moderate
#
# Pick the failed site with the largest clearance.
# This is useful because its failure is NOT simply an ultra-small-clearance case.
# -----------------------------------------------------------------------------

failed_moderate = (
    failed
    .sort_values(
        ["natural_clearance_A", "initial_max_force_eV_A"],
        ascending=[False, True]
    )
    .iloc[0]
)


# -----------------------------------------------------------------------------
# D) Failed / extreme
# -----------------------------------------------------------------------------

failed_extreme = (
    failed
    .sort_values(
        ["initial_max_force_eV_A", "natural_clearance_A"],
        ascending=[False, True]
    )
    .iloc[0]
)


selected = pd.DataFrame([
    successful_extreme,
    successful_ordinary,
    failed_moderate,
    failed_extreme,
])

selected["case"] = [
    "SUCCESS_EXTREME",
    "SUCCESS_ORDINARY",
    "FAILURE_MODERATE",
    "FAILURE_EXTREME",
]

selected = selected[
    [
        "case",
        "site_index",
        "converged_clean",
        "natural_clearance_A",
        "initial_energy_eV",
        "final_energy_eV",
        "initial_max_force_eV_A",
        "final_max_force_eV_A",
        "Li_displacement_A",
        "optimizer_steps",
    ]
]

print("\n" + "-" * 80)
print("SELECTED REPRESENTATIVE SITES")
print("-" * 80)

print(selected.to_string(index=False))


# =============================================================================
# 5. READ EXTXYZ
# =============================================================================

print("\n" + "-" * 80)
print("READING EXISTING EXTXYZ")
print("-" * 80)

if not os.path.exists(EXTXYZ_FILE):
    raise FileNotFoundError(
        f"EXTXYZ file not found:\n{EXTXYZ_FILE}"
    )

frames = read(
    EXTXYZ_FILE,
    index=":"
)

print(f"Total structures/frames read: {len(frames)}")


# =============================================================================
# 6. INSPECT FRAME METADATA
# =============================================================================
#
# Before assuming how the trajectory identifies a site, inspect the first
# frame's info keys.
# =============================================================================

if len(frames) == 0:
    raise ValueError("EXTXYZ contains no frames.")

print("\nFirst frame info keys:")
print(list(frames[0].info.keys()))

print("\nFirst frame arrays:")
print(list(frames[0].arrays.keys()))


# =============================================================================
# 7. FIND HOW SITE INDEX IS STORED
# =============================================================================

SITE_KEYS = [
    "site_index",
    "voronoi_site_index",
    "site",
    "test_number",
]

STEP_KEYS = [
    "optimizer_step",
    "step",
    "production_step",
]

ENERGY_KEYS = [
    "energy",
    "final_energy_eV",
    "initial_energy_eV",
]

FORCE_KEYS = [
    "max_force_eV_A",
    "final_max_force_eV_A",
]

print("\n" + "-" * 80)
print("METADATA DETECTION")
print("-" * 80)

print("Possible site keys:")
print([
    key for key in SITE_KEYS
    if key in frames[0].info
])

print("Possible step keys:")
print([
    key for key in STEP_KEYS
    if key in frames[0].info
])

print("Possible energy keys:")
print([
    key for key in ENERGY_KEYS
    if key in frames[0].info
])


# =============================================================================
# 8. AUTOMATIC SITE-KEY DETECTION
# =============================================================================

site_key = next(
    (key for key in SITE_KEYS if key in frames[0].info),
    None
)

if site_key is None:
    raise ValueError(
        "Could not find a site identifier in EXTXYZ metadata."
        "\nInspect the printed info keys before continuing."
    )

print(f"\nUsing site identifier: {site_key}")


# =============================================================================
# 9. BUILD TRAJECTORY RECORDS
# =============================================================================

records = []

selected_site_indices = set(
    selected["site_index"].astype(int).tolist()
)

for frame_number, atoms in enumerate(frames):

    info = atoms.info

    if site_key not in info:
        continue

    try:
        site_index = int(info[site_key])
    except Exception:
        continue

    if site_index not in selected_site_indices:
        continue

    record = {
        "frame_number": frame_number,
        "site_index": site_index,
    }

    # Copy all scalar metadata
    for key, value in info.items():

        if np.isscalar(value):
            record[key] = value

    # Store positions
    positions = atoms.get_positions()

    record["n_atoms"] = len(atoms)

    # Assume Li is the inserted atom
    li_indices = [
        i for i, symbol in enumerate(atoms.get_chemical_symbols())
        if symbol == "Li"
    ]

    if len(li_indices) == 1:
        li_index = li_indices[0]

        li_pos = positions[li_index]

        record["Li_x"] = li_pos[0]
        record["Li_y"] = li_pos[1]
        record["Li_z"] = li_pos[2]

    records.append(record)


# =============================================================================
# 10. CHECK WHETHER SELECTED SITES WERE FOUND
# =============================================================================

trajectory_df = pd.DataFrame(records)

print("\n" + "-" * 80)
print("TRAJECTORY COVERAGE")
print("-" * 80)

if trajectory_df.empty:
    raise ValueError(
        "No trajectory frames were matched to the selected site indices."
    )

found_sites = sorted(
    trajectory_df["site_index"].unique()
)

print("Selected sites:")
print(sorted(selected_site_indices))

print("\nFound sites:")
print(found_sites)

missing_sites = sorted(
    selected_site_indices - set(found_sites)
)

if missing_sites:
    print("\nWARNING — Missing sites:")
    print(missing_sites)


# =============================================================================
# 11. ATTACH CASE LABELS
# =============================================================================

case_map = dict(
    zip(
        selected["site_index"].astype(int),
        selected["case"]
    )
)

trajectory_df["case"] = trajectory_df["site_index"].map(case_map)


# =============================================================================
# 12. SORT
# =============================================================================

sort_columns = ["site_index"]

for possible_step in STEP_KEYS:
    if possible_step in trajectory_df.columns:
        sort_columns.append(possible_step)
        break

sort_columns.append("frame_number")

trajectory_df = trajectory_df.sort_values(
    sort_columns
).reset_index(drop=True)


# =============================================================================
# 13. PRINT TRAJECTORY SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("TRAJECTORY SUMMARY")
print("=" * 80)

for site_index in sorted(found_sites):

    sub = trajectory_df[
        trajectory_df["site_index"] == site_index
    ]

    case = case_map[site_index]

    print("\n" + "-" * 80)
    print(f"CASE: {case}")
    print(f"SITE: {site_index}")
    print(f"FRAMES: {len(sub)}")

    print("\nFirst frame:")
    print(sub.iloc[0].to_string())

    print("\nLast frame:")
    print(sub.iloc[-1].to_string())


# =============================================================================
# 14. ENERGY / FORCE COLUMN DISCOVERY
# =============================================================================

print("\n" + "-" * 80)
print("AVAILABLE TRAJECTORY COLUMNS")
print("-" * 80)

print(
    trajectory_df.columns.tolist()
)


# =============================================================================
# 15. SAVE
# =============================================================================

trajectory_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

print(f"\nSaved:")
print(OUTPUT_FILE)

print("\nREAD-ONLY ANALYSIS:")
print("- No MACE calculation")
print("- No BFGS relaxation")
print("- No structure modification")
print("- Original screening results unchanged")

In [ ]:
# =============================================================================
# CANDIDATE 81 — FAILED-SITE QUALITY CLASSIFICATION
# Read-only diagnostic analysis of the 79 non-converged Li sites
# =============================================================================

import os
import numpy as np
import pandas as pd


# =============================================================================
# 1. PATHS
# =============================================================================

CSV_FILE = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_Li_single_site_energy_screening/"
    "candidate_81_Li_site_energy_summary.csv"
)

OUTPUT_CSV = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_Li_single_site_energy_screening/"
    "candidate_81_failed_sites_classification.csv"
)


# =============================================================================
# 2. PARAMETERS
# =============================================================================

FMAX_TARGET = 0.01

# Diagnostic threshold:
# clearly absurd final states are flagged as numerically pathological.
PATHOLOGICAL_ENERGY = 1e6       # eV
PATHOLOGICAL_FORCE = 1e6        # eV/A

# "Near convergence" is only a diagnostic label.
# It does NOT change the original converged=False result.
NEAR_CONVERGED_FORCE = 0.05     # eV/A


# =============================================================================
# 3. LOAD DATA
# =============================================================================

if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(f"CSV not found:\n{CSV_FILE}")

df = pd.read_csv(CSV_FILE)

print("=" * 80)
print("CANDIDATE 81 — FAILED-SITE QUALITY CLASSIFICATION")
print("=" * 80)

print(f"Total rows: {len(df)}")


# =============================================================================
# 4. VALIDATE REQUIRED COLUMNS
# =============================================================================

required_columns = [
    "site_index",
    "natural_clearance_A",
    "initial_energy_eV",
    "final_energy_eV",
    "initial_max_force_eV_A",
    "final_max_force_eV_A",
    "Li_displacement_A",
    "optimizer_steps",
    "converged",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(
        "Missing required columns:\n"
        + "\n".join(missing)
    )


# =============================================================================
# 5. SEPARATE SUCCESS / FAILURE
# =============================================================================

failed = df[df["converged"] == False].copy()
successful = df[df["converged"] == True].copy()

print(f"Successful: {len(successful)}")
print(f"Failed:     {len(failed)}")

if len(failed) == 0:
    print("\nNo failed sites found.")
    raise SystemExit


# =============================================================================
# 6. NUMERIC CONVERSION
# =============================================================================

numeric_columns = [
    "natural_clearance_A",
    "initial_energy_eV",
    "final_energy_eV",
    "initial_max_force_eV_A",
    "final_max_force_eV_A",
    "Li_displacement_A",
    "optimizer_steps",
]

for col in numeric_columns:
    failed[col] = pd.to_numeric(
        failed[col],
        errors="coerce"
    )

    successful[col] = pd.to_numeric(
        successful[col],
        errors="coerce"
    )


# =============================================================================
# 7. BUILD A ROBUST "NORMAL ENERGY" BAND
#    FROM THE SUCCESSFUL SITES
# =============================================================================

success_energy = successful["final_energy_eV"].dropna()

median_energy = success_energy.median()

q25 = success_energy.quantile(0.25)
q75 = success_energy.quantile(0.75)

IQR = q75 - q25

# Robust diagnostic band.
LOWER_ENERGY = median_energy - 5 * IQR
UPPER_ENERGY = median_energy + 5 * IQR

print("\n" + "-" * 80)
print("SUCCESSFUL-SITE FINAL ENERGY REFERENCE")
print("-" * 80)

print(f"Median final energy : {median_energy:.6f} eV")
print(f"Q25                 : {q25:.6f} eV")
print(f"Q75                 : {q75:.6f} eV")
print(f"IQR                 : {IQR:.6f} eV")

print("\nDiagnostic normal-energy band:")
print(f"Lower = {LOWER_ENERGY:.6f} eV")
print(f"Upper = {UPPER_ENERGY:.6f} eV")


# =============================================================================
# 8. FINITE / NON-FINITE CHECK
# =============================================================================

failed["finite_final_energy"] = np.isfinite(
    failed["final_energy_eV"]
)

failed["finite_final_force"] = np.isfinite(
    failed["final_max_force_eV_A"]
)

failed["finite_final_state"] = (
    failed["finite_final_energy"]
    & failed["finite_final_force"]
)


# =============================================================================
# 9. PATHOLOGICAL FINAL STATE
# =============================================================================

failed["pathological_final_state"] = (
    (~failed["finite_final_state"])
    |
    (
        failed["final_energy_eV"].abs()
        > PATHOLOGICAL_ENERGY
    )
    |
    (
        failed["final_max_force_eV_A"].abs()
        > PATHOLOGICAL_FORCE
    )
)


# =============================================================================
# 10. DID THE ENERGY REACH THE NORMAL REGION?
# =============================================================================

failed["normal_energy_region"] = (
    failed["finite_final_state"]
    &
    (
        failed["final_energy_eV"] >= LOWER_ENERGY
    )
    &
    (
        failed["final_energy_eV"] <= UPPER_ENERGY
    )
)


# =============================================================================
# 11. FORCE-BASED DIAGNOSTICS
# =============================================================================

failed["near_converged_force"] = (
    failed["normal_energy_region"]
    &
    (
        failed["final_max_force_eV_A"] <= NEAR_CONVERGED_FORCE
    )
)

failed["persistent_force"] = (
    failed["normal_energy_region"]
    &
    (
        failed["final_max_force_eV_A"] > NEAR_CONVERGED_FORCE
    )
)


# =============================================================================
# 12. MAX-STEP CHECK
# =============================================================================

failed["hit_max_steps"] = (
    failed["optimizer_steps"] >= 1000
)


# =============================================================================
# 13. FINAL DIAGNOSTIC CLASSIFICATION
# =============================================================================

def classify(row):

    if row["pathological_final_state"]:
        return "PATHOLOGICAL_FINAL_STATE"

    if row["near_converged_force"]:
        return "NEAR_CONVERGED"

    if row["persistent_force"]:
        return "NORMAL_ENERGY_PERSISTENT_FORCE"

    if row["finite_final_state"]:
        return "FINITE_BUT_OUTSIDE_NORMAL_ENERGY_BAND"

    return "NON_FINITE_FINAL_STATE"


failed["diagnostic_class"] = failed.apply(
    classify,
    axis=1
)


# =============================================================================
# 14. PRINT CLASS COUNTS
# =============================================================================

print("\n" + "=" * 80)
print("FAILED-SITE DIAGNOSTIC CLASSES")
print("=" * 80)

class_counts = (
    failed["diagnostic_class"]
    .value_counts()
    .sort_index()
)

for category, count in class_counts.items():

    percentage = 100 * count / len(failed)

    print(
        f"{category:40s}"
        f"{count:5d}"
        f" ({percentage:6.2f}%)"
    )


# =============================================================================
# 15. DETAILED SUMMARY BY CLASS
# =============================================================================

print("\n" + "=" * 80)
print("CLASS SUMMARY")
print("=" * 80)

summary = (
    failed
    .groupby("diagnostic_class")
    .agg(
        sites=("site_index", "count"),
        mean_clearance_A=("natural_clearance_A", "mean"),
        mean_initial_force=("initial_max_force_eV_A", "mean"),
        mean_final_force=("final_max_force_eV_A", "mean"),
        mean_final_energy=("final_energy_eV", "mean"),
        mean_displacement_A=("Li_displacement_A", "mean"),
        mean_steps=("optimizer_steps", "mean"),
    )
    .sort_values("sites", ascending=False)
)

print(summary.to_string())


# =============================================================================
# 16. SHOW THE MOST IMPORTANT CASES
# =============================================================================

print("\n" + "=" * 80)
print("NEAR-CONVERGED FAILED SITES")
print("=" * 80)

near = failed[
    failed["diagnostic_class"] == "NEAR_CONVERGED"
].sort_values("final_max_force_eV_A")

if len(near) == 0:
    print("None.")

else:
    print(
        near[
            [
                "site_index",
                "natural_clearance_A",
                "initial_max_force_eV_A",
                "final_energy_eV",
                "final_max_force_eV_A",
                "Li_displacement_A",
                "optimizer_steps",
            ]
        ].to_string(index=False)
    )


print("\n" + "=" * 80)
print("PATHOLOGICAL FINAL STATES")
print("=" * 80)

pathological = failed[
    failed["diagnostic_class"] == "PATHOLOGICAL_FINAL_STATE"
].sort_values(
    "final_max_force_eV_A",
    ascending=False
)

if len(pathological) == 0:
    print("None.")

else:
    print(
        pathological[
            [
                "site_index",
                "natural_clearance_A",
                "initial_energy_eV",
                "final_energy_eV",
                "initial_max_force_eV_A",
                "final_max_force_eV_A",
                "Li_displacement_A",
                "optimizer_steps",
            ]
        ].head(20).to_string(index=False)
    )


# =============================================================================
# 17. CHECK HOW MANY FAILED SITES HIT MAX STEPS
# =============================================================================

print("\n" + "=" * 80)
print("MAX-STEP CHECK")
print("=" * 80)

print(
    f"Failed sites reaching 1000 steps: "
    f"{failed['hit_max_steps'].sum()} / {len(failed)}"
)


# =============================================================================
# 18. SAVE DIAGNOSTIC TABLE
# =============================================================================

failed = failed.sort_values(
    ["diagnostic_class", "final_energy_eV"],
    na_position="last"
)

failed.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\n" + "=" * 80)
print("OUTPUT")
print("=" * 80)

print(f"Saved diagnostic CSV:")
print(OUTPUT_CSV)

print("\nAnalysis complete.")

In [ ]:
# ============================================================
# CANDIDATE 81 — FAILED-SITE TRIAGE
# ============================================================
# PURPOSE:
#   Inspect the 79 non-converged Li sites from the existing CSV.
#
# IMPORTANT:
#   - NO relaxation
#   - NO MACE calculation
#   - NO modification of original results
#   - READ-ONLY diagnostic analysis
# ============================================================

import os
import numpy as np
import pandas as pd

# ============================================================
# 1) INPUT
# ============================================================

CSV_PATH = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_Li_single_site_energy_screening/"
    "candidate_81_Li_site_energy_summary.csv"
)

OUT_DIR = os.path.dirname(CSV_PATH)

TRIAGE_CSV = os.path.join(
    OUT_DIR,
    "candidate_81_failed_site_triage.csv"
)

# ============================================================
# 2) LOAD
# ============================================================

df = pd.read_csv(CSV_PATH)

required = [
    "site_index",
    "natural_clearance_A",
    "initial_energy_eV",
    "final_energy_eV",
    "initial_max_force_eV_A",
    "final_max_force_eV_A",
    "Li_displacement_A",
    "optimizer_steps",
    "converged",
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns:\n{missing}"
    )

# Normalize convergence column
df["converged"] = df["converged"].astype(bool)

success = df[df["converged"]].copy()
failed = df[~df["converged"]].copy()

print("=" * 70)
print("CANDIDATE 81 — FAILED-SITE TRIAGE")
print("=" * 70)

print(f"\nTotal sites      : {len(df)}")
print(f"Successful       : {len(success)}")
print(f"Failed           : {len(failed)}")

# ============================================================
# 3) SUCCESS REFERENCE
# ============================================================

success_energy = success["final_energy_eV"]

energy_median = success_energy.median()
energy_q25 = success_energy.quantile(0.25)
energy_q75 = success_energy.quantile(0.75)

iqr = energy_q75 - energy_q25

# Broad diagnostic energy band.
# This is NOT a physical cutoff; it is only for triage.
energy_low = energy_median - 5 * iqr
energy_high = energy_median + 5 * iqr

print("\n" + "-" * 70)
print("SUCCESS REFERENCE")
print("-" * 70)

print(f"Median final energy : {energy_median:.6f} eV")
print(f"Q25                 : {energy_q25:.6f} eV")
print(f"Q75                 : {energy_q75:.6f} eV")
print(f"IQR                 : {iqr:.6f} eV")
print(f"Diagnostic band     : [{energy_low:.6f}, {energy_high:.6f}] eV")

# ============================================================
# 4) TRIAGE RULES
# ============================================================
#
# These rules are deliberately conservative.
#
# PATHOLOGICAL:
#   Clearly non-physical final energy/force.
#
# NEAR_CONVERGED:
#   Final force is close to the actual convergence threshold.
#
# PERSISTENT_FORCE:
#   Final energy remains in a normal region, but force is
#   substantially above the convergence threshold.
#
# REVIEW:
#   Does not cleanly fit the above categories.
# ============================================================

FMAX = 0.01

# Clearly pathological numerical values.
PATH_ENERGY_LIMIT = 1e6       # eV
PATH_FORCE_LIMIT = 1e8         # eV/A
PATH_DISP_LIMIT = 10.0        # A

# "Near convergence" is diagnostic only.
NEAR_FORCE_LIMIT = 0.05       # eV/A

# Persistent but not obviously pathological.
PERSISTENT_FORCE_LIMIT = 0.05

# ============================================================
# 5) CLASSIFY
# ============================================================

def classify(row):

    E = row["final_energy_eV"]
    F = row["final_max_force_eV_A"]
    D = row["Li_displacement_A"]

    # ---- Pathological numerical failure ----
    if (
        not np.isfinite(E)
        or not np.isfinite(F)
        or not np.isfinite(D)
        or abs(E) > PATH_ENERGY_LIMIT
        or F > PATH_FORCE_LIMIT
        or D > PATH_DISP_LIMIT
    ):
        return "PATHOLOGICAL"

    # ---- Very close to convergence ----
    if F <= NEAR_FORCE_LIMIT:
        return "NEAR_CONVERGED"

    # ---- Normal-energy but persistent force ----
    if (
        energy_low <= E <= energy_high
        and F > PERSISTENT_FORCE_LIMIT
    ):
        return "PERSISTENT_FORCE"

    # ---- Everything else ----
    return "REVIEW"


failed["triage_class"] = failed.apply(classify, axis=1)

# ============================================================
# 6) PRIORITY
# ============================================================

priority = {
    "NEAR_CONVERGED": 1,
    "PERSISTENT_FORCE": 2,
    "REVIEW": 3,
    "PATHOLOGICAL": 4,
}

failed["priority"] = failed["triage_class"].map(priority)

failed = failed.sort_values(
    ["priority", "final_max_force_eV_A"],
    ascending=[True, True]
)

# ============================================================
# 7) SAVE
# ============================================================

failed.to_csv(TRIAGE_CSV, index=False)

# ============================================================
# 8) SUMMARY
# ============================================================

counts = (
    failed["triage_class"]
    .value_counts()
    .reindex(
        [
            "NEAR_CONVERGED",
            "PERSISTENT_FORCE",
            "REVIEW",
            "PATHOLOGICAL",
        ],
        fill_value=0,
    )
)

print("\n" + "=" * 70)
print("FAILURE CLASSIFICATION")
print("=" * 70)

for category, count in counts.items():
    pct = 100 * count / len(failed)
    print(f"{category:25s}: {count:3d} ({pct:6.2f}%)")

# ============================================================
# 9) SITES WORTH REVIEWING
# ============================================================

review_classes = [
    "NEAR_CONVERGED",
    "PERSISTENT_FORCE",
    "REVIEW",
]

worth_reviewing = failed[
    failed["triage_class"].isin(review_classes)
].copy()

print("\n" + "=" * 70)
print("SITES WORTH FURTHER REVIEW")
print("=" * 70)

if len(worth_reviewing) == 0:

    print("None.")

else:

    cols = [
        "site_index",
        "triage_class",
        "natural_clearance_A",
        "initial_energy_eV",
        "final_energy_eV",
        "initial_max_force_eV_A",
        "final_max_force_eV_A",
        "Li_displacement_A",
        "optimizer_steps",
    ]

    print(
        worth_reviewing[cols]
        .to_string(index=False)
    )

# ============================================================
# 10) MOST EXTREME PATHOLOGICAL CASES
# ============================================================

pathological = failed[
    failed["triage_class"] == "PATHOLOGICAL"
].copy()

print("\n" + "=" * 70)
print("MOST EXTREME PATHOLOGICAL CASES")
print("=" * 70)

if len(pathological) == 0:

    print("None.")

else:

    pathological["abs_final_energy"] = (
        pathological["final_energy_eV"].abs()
    )

    extreme = pathological.nlargest(
        10,
        "abs_final_energy"
    )

    cols = [
        "site_index",
        "final_energy_eV",
        "final_max_force_eV_A",
        "Li_displacement_A",
        "natural_clearance_A",
        "optimizer_steps",
    ]

    print(extreme[cols].to_string(index=False))

# ============================================================
# 11) FINAL MESSAGE
# ============================================================

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print(f"\nTriage CSV saved to:")
print(TRIAGE_CSV)

print(
    "\nIMPORTANT: This script only diagnoses the existing CSV."
)
print(
    "No site was re-relaxed and no original result was changed."
)


In [ ]:
# ============================================================
# CANDIDATE 81
# DIAGNOSTIC ANALYSIS OF THE 9 UNRESOLVED Li SITES
#
# PURPOSE:
#   Read-only investigation of the unresolved screening results.
#
#   NO MACE
#   NO relaxation
#   NO BFGS
#   NO modification of the original screening results
#   NO change to the original screening method
# ============================================================

import os
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_Li_single_site_energy_screening"
)

CSV_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_Li_site_energy_summary.csv"
)

OUTPUT_CSV = os.path.join(
    BASE_DIR,
    "candidate_81_9_unresolved_sites_diagnostic.csv"
)


# ============================================================
# 2. THE 9 UNRESOLVED SITES
# ============================================================

NEAR_CONVERGED = [
    461,
    101,
]

PERSISTENT_FORCE = [
    227,
    562,
    574,
    178,
    250,
    812,
    107,
]

TARGET_SITES = (
    NEAR_CONVERGED
    + PERSISTENT_FORCE
)


# ============================================================
# 3. LOAD CSV
# ============================================================

print("=" * 80)
print("CANDIDATE 81 — 9 UNRESOLVED Li SITES DIAGNOSTIC")
print("=" * 80)

print("\n[1] Loading CSV...")

if not os.path.isfile(CSV_FILE):
    raise FileNotFoundError(
        f"\nCSV file not found:\n{CSV_FILE}"
    )

df = pd.read_csv(CSV_FILE)

print(f"Total rows: {len(df)}")


# ============================================================
# 4. VALIDATE ACTUAL CSV COLUMNS
# ============================================================

required_columns = [
    "test_number",
    "site_index",
    "natural_clearance_A",
    "initial_energy_eV",
    "final_energy_eV",
    "initial_max_force_eV_A",
    "final_max_force_eV_A",
    "Li_displacement_A",
    "optimizer_steps",
    "converged",
]

missing = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        "\nMissing required columns:\n"
        + "\n".join(f" - {x}" for x in missing)
    )

print("\nPASS: All required CSV columns are present.")


# ============================================================
# 5. BASIC DATA VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("BASIC VALIDATION")
print("=" * 80)

print("Rows:", len(df))
print("Duplicate rows:", df.duplicated().sum())

if len(df) != 866:
    print(
        f"WARNING: Expected 866 rows, found {len(df)}."
    )
else:
    print("PASS: All 866 screening results are present.")


# ------------------------------------------------------------
# Validate site_index uniqueness
# ------------------------------------------------------------

duplicate_site_indices = (
    df["site_index"]
    .duplicated()
    .sum()
)

print(
    "Duplicate site_index values:",
    duplicate_site_indices
)


# ============================================================
# 6. NORMALIZE / VALIDATE NUMERIC COLUMNS
# ============================================================

numeric_columns = [
    "site_index",
    "natural_clearance_A",
    "initial_energy_eV",
    "final_energy_eV",
    "initial_max_force_eV_A",
    "final_max_force_eV_A",
    "Li_displacement_A",
    "optimizer_steps",
]

for col in numeric_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    if df[col].isna().any():

        bad_count = df[col].isna().sum()

        raise ValueError(
            f"\nColumn '{col}' contains "
            f"{bad_count} non-numeric values."
        )


# ------------------------------------------------------------
# Normalize convergence
# ------------------------------------------------------------

df["converged"] = (
    df["converged"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
    })
)

if df["converged"].isna().any():

    raise ValueError(
        "\nCould not interpret some values "
        "in the 'converged' column."
    )


# ============================================================
# 7. CHECK THE 9 TARGET SITES
# ============================================================

print("\n" + "=" * 80)
print("TARGET-SITE CHECK")
print("=" * 80)

found_sites = set(
    df["site_index"].astype(int)
)

missing_target_sites = [
    site
    for site in TARGET_SITES
    if site not in found_sites
]

if missing_target_sites:

    raise ValueError(
        "\nTarget sites missing from CSV:\n"
        + str(missing_target_sites)
    )

print(
    f"PASS: All {len(TARGET_SITES)} "
    "target sites exist in the CSV."
)


# ============================================================
# 8. EXTRACT THE 9 UNRESOLVED SITES
# ============================================================

diagnostic = (
    df[
        df["site_index"]
        .astype(int)
        .isin(TARGET_SITES)
    ]
    .copy()
)


# ------------------------------------------------------------
# Assign diagnostic category
# ------------------------------------------------------------

def classify_site(site):

    site = int(site)

    if site in NEAR_CONVERGED:
        return "NEAR_CONVERGED"

    if site in PERSISTENT_FORCE:
        return "PERSISTENT_FORCE"

    return "UNKNOWN"


diagnostic["category"] = (
    diagnostic["site_index"]
    .apply(classify_site)
)


# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

diagnostic = (
    diagnostic
    .sort_values("site_index")
    .reset_index(drop=True)
)


# ============================================================
# 9. CONFIRM THEIR CONVERGENCE STATUS
# ============================================================

print("\n" + "=" * 80)
print("CONVERGENCE STATUS OF THE 9 SITES")
print("=" * 80)

print(
    diagnostic[
        [
            "site_index",
            "category",
            "converged",
            "optimizer_steps",
        ]
    ].to_string(index=False)
)

unexpected_converged = diagnostic[
    diagnostic["converged"] == True
]

if len(unexpected_converged) > 0:

    print(
        "\nWARNING: Some target sites are marked "
        "as converged."
    )

else:

    print(
        "\nPASS: All 9 target sites are "
        "currently marked converged=False."
    )


# ============================================================
# 10. DETAILED 9-SITE TABLE
# ============================================================

print("\n" + "=" * 80)
print("THE 9 UNRESOLVED CASES")
print("=" * 80)

display_columns = [
    "site_index",
    "category",
    "natural_clearance_A",
    "initial_energy_eV",
    "final_energy_eV",
    "initial_max_force_eV_A",
    "final_max_force_eV_A",
    "Li_displacement_A",
    "optimizer_steps",
    "converged",
]

print(
    diagnostic[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)


# ============================================================
# 11. ENERGY / FORCE CHANGE
# ============================================================

diagnostic["energy_change_eV_calculated"] = (
    diagnostic["final_energy_eV"]
    - diagnostic["initial_energy_eV"]
)

diagnostic["force_reduction_factor"] = (
    diagnostic["initial_max_force_eV_A"]
    /
    diagnostic["final_max_force_eV_A"]
)

print("\n" + "=" * 80)
print("ENERGY / FORCE BEHAVIOR")
print("=" * 80)

print(
    diagnostic[
        [
            "site_index",
            "category",
            "energy_change_eV_calculated",
            "initial_max_force_eV_A",
            "final_max_force_eV_A",
            "force_reduction_factor",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.8e}"
    )
)


# ============================================================
# 12. COMPARE THE 9 SITES WITH SUCCESSFUL RESULTS
# ============================================================

successful = df[
    df["converged"] == True
].copy()

failed = df[
    df["converged"] == False
].copy()

print("\n" + "=" * 80)
print("SUCCESSFUL-SITE REFERENCE")
print("=" * 80)

print(
    f"Successful sites : {len(successful)}"
)

print(
    f"Failed sites     : {len(failed)}"
)


# ------------------------------------------------------------
# Successful final-energy statistics
# ------------------------------------------------------------

success_median_energy = (
    successful["final_energy_eV"]
    .median()
)

success_q25 = (
    successful["final_energy_eV"]
    .quantile(0.25)
)

success_q75 = (
    successful["final_energy_eV"]
    .quantile(0.75)
)

success_min = (
    successful["final_energy_eV"]
    .min()
)

success_max = (
    successful["final_energy_eV"]
    .max()
)

print(
    f"\nSuccessful median final energy : "
    f"{success_median_energy:.10f} eV"
)

print(
    f"Successful Q25                 : "
    f"{success_q25:.10f} eV"
)

print(
    f"Successful Q75                 : "
    f"{success_q75:.10f} eV"
)

print(
    f"Successful minimum             : "
    f"{success_min:.10f} eV"
)

print(
    f"Successful maximum             : "
    f"{success_max:.10f} eV"
)


# ============================================================
# 13. DISTANCE FROM SUCCESSFUL ENERGY MEDIAN
# ============================================================

diagnostic[
    "energy_difference_from_success_median_eV"
] = (
    diagnostic["final_energy_eV"]
    - success_median_energy
)

print("\n" + "=" * 80)
print("DISTANCE FROM SUCCESSFUL ENERGY MEDIAN")
print("=" * 80)

print(
    diagnostic[
        [
            "site_index",
            "category",
            "final_energy_eV",
            "energy_difference_from_success_median_eV",
            "final_max_force_eV_A",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.10f}"
    )
)


# ============================================================
# 14. SPECIAL GROUP
#     227 / 250 / 562 / 574
# ============================================================

SPECIAL_GROUP = [
    227,
    250,
    562,
    574,
]

group_df = diagnostic[
    diagnostic["site_index"]
    .astype(int)
    .isin(SPECIAL_GROUP)
].copy()

print("\n" + "=" * 80)
print("SPECIAL GROUP: 227 / 250 / 562 / 574")
print("=" * 80)

print(
    group_df[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.10f}"
    )
)


if len(group_df) == len(SPECIAL_GROUP):

    clearance_range = (
        group_df["natural_clearance_A"].max()
        -
        group_df["natural_clearance_A"].min()
    )

    initial_energy_range = (
        group_df["initial_energy_eV"].max()
        -
        group_df["initial_energy_eV"].min()
    )

    final_energy_range = (
        group_df["final_energy_eV"].max()
        -
        group_df["final_energy_eV"].min()
    )

    initial_force_range = (
        group_df["initial_max_force_eV_A"].max()
        -
        group_df["initial_max_force_eV_A"].min()
    )

    final_force_range = (
        group_df["final_max_force_eV_A"].max()
        -
        group_df["final_max_force_eV_A"].min()
    )

    print("\nRanges inside this group:")

    print(
        f"Clearance range       : "
        f"{clearance_range:.10f} Å"
    )

    print(
        f"Initial energy range  : "
        f"{initial_energy_range:.10f} eV"
    )

    print(
        f"Final energy range    : "
        f"{final_energy_range:.10f} eV"
    )

    print(
        f"Initial force range   : "
        f"{initial_force_range:.10f} eV/Å"
    )

    print(
        f"Final force range     : "
        f"{final_force_range:.10f} eV/Å"
    )


# ============================================================
# 15. CHECK MAX-STEP CONDITION
# ============================================================

print("\n" + "=" * 80)
print("MAX-STEP CHECK")
print("=" * 80)

max_step_sites = diagnostic[
    diagnostic["optimizer_steps"] >= 1000
]

print(
    f"Sites reaching 1000 steps : "
    f"{len(max_step_sites)} / {len(diagnostic)}"
)


# ============================================================
# 16. SIMPLE TRIAGE INDICATORS
#
# IMPORTANT:
# These are diagnostic indicators only.
# They do NOT automatically decide which sites should
# be re-relaxed.
# ============================================================

diagnostic["force_near_target"] = (
    diagnostic["final_max_force_eV_A"]
    <= 0.02
)

diagnostic["energy_near_success_median"] = (
    np.abs(
        diagnostic["final_energy_eV"]
        -
        success_median_energy
    )
    <= 0.10
)

diagnostic["reasonable_final_energy"] = (
    diagnostic["final_energy_eV"]
    >= success_min - 1.0
)

print("\n" + "=" * 80)
print("DIAGNOSTIC INDICATORS")
print("=" * 80)

print(
    diagnostic[
        [
            "site_index",
            "category",
            "force_near_target",
            "energy_near_success_median",
            "reasonable_final_energy",
        ]
    ].to_string(index=False)
)


# ============================================================
# 17. SAVE READ-ONLY DIAGNOSTIC TABLE
# ============================================================

diagnostic.to_csv(
    OUTPUT_CSV,
    index=False
)


# ============================================================
# 18. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

print("\nSaved diagnostic file:")
print(OUTPUT_CSV)

print("\nIMPORTANT:")
print("No MACE calculation was performed.")
print("No BFGS relaxation was performed.")
print("No structure was modified.")
print("No original screening result was changed.")
print("No screening parameter was changed.")
print("The original 866-site screening remains untouched.")

In [ ]:
# =============================================================================
# CANDIDATE 81
# CONTROLLED RE-RELAXATION TEST OF SELECTED FAILED Li SITES
#
# PURPOSE
# -------
# Small, controlled scientific experiment.
#
# We DO NOT re-relax all 79 failed sites.
# We test only representative cases selected from the diagnostic analysis.
#
# SELECTED SITES
# --------------
# 461 : NEAR_CONVERGED
# 101 : NEAR_CONVERGED
#
# 227 : representative of the nearly identical group
#       [227, 250, 562, 574]
#
# 812 : different moderate-clearance persistent-force case
#
# 107 : optional middle case
# 178 : optional middle case
#
# SCIENTIFIC RULES
# ----------------
# 1. Same Candidate 81 host.
# 2. Same periodic Voronoi site coordinates.
# 3. Same MACE method.
# 4. Same BFGS relaxation method.
# 5. Same fixed cell.
# 6. Same FMAX = 0.01 eV/A.
# 7. Same MAX_STEPS = 1000.
# 8. NO Li-host distance cutoff.
# 9. NO artificial distance criterion.
# 10. Original screening CSV is NEVER modified.
# 11. Original screening EXTXYZ is NEVER modified.
#
# OUTPUTS ARE WRITTEN TO A NEW DIRECTORY.
# =============================================================================


# =============================================================================
# 1. IMPORTS
# =============================================================================

import os
import csv
import numpy as np
import pandas as pd
import torch

from google.colab import drive

from ase import Atoms
from ase.io import read, write
from ase.optimize import BFGS

from mace.calculators import mace_mp


# =============================================================================
# 2. MOUNT GOOGLE DRIVE
# =============================================================================

drive.mount("/content/drive")


# =============================================================================
# 3. PATHS
# =============================================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

HOST_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_40atom_relaxed_host.extxyz"
)

SCREENING_DIR = os.path.join(
    BASE_DIR,
    "candidate_81_Li_single_site_energy_screening"
)

SUMMARY_FILE = os.path.join(
    SCREENING_DIR,
    "candidate_81_Li_site_energy_summary.csv"
)

SITE_FILE = os.path.join(
    SCREENING_DIR,
    "candidate_81_all_periodic_voronoi_sites.npy"
)

# IMPORTANT:
# Completely separate output directory.
OUTPUT_DIR = os.path.join(
    SCREENING_DIR,
    "candidate_81_selected_failed_sites_re_relaxation"
)

OUTPUT_CSV = os.path.join(
    OUTPUT_DIR,
    "candidate_81_selected_failed_sites_re_relaxation.csv"
)

OUTPUT_EXTXYZ = os.path.join(
    OUTPUT_DIR,
    "candidate_81_selected_failed_sites_re_relaxation.extxyz"
)


# =============================================================================
# 4. EXPERIMENT SETTINGS
# =============================================================================

CANDIDATE_ID = 81

FMAX = 0.01

MAX_STEPS = 1000

# -------------------------------------------------------------------------
# PRIMARY TEST SET
# -------------------------------------------------------------------------
#
# 461 = near-converged
# 101 = second near-converged / different behavior
# 227 = representative of nearly identical group
# 812 = different persistent-force failure
#
# -------------------------------------------------------------------------

PRIMARY_SITES = [
    461,
    101,
    227,
    812,
]

# -------------------------------------------------------------------------
# OPTIONAL MIDDLE CASES
# -------------------------------------------------------------------------
#
# Set ADD_MIDDLE_CASES = True if you want to include them.
#
# 107 = moderate clearance / persistent force
# 178 = very high initial force / unusual displacement
#
# -------------------------------------------------------------------------

ADD_MIDDLE_CASES = True

MIDDLE_SITES = [
    107,
    178,
]

if ADD_MIDDLE_CASES:
    TARGET_SITES = PRIMARY_SITES + MIDDLE_SITES
else:
    TARGET_SITES = PRIMARY_SITES


# Remove accidental duplicates while preserving order.
TARGET_SITES = list(dict.fromkeys(TARGET_SITES))


# =============================================================================
# 5. CREATE NEW OUTPUT DIRECTORY
# =============================================================================

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# =============================================================================
# 6. HEADER
# =============================================================================

print("=" * 100)
print("CANDIDATE 81 — CONTROLLED FAILED-SITE RE-RELAXATION TEST")
print("=" * 100)

print()
print("Host file          :", HOST_FILE)
print("Original CSV       :", SUMMARY_FILE)
print("Voronoi site file  :", SITE_FILE)
print("New output dir     :", OUTPUT_DIR)

print()
print("Method             :", "Periodic Voronoi + single-Li MACE relaxation")
print("Cell               :", "FIXED")
print("Li-host cutoff     :", "NONE")
print("Artificial distance:", "NONE")
print("FMAX               :", FMAX, "eV/A")
print("MAX_STEPS          :", MAX_STEPS)

print()
print("Target sites       :", TARGET_SITES)

print()
print("IMPORTANT:")
print("Original screening files will NOT be modified.")
print("Results will be written to a NEW directory.")

print("=" * 100)


# =============================================================================
# 7. CHECK INPUT FILES
# =============================================================================

for path, label in [
    (HOST_FILE, "HOST_FILE"),
    (SUMMARY_FILE, "SUMMARY_FILE"),
    (SITE_FILE, "SITE_FILE"),
]:

    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"\n{label} not found:\n{path}"
        )


# =============================================================================
# 8. LOAD ORIGINAL CSV
# =============================================================================

print()
print("=" * 100)
print("LOADING ORIGINAL SCREENING CSV")
print("=" * 100)

df = pd.read_csv(SUMMARY_FILE)

print("Rows:", len(df))

required_columns = [
    "site_index",
    "natural_clearance_A",
    "initial_energy_eV",
    "final_energy_eV",
    "initial_max_force_eV_A",
    "final_max_force_eV_A",
    "Li_displacement_A",
    "optimizer_steps",
    "converged",
]

missing = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        "Missing required CSV columns:\n"
        + "\n".join(missing)
    )

print("PASS: Required columns found.")


# =============================================================================
# 9. VALIDATE TARGET SITES
# =============================================================================

print()
print("=" * 100)
print("TARGET-SITE VALIDATION")
print("=" * 100)

csv_sites = set(
    df["site_index"].astype(int).tolist()
)

missing_sites = [
    site
    for site in TARGET_SITES
    if site not in csv_sites
]

if missing_sites:
    raise ValueError(
        f"Target sites missing from original CSV: {missing_sites}"
    )

target_df = (
    df[
        df["site_index"].isin(TARGET_SITES)
    ]
    .copy()
)

target_df["site_index"] = (
    target_df["site_index"].astype(int)
)

target_df = (
    target_df
    .set_index("site_index")
    .loc[TARGET_SITES]
    .reset_index()
)

print("PASS: All target sites exist.")

print()
print(
    target_df[
        [
            "site_index",
            "natural_clearance_A",
            "initial_energy_eV",
            "final_energy_eV",
            "initial_max_force_eV_A",
            "final_max_force_eV_A",
            "Li_displacement_A",
            "optimizer_steps",
            "converged",
        ]
    ].to_string(index=False)
)


# =============================================================================
# 10. LOAD HOST
# =============================================================================

print()
print("=" * 100)
print("LOADING HOST")
print("=" * 100)

host = read(
    HOST_FILE,
    index=0
).copy()

print("Formula :", host.get_chemical_formula())
print("Atoms   :", len(host))
print("Volume  :", host.get_volume(), "A^3")
print("PBC     :", host.get_pbc())
print("Cell:")
print(host.cell.array)

if len(host) != 40:
    raise ValueError(
        f"Expected 40 host atoms, found {len(host)}."
    )

if not np.all(host.get_pbc()):
    raise ValueError(
        "Host must be fully periodic."
    )


# =============================================================================
# 11. LOAD VORONOI SITES
# =============================================================================

print()
print("=" * 100)
print("LOADING VORONOI SITES")
print("=" * 100)

all_sites = np.load(
    SITE_FILE
)

print("Total Voronoi sites:", len(all_sites))

if all_sites.ndim != 2 or all_sites.shape[1] != 3:
    raise ValueError(
        f"Unexpected Voronoi site array shape: {all_sites.shape}"
    )

for site in TARGET_SITES:

    if site < 0 or site >= len(all_sites):
        raise IndexError(
            f"Site index {site} is outside Voronoi site array."
        )

print("PASS: All selected site indices are valid.")


# =============================================================================
# 12. LOAD MACE
# =============================================================================

print()
print("=" * 100)
print("LOADING MACE")
print("=" * 100)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64",
    device=DEVICE
)

print("MACE       : medium")
print("Dispersion : True")
print("dtype      : float64")


# =============================================================================
# 13. RELAXATION FUNCTION
# =============================================================================

def relax_one_site(site_index):

    # ------------------------------------------------------------
    # Retrieve EXACT original Voronoi coordinate.
    # ------------------------------------------------------------

    li_position = np.asarray(
        all_sites[site_index],
        dtype=float
    )

    # ------------------------------------------------------------
    # Copy the original relaxed 40-atom host.
    # ------------------------------------------------------------

    atoms = host.copy()

    # ------------------------------------------------------------
    # Insert ONE Li atom.
    # ------------------------------------------------------------

    atoms += Atoms(
        "Li",
        positions=[li_position]
    )

    # ------------------------------------------------------------
    # IMPORTANT:
    # Cell is inherited from host.
    # No cell optimizer/filter is used.
    # Therefore the cell remains FIXED.
    # ------------------------------------------------------------

    atoms.set_calculator(calc)

    # ------------------------------------------------------------
    # Initial state
    # ------------------------------------------------------------

    initial_energy = atoms.get_potential_energy()

    initial_forces = atoms.get_forces()

    initial_max_force = np.max(
        np.linalg.norm(
            initial_forces,
            axis=1
        )
    )

    initial_li_position = (
        atoms.positions[-1].copy()
    )

    # ------------------------------------------------------------
    # BFGS
    # ------------------------------------------------------------

    optimizer = BFGS(
        atoms,
        logfile=None
    )

    converged = optimizer.run(
        fmax=FMAX,
        steps=MAX_STEPS
    )

    optimizer_steps = optimizer.nsteps

    # ------------------------------------------------------------
    # Final state
    # ------------------------------------------------------------

    final_energy = atoms.get_potential_energy()

    final_forces = atoms.get_forces()

    final_max_force = np.max(
        np.linalg.norm(
            final_forces,
            axis=1
        )
    )

    final_li_position = (
        atoms.positions[-1].copy()
    )

    li_displacement = np.linalg.norm(
        final_li_position
        - initial_li_position
    )

    energy_change = (
        final_energy
        - initial_energy
    )

    # ------------------------------------------------------------
    # Store metadata in EXTXYZ frame.
    # ------------------------------------------------------------

    atoms.info["candidate_id"] = CANDIDATE_ID
    atoms.info["structure_role"] = (
        "40-atom fully relaxed host + one Li "
        "for controlled failed-site re-relaxation"
    )

    atoms.info["relaxation"] = (
        "MACE fixed-cell BFGS relaxation"
    )

    atoms.info["relaxation_fmax_eV_per_A"] = FMAX

    atoms.info["site_index"] = int(site_index)
    atoms.info["voronoi_site_index"] = int(site_index)

    atoms.info["initial_li_x"] = float(
        initial_li_position[0]
    )
    atoms.info["initial_li_y"] = float(
        initial_li_position[1]
    )
    atoms.info["initial_li_z"] = float(
        initial_li_position[2]
    )

    atoms.info["final_li_x"] = float(
        final_li_position[0]
    )
    atoms.info["final_li_y"] = float(
        final_li_position[1]
    )
    atoms.info["final_li_z"] = float(
        final_li_position[2]
    )

    atoms.info["initial_energy_eV"] = float(
        initial_energy
    )

    atoms.info["final_energy_eV"] = float(
        final_energy
    )

    atoms.info["energy_change_eV"] = float(
        energy_change
    )

    atoms.info["initial_max_force_eV_A"] = float(
        initial_max_force
    )

    atoms.info["final_max_force_eV_A"] = float(
        final_max_force
    )

    atoms.info["Li_displacement_A"] = float(
        li_displacement
    )

    atoms.info["optimizer_steps"] = int(
        optimizer_steps
    )

    atoms.info["converged"] = bool(
        converged
    )

    atoms.info["original_converged"] = bool(
        target_df.loc[
            target_df["site_index"] == site_index,
            "converged"
        ].iloc[0]
    )

    return {
        "site_index": int(site_index),

        "natural_clearance_A": float(
            target_df.loc[
                target_df["site_index"] == site_index,
                "natural_clearance_A"
            ].iloc[0]
        ),

        "original_initial_energy_eV": float(
            target_df.loc[
                target_df["site_index"] == site_index,
                "initial_energy_eV"
            ].iloc[0]
        ),

        "original_final_energy_eV": float(
            target_df.loc[
                target_df["site_index"] == site_index,
                "final_energy_eV"
            ].iloc[0]
        ),

        "original_initial_max_force_eV_A": float(
            target_df.loc[
                target_df["site_index"] == site_index,
                "initial_max_force_eV_A"
            ].iloc[0]
        ),

        "original_final_max_force_eV_A": float(
            target_df.loc[
                target_df["site_index"] == site_index,
                "final_max_force_eV_A"
            ].iloc[0]
        ),

        "original_Li_displacement_A": float(
            target_df.loc[
                target_df["site_index"] == site_index,
                "Li_displacement_A"
            ].iloc[0]
        ),

        "original_optimizer_steps": int(
            target_df.loc[
                target_df["site_index"] == site_index,
                "optimizer_steps"
            ].iloc[0]
        ),

        "original_converged": bool(
            target_df.loc[
                target_df["site_index"] == site_index,
                "converged"
            ].iloc[0]
        ),

        "new_initial_energy_eV": float(
            initial_energy
        ),

        "new_final_energy_eV": float(
            final_energy
        ),

        "new_energy_change_eV": float(
            energy_change
        ),

        "new_initial_max_force_eV_A": float(
            initial_max_force
        ),

        "new_final_max_force_eV_A": float(
            final_max_force
        ),

        "new_Li_displacement_A": float(
            li_displacement
        ),

        "new_optimizer_steps": int(
            optimizer_steps
        ),

        "new_converged": bool(
            converged
        ),
    }, atoms


# =============================================================================
# 14. RUN SELECTED SITES
# =============================================================================

print()
print("=" * 100)
print("STARTING CONTROLLED RE-RELAXATION")
print("=" * 100)

results = []

# New EXTXYZ is created independently.
if os.path.exists(OUTPUT_EXTXYZ):
    os.remove(OUTPUT_EXTXYZ)

for counter, site_index in enumerate(
    TARGET_SITES,
    start=1
):

    print()
    print("-" * 100)
    print(
        f"[{counter}/{len(TARGET_SITES)}] "
        f"Site {site_index}"
    )
    print("-" * 100)

    result, relaxed_atoms = relax_one_site(
        site_index
    )

    results.append(result)

    write(
        OUTPUT_EXTXYZ,
        relaxed_atoms,
        format="extxyz",
        append=True
    )

    print(
        f"Original converged : "
        f"{result['original_converged']}"
    )

    print(
        f"New converged      : "
        f"{result['new_converged']}"
    )

    print(
        f"Initial energy     : "
        f"{result['new_initial_energy_eV']:.8f} eV"
    )

    print(
        f"Final energy       : "
        f"{result['new_final_energy_eV']:.8f} eV"
    )

    print(
        f"Final max force    : "
        f"{result['new_final_max_force_eV_A']:.8f} eV/A"
    )

    print(
        f"Li displacement     : "
        f"{result['new_Li_displacement_A']:.8f} A"
    )

    print(
        f"Optimizer steps     : "
        f"{result['new_optimizer_steps']}"
    )


# =============================================================================
# 15. SAVE COMPARISON CSV
# =============================================================================

results_df = pd.DataFrame(results)

results_df.to_csv(
    OUTPUT_CSV,
    index=False
)


# =============================================================================
# 16. FINAL COMPARISON
# =============================================================================

print()
print("=" * 100)
print("FINAL ORIGINAL vs NEW COMPARISON")
print("=" * 100)

comparison_columns = [
    "site_index",
    "natural_clearance_A",

    "original_final_energy_eV",
    "new_final_energy_eV",

    "original_final_max_force_eV_A",
    "new_final_max_force_eV_A",

    "original_Li_displacement_A",
    "new_Li_displacement_A",

    "original_optimizer_steps",
    "new_optimizer_steps",

    "original_converged",
    "new_converged",
]

print(
    results_df[
        comparison_columns
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)


# =============================================================================
# 17. SCIENTIFIC INTERPRETATION FLAGS
# =============================================================================

print()
print("=" * 100)
print("INTERPRETATION FLAGS")
print("=" * 100)

for _, row in results_df.iterrows():

    site = int(row["site_index"])

    old_conv = bool(
        row["original_converged"]
    )

    new_conv = bool(
        row["new_converged"]
    )

    old_force = row[
        "original_final_max_force_eV_A"
    ]

    new_force = row[
        "new_final_max_force_eV_A"
    ]

    old_energy = row[
        "original_final_energy_eV"
    ]

    new_energy = row[
        "new_final_energy_eV"
    ]

    print(f"\nSite {site}")

    if (
        not old_conv
        and new_conv
    ):
        print(
            "  RESULT: Previously failed -> "
            "converged under the controlled re-relaxation."
        )

    elif (
        not old_conv
        and not new_conv
    ):
        print(
            "  RESULT: Still not converged."
        )

    else:
        print(
            "  RESULT: Original status was already converged."
        )

    print(
        f"  Force change : "
        f"{old_force:.6e} -> {new_force:.6e} eV/A"
    )

    print(
        f"  Energy change: "
        f"{old_energy:.8f} -> {new_energy:.8f} eV"
    )


# =============================================================================
# 18. FINAL OUTPUT
# =============================================================================

print()
print("=" * 100)
print("CONTROLLED RE-RELAXATION COMPLETE")
print("=" * 100)

print()
print("Saved comparison CSV:")
print(OUTPUT_CSV)

print()
print("Saved relaxed structures:")
print(OUTPUT_EXTXYZ)

print()
print("Original screening files were NOT modified.")

print()
print("No change was made to:")
print("- Voronoi site generation")
print("- MACE model")
print("- dispersion setting")
print("- dtype")
print("- BFGS optimizer")
print("- fixed-cell condition")
print("- force threshold")
print("- maximum step count")
print("- Li-host cutoff (none)")
print("- artificial distance criterion (none)")

print("=" * 100)

In [ ]:
# ============================================================
# CANDIDATE 81
# LI-SITE LANDSCAPE — BASIN / SITE CLUSTERING
#
# PURPOSE:
#   Convert the converged Voronoi-site results into a physical
#   Li-site landscape by identifying distinct relaxed Li basins.
#
# IMPORTANT:
#   READ-ONLY ANALYSIS.
#
#   NO MACE
#   NO RELAXATION
#   NO STRUCTURE MODIFICATION
#   NO CHANGE TO ORIGINAL SCREENING RESULTS
#
# METHOD:
#   1. Load the original 866-site screening CSV
#   2. Select converged sites
#   3. Read final Li positions from the screening EXTXYZ
#   4. Convert final Li positions to fractional coordinates
#   5. Apply periodic minimum-image distances
#   6. Cluster spatially equivalent final Li positions
#   7. Build distinct Li basins
#   8. Calculate basin energies and basin-to-basin distances
# ============================================================

import os
import numpy as np
import pandas as pd

from ase.io import read


# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_Li_single_site_energy_screening"
)

CSV_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_Li_site_energy_summary.csv"
)

EXTXYZ_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_all_tested_Li_sites_results.extxyz"
)

OUTPUT_BASINS = os.path.join(
    BASE_DIR,
    "candidate_81_Li_site_basins.csv"
)

OUTPUT_SITE_MAP = os.path.join(
    BASE_DIR,
    "candidate_81_Li_site_to_basin.csv"
)

OUTPUT_DISTANCES = os.path.join(
    BASE_DIR,
    "candidate_81_Li_basin_distances.csv"
)


# ============================================================
# 2. ANALYSIS PARAMETERS
# ============================================================

# Spatial tolerance used to identify the same relaxed basin.
#
# IMPORTANT:
# This is NOT a physical diffusion distance.
# It is only a clustering tolerance.
#
# Start conservatively.
BASIN_DISTANCE_CUTOFF_A = 0.50

# Energy tolerance is NOT used to merge basins.
#
# Two positions are considered the same basin primarily from
# their spatial location.
#
# Energies are reported afterward.
#
# This prevents two physically distinct sites with similar
# energies from being incorrectly merged.

MIN_SITES_PER_BASIN = 1


# ============================================================
# 3. HEADER
# ============================================================

print("=" * 80)
print("CANDIDATE 81 — LI-SITE LANDSCAPE / BASIN CLUSTERING")
print("=" * 80)

print()
print("Purpose:")
print("Convert converged Voronoi starting sites into distinct")
print("relaxed Li-site basins using periodic spatial clustering.")
print()
print("READ-ONLY ANALYSIS:")
print("No MACE")
print("No BFGS relaxation")
print("No structure modification")
print("Original 866-site screening remains untouched.")


# ============================================================
# 4. LOAD CSV
# ============================================================

print("\n" + "=" * 80)
print("[1] LOADING SCREENING CSV")
print("=" * 80)

df = pd.read_csv(CSV_FILE)

print(f"Total CSV rows: {len(df)}")


# ============================================================
# 5. VALIDATE CSV
# ============================================================

required_columns = [
    "site_index",
    "final_x",
    "final_y",
    "final_z",
    "final_energy_eV",
    "converged",
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        "Missing required CSV columns:\n"
        + "\n".join(missing)
    )

if df["site_index"].duplicated().any():
    duplicates = df.loc[
        df["site_index"].duplicated(),
        "site_index"
    ].tolist()

    raise ValueError(
        f"Duplicate site_index values found: {duplicates}"
    )

print("PASS: Required columns found.")
print("PASS: No duplicate site_index values.")


# ============================================================
# 6. BASIC SCREENING STATISTICS
# ============================================================

n_total = len(df)

converged_df = df[
    df["converged"] == True
].copy()

failed_df = df[
    df["converged"] != True
].copy()

print("\n" + "=" * 80)
print("SCREENING STATUS")
print("=" * 80)

print(f"Total sites      : {n_total}")
print(f"Converged sites  : {len(converged_df)}")
print(f"Non-converged    : {len(failed_df)}")

if len(converged_df) == 0:
    raise RuntimeError(
        "No converged sites are available for basin analysis."
    )


# ============================================================
# 7. LOAD EXTXYZ
# ============================================================

print("\n" + "=" * 80)
print("[2] LOADING FINAL STRUCTURES")
print("=" * 80)

frames = read(
    EXTXYZ_FILE,
    index=":"
)

print(f"EXTXYZ structures/frames: {len(frames)}")


# ============================================================
# 8. CHECK EXTXYZ COVERAGE
# ============================================================

print("\n" + "=" * 80)
print("[3] CHECKING EXTXYZ SITE COVERAGE")
print("=" * 80)

extxyz_site_map = {}

for frame_number, atoms in enumerate(frames):

    info = atoms.info

    site_index = None

    if "voronoi_site_index" in info:
        site_index = int(info["voronoi_site_index"])

    elif "site_index" in info:
        site_index = int(info["site_index"])

    elif "test_number" in info:
        # Fallback only.
        # We do NOT assume test_number == site_index.
        print(
            f"WARNING: frame {frame_number} has "
            f"test_number but no site index."
        )

    if site_index is not None:

        if site_index in extxyz_site_map:
            raise ValueError(
                f"Duplicate EXTXYZ site_index found: {site_index}"
            )

        extxyz_site_map[site_index] = (
            frame_number,
            atoms
        )


print(
    f"Unique site indices in EXTXYZ: "
    f"{len(extxyz_site_map)}"
)


# ============================================================
# 9. VERIFY CONVERGED SITE COVERAGE
# ============================================================

converged_site_indices = set(
    converged_df["site_index"].astype(int)
)

extxyz_site_indices = set(
    extxyz_site_map.keys()
)

missing_converged = sorted(
    converged_site_indices - extxyz_site_indices
)

if missing_converged:

    print("\nWARNING:")
    print(
        "Some converged CSV sites are missing from EXTXYZ:"
    )

    print(missing_converged)

    raise ValueError(
        "Cannot safely perform basin analysis because "
        "some converged sites have no final structure."
    )

print(
    "PASS: Every converged CSV site has a corresponding "
    "EXTXYZ structure."
)


# ============================================================
# 10. DETERMINE REFERENCE CELL
# ============================================================

print("\n" + "=" * 80)
print("[4] REFERENCE CELL")
print("=" * 80)

reference_site = int(
    converged_df.iloc[0]["site_index"]
)

reference_atoms = extxyz_site_map[
    reference_site
][1]

cell = np.asarray(
    reference_atoms.get_cell()
)

if cell.shape != (3, 3):
    raise ValueError(
        f"Unexpected cell shape: {cell.shape}"
    )

if not reference_atoms.pbc.all():
    raise ValueError(
        "Reference structure does not have full PBC."
    )

volume = reference_atoms.get_volume()

print("Reference site :", reference_site)
print("Cell:")
print(cell)

print(f"Cell volume     : {volume:.8f} A^3")
print(
    f"PBC             : "
    f"{reference_atoms.pbc.tolist()}"
)


# ============================================================
# 11. VERIFY CELL CONSISTENCY
# ============================================================

print("\nChecking cell consistency across converged frames...")

cell_tolerance = 1e-8

for site_index in converged_site_indices:

    atoms = extxyz_site_map[site_index][1]

    current_cell = np.asarray(
        atoms.get_cell()
    )

    if not np.allclose(
        current_cell,
        cell,
        atol=cell_tolerance,
        rtol=0.0
    ):

        raise ValueError(
            f"Cell mismatch detected for site "
            f"{site_index}."
        )

print(
    "PASS: All converged structures use the same cell "
    "within tolerance."
)


# ============================================================
# 12. COORDINATE CONVERSION
# ============================================================

print("\n" + "=" * 80)
print("[5] EXTRACTING FINAL LI POSITIONS")
print("=" * 80)

records = []

for _, row in converged_df.iterrows():

    site_index = int(
        row["site_index"]
    )

    atoms = extxyz_site_map[
        site_index
    ][1]

    symbols = atoms.get_chemical_symbols()

    li_indices = [
        i
        for i, symbol in enumerate(symbols)
        if symbol == "Li"
    ]

    if len(li_indices) != 1:
        raise ValueError(
            f"Site {site_index}: expected exactly "
            f"one Li atom, found {len(li_indices)}."
        )

    li_index = li_indices[0]

    cartesian = np.asarray(
        atoms.positions[li_index],
        dtype=float
    )

    scaled = np.asarray(
        atoms.get_scaled_positions(
            wrap=False
        )[li_index],
        dtype=float
    )

    # Wrap into the primary periodic cell.
    fractional = scaled % 1.0

    records.append({

        "site_index":
            site_index,

        "final_x_A":
            cartesian[0],

        "final_y_A":
            cartesian[1],

        "final_z_A":
            cartesian[2],

        "frac_x":
            fractional[0],

        "frac_y":
            fractional[1],

        "frac_z":
            fractional[2],

        "final_energy_eV":
            float(row["final_energy_eV"]),

        "final_force_eV_A":
            float(
                row["final_max_force_eV_A"]
            )
            if "final_max_force_eV_A" in row
            else np.nan,

        "initial_site_clearance_A":
            float(
                row["natural_clearance_A"]
            )
            if "natural_clearance_A" in row
            else np.nan,

        "initial_site_x_A":
            float(row["initial_x"]),

        "initial_site_y_A":
            float(row["initial_y"]),

        "initial_site_z_A":
            float(row["initial_z"]),

        "li_displacement_A":
            float(
                row["Li_displacement_A"]
            )
            if "Li_displacement_A" in row
            else np.nan,

        "optimizer_steps":
            int(
                row["optimizer_steps"]
            )
    })


site_df = pd.DataFrame(records)

print(
    f"Converged final Li positions extracted: "
    f"{len(site_df)}"
)


# ============================================================
# 13. PERIODIC DISTANCE FUNCTION
# ============================================================

def periodic_distance(
    frac_a,
    frac_b,
    cell_matrix
):
    """
    Calculate minimum-image distance between two
    fractional-coordinate positions under PBC.
    """

    delta_frac = (
        np.asarray(frac_a)
        - np.asarray(frac_b)
    )

    # Minimum-image displacement in fractional space.
    delta_frac -= np.round(
        delta_frac
    )

    # Convert fractional displacement to Cartesian.
    delta_cart = (
        delta_frac @ cell_matrix
    )

    return float(
        np.linalg.norm(delta_cart)
    )


# ============================================================
# 14. BUILD PAIRWISE DISTANCE MATRIX
# ============================================================

print("\n" + "=" * 80)
print("[6] BUILDING PERIODIC DISTANCE MATRIX")
print("=" * 80)

site_indices = (
    site_df["site_index"]
    .astype(int)
    .to_numpy()
)

fractional_positions = (
    site_df[
        [
            "frac_x",
            "frac_y",
            "frac_z"
        ]
    ]
    .to_numpy()
)

n_sites = len(site_df)

distance_matrix = np.zeros(
    (n_sites, n_sites),
    dtype=float
)

for i in range(n_sites):

    for j in range(i + 1, n_sites):

        d = periodic_distance(
            fractional_positions[i],
            fractional_positions[j],
            cell
        )

        distance_matrix[i, j] = d
        distance_matrix[j, i] = d

print(
    f"Distance matrix constructed: "
    f"{n_sites} x {n_sites}"
)


# ============================================================
# 15. CONNECTED-COMPONENT CLUSTERING
# ============================================================
#
# Two relaxed positions are connected when their periodic
# distance <= BASIN_DISTANCE_CUTOFF_A.
#
# Connected components define the preliminary basins.
#
# IMPORTANT:
# This is intentionally transparent and deterministic.
# We are NOT using an opaque ML clustering algorithm.
# ============================================================

print("\n" + "=" * 80)
print("[7] BASIN CLUSTERING")
print("=" * 80)

print(
    f"Spatial clustering cutoff : "
    f"{BASIN_DISTANCE_CUTOFF_A:.3f} A"
)

parent = np.arange(
    n_sites,
    dtype=int
)


def find_root(
    x
):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]

    return x


def union(
    a,
    b
):
    root_a = find_root(a)
    root_b = find_root(b)

    if root_a != root_b:
        parent[root_b] = root_a


for i in range(n_sites):

    for j in range(i + 1, n_sites):

        if (
            distance_matrix[i, j]
            <= BASIN_DISTANCE_CUTOFF_A
        ):
            union(i, j)


roots = np.array([
    find_root(i)
    for i in range(n_sites)
])


# Convert arbitrary roots into consecutive basin IDs.
unique_roots = sorted(
    np.unique(roots)
)

root_to_basin = {
    root: basin_id + 1
    for basin_id, root in enumerate(unique_roots)
}

basin_ids = np.array([
    root_to_basin[root]
    for root in roots
])

site_df["basin_id"] = basin_ids


print(
    f"Distinct preliminary basins: "
    f"{len(unique_roots)}"
)


# ============================================================
# 16. BUILD BASIN SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("[8] BUILDING BASIN SUMMARY")
print("=" * 80)

basin_records = []

global_min_energy = (
    site_df["final_energy_eV"].min()
)

for basin_id, group in site_df.groupby(
    "basin_id",
    sort=True
):

    group = group.copy()

    # Lowest-energy member of the basin.
    best_idx = group[
        "final_energy_eV"
    ].idxmin()

    best = group.loc[
        best_idx
    ]

    # Choose the lowest-energy relaxed site as the
    # representative of this basin.
    representative_site = int(
        best["site_index"]
    )

    representative_frac = np.array([
        best["frac_x"],
        best["frac_y"],
        best["frac_z"]
    ])

    # Distances from the representative to all
    # members of the same basin.
    internal_distances = []

    for _, member in group.iterrows():

        member_frac = np.array([
            member["frac_x"],
            member["frac_y"],
            member["frac_z"]
        ])

        internal_distances.append(
            periodic_distance(
                representative_frac,
                member_frac,
                cell
            )
        )

    basin_records.append({

        "basin_id":
            int(basin_id),

        "n_starting_sites":
            int(len(group)),

        "representative_site_index":
            representative_site,

        "representative_frac_x":
            representative_frac[0],

        "representative_frac_y":
            representative_frac[1],

        "representative_frac_z":
            representative_frac[2],

        "representative_energy_eV":
            float(
                best["final_energy_eV"]
            ),

        "relative_energy_eV":
            float(
                best["final_energy_eV"]
                - global_min_energy
            ),

        "basin_mean_energy_eV":
            float(
                group["final_energy_eV"].mean()
            ),

        "basin_energy_std_eV":
            float(
                group["final_energy_eV"].std()
                if len(group) > 1
                else 0.0
            ),

        "basin_min_energy_eV":
            float(
                group["final_energy_eV"].min()
            ),

        "basin_max_energy_eV":
            float(
                group["final_energy_eV"].max()
            ),

        "basin_internal_spread_A":
            float(
                max(internal_distances)
                if internal_distances
                else 0.0
            ),

        "mean_final_force_eV_A":
            float(
                group["final_force_eV_A"].mean()
            ),

        "max_final_force_eV_A":
            float(
                group["final_force_eV_A"].max()
            ),

        "mean_li_displacement_A":
            float(
                group["li_displacement_A"].mean()
            ),

        "min_clearance_A":
            float(
                group[
                    "initial_site_clearance_A"
                ].min()
            ),

        "max_clearance_A":
            float(
                group[
                    "initial_site_clearance_A"
                ].max()
            )
    })


basin_df = pd.DataFrame(
    basin_records
)

basin_df = basin_df.sort_values(
    [
        "relative_energy_eV",
        "basin_id"
    ]
).reset_index(
    drop=True
)

# Rank basins by relative energy.
basin_df[
    "energy_rank"
] = np.arange(
    1,
    len(basin_df) + 1
)


# ============================================================
# 17. REASSIGN ENERGY RANK AFTER SORTING
# ============================================================

basin_rank_map = dict(
    zip(
        basin_df["basin_id"],
        basin_df["energy_rank"]
    )
)

site_df[
    "basin_energy_rank"
] = site_df[
    "basin_id"
].map(
    basin_rank_map
)


# ============================================================
# 18. BASIN-TO-BASIN DISTANCES
# ============================================================

print("\n" + "=" * 80)
print("[9] BASIN-TO-BASIN DISTANCES")
print("=" * 80)

distance_records = []

for i in range(
    len(basin_df)
):

    basin_a = int(
        basin_df.iloc[i]["basin_id"]
    )

    frac_a = np.array([
        basin_df.iloc[i][
            "representative_frac_x"
        ],
        basin_df.iloc[i][
            "representative_frac_y"
        ],
        basin_df.iloc[i][
            "representative_frac_z"
        ]
    ])

    for j in range(
        i + 1,
        len(basin_df)
    ):

        basin_b = int(
            basin_df.iloc[j]["basin_id"]
        )

        frac_b = np.array([
            basin_df.iloc[j][
                "representative_frac_x"
            ],
            basin_df.iloc[j][
                "representative_frac_y"
            ],
            basin_df.iloc[j][
                "representative_frac_z"
            ]
        ])

        distance_A = periodic_distance(
            frac_a,
            frac_b,
            cell
        )

        energy_a = float(
            basin_df.iloc[i][
                "representative_energy_eV"
            ]
        )

        energy_b = float(
            basin_df.iloc[j][
                "representative_energy_eV"
            ]
        )

        distance_records.append({

            "basin_a":
                basin_a,

            "basin_b":
                basin_b,

            "distance_A":
                distance_A,

            "energy_a_eV":
                energy_a,

            "energy_b_eV":
                energy_b,

            "energy_difference_eV":
                abs(
                    energy_a
                    - energy_b
                )
        })


distance_df = pd.DataFrame(
    distance_records
)

if len(distance_df) > 0:

    distance_df = distance_df.sort_values(
        "distance_A"
    ).reset_index(
        drop=True
    )


# ============================================================
# 19. BASIN SIZE / OCCUPANCY STATISTICS
# ============================================================

print("\n" + "=" * 80)
print("[10] LANDSCAPE SUMMARY")
print("=" * 80)

print(
    f"Original Voronoi sites      : "
    f"{n_total}"
)

print(
    f"Converged relaxed sites     : "
    f"{len(site_df)}"
)

print(
    f"Distinct relaxed basins     : "
    f"{len(basin_df)}"
)

print(
    f"Non-converged sites         : "
    f"{len(failed_df)}"
)

print(
    f"Basin clustering cutoff     : "
    f"{BASIN_DISTANCE_CUTOFF_A:.3f} A"
)


# ============================================================
# 20. PRINT BASIN TABLE
# ============================================================

print("\n" + "=" * 80)
print("DISTINCT LI BASINS")
print("=" * 80)

summary_columns = [
    "energy_rank",
    "basin_id",
    "n_starting_sites",
    "representative_site_index",
    "representative_energy_eV",
    "relative_energy_eV",
    "basin_energy_std_eV",
    "basin_internal_spread_A",
]

print(
    basin_df[
        summary_columns
    ].to_string(
        index=False,
        float_format=lambda x:
            f"{x:.6f}"
    )
)


# ============================================================
# 21. SAVE SITE -> BASIN MAP
# ============================================================

site_df = site_df.sort_values(
    [
        "basin_energy_rank",
        "basin_id",
        "final_energy_eV"
    ]
).reset_index(
    drop=True
)

site_df.to_csv(
    OUTPUT_SITE_MAP,
    index=False
)


# ============================================================
# 22. SAVE BASIN TABLE
# ============================================================

basin_df.to_csv(
    OUTPUT_BASINS,
    index=False
)


# ============================================================
# 23. SAVE BASIN DISTANCES
# ============================================================

distance_df.to_csv(
    OUTPUT_DISTANCES,
    index=False
)


# ============================================================
# 24. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 80)
print("[11] VALIDATION")
print("=" * 80)

# Every converged site must belong to exactly one basin.
if (
    site_df["basin_id"].isna().any()
):
    raise RuntimeError(
        "Some converged sites have no basin assignment."
    )

# Every basin must have at least one site.
if (
    basin_df["n_starting_sites"] < 1
).any():
    raise RuntimeError(
        "Invalid empty basin detected."
    )

# Total basin occupancy must equal number of converged sites.
if (
    basin_df["n_starting_sites"].sum()
    != len(site_df)
):
    raise RuntimeError(
        "Basin occupancy does not match "
        "number of converged sites."
    )

print(
    "PASS: Every converged site belongs to one basin."
)

print(
    "PASS: No empty basins."
)

print(
    "PASS: Basin occupancy equals converged-site count."
)


# ============================================================
# 25. OUTPUT PATHS
# ============================================================

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

print("\nSaved basin summary:")
print(OUTPUT_BASINS)

print("\nSaved site-to-basin mapping:")
print(OUTPUT_SITE_MAP)

print("\nSaved basin-to-basin distances:")
print(OUTPUT_DISTANCES)

print("\nIMPORTANT:")
print("This analysis is READ-ONLY.")
print("No MACE calculation was performed.")
print("No BFGS relaxation was performed.")
print("No structure was modified.")
print("No original CSV result was modified.")
print("No screening parameter was changed.")

In [ ]:
# ============================================================
# Candidate 81
# Random 5 Converged + Random 5 Failed
# Independent Re-Relaxation Validation
#
# IMPORTANT:
# - Original results are NEVER modified.
# - Same MACE / D3 / BFGS / fixed-cell protocol.
# - Uses ORIGINAL initial Li coordinates.
# ============================================================

import os
import random
import numpy as np
import pandas as pd

from ase import Atoms
from ase.io import read, write
from ase.optimize import BFGS
from mace.calculators import mace_mp


# ============================================================
# 1. PATHS
# ============================================================

BASE_DIR = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive"
)

HOST_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_40atom_relaxed_host.extxyz"
)

CSV_FILE = os.path.join(
    BASE_DIR,
    "candidate_81_Li_single_site_energy_screening",
    "candidate_81_Li_site_energy_summary.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "candidate_81_random_5plus5_re_relaxation"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 2. SETTINGS
# ============================================================

N_SUCCESSFUL = 5
N_FAILED = 5

RANDOM_SEED = 20261002

FMAX = 0.01
MAX_STEPS = 1000

DEVICE = "cuda"
DTYPE = "float64"

# Previous failures that were already independently re-relaxed.
# We exclude them so that the new random test is genuinely new.
PREVIOUSLY_RERELAXED_FAILED = {
    461,
    101,
    227,
    812,
    107,
    178,
}


# ============================================================
# 3. RANDOM NUMBER GENERATOR
# ============================================================

rng = random.Random(RANDOM_SEED)


# ============================================================
# 4. LOAD ORIGINAL CSV
# ============================================================

print("=" * 70)
print("CANDIDATE 81 — RANDOM 5 + 5 RE-RELAXATION VALIDATION")
print("=" * 70)

print("\nLoading original CSV...")

df = pd.read_csv(CSV_FILE)

print(f"Total rows in original CSV : {len(df)}")


# ============================================================
# 5. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "site_index",
    "initial_x",
    "initial_y",
    "initial_z",
    "initial_energy_eV",
    "final_energy_eV",
    "final_max_force_eV_A",
    "converged",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise RuntimeError(
        f"Missing required CSV columns: {missing}"
    )


# ============================================================
# 6. NORMALIZE CONVERGED COLUMN
# ============================================================

def normalize_bool(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, (int, np.integer)):
        return bool(value)

    if isinstance(value, str):
        return value.strip().lower() in {
            "true",
            "1",
            "yes",
            "y",
        }

    return False


df["converged_bool"] = df["converged"].apply(normalize_bool)


# ============================================================
# 7. SEPARATE SUCCESSFUL / FAILED
# ============================================================

successful = df[df["converged_bool"]].copy()

failed = df[~df["converged_bool"]].copy()

print(f"Converged sites : {len(successful)}")
print(f"Failed sites    : {len(failed)}")


# ============================================================
# 8. REMOVE PREVIOUSLY RE-RELAXED FAILURES
# ============================================================

failed_new_pool = failed[
    ~failed["site_index"].isin(PREVIOUSLY_RERELAXED_FAILED)
].copy()

print(
    "\nFailed sites available for NEW random test "
    f"(after excluding previous 6): {len(failed_new_pool)}"
)


# ============================================================
# 9. RANDOM SELECTION
# ============================================================

if len(successful) < N_SUCCESSFUL:
    raise RuntimeError(
        f"Not enough successful sites: {len(successful)}"
    )

if len(failed_new_pool) < N_FAILED:
    raise RuntimeError(
        f"Not enough new failed sites: {len(failed_new_pool)}"
    )


selected_successful = rng.sample(
    successful["site_index"].tolist(),
    N_SUCCESSFUL
)

selected_failed = rng.sample(
    failed_new_pool["site_index"].tolist(),
    N_FAILED
)

selected_sites = selected_successful + selected_failed


print("\n" + "=" * 70)
print("RANDOMLY SELECTED SITES")
print("=" * 70)

print("\nSuccessful:")
print(selected_successful)

print("\nFailed:")
print(selected_failed)

print("\nAll selected:")
print(selected_sites)


# ============================================================
# 10. SAVE SELECTION BEFORE RUNNING
# ============================================================

selection_rows = []

for site in selected_successful:
    selection_rows.append({
        "site_index": site,
        "original_status": "converged",
    })

for site in selected_failed:
    selection_rows.append({
        "site_index": site,
        "original_status": "failed",
    })

selection_df = pd.DataFrame(selection_rows)

selection_file = os.path.join(
    OUTPUT_DIR,
    "random_5_converged_plus_5_failed_selection.csv"
)

selection_df.to_csv(
    selection_file,
    index=False
)

print(
    f"\nSelection saved to:\n{selection_file}"
)


# ============================================================
# 11. LOAD HOST
# ============================================================

print("\nLoading host structure...")

host = read(HOST_FILE)

print(f"Host atoms : {len(host)}")
print(f"Host volume: {host.get_volume():.6f} Å³")
print(f"PBC        : {host.get_pbc()}")


# ============================================================
# 12. VALIDATE HOST
# ============================================================

if len(host) != 40:
    raise RuntimeError(
        f"Host atom count is {len(host)}, expected 40."
    )

if not np.all(host.get_pbc()):
    raise RuntimeError(
        "Host is not fully periodic."
    )


# ============================================================
# 13. CREATE MACE CALCULATOR
# ============================================================

print("\nLoading MACE calculator...")

calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype=DTYPE,
    device=DEVICE,
)

print("MACE loaded.")
print(f"Device : {DEVICE}")
print(f"Dtype  : {DTYPE}")
print("D3     : enabled")


# ============================================================
# 14. RE-LAXATION FUNCTION
# ============================================================

def relax_single_site(row, original_status):

    site_index = int(row["site_index"])

    print("\n" + "=" * 70)
    print(
        f"SITE {site_index} | ORIGINAL STATUS: {original_status}"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # Original initial Li position
    # --------------------------------------------------------

    li_position = np.array([
        row["initial_x"],
        row["initial_y"],
        row["initial_z"],
    ], dtype=float)

    print(
        "Initial Li position:",
        li_position
    )

    # --------------------------------------------------------
    # Build host + one Li
    # --------------------------------------------------------

    atoms = host.copy()

    atoms += Atoms(
        "Li",
        positions=[li_position],
        cell=atoms.get_cell(),
        pbc=atoms.get_pbc(),
    )

    # Fixed cell
    atoms.set_cell(host.get_cell())
    atoms.set_pbc(host.get_pbc())

    # Same calculator
    atoms.calc = calc

    # --------------------------------------------------------
    # Initial energy
    # --------------------------------------------------------

    initial_energy = atoms.get_potential_energy()

    initial_forces = atoms.get_forces()

    initial_max_force = np.max(
        np.linalg.norm(initial_forces, axis=1)
    )

    print(
        f"Initial energy      : {initial_energy:.10f} eV"
    )

    print(
        f"Initial max force   : {initial_max_force:.10f} eV/Å"
    )

    # --------------------------------------------------------
    # BFGS
    # Fixed cell automatically because we optimize Atoms,
    # not a cell filter.
    # --------------------------------------------------------

    trajectory_file = os.path.join(
        OUTPUT_DIR,
        f"site_{site_index}_relaxation.traj"
    )

    optimizer = BFGS(
        atoms,
        trajectory=trajectory_file,
        logfile=None,
    )

    optimizer.run(
        fmax=FMAX,
        steps=MAX_STEPS,
    )

    # --------------------------------------------------------
    # Final properties
    # --------------------------------------------------------

    final_energy = atoms.get_potential_energy()

    final_forces = atoms.get_forces()

    final_max_force = np.max(
        np.linalg.norm(final_forces, axis=1)
    )

    final_li_position = atoms.positions[-1].copy()

    displacement = np.linalg.norm(
        final_li_position - li_position
    )

    # --------------------------------------------------------
    # Convergence
    # --------------------------------------------------------

    converged = final_max_force <= FMAX

    optimizer_steps = optimizer.nsteps

    # --------------------------------------------------------
    # Nearest host atom
    # --------------------------------------------------------

    distances = atoms.get_distances(
        -1,
        np.arange(len(host)),
        mic=True,
    )

    nearest_idx = int(np.argmin(distances))
    nearest_distance = float(distances[nearest_idx])
    nearest_element = host[nearest_idx].symbol

    # --------------------------------------------------------
    # Energy change
    # --------------------------------------------------------

    energy_change = (
        final_energy - initial_energy
    )

    # --------------------------------------------------------
    # Save final structure
    # --------------------------------------------------------

    final_extxyz = os.path.join(
        OUTPUT_DIR,
        f"site_{site_index}_final.extxyz"
    )

    write(
        final_extxyz,
        atoms,
        format="extxyz",
    )

    print("\nRESULT")
    print("-" * 50)

    print(
        f"Final energy        : {final_energy:.10f} eV"
    )

    print(
        f"Final max force     : {final_max_force:.10f} eV/Å"
    )

    print(
        f"Optimizer steps     : {optimizer_steps}"
    )

    print(
        f"Converged           : {converged}"
    )

    print(
        f"Li displacement     : {displacement:.10f} Å"
    )

    print(
        f"Nearest host        : "
        f"{nearest_element} #{nearest_idx}"
    )

    print(
        f"Nearest distance    : "
        f"{nearest_distance:.10f} Å"
    )

    print(
        f"Energy change       : "
        f"{energy_change:.10f} eV"
    )

    return {
        "site_index": site_index,
        "original_status": original_status,

        "initial_x": li_position[0],
        "initial_y": li_position[1],
        "initial_z": li_position[2],

        "new_final_x": final_li_position[0],
        "new_final_y": final_li_position[1],
        "new_final_z": final_li_position[2],

        "initial_energy_eV": initial_energy,
        "new_final_energy_eV": final_energy,

        "energy_change_eV": energy_change,

        "initial_max_force_eV_A": initial_max_force,
        "new_final_max_force_eV_A": final_max_force,

        "Li_displacement_A": displacement,

        "final_nearest_host_distance_A":
            nearest_distance,

        "final_nearest_host_element":
            nearest_element,

        "final_nearest_host_index":
            nearest_idx,

        "optimizer_steps":
            optimizer_steps,

        "new_converged":
            converged,
    }


# ============================================================
# 15. RUN THE 10 TESTS
# ============================================================

results = []

selected_df = df[
    df["site_index"].isin(selected_sites)
].copy()

# Preserve the intended order:
# 5 successful first, then 5 failed.
ordered_sites = selected_successful + selected_failed

for site_index in ordered_sites:

    row = selected_df[
        selected_df["site_index"] == site_index
    ].iloc[0]

    original_status = (
        "converged"
        if bool(row["converged_bool"])
        else "failed"
    )

    try:

        result = relax_single_site(
            row,
            original_status,
        )

        results.append(result)

    except Exception as exc:

        print(
            f"\n!!! ERROR at site {site_index} !!!"
        )

        print(exc)

        results.append({
            "site_index": site_index,
            "original_status": original_status,
            "new_converged": False,
            "error": str(exc),
        })


# ============================================================
# 16. SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)

results_file = os.path.join(
    OUTPUT_DIR,
    "random_5plus5_re_relaxation_results.csv"
)

results_df.to_csv(
    results_file,
    index=False
)


# ============================================================
# 17. FINAL SUMMARY
# ============================================================

print("\n\n" + "=" * 70)
print("FINAL VALIDATION SUMMARY")
print("=" * 70)

print(
    f"\nResults saved to:\n{results_file}"
)

print(
    f"\nSuccessful originally : {N_SUCCESSFUL}"
)

print(
    f"Failed originally    : {N_FAILED}"
)

print("\nNew convergence:")

if len(results_df) > 0:

    for status in ["converged", "failed"]:

        subset = results_df[
            results_df["original_status"] == status
        ]

        if len(subset) == 0:
            continue

        n_new_converged = int(
            subset["new_converged"]
            .fillna(False)
            .astype(bool)
            .sum()
        )

        print(
            f"  Original {status:9s} → "
            f"{n_new_converged}/{len(subset)} converged"
        )


print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

In [ ]:
# =============================================================================
# CANDIDATE 81
# FINAL 117-BASIN FREEZE
# =============================================================================
#
# PURPOSE
# -------
# Create the FINAL, internally consistent 793-structure MASTER CSV
# and 117-basin SUMMARY CSV after the already-decided reassignment:
#
#       site 18 : basin 19 -> basin 80
#
# IMPORTANT
# ---------
# NO new clustering
# NO MACE
# NO relaxation
# NO new threshold
# NO deletion of structures
# NO change in number of basins
#
# The original input files are NOT overwritten.
# New FINAL files are written instead.
#
# =============================================================================

import os
import numpy as np
import pandas as pd

from ase.io import read


# =============================================================================
# CONTRACT
# =============================================================================
#
# INPUT
#   - Original 793-row MASTER CSV
#   - Original 117-row BASIN SUMMARY CSV
#   - Original 40-atom relaxed host
#
# PROCESS
#   - Copy MASTER
#   - Move only site 18: basin 19 -> basin 80
#   - Rebuild the 117-basin summary from the corrected MASTER
#   - Use the actual periodic host cell for PBC distances
#
# OUTPUT
#   - FINAL MASTER CSV
#   - FINAL BASIN SUMMARY CSV
#
# INVARIANTS
#   - Host atoms = 40
#   - MASTER rows = 793
#   - Basin count = 117
#   - Site count = 793 unique
#   - Only site 18 changes basin_id
#   - Site 18 changes 19 -> 80
#
# =============================================================================


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_validated_Li_basin_analysis"
)

HOST_FILE = (
    "/content/drive/MyDrive/Materials_Project/"
    "Top5_search_archive/generation_3_MACE_TOP5_archive/"
    "candidate_81_40atom_relaxed_host.extxyz"
)

MASTER_FILE = os.path.join(
    ROOT,
    "candidate_81_793_converged_master.csv"
)

BASIN_FILE = os.path.join(
    ROOT,
    "candidate_81_validated_Li_basins.csv"
)


# FINAL OUTPUTS
FINAL_MASTER_FILE = os.path.join(
    ROOT,
    "candidate_81_793_converged_master_FINAL_117BASINS.csv"
)

FINAL_BASIN_FILE = os.path.join(
    ROOT,
    "candidate_81_validated_Li_basins_FINAL_117.csv"
)


SITE_TO_MOVE = 18
OLD_BASIN = 19
NEW_BASIN = 80


# =============================================================================
# 2. LOAD
# =============================================================================

print("=" * 100)
print("CANDIDATE 81 — FINAL 117-BASIN FREEZE")
print("=" * 100)

host = read(HOST_FILE)

master_original = pd.read_csv(MASTER_FILE)
basins_original = pd.read_csv(BASIN_FILE)


# =============================================================================
# 3. INPUT VALIDATION
# =============================================================================

print("\nINPUT VALIDATION")
print("-" * 100)

print(f"Host atoms       : {len(host)}")
print(f"MASTER rows      : {len(master_original)}")
print(f"BASIN rows       : {len(basins_original)}")
print(
    f"Original basins  : "
    f"{basins_original['basin_id'].nunique()}"
)


assert len(host) == 40, (
    "Host must contain exactly 40 atoms."
)

assert len(master_original) == 793, (
    "MASTER must contain exactly 793 rows."
)

assert basins_original["basin_id"].nunique() == 117, (
    "Original BASIN summary must contain exactly 117 basins."
)


required_master = {
    "site_index",
    "final_x",
    "final_y",
    "final_z",
    "final_energy_eV",
    "basin_id",
}

missing_master = required_master - set(master_original.columns)

assert not missing_master, (
    f"MASTER is missing columns: {sorted(missing_master)}"
)


required_basin = {
    "basin_id",
    "population",
    "representative_site_index",
    "representative_source",
    "minimum_energy_eV",
    "maximum_energy_eV",
    "energy_range_eV",
    "energy_std_eV",
    "basin_diameter_A",
    "rms_spread_A",
    "representative_x",
    "representative_y",
    "representative_z",
    "energy_rank",
    "diameter_within_cutoff",
}

missing_basin = required_basin - set(basins_original.columns)

assert not missing_basin, (
    f"BASIN summary is missing columns: {sorted(missing_basin)}"
)


assert master_original["site_index"].is_unique, (
    "MASTER site_index must be unique."
)


# =============================================================================
# 4. CREATE FINAL MASTER
# =============================================================================
#
# IMPORTANT:
# ----------
# We work on a COPY.
# The original MASTER CSV is untouched.
#
# Only one value is changed:
#
#       site 18: 19 -> 80
#
# =============================================================================

master = master_original.copy()

site18_mask = master["site_index"] == SITE_TO_MOVE

assert site18_mask.sum() == 1, (
    "Site 18 must appear exactly once in MASTER."
)

old_value = int(
    master.loc[site18_mask, "basin_id"].iloc[0]
)

assert old_value == OLD_BASIN, (
    f"Expected site 18 to currently belong to basin "
    f"{OLD_BASIN}, found basin {old_value}."
)


print("\nSITE 18 REASSIGNMENT")
print("-" * 100)

print(f"Site index       : {SITE_TO_MOVE}")
print(f"Old basin        : {old_value}")
print(f"New basin        : {NEW_BASIN}")


master.loc[site18_mask, "basin_id"] = NEW_BASIN


# =============================================================================
# 5. MASTER INVARIANTS AFTER REASSIGNMENT
# =============================================================================

assert len(master) == 793

assert master["site_index"].is_unique

assert master["basin_id"].nunique() == 117

assert int(
    master.loc[
        master["site_index"] == SITE_TO_MOVE,
        "basin_id"
    ].iloc[0]
) == NEW_BASIN


# Verify that NO other site changed basin assignment
comparison = master_original[
    ["site_index", "basin_id"]
].merge(
    master[
        ["site_index", "basin_id"]
    ],
    on="site_index",
    suffixes=("_old", "_new")
)

changed_sites = comparison[
    comparison["basin_id_old"] != comparison["basin_id_new"]
]

assert len(changed_sites) == 1, (
    "More than one site changed basin assignment."
)

assert int(
    changed_sites.iloc[0]["site_index"]
) == SITE_TO_MOVE


print("\nPASS: Exactly one membership changed.")


# =============================================================================
# 6. PBC DISTANCE FUNCTION
# =============================================================================
#
# Same periodic-cell logic used in the focused site-18 analysis.
#
# =============================================================================

cell = host.get_cell().array


def pbc_distances(point, points, cell):
    """
    Minimum-image distances under the actual periodic cell.
    """

    point = np.asarray(point, dtype=float)
    points = np.asarray(points, dtype=float)

    delta = points - point

    fractional = delta @ np.linalg.inv(cell)

    fractional -= np.round(fractional)

    cartesian = fractional @ cell

    return np.linalg.norm(
        cartesian,
        axis=1
    )


# =============================================================================
# 7. BASIN SUMMARY REBUILD
# =============================================================================
#
# The MASTER is the source of membership.
#
# The BASIN CSV is only a summary.
#
# Therefore, after changing site 18, we rebuild the summary from MASTER.
#
# =============================================================================

print("\nREBUILDING FINAL 117-BASIN SUMMARY")
print("-" * 100)


def calculate_basin_diameter(group):
    """
    Maximum pairwise PBC distance among Li positions in the basin.
    """

    coords = group[
        ["final_x", "final_y", "final_z"]
    ].to_numpy(dtype=float)

    n = len(coords)

    if n <= 1:
        return 0.0

    max_distance = 0.0

    for i in range(n):
        distances = pbc_distances(
            coords[i],
            coords[i + 1:],
            cell
        )

        if len(distances) > 0:
            max_distance = max(
                max_distance,
                float(np.max(distances))
            )

    return max_distance


def calculate_rms_spread(group, representative_position):
    """
    RMS PBC distance of basin members from the representative.
    """

    coords = group[
        ["final_x", "final_y", "final_z"]
    ].to_numpy(dtype=float)

    distances = pbc_distances(
        representative_position,
        coords,
        cell
    )

    return float(
        np.sqrt(
            np.mean(
                distances ** 2
            )
        )
    )


summary_rows = []


for basin_id in sorted(
    master["basin_id"].unique()
):

    group = master[
        master["basin_id"] == basin_id
    ].copy()

    assert len(group) > 0

    # -------------------------------------------------------------------------
    # Representative = lowest-energy converged member
    # -------------------------------------------------------------------------

    representative_row = (
        group
        .sort_values(
            by=[
                "final_energy_eV",
                "site_index"
            ],
            ascending=[
                True,
                True
            ]
        )
        .iloc[0]
    )

    representative_site = int(
        representative_row["site_index"]
    )

    representative_position = (
        representative_row[
            [
                "final_x",
                "final_y",
                "final_z"
            ]
        ]
        .to_numpy(dtype=float)
    )

    energies = group[
        "final_energy_eV"
    ].astype(float)

    minimum_energy = float(
        energies.min()
    )

    maximum_energy = float(
        energies.max()
    )

    energy_range = (
        maximum_energy
        - minimum_energy
    )

    energy_std = float(
        energies.std(
            ddof=0
        )
    )

    diameter = calculate_basin_diameter(
        group
    )

    rms_spread = calculate_rms_spread(
        group,
        representative_position
    )

    # Keep the same cutoff convention already represented
    # by the existing summary column.
    #
    # The cutoff is NOT used to define or modify basins here.
    DIAMETER_CUTOFF_A = 0.20

    diameter_within_cutoff = (
        diameter <= DIAMETER_CUTOFF_A
    )

    representative_source = (
        representative_row["source"]
        if "source" in representative_row.index
        else np.nan
    )

    summary_rows.append(
        {
            "basin_id": int(basin_id),

            "population": int(
                len(group)
            ),

            "representative_site_index": (
                representative_site
            ),

            "representative_source": (
                representative_source
            ),

            "minimum_energy_eV": (
                minimum_energy
            ),

            "maximum_energy_eV": (
                maximum_energy
            ),

            "energy_range_eV": (
                energy_range
            ),

            "energy_std_eV": (
                energy_std
            ),

            "basin_diameter_A": (
                diameter
            ),

            "rms_spread_A": (
                rms_spread
            ),

            "representative_x": (
                representative_position[0]
            ),

            "representative_y": (
                representative_position[1]
            ),

            "representative_z": (
                representative_position[2]
            ),

            "diameter_within_cutoff": (
                bool(
                    diameter_within_cutoff
                )
            ),
        }
    )


final_basins = pd.DataFrame(
    summary_rows
)


# =============================================================================
# 8. ENERGY RANK
# =============================================================================
#
# Rank basins by their minimum energy.
# Lowest energy gets rank 1.
#
# =============================================================================

final_basins = final_basins.sort_values(
    by=[
        "minimum_energy_eV",
        "basin_id"
    ],
    ascending=[
        True,
        True
    ]
).reset_index(
    drop=True
)

final_basins["energy_rank"] = (
    np.arange(
        1,
        len(final_basins) + 1
    )
)


# Restore exact column order of original BASIN CSV
final_basins = final_basins[
    basins_original.columns
]


# =============================================================================
# 9. FINAL INVARIANTS
# =============================================================================

print("\nFINAL INVARIANTS")
print("-" * 100)

assert len(master) == 793

assert len(final_basins) == 117

assert master["basin_id"].nunique() == 117

assert final_basins["basin_id"].nunique() == 117

assert set(
    master["basin_id"].unique()
) == set(
    final_basins["basin_id"].unique()
)


# Population consistency
master_populations = (
    master
    .groupby("basin_id")
    .size()
    .sort_index()
)

summary_populations = (
    final_basins
    .set_index("basin_id")[
        "population"
    ]
    .sort_index()
    .astype(int)
)

assert master_populations.equals(
    summary_populations
), (
    "MASTER populations and BASIN summary populations disagree."
)


# Site 18 final assignment
assert int(
    master.loc[
        master["site_index"] == SITE_TO_MOVE,
        "basin_id"
    ].iloc[0]
) == NEW_BASIN


# Confirm only 19 and 80 populations changed
original_pop = (
    master_original
    .groupby("basin_id")
    .size()
)

final_pop = (
    master
    .groupby("basin_id")
    .size()
)

for basin_id in sorted(
    set(original_pop.index)
    | set(final_pop.index)
):

    old_n = int(
        original_pop.get(
            basin_id,
            0
        )
    )

    new_n = int(
        final_pop.get(
            basin_id,
            0
        )
    )

    if basin_id not in {
        OLD_BASIN,
        NEW_BASIN
    }:
        assert old_n == new_n, (
            f"Unexpected population change in basin "
            f"{basin_id}: {old_n} -> {new_n}"
        )


assert int(
    original_pop[OLD_BASIN]
) == 9

assert int(
    final_pop[OLD_BASIN]
) == 8

assert int(
    original_pop[NEW_BASIN]
) == 7

assert int(
    final_pop[NEW_BASIN]
) == 8


print("PASS: MASTER = 793 structures.")
print("PASS: FINAL basin count = 117.")
print("PASS: MASTER and BASIN summary populations agree.")
print("PASS: Site 18 is now in basin 80.")
print("PASS: Basin 19 population = 8.")
print("PASS: Basin 80 population = 8.")
print("PASS: No other basin population changed.")


# =============================================================================
# 10. SHOW THE TWO MODIFIED BASINS
# =============================================================================

print("\nFINAL BASIN 19 / BASIN 80")
print("-" * 100)

print(
    final_basins[
        final_basins["basin_id"].isin(
            [19, 80]
        )
    ].to_string(
        index=False
    )
)


# =============================================================================
# 11. SAVE — NEW FILES ONLY
# =============================================================================

master.to_csv(
    FINAL_MASTER_FILE,
    index=False
)

final_basins.to_csv(
    FINAL_BASIN_FILE,
    index=False
)


# =============================================================================
# 12. FINAL FILE CHECK
# =============================================================================

assert os.path.isfile(
    FINAL_MASTER_FILE
)

assert os.path.isfile(
    FINAL_BASIN_FILE
)

# Reload once to ensure the files are actually readable.
master_check = pd.read_csv(
    FINAL_MASTER_FILE
)

basins_check = pd.read_csv(
    FINAL_BASIN_FILE
)

assert len(master_check) == 793
assert basins_check["basin_id"].nunique() == 117

assert int(
    master_check.loc[
        master_check["site_index"] == SITE_TO_MOVE,
        "basin_id"
    ].iloc[0]
) == NEW_BASIN


print("\n" + "=" * 100)
print("FINAL 117-BASIN FILES CREATED")
print("=" * 100)

print(f"\nFINAL MASTER:")
print(FINAL_MASTER_FILE)

print(f"\nFINAL BASIN SUMMARY:")
print(FINAL_BASIN_FILE)

print("\nFINAL STATE")
print("-" * 100)
print("793 converged single-Li structures")
print("117 final basins")
print("Site 18: basin 19 -> basin 80")
print("Basin 19: 9 -> 8")
print("Basin 80: 7 -> 8")
print("Original files preserved.")
print("\nREADY FOR 2-Li STAGE.")
print("=" * 100)

In [1]:
# CANDIDATE 81
# MULTI-Li LOADING — 2-Li STAGE 1
# FINAL 117-BASIN PRODUCTION
# Parallel execution is supported through deterministic configuration chunks.

# =============================================================================
#
# SCIENTIFIC PURPOSE
# ------------------
# Enumerate ALL unique 2-Li configurations generated from the FINAL
# 117 single-Li basins, allowing same-basin occupancy.
#
# Number of configurations:
#
#       C(117 + 2 - 1, 2)
#     = C(118, 2)
#     = 6903
#
# IMPORTANT SCIENTIFIC RULE
# -------------------------
#
# A+B, A != B:
#     Use the lowest-energy representative of basin A
#     + lowest-energy representative of basin B.
#
# A+A:
#     Same-basin occupancy is allowed.
#
#     If basin A has >= 2 real MASTER members:
#         use TWO DISTINCT REAL MASTER MEMBERS.
#         The two lowest-energy distinct members are used.
#         This configuration enters MACE.
#
#     If basin A has exactly 1 MASTER member:
#         no physically distinct second position is available.
#         DO NOT invent a second position.
#         DO NOT run MACE.
#         Record the configuration officially as:
#
#             singleton_same_basin_no_distinct_second_member
#
# Therefore:
#
#     6903 configurations are enumerated.
#
#     104 A+A configurations are physically constructible
#     and enter MACE.
#
#     13 singleton A+A configurations are recorded without MACE.
#
# NO NEW CLUSTERING
# NO NEW BASIN DEFINITION
# NO MODIFICATION OF SINGLE-Li MASTER
# NO MODIFICATION OF SINGLE-Li BASIN SUMMARY
#

# =============================================================================

# =============================================================================

# [0] IMPORTS

# =============================================================================
import os
import json
import itertools
import shutil
import warnings
import re
import subprocess
import numpy as np
import pandas as pd
from ase import Atoms
from ase.io import read, write
from ase.optimize import BFGS
from ase.filters import FrechetCellFilter
from mace.calculators import mace_mp

# =============================================================================
# PARALLEL EXECUTION CONFIGURATION
# =============================================================================
# Run five independent notebook instances by changing CHUNK_INDEX to 0, 1, 2,
# 3, or 4 in each copy. Keep TOTAL_CHUNKS set to 5 in every notebook.
# Each worker processes a deterministic, non-overlapping slice of the complete
# configuration list and writes all outputs to its own chunk directory.
os.environ["TOTAL_CHUNKS"] = "5"
os.environ["CHUNK_INDEX"] = "0"  # Change this value in each notebook: 0 through 4.

TOTAL_CHUNKS = int(os.environ["TOTAL_CHUNKS"])
CHUNK_INDEX = int(os.environ["CHUNK_INDEX"])
if TOTAL_CHUNKS < 1:
    raise ValueError("TOTAL_CHUNKS must be a positive integer.")
if not 0 <= CHUNK_INDEX < TOTAL_CHUNKS:
    raise ValueError(
        f"CHUNK_INDEX must be in the range 0..{TOTAL_CHUNKS - 1}; "
        f"received {CHUNK_INDEX}."
    )
CHUNK_FOLDER_NAME = f"chunk_{CHUNK_INDEX}"

# =============================================================================

# [1] PROJECT CONFIGURATION

# =============================================================================
CANDIDATE_ID = 81

# =============================================================================
# ENVIRONMENT-AWARE GOOGLE DRIVE ACCESS
# =============================================================================
# The scientific workflow below is kept unchanged. This block only selects
# the working path and configures Google Drive access for Colab or Kaggle.

PROJECT_RELATIVE_ROOT = os.path.join(
    "Materials_Project",
    "Top5_search_archive",
    "generation_3_MACE_TOP5_archive"
)
DRIVE_PROJECT_RELATIVE_ROOT = PROJECT_RELATIVE_ROOT.replace(os.sep, "/")
RCLONE_REMOTE_NAME = None
RCLONE_CONFIG_PATH = None

if os.path.isdir("/kaggle/working") and (
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
    or os.path.isdir("/kaggle/input")
):
    EXECUTION_ENVIRONMENT = "KAGGLE"
    PROJECT_WORKSPACE = "/kaggle/working"

    # Kaggle Secrets: create a private rclone configuration file from the
    # RCLONE_CONFIG secret. The secret may contain config text or a file path.
    rclone_config_value = os.environ.get("RCLONE_CONFIG")
    if not rclone_config_value:
        try:
            from kaggle_secrets import UserSecretsClient
            rclone_config_value = UserSecretsClient().get_secret("RCLONE_CONFIG")
        except Exception as exc:
            raise RuntimeError(
                "Kaggle requires a RCLONE_CONFIG secret containing the rclone "
                "configuration text (or a path to that configuration file)."
            ) from exc

    if (
        "\n" not in rclone_config_value
        and os.path.isfile(rclone_config_value)
    ):
        with open(rclone_config_value, "r", encoding="utf-8") as config_file:
            rclone_config_text = config_file.read()
    else:
        rclone_config_text = rclone_config_value

    RCLONE_CONFIG_PATH = os.path.expanduser("~/.config/rclone/rclone.conf")
    os.makedirs(os.path.dirname(RCLONE_CONFIG_PATH), exist_ok=True)
    with open(RCLONE_CONFIG_PATH, "w", encoding="utf-8") as config_file:
        config_file.write(rclone_config_text)
    os.chmod(RCLONE_CONFIG_PATH, 0o600)

    remote_sections = re.findall(
        r"^\s*\[([^\]]+)\]\s*$",
        rclone_config_text,
        flags=re.MULTILINE
    )
    if not remote_sections:
        raise ValueError(
            "RCLONE_CONFIG does not contain an rclone remote section such as [gdrive]."
        )
    RCLONE_REMOTE_NAME = remote_sections[0]
    if shutil.which("rclone") is None:
        raise RuntimeError(
            "rclone is not installed in this Kaggle environment. Install rclone "
            "before running this script; no scientific code was changed."
        )

    def rclone_run(arguments, required=True):
        command = ["rclone", "--config", RCLONE_CONFIG_PATH] + list(arguments)
        completed = subprocess.run(
            command,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )
        if required and completed.returncode != 0:
            raise RuntimeError(
                "rclone command failed:\n"
                + " ".join(command)
                + "\n"
                + completed.stderr.strip()
            )
        return completed

    def rclone_copy_from_drive(remote_relative_path, local_path):
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        return rclone_run([
            "copy",
            f"{RCLONE_REMOTE_NAME}:{remote_relative_path}",
            local_path
        ])

    def sync_file_to_drive(local_path):
        if not os.path.isfile(local_path):
            return
        relative_path = os.path.relpath(local_path, PROJECT_ROOT).replace(os.sep, "/")
        remote_path = (
            DRIVE_PROJECT_RELATIVE_ROOT.rstrip("/")
            + "/"
            + relative_path
        )
        rclone_run([
            "copyto",
            local_path,
            f"{RCLONE_REMOTE_NAME}:{remote_path}"
        ])

    def sync_output_to_drive():
        if os.path.isdir(OUTPUT_ROOT):
            remote_output_path = (
                DRIVE_PROJECT_RELATIVE_ROOT.rstrip("/")
                + "/candidate_81_multi_Li_capacity_SCREENING/2_Li_FINAL_117BASINS/"
                + CHUNK_FOLDER_NAME
            )
            rclone_run([
                "copy",
                OUTPUT_ROOT,
                f"{RCLONE_REMOTE_NAME}:{remote_output_path}"
            ])

elif os.path.isdir("/content"):
    EXECUTION_ENVIRONMENT = "COLAB"
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_WORKSPACE = "/content/drive/MyDrive"

    def sync_file_to_drive(local_path):
        # Colab writes directly into the mounted Google Drive filesystem.
        return None

    def sync_output_to_drive():
        # Colab writes directly into the mounted Google Drive filesystem.
        return None

else:
    EXECUTION_ENVIRONMENT = "LOCAL"
    PROJECT_WORKSPACE = os.getcwd()

    def sync_file_to_drive(local_path):
        return None

    def sync_output_to_drive():
        return None

PROJECT_ROOT = os.path.join(
    PROJECT_WORKSPACE,
    PROJECT_RELATIVE_ROOT
)
print(f"Execution environment: {EXECUTION_ENVIRONMENT}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Parallel worker: {CHUNK_INDEX + 1}/{TOTAL_CHUNKS} ({CHUNK_FOLDER_NAME})")
BASIN_DIR = os.path.join(
    PROJECT_ROOT,
    "candidate_81_validated_Li_basin_analysis"
)
HOST_PATH = os.path.join(
    PROJECT_ROOT,
    "candidate_81_40atom_relaxed_host.extxyz"
)

# -------------------------------------------------------------------------
# FINAL INPUTS

# -------------------------------------------------------------------------
#
# IMPORTANT:
# These are the FINAL files created after the site-18 decision.
#
# Site 18:
#     basin 19 -> basin 80
#

# -------------------------------------------------------------------------
MASTER_CSV = os.path.join(
    BASIN_DIR,
    "candidate_81_793_converged_master_FINAL_117BASINS.csv"
)
VALIDATED_BASINS_CSV = os.path.join(
    BASIN_DIR,
    "candidate_81_validated_Li_basins_FINAL_117.csv"
)

# -------------------------------------------------------------------------
# OPTIONAL SINGLE-Li ARCHIVE

# -------------------------------------------------------------------------
#
# Reference only.
# It is NOT used to determine basin coordinates.
#
SINGLE_LI_ARCHIVE = os.path.join(
    PROJECT_ROOT,
    "candidate_81_Li_single_site_energy_screening",
    "candidate_81_all_tested_Li_sites_results.extxyz"
)

# =============================================================================
# OUTPUT

# =============================================================================
#
# A NEW FINAL DIRECTORY is used so that an older 6786-configuration run
# cannot silently contaminate the final 6903-configuration production.
#

# =============================================================================
OUTPUT_ROOT = os.path.join(
    PROJECT_ROOT,
    "candidate_81_multi_Li_capacity_SCREENING",
    "2_Li_FINAL_117BASINS",
    CHUNK_FOLDER_NAME
)
RESULTS_CSV = os.path.join(
    OUTPUT_ROOT,
    "candidate_81_2Li_FINAL_117BASINS_results.csv"
)
CONVERGED_EXTXYZ = os.path.join(
    OUTPUT_ROOT,
    "candidate_81_2Li_FINAL_117BASINS_converged.extxyz"
)
CHECKPOINT_DIR = os.path.join(
    OUTPUT_ROOT,
    "checkpoints"
)
FINAL_STRUCTURES_DIR = os.path.join(
    OUTPUT_ROOT,
    "final_structures"
)

# =============================================================================
# KAGGLE: DOWNLOAD REQUIRED INPUTS AND RESUME DATA FROM GOOGLE DRIVE
# =============================================================================
if EXECUTION_ENVIRONMENT == "KAGGLE":
    print("Preparing Google Drive inputs through rclone...")
    rclone_copy_from_drive(
        DRIVE_PROJECT_RELATIVE_ROOT + "/candidate_81_validated_Li_basin_analysis",
        BASIN_DIR
    )
    rclone_run([
        "copyto",
        f"{RCLONE_REMOTE_NAME}:{DRIVE_PROJECT_RELATIVE_ROOT}/candidate_81_40atom_relaxed_host.extxyz",
        HOST_PATH
    ])
    remote_output_path = (
        DRIVE_PROJECT_RELATIVE_ROOT
        + "/candidate_81_multi_Li_capacity_SCREENING/2_Li_FINAL_117BASINS/"
        + CHUNK_FOLDER_NAME
    )
    output_check = rclone_run([
        "lsf",
        f"{RCLONE_REMOTE_NAME}:{remote_output_path}"
    ], required=False)
    if output_check.returncode == 0:
        rclone_copy_from_drive(remote_output_path, OUTPUT_ROOT)
    else:
        print("No previous output directory found on Drive; starting a new output folder.")

# =============================================================================

# [2] SCIENTIFIC CONSTANTS

# =============================================================================
EXPECTED_HOST_ATOMS = 40
EXPECTED_MASTER_ROWS = 793
EXPECTED_VALIDATED_BASINS = 117
# Correct combinatorial count INCLUDING A+A.
EXPECTED_CONFIGURATIONS = (
    EXPECTED_VALIDATED_BASINS
    * (EXPECTED_VALIDATED_BASINS + 1)
    // 2
)
# 117 * 118 / 2 = 6903
assert EXPECTED_CONFIGURATIONS == 6903
# Number of BFGS steps allowed per MACE relaxation.
FMAX = 0.01
MAX_STEPS = 1000
# Save a structural checkpoint every N optimizer steps.
CHECKPOINT_INTERVAL = 5

# -------------------------------------------------------------------------
# INITIAL GEOMETRIC SAFEGUARD

# -------------------------------------------------------------------------
#
# This is NOT used to delete configurations from enumeration.
#
# A configuration with an initial Li-Li distance below this value is
# recorded as skipped AFTER enumeration.
#
INITIAL_MIN_LI_LI_DISTANCE = 0.15

# -------------------------------------------------------------------------
# FINAL PHYSICAL STABILITY SCREEN

# -------------------------------------------------------------------------
FINAL_MIN_LI_LI_STABILITY = 1.20
MIN_LI_HOST_DISTANCE = 1.50
MAX_RELATIVE_VOLUME_CHANGE = 0.15
MAX_LATTICE_RELATIVE_CHANGE = 0.15
MAX_HOST_DISTORTION_RMS = 0.50
MAX_HOST_DISTORTION_MAX = 1.50

# -------------------------------------------------------------------------
# INTENTIONAL STOP

# -------------------------------------------------------------------------
#
# MUST remain None for real production.
#
# Example for testing:
#
# FORCED_STOP_AFTER_STEPS = 25
#

# -------------------------------------------------------------------------
FORCED_STOP_AFTER_STEPS = None

# =============================================================================

# [3] CREATE OUTPUT DIRECTORIES

# =============================================================================
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(FINAL_STRUCTURES_DIR, exist_ok=True)

# =============================================================================

# [4] HEADER

# =============================================================================
print("=" * 100)
print(
    "CANDIDATE 81 — MULTI-Li LOADING "
    "2-Li STAGE 1 — FINAL 117-BASIN PRODUCTION"
)
print("=" * 100)
print()
print("CONTRACT")
print("-" * 100)
print("INPUT")
print("  - Final 793-row MASTER")
print("  - Final 117-basin summary")
print("  - Original 40-atom relaxed host")
print()
print("PROCESS")
print("  - 117 frozen single-Li basins")
print("  - combinations_with_replacement")
print("  - Same-basin occupancy ALLOWED")
print("  - A+B uses lowest-energy representative of each basin")
print("  - A+A uses two DISTINCT real MASTER members when available")
print("  - Singleton A+A is recorded WITHOUT inventing a second position")
print("  - MACE-MP medium + D3")
print("  - Fully flexible cell")
print("  - BFGS relaxation")
print()
print("OUTPUT")
print("  - 6903 enumerated 2-Li configurations")
print("  - 104 constructible A+A configurations enter MACE")
print("  - 13 singleton A+A configurations recorded without MACE")
print("  - Relaxation result for every MACE-eligible configuration")
print("  - Stable / unstable classification")
print()
print("INVARIANTS")
print("  - Host = 40 atoms")
print("  - Single-Li MASTER = 793")
print("  - Frozen basin count = 117")
print("  - Expected configurations = 6903")
print("  - Same-basin occupancy = YES")
print("  - Cell = FULLY FLEXIBLE")
print("  - MACE = medium")
print("  - D3 = True")
print("  - dtype = float64")
print(f"  - FMAX = {FMAX} eV/A")
print("=" * 100)

# =============================================================================

# [5] HELPER FUNCTIONS

# =============================================================================

def require_file(path, label):
    """Require an input file to exist."""
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"{label} not found:\n{path}"
        )
    print(f"FOUND: {label}")

def require_columns(df, required_columns, label):
    """Validate required dataframe columns."""
    missing = [
        col for col in required_columns
        if col not in df.columns
    ]
    if missing:
        raise ValueError(
            f"{label} is missing required columns:\n{missing}"
        )

def get_li_indices(atoms):
    """Return indices of Li atoms."""
    return [
        i
        for i, symbol in enumerate(atoms.get_chemical_symbols())
        if symbol == "Li"
    ]

def minimum_image_distance(atoms, index_a, index_b):
    """Periodic minimum-image distance between two atoms."""
    return float(
        atoms.get_distance(
            index_a,
            index_b,
            mic=True
        )
    )

def min_li_li_distance(atoms):
    """Minimum periodic Li-Li distance."""
    li_indices = get_li_indices(atoms)
    if len(li_indices) < 2:
        return np.nan
    distances = [
        minimum_image_distance(atoms, i, j)
        for i, j in itertools.combinations(li_indices, 2)
    ]
    return float(min(distances))

def min_li_host_distance(atoms):
    """Minimum periodic Li-host distance."""
    li_indices = get_li_indices(atoms)
    host_indices = [
        i
        for i in range(len(atoms))
        if atoms[i].symbol != "Li"
    ]
    if not li_indices or not host_indices:
        return np.nan
    distances = []
    for li_index in li_indices:
        for host_index in host_indices:
            distances.append(
                atoms.get_distance(
                    li_index,
                    host_index,
                    mic=True
                )
            )
    return float(min(distances))

def maximum_force(atoms):
    """Maximum atomic force magnitude."""
    forces = atoms.get_forces()
    if len(forces) == 0:
        return np.nan
    return float(
        np.max(
            np.linalg.norm(
                forces,
                axis=1
            )
        )
    )

def relative_lattice_change(initial_cell, final_cell):
    """
    Frobenius-norm relative change of the cell matrix.
    """
    initial = np.asarray(
        initial_cell,
        dtype=float
    )
    final = np.asarray(
        final_cell,
        dtype=float
    )
    denominator = np.linalg.norm(initial)
    if denominator == 0:
        return np.nan
    return float(
        np.linalg.norm(final - initial)
        / denominator
    )

def host_distortion_metrics(
    initial_host_positions,
    final_atoms,
    initial_cell
):
    """
    Estimate host-atom displacement after relaxation.
    Initial and final host coordinates are compared through fractional
    minimum-image differences.
    """
    host_count = len(initial_host_positions)
    if len(final_atoms) < host_count:
        return np.nan, np.nan
    initial_positions = np.asarray(
        initial_host_positions,
        dtype=float
    )
    final_positions = np.asarray(
        final_atoms.positions[:host_count],
        dtype=float
    )
    initial_cell = np.asarray(
        initial_cell,
        dtype=float
    )
    final_cell = np.asarray(
        final_atoms.cell.array,
        dtype=float
    )
    initial_frac = (
        initial_positions
        @ np.linalg.inv(initial_cell)
    )
    final_frac = (
        final_positions
        @ np.linalg.inv(final_cell)
    )
    delta_frac = final_frac - initial_frac
    delta_frac -= np.round(delta_frac)
    delta_cart = delta_frac @ final_cell
    magnitudes = np.linalg.norm(
        delta_cart,
        axis=1
    )
    return (
        float(np.sqrt(np.mean(magnitudes ** 2))),
        float(np.max(magnitudes))
    )

def checkpoint_xyz_path(configuration_id):
    return os.path.join(
        CHECKPOINT_DIR,
        f"{configuration_id}.extxyz"
    )

def checkpoint_json_path(configuration_id):
    return os.path.join(
        CHECKPOINT_DIR,
        f"{configuration_id}.json"
    )

def save_checkpoint(
    atoms,
    configuration_id,
    step
):
    """
    Save atomic/cell state and checkpoint metadata.
    BFGS internal state is intentionally not serialized.
    """
    xyz_path = checkpoint_xyz_path(
        configuration_id
    )
    json_path = checkpoint_json_path(
        configuration_id
    )
    write(
        xyz_path,
        atoms,
        format="extxyz"
    )
    metadata = {
        "configuration_id": configuration_id,
        "checkpoint_step": int(step),
        "n_atoms": int(len(atoms)),
        "cell": atoms.cell.array.tolist(),
        "pbc": atoms.pbc.tolist(),
    }
    with open(
        json_path,
        "w"
    ) as f:
        json.dump(
            metadata,
            f,
            indent=2
        )
    sync_file_to_drive(xyz_path)
    sync_file_to_drive(json_path)

def load_checkpoint(configuration_id):
    """
    Load checkpoint if both structure and metadata exist.
    Returns:
        atoms, checkpoint_step
    """
    xyz_path = checkpoint_xyz_path(
        configuration_id
    )
    json_path = checkpoint_json_path(
        configuration_id
    )
    if not (
        os.path.isfile(xyz_path)
        and os.path.isfile(json_path)
    ):
        return None, 0
    atoms = read(
        xyz_path,
        index=-1
    )
    with open(
        json_path,
        "r"
    ) as f:
        metadata = json.load(f)
    step = int(
        metadata.get(
            "checkpoint_step",
            0
        )
    )
    return atoms, step

def remove_checkpoint(configuration_id):
    """Remove checkpoint files after successful completion."""
    for path in [
        checkpoint_xyz_path(configuration_id),
        checkpoint_json_path(configuration_id),
    ]:
        if os.path.isfile(path):
            os.remove(path)

def load_completed_ids():
    """
    Load configuration IDs already recorded in the final results CSV.
    """
    if not os.path.isfile(RESULTS_CSV):
        return set()
    df = pd.read_csv(
        RESULTS_CSV
    )
    if "configuration_id" not in df.columns:
        raise ValueError(
            "Existing results CSV does not contain "
            "'configuration_id'."
        )
    if df["configuration_id"].duplicated().any():
        raise ValueError(
            "Existing results CSV contains duplicate "
            "configuration IDs."
        )
    return set(
        df["configuration_id"]
        .dropna()
        .astype(str)
    )

def append_result(row):
    """Append one result safely to the results CSV."""
    df = pd.DataFrame([row])
    file_exists = os.path.isfile(
        RESULTS_CSV
    )
    df.to_csv(
        RESULTS_CSV,
        mode="a",
        header=not file_exists,
        index=False
    )
    sync_file_to_drive(RESULTS_CSV)

def build_two_li_system(
    host,
    position_a,
    position_b
):
    """
    Construct a fresh 2-Li system from the original host.
    """
    atoms = host.copy()
    li_atoms = Atoms(
        symbols=["Li", "Li"],
        positions=[
            np.asarray(
                position_a,
                dtype=float
            ),
            np.asarray(
                position_b,
                dtype=float
            )
        ]
    )
    atoms += li_atoms
    atoms.set_cell(
        host.cell.copy()
    )
    atoms.set_pbc(
        host.pbc
    )
    return atoms

def make_configuration_id(
    basin_a,
    basin_b
):
    """
    Deterministic configuration ID.
    """
    a = int(basin_a)
    b = int(basin_b)
    if a > b:
        a, b = b, a
    return f"2Li_B{a}_B{b}"

# =============================================================================

# [6] VALIDATE INPUT PATHS

# =============================================================================
print()
print("1 — VALIDATING FINAL INPUTS")
print("-" * 100)
require_file(
    HOST_PATH,
    "Final host"
)
require_file(
    MASTER_CSV,
    "FINAL 793-row MASTER"
)
require_file(
    VALIDATED_BASINS_CSV,
    "FINAL 117-basin summary"
)
print("All required final inputs found.")

# =============================================================================

# [7] LOAD HOST

# =============================================================================
print()
print("2 — LOADING HOST")
print("-" * 100)
host = read(
    HOST_PATH
)
if len(host) != EXPECTED_HOST_ATOMS:
    raise ValueError(
        f"Host must contain exactly "
        f"{EXPECTED_HOST_ATOMS} atoms; "
        f"found {len(host)}."
    )
if not np.all(host.pbc):
    raise ValueError(
        "Host must be fully periodic."
    )
print(
    f"Host atoms : {len(host)}"
)
print(
    f"Formula    : {host.get_chemical_formula()}"
)
print(
    f"Volume     : {host.get_volume():.12f} A^3"
)
print(
    f"PBC        : {host.pbc}"
)
INITIAL_HOST_POSITIONS = (
    host.positions.copy()
)
INITIAL_HOST_CELL = (
    host.cell.array.copy()
)
INITIAL_HOST_VOLUME = (
    host.get_volume()
)

# =============================================================================

# [8] LOAD FINAL MASTER

# =============================================================================
print()
print("3 — LOADING FINAL 793 MASTER")
print("-" * 100)
master_df = pd.read_csv(
    MASTER_CSV
)
print(
    f"MASTER rows : {len(master_df)}"
)
required_master_columns = [
    "site_index",
    "source",
    "final_x",
    "final_y",
    "final_z",
    "final_energy_eV",
    "final_max_force_eV_A",
    "optimizer_steps",
    "converged",
    "basin_id"
]
require_columns(
    master_df,
    required_master_columns,
    "FINAL MASTER"
)
if len(master_df) != EXPECTED_MASTER_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_MASTER_ROWS} MASTER rows, "
        f"found {len(master_df)}."
    )
if not master_df["site_index"].is_unique:
    raise ValueError(
        "MASTER site_index values are not unique."
    )
master_df["basin_id"] = (
    master_df["basin_id"]
    .astype(int)
)
master_df["site_index"] = (
    master_df["site_index"]
    .astype(int)
)
numeric_columns = [
    "final_x",
    "final_y",
    "final_z",
    "final_energy_eV",
    "final_max_force_eV_A"
]
for column in numeric_columns:
    master_df[column] = pd.to_numeric(
        master_df[column],
        errors="raise"
    )
converged_values = (
    master_df["converged"]
    .astype(str)
    .str.lower()
    .isin(
        [
            "true",
            "1",
            "yes"
        ]
    )
)
if not converged_values.all():
    raise ValueError(
        "FINAL MASTER contains non-converged structures."
    )
print(
    "PASS: FINAL MASTER validation."
)

# =============================================================================

# [9] LOAD FINAL BASIN SUMMARY

# =============================================================================
print()
print("4 — LOADING FINAL 117-BASIN SUMMARY")
print("-" * 100)
basin_df = pd.read_csv(
    VALIDATED_BASINS_CSV
)
print(
    f"BASIN rows : {len(basin_df)}"
)
if len(basin_df) != EXPECTED_VALIDATED_BASINS:
    raise ValueError(
        f"Expected {EXPECTED_VALIDATED_BASINS} basin rows, "
        f"found {len(basin_df)}."
    )
required_basin_columns = [
    "basin_id",
    "population",
    "representative_site_index",
    "minimum_energy_eV"
]
require_columns(
    basin_df,
    required_basin_columns,
    "FINAL BASIN summary"
)
basin_df["basin_id"] = (
    basin_df["basin_id"]
    .astype(int)
)
if not basin_df["basin_id"].is_unique:
    raise ValueError(
        "FINAL BASIN summary contains duplicate basin IDs."
    )
if basin_df["basin_id"].nunique() != 117:
    raise ValueError(
        "FINAL BASIN summary does not contain exactly 117 basins."
    )
master_basin_ids = set(
    master_df["basin_id"]
)
summary_basin_ids = set(
    basin_df["basin_id"]
)
if master_basin_ids != summary_basin_ids:
    raise ValueError(
        "MASTER and FINAL BASIN summary contain different basin IDs."
    )

# =============================================================================

# [10] MASTER ↔ BASIN POPULATION CONSISTENCY

# =============================================================================
print()
print("5 — MASTER ↔ BASIN CONSISTENCY")
print("-" * 100)
master_populations = (
    master_df
    .groupby("basin_id")
    .size()
    .sort_index()
)
summary_populations = (
    basin_df
    .set_index("basin_id")["population"]
    .astype(int)
    .sort_index()
)
if not master_populations.equals(
    summary_populations
):
    raise ValueError(
        "MASTER and FINAL BASIN populations disagree."
    )
print(
    "PASS: MASTER and final basin summary populations agree."
)

# =============================================================================

# [11] SELECT REPRESENTATIVES

# =============================================================================
#
# For each basin:
#
#   representative:
#       lowest-energy member.
#
# For A+A:
#
#   second member:
#       second-lowest-energy DISTINCT MASTER member.
#
# A+A is therefore constructed only from actual observed MASTER members.
#
# No position is invented.
#

# =============================================================================
print()
print("6 — SELECTING BASIN REPRESENTATIVES")
print("-" * 100)
sorted_master = master_df.sort_values(
    by=[
        "basin_id",
        "final_energy_eV",
        "site_index"
    ],
    ascending=[
        True,
        True,
        True
    ]
)
basin_records = {}
for basin_id, group in sorted_master.groupby(
    "basin_id",
    sort=True
):
    members = []
    for _, row in group.iterrows():
        position = np.array(
            [
                row["final_x"],
                row["final_y"],
                row["final_z"]
            ],
            dtype=float
        )
        if not np.all(
            np.isfinite(position)
        ):
            raise ValueError(
                f"Non-finite coordinates in basin {basin_id}."
            )
        members.append(
            {
                "site_index": int(
                    row["site_index"]
                ),
                "source": row["source"],
                "energy_eV": float(
                    row["final_energy_eV"]
                ),
                "position": position
            }
        )
    if len(members) == 0:
        raise ValueError(
            f"Basin {basin_id} has no MASTER members."
        )
    basin_records[int(basin_id)] = {
        "basin_id": int(basin_id),
        "members": members,
        "representative": members[0],
        "second_member": (
            members[1]
            if len(members) >= 2
            else None
        )
    }
if len(basin_records) != 117:
    raise ValueError(
        f"Expected 117 basin records; "
        f"found {len(basin_records)}."
    )
constructible_AA_basins = [
    basin_id
    for basin_id, record in basin_records.items()
    if record["second_member"] is not None
]
singleton_AA_basins = [
    basin_id
    for basin_id, record in basin_records.items()
    if record["second_member"] is None
]
print(
    f"PASS: 117 basin representatives selected."
)
print(
    f"Basins with >=2 real MASTER members for A+A : "
    f"{len(constructible_AA_basins)}"
)
print(
    f"Singleton basins for A+A                  : "
    f"{len(singleton_AA_basins)}"
)
print(
    f"Expected: 104 constructible + 13 singleton = 117"
)
if len(constructible_AA_basins) != 104:
    raise ValueError(
        "Expected exactly 104 basins with a real second member "
        "for A+A."
    )
if len(singleton_AA_basins) != 13:
    raise ValueError(
        "Expected exactly 13 singleton basins."
    )

# =============================================================================

# [12] INITIALIZE MACE

# =============================================================================
print()
print("7 — INITIALIZING MACE")
print("-" * 100)
calc = mace_mp(
    model="medium",
    dispersion=True,
    default_dtype="float64"
)
print(
    "PASS: MACE-MP medium + D3 initialized."
)

# =============================================================================

# [13] ENUMERATE ALL 6903 CONFIGURATIONS

# =============================================================================
#
# combinations_with_replacement gives:
#
#   B1+B1
#   B1+B2
#   ...
#   B1+B117
#   B2+B2
#   ...
#   B117+B117
#

# =============================================================================
print()
print("8 — ENUMERATING ALL FINAL 2-Li CONFIGURATIONS")
print("-" * 100)
basin_ids = sorted(
    basin_records.keys()
)
configuration_list = []
for basin_a, basin_b in itertools.combinations_with_replacement(
    basin_ids,
    2
):
    record_a = basin_records[basin_a]
    record_b = basin_records[basin_b]
    representative_a = record_a["representative"]
    representative_b = record_b["representative"]
    same_basin = (
        basin_a == basin_b
    )
    configuration_id = make_configuration_id(
        basin_a,
        basin_b
    )
    if same_basin:
        second_member = (
            record_a["second_member"]
        )
        if second_member is None:
            configuration_list.append(
                {
                    "configuration_id":
                        configuration_id,
                    "basin_1":
                        basin_a,
                    "basin_2":
                        basin_b,
                    "site_1":
                        representative_a["site_index"],
                    "site_2":
                        np.nan,
                    "energy_1_eV":
                        representative_a["energy_eV"],
                    "energy_2_eV":
                        np.nan,
                    "position_1":
                        representative_a["position"].copy(),
                    "position_2":
                        None,
                    "same_basin":
                        True,
                    "aa_constructible":
                        False,
                    "enumeration_status":
                        "singleton_same_basin_no_distinct_second_member"
                }
            )
        else:
            configuration_list.append(
                {
                    "configuration_id":
                        configuration_id,
                    "basin_1":
                        basin_a,
                    "basin_2":
                        basin_b,
                    "site_1":
                        representative_a["site_index"],
                    "site_2":
                        second_member["site_index"],
                    "energy_1_eV":
                        representative_a["energy_eV"],
                    "energy_2_eV":
                        second_member["energy_eV"],
                    "position_1":
                        representative_a["position"].copy(),
                    "position_2":
                        second_member["position"].copy(),
                    "same_basin":
                        True,
                    "aa_constructible":
                        True,
                    "enumeration_status":
                        "constructible_same_basin"
                }
            )
    else:
        configuration_list.append(
            {
                "configuration_id":
                    configuration_id,
                "basin_1":
                    basin_a,
                "basin_2":
                    basin_b,
                "site_1":
                    representative_a["site_index"],
                "site_2":
                    representative_b["site_index"],
                "energy_1_eV":
                    representative_a["energy_eV"],
                "energy_2_eV":
                    representative_b["energy_eV"],
                "position_1":
                    representative_a["position"].copy(),
                "position_2":
                    representative_b["position"].copy(),
                "same_basin":
                    False,
                "aa_constructible":
                    True,
                "enumeration_status":
                    "constructible_distinct_basin"
            }
        )

# =============================================================================

# [14] ENUMERATION VALIDATION

# =============================================================================
print()
print("9 — ENUMERATION VALIDATION")
print("-" * 100)
if len(configuration_list) != EXPECTED_CONFIGURATIONS:
    raise ValueError(
        f"Expected {EXPECTED_CONFIGURATIONS} configurations; "
        f"found {len(configuration_list)}."
    )
configuration_ids = [
    item["configuration_id"]
    for item in configuration_list
]
if len(configuration_ids) != len(
    set(configuration_ids)
):
    raise ValueError(
        "Duplicate configuration IDs detected."
    )
same_basin_configs = [
    item
    for item in configuration_list
    if item["same_basin"]
]
constructible_AA_configs = [
    item
    for item in configuration_list
    if (
        item["same_basin"]
        and item["aa_constructible"]
    )
]
singleton_AA_configs = [
    item
    for item in configuration_list
    if (
        item["same_basin"]
        and not item["aa_constructible"]
    )
]
distinct_basin_configs = [
    item
    for item in configuration_list
    if not item["same_basin"]
]
expected_AA = 117
expected_distinct = (
    117 *
    116 // 2
)
if len(same_basin_configs) != expected_AA:
    raise ValueError(
        f"Expected {expected_AA} A+A configurations; "
        f"found {len(same_basin_configs)}."
    )
if len(distinct_basin_configs) != expected_distinct:
    raise ValueError(
        f"Expected {expected_distinct} A+B configurations; "
        f"found {len(distinct_basin_configs)}."
    )
if len(constructible_AA_configs) != 104:
    raise ValueError(
        f"Expected 104 constructible A+A configurations; "
        f"found {len(constructible_AA_configs)}."
    )
if len(singleton_AA_configs) != 13:
    raise ValueError(
        f"Expected 13 singleton A+A configurations; "
        f"found {len(singleton_AA_configs)}."
    )
print(
    f"PASS: total enumerated configurations = "
    f"{len(configuration_list)}"
)
print(
    f"PASS: A+A configurations = "
    f"{len(same_basin_configs)}"
)
print(
    f"PASS: constructible A+A = "
    f"{len(constructible_AA_configs)}"
)
print(
    f"PASS: singleton A+A = "
    f"{len(singleton_AA_configs)}"
)
print(
    f"PASS: distinct-basin A+B = "
    f"{len(distinct_basin_configs)}"
)
print()
print(
    "6903 = 117 A+A + 6786 distinct-basin A+B"
)

# =============================================================================
# [14A] ASSIGN THIS WORKER A NON-OVERLAPPING CONFIGURATION SLICE
# =============================================================================
# Python list slicing assigns each configuration to exactly one worker:
# worker 0 receives indices 0, 5, 10, ...; worker 1 receives 1, 6, 11, ...;
# and so on. The complete list above remains available for global validation.
all_configuration_ids = list(configuration_ids)
configuration_list = configuration_list[CHUNK_INDEX::TOTAL_CHUNKS]
chunk_configuration_ids = [
    item["configuration_id"] for item in configuration_list
]
EXPECTED_CHUNK_CONFIGURATIONS = len(configuration_list)
if len(chunk_configuration_ids) != len(set(chunk_configuration_ids)):
    raise ValueError(f"Duplicate configuration IDs assigned to {CHUNK_FOLDER_NAME}.")
print()
print("PARALLEL WORK ALLOCATION")
print("-" * 100)
print(f"Worker index             : {CHUNK_INDEX}")
print(f"Total workers            : {TOTAL_CHUNKS}")
print(f"Output directory         : {OUTPUT_ROOT}")
print(f"Configurations assigned  : {EXPECTED_CHUNK_CONFIGURATIONS}")
print(f"Full configuration count : {len(all_configuration_ids)}")

# =============================================================================

# [15] LOAD COMPLETED RESULTS

# =============================================================================
print()
print("10 — CHECKING EXISTING RESULTS")
print("-" * 100)
completed_ids = load_completed_ids()
unknown_completed = (
    completed_ids
    - set(chunk_configuration_ids)
)
if unknown_completed:
    raise ValueError(
        "Existing RESULTS_CSV contains configuration IDs "
        f"that are not assigned to {CHUNK_FOLDER_NAME}:\n"
        f"{sorted(unknown_completed)[:20]}"
    )
print(
    f"Already completed : {len(completed_ids)}"
)
remaining_count = (
    len(configuration_list)
    - len(
        completed_ids.intersection(
            chunk_configuration_ids
        )
    )

# =============================================================================
)

# [16] RELAXATION LOOP

# =============================================================================
print()
print("11 — STARTING FINAL 2-Li PRODUCTION")
print("=" * 100)
for config_number, config in enumerate(
    configuration_list,
    start=1
):
    configuration_id = (
        config["configuration_id"]
    )
    if configuration_id in completed_ids:
        print(
            f"[{config_number}/{EXPECTED_CHUNK_CONFIGURATIONS}] "
            f"{configuration_id} -> ALREADY COMPLETED"
        )
        continue
    basin_1 = config["basin_1"]
    basin_2 = config["basin_2"]
    site_1 = config["site_1"]
    site_2 = config["site_2"]
    same_basin = config["same_basin"]
    # =========================================================================
    # SINGLETON A+A
    # =========================================================================
    #
    # This configuration is enumerated but cannot be physically constructed
    # from two distinct real MASTER members.
    #
    # DO NOT invent a second coordinate.
    # DO NOT run MACE.
    #
    # =========================================================================
    if (
        same_basin
        and not config["aa_constructible"]
    ):
        print()
        print("-" * 100)
        print(
            f"[{config_number}/{EXPECTED_CHUNK_CONFIGURATIONS}] "
            f"{configuration_id}"
        )
        print(
            "SINGLETON A+A -> OFFICIAL RECORD ONLY"
        )
        print(
            f"Basin {basin_1} has only one real MASTER member."
        )
        print(
            "No artificial second Li position will be invented."
        )
        print(
            "MACE relaxation: NOT RUN."
        )
        result = {
            "configuration_id":
                configuration_id,
            "status":
                "singleton_same_basin_no_distinct_second_member",
            "enumeration_status":
                config["enumeration_status"],
            "basin_1":
                basin_1,
            "basin_2":
                basin_2,
            "site_1":
                site_1,
            "site_2":
                np.nan,
            "same_basin":
                True,
            "aa_constructible":
                False,
            "initial_Li_Li_distance_A":
                np.nan,
            "final_Li_Li_distance_A":
                np.nan,
            "initial_Li_host_distance_A":
                np.nan,
            "final_Li_host_distance_A":
                np.nan,
            "initial_energy_eV":
                np.nan,
            "final_energy_eV":
                np.nan,
            "final_max_force_eV_A":
                np.nan,
            "optimizer_steps":
                0,
            "converged":
                False,
            "stable":
                False,
            "initial_volume_A3":
                INITIAL_HOST_VOLUME,
            "final_volume_A3":
                np.nan,
            "volume_change_percent":
                np.nan,
            "relative_lattice_change":
                np.nan,
            "host_distortion_rms_A":
                np.nan,
            "host_distortion_max_A":
                np.nan,
            "resumed_from_checkpoint":
                False,
            "final_structure":
                np.nan
        }
        append_result(
            result
        )
        completed_ids.add(
            configuration_id
        )
        continue
    # =========================================================================
    # CHECKPOINT / FRESH START
    # =========================================================================
    print()
    print("-" * 100)
    print(
        f"[{config_number}/{EXPECTED_CHUNK_CONFIGURATIONS}] "
        f"{configuration_id}"
    )
    print(
        f"Basins: {basin_1} + {basin_2}"
    )
    print(
        f"Sites : {site_1} + {site_2}"
    )
    if same_basin:
        print(
            "Type  : CONSTRUCTIBLE SAME-BASIN A+A"
        )
    else:
        print(
            "Type  : DISTINCT-BASIN A+B"
        )
    checkpoint_atoms, checkpoint_step = (
        load_checkpoint(
            configuration_id
        )
    )
    resumed = (
        checkpoint_atoms is not None
    )
    if resumed:
        atoms = checkpoint_atoms
        print(
            f"RESUME: checkpoint at step "
            f"{checkpoint_step}"
        )
        if len(atoms) != 42:
            raise ValueError(
                f"Checkpoint {configuration_id} has "
                f"{len(atoms)} atoms; expected 42."
            )
        if not np.all(atoms.pbc):
            raise ValueError(
                f"Checkpoint {configuration_id} "
                f"is not fully periodic."
            )
    else:
        atoms = build_two_li_system(
            host,
            config["position_1"],
            config["position_2"]
        )
        checkpoint_step = 0
        print(
            "START: fresh 2-Li configuration"
        )
    # =========================================================================
    # STRUCTURAL VALIDATION
    # =========================================================================
    li_indices = get_li_indices(
        atoms
    )
    if len(li_indices) != 2:
        raise ValueError(
            f"{configuration_id}: expected exactly 2 Li atoms; "
            f"found {len(li_indices)}."
        )
    if len(atoms) != 42:
        raise ValueError(
            f"{configuration_id}: expected 42 total atoms; "
            f"found {len(atoms)}."
        )
    # =========================================================================
    # INITIAL / CURRENT Li-Li DISTANCE
    # =========================================================================
    initial_li_li_distance = (
        min_li_li_distance(atoms)
    )
    print(
        f"Initial/current Li-Li distance: "
        f"{initial_li_li_distance:.6f} A"
    )
    # =========================================================================
    # INITIAL Li-Li SAFETY SCREEN
    # =========================================================================
    #
    # The configuration has already been enumerated.
    #
    # Therefore a rejected initial geometry remains represented in the
    # official 6903 records.
    #
    # It simply does not enter MACE.
    #
    # =========================================================================
    if (
        not resumed
        and
        initial_li_li_distance
        < INITIAL_MIN_LI_LI_DISTANCE
    ):
        print(
            f"SKIPPED: initial Li-Li distance "
            f"{initial_li_li_distance:.6f} A < "
            f"{INITIAL_MIN_LI_LI_DISTANCE:.2f} A"
        )
        result = {
            "configuration_id":
                configuration_id,
            "status":
                "skipped_initial_Li_Li_distance",
            "enumeration_status":
                config["enumeration_status"],
            "basin_1":
                basin_1,
            "basin_2":
                basin_2,
            "site_1":
                site_1,
            "site_2":
                site_2,
            "same_basin":
                same_basin,
            "aa_constructible":
                config["aa_constructible"],
            "initial_Li_Li_distance_A":
                initial_li_li_distance,
            "final_Li_Li_distance_A":
                np.nan,
            "initial_Li_host_distance_A":
                min_li_host_distance(atoms),
            "final_Li_host_distance_A":
                np.nan,
            "initial_energy_eV":
                np.nan,
            "final_energy_eV":
                np.nan,
            "final_max_force_eV_A":
                np.nan,
            "optimizer_steps":
                0,
            "converged":
                False,
            "stable":
                False,
            "initial_volume_A3":
                INITIAL_HOST_VOLUME,
            "final_volume_A3":
                np.nan,
            "volume_change_percent":
                np.nan,
            "relative_lattice_change":
                np.nan,
            "host_distortion_rms_A":
                np.nan,
            "host_distortion_max_A":
                np.nan,
            "resumed_from_checkpoint":
                False,
            "final_structure":
                np.nan
        }
        append_result(
            result
        )
        completed_ids.add(
            configuration_id
        )
        continue
    # =========================================================================
    # MACE
    # =========================================================================
    atoms.calc = calc
    # =========================================================================
    # INITIAL ENERGY
    # =========================================================================
    initial_energy = (
        atoms.get_potential_energy()
    )
    print(
        f"Initial energy: "
        f"{initial_energy:.10f} eV"
    )
    # =========================================================================
    # INITIAL STRUCTURAL STATE
    # =========================================================================
    initial_volume = (
        atoms.get_volume()
    )
    initial_cell = (
        atoms.cell.array.copy()
    )
    initial_host_positions = (
        atoms.positions[
            :EXPECTED_HOST_ATOMS
        ].copy()
    )
    # =========================================================================
    # FULLY FLEXIBLE CELL RELAXATION
    # =========================================================================
    cell_filter = FrechetCellFilter(
        atoms
    )
    dyn = BFGS(
        cell_filter,
        logfile=None
    )
    current_step = int(
        checkpoint_step
    )
    # =========================================================================
    # CHECKPOINT CALLBACK
    # =========================================================================
    def checkpoint_callback():
        global current_step
        current_step += 1
        if (
            current_step
            % CHECKPOINT_INTERVAL
            == 0
        ):
            save_checkpoint(
                atoms,
                configuration_id,
                current_step
            )
            print(
                f"  CHECKPOINT saved at step "
                f"{current_step}"
            )
        if (
            FORCED_STOP_AFTER_STEPS is not None
            and
            current_step
            >= FORCED_STOP_AFTER_STEPS
        ):
            save_checkpoint(
                atoms,
                configuration_id,
                current_step
            )
            raise RuntimeError(
                f"INTENTIONAL TEST STOP at "
                f"step {current_step}"
            )
    dyn.attach(
        checkpoint_callback,
        interval=1
    )
    # =========================================================================
    # RUN RELAXATION
    # =========================================================================
    remaining_steps = max(
        0,
        MAX_STEPS - checkpoint_step
    )
    print(
        f"Running BFGS for up to "
        f"{remaining_steps} additional steps..."
    )
    try:
        dyn.run(
            fmax=FMAX,
            steps=remaining_steps
        )
    except RuntimeError as exc:
        if (
            FORCED_STOP_AFTER_STEPS is not None
            and
            "INTENTIONAL TEST STOP"
            in str(exc)
        ):
            raise
        print(
            f"WARNING: relaxation interrupted for "
            f"{configuration_id}: {exc}"
        )
        save_checkpoint(
            atoms,
            configuration_id,
            current_step
        )
        raise
    # =========================================================================
    # FINAL CALCULATIONS
    # =========================================================================
    final_energy = (
        atoms.get_potential_energy()
    )
    final_forces = (
        atoms.get_forces()
    )
    final_max_force = float(
        np.max(
            np.linalg.norm(
                final_forces,
                axis=1
            )
        )
    )
    final_li_li_distance = (
        min_li_li_distance(atoms)
    )
    final_li_host_distance = (
        min_li_host_distance(atoms)
    )
    final_volume = (
        atoms.get_volume()
    )
    volume_change_percent = (
        100.0
        *
        (
            final_volume
            - initial_volume
        )
        / initial_volume
    )
    relative_lattice = (
        relative_lattice_change(
            initial_cell,
            atoms.cell.array
        )
    )
    host_rms, host_max = (
        host_distortion_metrics(
            initial_host_positions,
            atoms,
            initial_cell
        )
    )
    # =========================================================================
    # CONVERGENCE
    # =========================================================================
    converged = (
        final_max_force
        <= FMAX
    )
    # =========================================================================
    # PHYSICAL STABILITY SCREEN
    # =========================================================================
    stable = (
        converged
        and
        final_li_li_distance
        >= FINAL_MIN_LI_LI_STABILITY
        and
        final_li_host_distance
        >= MIN_LI_HOST_DISTANCE
        and
        abs(
            volume_change_percent
        )
        <= (
            MAX_RELATIVE_VOLUME_CHANGE
            * 100.0
        )
        and
        relative_lattice
        <= MAX_LATTICE_RELATIVE_CHANGE
        and
        host_rms
        <= MAX_HOST_DISTORTION_RMS
        and
        host_max
        <= MAX_HOST_DISTORTION_MAX
    )
    # =========================================================================
    # STATUS
    # =========================================================================
    if stable:
        status = (
            "converged_stable"
        )
    elif converged:
        status = (
            "converged_but_failed_stability"
        )
    else:
        status = (
            "not_converged"
        )
    # =========================================================================
    # REPORT
    # =========================================================================
    print()
    print("FINAL RESULT")
    print("-" * 100)
    print(
        f"Final energy             : "
        f"{final_energy:.10f} eV"
    )
    print(
        f"Final max force          : "
        f"{final_max_force:.8f} eV/A"
    )
    print(
        f"Final Li-Li distance     : "
        f"{final_li_li_distance:.6f} A"
    )
    print(
        f"Final Li-host distance   : "
        f"{final_li_host_distance:.6f} A"
    )
    print(
        f"Final volume             : "
        f"{final_volume:.6f} A^3"
    )
    print(
        f"Volume change            : "
        f"{volume_change_percent:.4f} %"
    )
    print(
        f"Relative lattice change  : "
        f"{relative_lattice:.6f}"
    )
    print(
        f"Host distortion RMS      : "
        f"{host_rms:.6f} A"
    )
    print(
        f"Host distortion MAX      : "
        f"{host_max:.6f} A"
    )
    print(
        f"Converged                : "
        f"{converged}"
    )
    print(
        f"Stable                   : "
        f"{stable}"
    )
    print(
        f"STATUS                   : "
        f"{status}"
    )
    # =========================================================================
    # SAVE FINAL STRUCTURE
    # =========================================================================
    atoms.info[
        "configuration_id"
    ] = configuration_id
    atoms.info[
        "basin_1"
    ] = int(basin_1)
    atoms.info[
        "basin_2"
    ] = int(basin_2)
    atoms.info[
        "site_1"
    ] = int(site_1)
    atoms.info[
        "site_2"
    ] = int(site_2)
    atoms.info[
        "same_basin"
    ] = bool(same_basin)
    atoms.info[
        "aa_constructible"
    ] = bool(
        config["aa_constructible"]
    )
    atoms.info[
        "final_energy_eV"
    ] = float(final_energy)
    atoms.info[
        "final_max_force_eV_A"
    ] = float(final_max_force)
    atoms.info[
        "final_Li_Li_distance_A"
    ] = float(final_li_li_distance)
    atoms.info[
        "final_Li_host_distance_A"
    ] = float(final_li_host_distance)
    atoms.info[
        "converged"
    ] = bool(converged)
    atoms.info[
        "stable"
    ] = bool(stable)
    final_structure_path = os.path.join(
        FINAL_STRUCTURES_DIR,
        f"{configuration_id}.extxyz"
    )
    write(
        final_structure_path,
        atoms,
        format="extxyz"
    )
    sync_file_to_drive(final_structure_path)
    # =========================================================================
    # APPEND FINAL RESULT
    # =========================================================================
    result = {
        "configuration_id":
            configuration_id,
        "status":
            status,
        "enumeration_status":
            config["enumeration_status"],
        "basin_1":
            basin_1,
        "basin_2":
            basin_2,
        "site_1":
            site_1,
        "site_2":
            site_2,
        "same_basin":
            same_basin,
        "aa_constructible":
            config["aa_constructible"],
        "initial_Li_Li_distance_A":
            initial_li_li_distance,
        "final_Li_Li_distance_A":
            final_li_li_distance,
        "initial_Li_host_distance_A":
            min_li_host_distance(
                build_two_li_system(
                    host,
                    config["position_1"],
                    config["position_2"]
                )
            ),
        "final_Li_host_distance_A":
            final_li_host_distance,
        "initial_energy_eV":
            initial_energy,
        "final_energy_eV":
            final_energy,
        "final_max_force_eV_A":
            final_max_force,
        "optimizer_steps":
            current_step,
        "converged":
            converged,
        "stable":
            stable,
        "initial_volume_A3":
            initial_volume,
        "final_volume_A3":
            final_volume,
        "volume_change_percent":
            volume_change_percent,
        "relative_lattice_change":
            relative_lattice,
        "host_distortion_rms_A":
            host_rms,
        "host_distortion_max_A":
            host_max,
        "resumed_from_checkpoint":
            resumed,
        "final_structure":
            final_structure_path
    }
    append_result(
        result
    )
    completed_ids.add(
        configuration_id
    )
    # =========================================================================
    # REMOVE CHECKPOINT ONLY AFTER RESULT IS SAFE
    # =========================================================================
    remove_checkpoint(
        configuration_id
    )
    print(
        f"COMPLETED: {configuration_id}"
    )

# =============================================================================

# [17] FINAL VALIDATION
# =============================================================================
sync_output_to_drive()
print()
print("=" * 100)
print(f"2-Li PRODUCTION VALIDATION — {CHUNK_FOLDER_NAME}")
print("=" * 100)
if not os.path.isfile(RESULTS_CSV):
    raise FileNotFoundError("This worker's results CSV does not exist.")

final_results = pd.read_csv(RESULTS_CSV)
print(f"Result rows recorded     : {len(final_results)}")
unique_result_ids = final_results["configuration_id"].nunique()
print(f"Unique configuration IDs : {unique_result_ids}")

if len(final_results) != EXPECTED_CHUNK_CONFIGURATIONS:
    raise ValueError(
        f"FINAL VALIDATION FAILED for {CHUNK_FOLDER_NAME}: expected "
        f"{EXPECTED_CHUNK_CONFIGURATIONS} rows, found {len(final_results)}. "
        "If this run was interrupted, rerun the same chunk to resume it."
    )
if unique_result_ids != EXPECTED_CHUNK_CONFIGURATIONS:
    raise ValueError(
        f"FINAL VALIDATION FAILED for {CHUNK_FOLDER_NAME}: "
        "configuration IDs are not unique."
    )

expected_id_set = set(chunk_configuration_ids)
actual_id_set = set(final_results["configuration_id"].astype(str))
if expected_id_set != actual_id_set:
    missing = expected_id_set - actual_id_set
    extra = actual_id_set - expected_id_set
    raise ValueError(
        f"Configuration assignment mismatch in {CHUNK_FOLDER_NAME}.\n"
        f"Missing IDs: {sorted(missing)[:20]}\n"
        f"Unexpected IDs: {sorted(extra)[:20]}"
    )

# Check the chunk's result accounting against the configuration slice assigned
# to this worker. The scientific definitions and global counts remain unchanged.
same_basin_final = final_results[final_results["same_basin"] == True]
constructible_AA_final = final_results[
    (final_results["same_basin"] == True)
    & (final_results["aa_constructible"] == True)
]
singleton_AA_final = final_results[
    (final_results["same_basin"] == True)
    & (final_results["aa_constructible"] == False)
]
distinct_final = final_results[final_results["same_basin"] == False]
expected_same_basin = sum(item["same_basin"] for item in configuration_list)
expected_constructible_AA = sum(
    item["same_basin"] and item["aa_constructible"]
    for item in configuration_list
)
expected_singleton_AA = sum(
    item["same_basin"] and not item["aa_constructible"]
    for item in configuration_list
)
expected_distinct = sum(not item["same_basin"] for item in configuration_list)

if len(same_basin_final) != expected_same_basin:
    raise ValueError("Same-basin accounting does not match this worker's assignment.")
if len(constructible_AA_final) != expected_constructible_AA:
    raise ValueError("Constructible A+A accounting does not match this worker's assignment.")
if len(singleton_AA_final) != expected_singleton_AA:
    raise ValueError("Singleton A+A accounting does not match this worker's assignment.")
if len(distinct_final) != expected_distinct:
    raise ValueError("Distinct-basin A+B accounting does not match this worker's assignment.")

if expected_singleton_AA:
    singleton_statuses = set(singleton_AA_final["status"])
    if singleton_statuses != {"singleton_same_basin_no_distinct_second_member"}:
        raise ValueError("A singleton A+A configuration has an unexpected status.")

if expected_constructible_AA:
    constructible_site_pairs = constructible_AA_final[["site_1", "site_2"]]
    if (constructible_site_pairs["site_1"] == constructible_site_pairs["site_2"]).any():
        raise ValueError(
            "A constructible A+A configuration contains the same site twice."
        )

print()
print("FINAL STATUS COUNTS")
print("-" * 100)
print(final_results["status"].value_counts(dropna=False))
print()
print("CHUNK CONFIGURATION ACCOUNTING")
print("-" * 100)
print(f"Worker                      : {CHUNK_FOLDER_NAME}")
print(f"Total rows                  : {len(final_results)}")
print(f"Same-basin A+A              : {len(same_basin_final)}")
print(f"Constructible A+A           : {len(constructible_AA_final)}")
print(f"Singleton A+A               : {len(singleton_AA_final)}")
print(f"Distinct-basin A+B          : {len(distinct_final)}")
print()
print("=" * 100)
print(f"CHUNK VALIDATION PASSED: {CHUNK_FOLDER_NAME}")
print("=" * 100)
print(f"This worker accounted for {len(final_results)} / {EXPECTED_CHUNK_CONFIGURATIONS} assigned configurations.")

print()
print("OUTPUTS")
print("-" * 100)
print(
    f"Results CSV      : {RESULTS_CSV}"
)
print(
    f"Final structures : {FINAL_STRUCTURES_DIR}"
)
print(
    f"Checkpoints      : {CHECKPOINT_DIR}"
)
print()
print("FINAL 117-BASIN 2-Li STAGE READY.")
print("=" * 100)

ModuleNotFoundError: No module named 'ase'